### This analysis is based on the pull aligned continuous bhv variables and neural activity analysis
### The goal of this code is to define DHHM model and test the hypothesis that social gaze before pull serves as a evidence accumulation process, and test if the neural profile matches the accumulation hypothesis

In [ ]:
import numpy as np
import pandas as pd # Added import for Pandas

import matplotlib.pyplot as plt
import seaborn as sns
import scipy
import scipy.stats as st
import scipy.io
from scipy.stats import pearsonr

# import hddm
# import pymc as pm # Explicitly import pymc for summary function
# import arviz as az # Explicitly import arviz for summary function

import statsmodels.formula.api as smf

from sklearn.neighbors import KernelDensity
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_samples, silhouette_score
# # from dPCA import dPCA
import string
import warnings
import pickle
import json

from scipy.ndimage import gaussian_filter1d

import sys
import os
import glob
import random
from time import time


In [ ]:
# !pip install lifelines
# from lifelines import CoxPHFitter


In [ ]:
# !pip install matplotlib-venn
# from matplotlib_venn import venn3

In [ ]:
# to be able to use the functions in the ana_functions under /3d_recontruction_analysis_self_and_coop_task_neural_analysis/
sys.path.append(os.path.abspath('../3d_recontruction_analysis_self_and_coop_task_neural_analysis/'))

### function - get body part location for each pair of cameras

In [ ]:
from ana_functions.body_part_locs_eachpair import body_part_locs_eachpair
from ana_functions.body_part_locs_singlecam import body_part_locs_singlecam

### function - align the two cameras

In [ ]:
from ana_functions.camera_align import camera_align       

### function - merge the two pairs of cameras

In [ ]:
from ana_functions.camera_merge import camera_merge

### function - find social gaze time point

In [ ]:
from ana_functions.find_socialgaze_timepoint import find_socialgaze_timepoint
from ana_functions.find_socialgaze_timepoint_singlecam import find_socialgaze_timepoint_singlecam
from ana_functions.find_socialgaze_timepoint_singlecam_wholebody import find_socialgaze_timepoint_singlecam_wholebody
from ana_functions.find_socialgaze_timepoint_singlecam_wholebody_2 import find_socialgaze_timepoint_singlecam_wholebody_2


### function - define time point of behavioral events

In [ ]:
from ana_functions.bhv_events_timepoint import bhv_events_timepoint
from ana_functions.bhv_events_timepoint_singlecam import bhv_events_timepoint_singlecam

### function - plot behavioral events

In [ ]:
from ana_functions.plot_bhv_events import plot_bhv_events
from ana_functions.plot_bhv_events_levertube import plot_bhv_events_levertube
from ana_functions.plot_continuous_bhv_var_singlecam import plot_continuous_bhv_var_singlecam
from ana_functions.draw_self_loop import draw_self_loop
import matplotlib.patches as mpatches 
from matplotlib.collections import PatchCollection

### function - plot inter-pull interval

In [ ]:
from ana_functions.plot_interpull_interval import plot_interpull_interval

### function - make demo videos with skeleton and inportant vectors

In [ ]:
from ana_functions.tracking_video_singlecam_demo import tracking_video_singlecam_demo
from ana_functions.tracking_video_singlecam_wholebody_demo import tracking_video_singlecam_wholebody_demo
from ana_functions.tracking_video_singlecam_wholebody_withNeuron_demo import tracking_video_singlecam_wholebody_withNeuron_demo
from ana_functions.tracking_video_singlecam_wholebody_withNeuron_sepbhv_demo import tracking_video_singlecam_wholebody_withNeuron_sepbhv_demo
from ana_functions.tracking_frame_singlecam_wholebody_withNeuron_sepbhv_demo import tracking_frame_singlecam_wholebody_withNeuron_sepbhv_demo

### function - interval between all behavioral events

In [ ]:
from ana_functions.bhv_events_interval import bhv_events_interval

### function - spike analysis

In [ ]:
from ana_functions.spike_analysis_FR_calculation import spike_analysis_FR_calculation
from ana_functions.plot_spike_triggered_singlecam_bhvevent import plot_spike_triggered_singlecam_bhvevent
from ana_functions.plot_bhv_events_aligned_FR import plot_bhv_events_aligned_FR
from ana_functions.plot_strategy_aligned_FR import plot_strategy_aligned_FR

### function - PCA projection

In [ ]:
from ana_functions.PCA_around_bhv_events import PCA_around_bhv_events
from ana_functions.PCA_around_bhv_events_video import PCA_around_bhv_events_video
from ana_functions.confidence_ellipse import confidence_ellipse

### function - HDDM related function

In [ ]:
from functions._apply_gaussian_burst import _apply_gaussian_burst
from functions.generate_marmoset_pull_data import generate_marmoset_pull_data # create simulated data
from functions.compute_stats import compute_stats
from functions.align_and_plot_data import align_and_plot_data
from functions.get_aligned_segment import get_aligned_segment
from functions.analyze_pull_aligned_data import analyze_pull_aligned_data
from functions.analyze_pull_aligned_data_flexibleTW import analyze_pull_aligned_data_flexibleTW
from functions.analyze_pull_aligned_data_flexibleTW_newRTdefinition import analyze_pull_aligned_data_flexibleTW_newRTdefinition

# from functions.run_hddmnn_modeling import run_hddmnn_modeling # do not work
from functions.do_hddm_model_fitted_plot import do_hddm_model_fitted_plot

from functions.run_hddm_modeling import run_hddm_modeling
from functions.run_hddm_modeling_exaustModel import run_hddm_modeling_exaustModel
from functions.run_hddm_modeling_bhvPCA import run_hddm_modeling_bhvPCA

from functions.run_hddm_modeling_gazefilter import run_hddm_modeling_gazefilter
from functions.run_hddm_modeling_bhvPCA_gazefilter import run_hddm_modeling_bhvPCA_gazefilter
from functions.run_hddm_modeling_bhvPCA_gazefilter_ZHistOut import run_hddm_modeling_bhvPCA_gazefilter_ZHistOut
from functions.run_hddm_modeling_bhvPCA_gazefilter_ZHistOutCumRew import run_hddm_modeling_bhvPCA_gazefilter_ZHistOutCumRew



from functions.run_hddm_modeling_gazeInteract import run_hddm_modeling_gazeInteract
from functions.run_hddm_modeling_bhvPCA_gazeInteract import run_hddm_modeling_bhvPCA_gazeInteract

from functions.run_hddm_modeling_withNeuroActivity import run_hddm_modeling_withNeuroActivity
from functions.run_hddm_modeling_withNeuroActivity_noz import run_hddm_modeling_withNeuroActivity_noz





### function - other useful functions

In [ ]:
# for defining the meaningful social gaze (the continuous gaze distribution that is closest to the pull) 
from ana_functions.keep_closest_cluster_single_trial import keep_closest_cluster_single_trial

In [ ]:
# get more information for each pull: number of preceding failed pull and time since last reward/successful pull
from ana_functions.get_pull_infos import get_pull_infos


## Analyze each session

### prepare the basic behavioral data (especially the time stamps for each bhv events)

In [ ]:
# instead of using gaze angle threshold, use the target rectagon to deside gaze info
# ...need to update
sqr_thres_tubelever = 75 # draw the square around tube and lever
sqr_thres_face = 1.15 # a ratio for defining face boundary
sqr_thres_body = 4 # how many times to enlongate the face box boundry to the body


# get the fps of the analyzed video
fps = 30

# get the fs for neural recording
fs_spikes = 20000
fs_lfp = 1000

# frame number of the demo video
nframes = 0.5*30 # second*30fps
# nframes = 45*30 # second*30fps

# re-analyze the video or not
reanalyze_video = 0
redo_anystep = 0

# do OFC sessions or DLPFC sessions
do_OFC = 1
do_DLPFC  = 0
if do_OFC:
    savefile_sufix = '_OFCs'
elif do_DLPFC:
    savefile_sufix = '_DLPFCs'
else:
    savefile_sufix = ''
    
# all the videos (no misaligned ones)
# aligned with the audio
# get the session start time from "videosound_bhv_sync.py/.ipynb"
# currently the session_start_time will be manually typed in. It can be updated after a better method is used


# dodson ginger for dlpfc (dmpfc)
# dodson selene for ofc
if 1:
    if do_DLPFC:
        neural_record_conditions = [
                        '20240531_Dodson_MC', '20240603_Dodson_MC_and_SR', '20240603_Dodson_MC_and_SR',
                        '20240604_Dodson_MC', '20240605_Dodson_MC_and_SR', '20240605_Dodson_MC_and_SR',

                        '20240606_Dodson_MC_and_SR', '20240606_Dodson_MC_and_SR', '20240607_Dodson_SR',
                        '20240610_Dodson_MC', '20240611_Dodson_SR', '20240612_Dodson_MC',

                        '20240613_Dodson_SR', '20240620_Dodson_SR', '20240719_Dodson_MC',
                        '20250129_Dodson_MC', '20250130_Dodson_SR', '20250131_Dodson_MC',

                        '20250210_Dodson_SR_withKoala', '20250211_Dodson_MC_withKoala',
                        '20250212_Dodson_SR_withKoala', '20250214_Dodson_MC_withKoala',
                        '20250217_Dodson_SR_withKoala', '20250218_Dodson_MC_withKoala',

                        '20250219_Dodson_SR_withKoala', '20250220_Dodson_MC_withKoala',
                        '20250224_Dodson_KoalaAL_withKoala', '20250226_Dodson_MC_withKoala',
                        '20250227_Dodson_KoalaAL_withKoala', '20250228_Dodson_DodsonAL_withKoala',

                        '20250304_Dodson_DodsonAL_withKoala', '20250305_Dodson_MC_withKoala',
                        '20250306_Dodson_KoalaAL_withKoala', '20250307_Dodson_DodsonAL_withKoala',
                        '20250310_Dodson_MC_withKoala', '20250312_Dodson_NV_withKoala',

                        '20250313_Dodson_NV_withKoala', '20250314_Dodson_NV_withKoala',
                        '20250401_Dodson_MC_withKanga', '20250402_Dodson_MC_withKanga',
                        '20250403_Dodson_MC_withKanga', '20250404_Dodson_SR_withKanga',

                        '20250407_Dodson_SR_withKanga', '20250408_Dodson_SR_withKanga',
                        '20250409_Dodson_MC_withKanga', '20250415_Dodson_MC_withKanga',
                        # '20250416_Dodson_SR_withKanga',
                        '20250417_Dodson_MC_withKanga',

                        '20250418_Dodson_SR_withKanga', '20250421_Dodson_SR_withKanga',
                        '20250422_Dodson_MC_withKanga', '20250422_Dodson_SR_withKanga',
                        '20250423_Dodson_MC_withKanga', '20250423_Dodson_SR_withKanga',

                        '20250424_Dodson_NV_withKanga', '20250424_Dodson_MC_withKanga',
                        '20250424_Dodson_SR_withKanga', '20250425_Dodson_NV_withKanga',
                        '20250425_Dodson_SR_withKanga', '20250428_Dodson_NV_withKanga',

                        '20250428_Dodson_MC_withKanga', '20250428_Dodson_SR_withKanga',
                        '20250429_Dodson_NV_withKanga', '20250429_Dodson_MC_withKanga',
                        '20250429_Dodson_SR_withKanga', '20250430_Dodson_NV_withKanga',

                        '20250430_Dodson_MC_withKanga', '20250430_Dodson_SR_withKanga',
                    ]
        task_conditions = [
                        'MC', 'MC', 'SR', 'MC', 'MC', 'SR',
                        'MC', 'SR', 'SR', 'MC', 'SR', 'MC',
                        'SR', 'SR', 'MC', 'MC_withGingerNew', 'SR_withGingerNew', 'MC_withGingerNew',

                        'SR_withKoala', 'MC_withKoala', 'SR_withKoala',
                        'MC_withKoala', 'SR_withKoala', 'MC_withKoala',

                        'SR_withKoala', 'MC_withKoala', 'MC_KoalaAuto_withKoala',
                        'MC_withKoala', 'MC_KoalaAuto_withKoala', 'MC_DodsonAuto_withKoala',

                        'MC_DodsonAuto_withKoala', 'MC_withKoala', 'MC_KoalaAuto_withKoala',
                        'MC_DodsonAuto_withKoala', 'MC_withKoala', 'NV_withKoala',

                        'NV_withKoala', 'NV_withKoala', 'MC_withKanga',
                        'MC_withKanga', 'MC_withKanga', 'SR_withKanga',

                        'SR_withKanga', 'SR_withKanga', 'MC_withKanga',
                        'MC_withKanga',
                        # 'SR_withKanga',
                        'MC_withKanga', 'SR_withKanga', 'SR_withKanga', 'MC_withKanga', 'SR_withKanga',

                        'MC_withKanga', 'SR_withKanga', 'NV_withKanga',
                        'MC_withKanga', 'SR_withKanga', 'NV_withKanga',

                        'SR_withKanga', 'NV_withKanga', 'MC_withKanga',
                        'SR_withKanga', 'NV_withKanga', 'MC_withKanga',

                        'SR_withKanga', 'NV_withKanga', 'MC_withKanga', 'SR_withKanga',
                    ]
        dates_list = [
                        '20240531', '20240603_MC', '20240603_SR', '20240604', '20240605_MC', '20240605_SR',
                        '20240606_MC', '20240606_SR', '20240607', '20240610_MC', '20240611', '20240612',

                        '20240613', '20240620', '20240719',
                        '20250129', '20250130', '20250131',

                        '20250210', '20250211', '20250212', '20250214', '20250217', '20250218',
                        '20250219', '20250220', '20250224', '20250226', '20250227', '20250228',

                        '20250304', '20250305', '20250306', '20250307', '20250310', '20250312',
                        '20250313', '20250314',

                        '20250401', '20250402', '20250403', '20250404', '20250407', '20250408',
                        '20250409',

                        '20250415',
                        # '20250416',
                        '20250417', '20250418', '20250421', '20250422', '20250422_SR',

                        '20250423', '20250423_SR', '20250424', '20250424_MC', '20250424_SR',
                        '20250425', '20250425_SR',

                        '20250428_NV', '20250428_MC', '20250428_SR',
                        '20250429_NV', '20250429_MC', '20250429_SR',

                        '20250430_NV', '20250430_MC', '20250430_SR',
                    ]
        videodates_list = [
                        '20240531', '20240603', '20240603', '20240604', '20240605', '20240605',
                        '20240606', '20240606', '20240607', '20240610_MC', '20240611', '20240612',

                        '20240613', '20240620', '20240719',
                        '20250129', '20250130', '20250131',

                        '20250210', '20250211', '20250212', '20250214', '20250217', '20250218',
                        '20250219', '20250220', '20250224', '20250226', '20250227', '20250228',

                        '20250304', '20250305', '20250306', '20250307', '20250310', '20250312',
                        '20250313', '20250314',

                        '20250401', '20250402', '20250403', '20250404', '20250407', '20250408',
                        '20250409',

                        '20250415',
                        # '20250416',
                        '20250417', '20250418', '20250421', '20250422', '20250422_SR',

                        '20250423', '20250423_SR', '20250424', '20250424_MC', '20250424_SR',
                        '20250425', '20250425_SR',

                        '20250428_NV', '20250428_MC', '20250428_SR',
                        '20250429_NV', '20250429_MC', '20250429_SR',

                        '20250430_NV', '20250430_MC', '20250430_SR',
                    ] # to deal with the sessions that MC and SR were in the same session
        session_start_times = [
                        0.00, 340, 340, 72.0, 60.1, 60.1,
                        82.2, 82.2, 35.8, 0.00, 29.2, 35.8,

                        62.5, 71.5, 54.4,
                        0.00, 0.00, 0.00,

                        0.00, 0.00, 0.00, 0.00, 0.00, 0.00,
                        0.00, 0.00, 0.00, 0.00, 0.00, 0.00,

                        0.00, 0.00, 0.00, 0.00, 0.00, 0.00,
                        0.00, 0.00,

                        0.00, 0.00, 73.5, 0.00, 76.1, 81.5,
                        0.00,

                        363,
                        # 0.00,
                        79.0, 162.6, 231.9, 109, 0.00,

                        0.00, 0.00, 0.00, 0.00, 0.00,
                        0.00, 93.0,

                        0.00, 0.00, 0.00,
                        0.00, 0.00, 0.00,

                        0.00, 274.4, 0.00,
                    ]
        
        kilosortvers = list((np.ones(np.shape(dates_list))*4).astype(int))
        
        trig_channelnames = [ 'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0',
                              'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0',
                              'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0',
                              'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0',
                              'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0', #'Dev1/ai0',
                              'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9',
                              'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0',
                              'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0',
                             
                              ]
        animal1_fixedorders = ['dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson',
                               'dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson',
                               'dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson',
                               'dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson',
                               'dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson',# 'dodson',
                               'dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson',
                               'dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson',
                               'dodson','dodson','dodson','dodson','dodson',
                              ]
        recordedanimals = animal1_fixedorders 
        animal2_fixedorders = ['ginger','ginger','ginger','ginger','ginger','ginger','ginger','ginger','ginger',
                               'ginger','ginger','ginger','ginger','ginger','ginger','gingerNew','gingerNew','gingerNew',
                               'koala', 'koala', 'koala', 'koala', 'koala', 'koala', 'koala', 'koala', 'koala',
                               'koala', 'koala', 'koala', 'koala', 'koala', 'koala', 'koala', 'koala', 'koala',
                               'koala', 'koala', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga', # 'kanga',
                               'kanga', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga',
                               'kanga', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga',
                               'kanga', 'kanga', 'kanga', 'kanga', 'kanga',
                              ]

        animal1_filenames = ["Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson",
                             "Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson",
                             "Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson",
                             "Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson",
                             "Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson",# "Dodson",
                             'Dodson','Dodson','Dodson','Dodson','Dodson','Dodson','Dodson','Dodson','Dodson',
                             'Dodson','Dodson','Dodson','Dodson','Dodson','Dodson','Dodson','Dodson','Dodson',
                             'Dodson','Dodson','Dodson','Dodson','Dodson',
                            ]
        animal2_filenames = ["Ginger","Ginger","Ginger","Ginger","Ginger","Ginger","Ginger","Ginger","Ginger",
                             "Ginger","Ginger","Ginger","Ginger","Ginger","Ginger","Ginger","Ginger","Ginger",
                             "Koala", "Koala", "Koala", "Koala", "Koala", "Koala", "Koala", "Koala", "Koala",
                             "Koala", "Koala", "Koala", "Koala", "Koala", "Koala", "Koala", "Koala", "Koala",
                             "Koala", "Koala", "Kanga", "Kanga", "Kanga", "Kanga", "Kanga", "Kanga", # "Kanga",
                             'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga',
                             'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga',
                             'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga',
                            ]
        
    elif do_OFC:
        neural_record_conditions = [
                        '20260219_Dodson_OFC_33turns_SRwithSelene',  '20260303_Dodson_OFC_33turns_1sMCwithSelene',
                        '20260303_Dodson_OFC_33turns_SRwithSelene',  '20260304_Dodson_OFC_33turns_1sMCwithSelene',
                        '20260304_Dodson_OFC_33turns_SRwithSelene',  '20260305_Dodson_OFC_33turns_1sMCwithSelene',
                        '20260309_Dodson_OFC_33turns_1sMCwithKanga', '20260309_Dodson_OFC_33turns_SRwithKanga',
                        '20260310_Dodson_OFC_33turns_1sMCwithKanga', '20260310_Dodson_OFC_33turns_SRwithKanga',
                        '20260311_Dodson_OFC_33turns_1sMCwithKanga', '20260311_Dodson_OFC_33turns_SRwithKanga',
                        '20260312_Dodson_OFC_33turns_1sMCwithKanga', '20260312_Dodson_OFC_33turns_SRwithKanga',
                        '20260313_Dodson_OFC_33turns_1sMCwithKanga', '20260313_Dodson_OFC_33turns_SRwithKanga',
                    ]
        task_conditions = [
                        'SR', 'MC', 'SR', 'MC', 'SR', 'MC', 
                        'MC', 'SR', 'MC', 'SR', 'MC', 'SR',
                        'MC', 'SR', 'MC', 'SR',
                    ]
        dates_list = [
                        '20260219', '20260303',    '20260303_SR', '20260304',    '20260304_SR', '20260305', 
                        '20260309', '20260309_SR', '20260310',    '20260310_SR', '20260311',    '20260311_SR',
                        '20260312', '20260312_SR', '20260313',    '20260313_SR',
                    ]
        videodates_list = [
                        '20260219', '20260303',    '20260303_SR', '20260304',    '20260304_SR', '20260305', 
                        '20260309', '20260309_SR', '20260310',    '20260310_SR', '20260311',    '20260311_SR',
                        '20260312', '20260312_SR', '20260313',    '20260313_SR',
                    ] 
        
        session_start_times = [
                        188.7, 0.00, 0.00, 0.00, 0.00, 0.00, 
                         0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 
                         0.00, 0.00, 0.00, 0.00, 
                    ]
        
        kilosortvers = list((np.ones(np.shape(dates_list))*4).astype(int))
        
        trig_channelnames = [ 'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0',
                              'Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9',
                              'Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9',
                              ]
        animal1_fixedorders = ['dodson','dodson','dodson','dodson','dodson','dodson',
                               'dodson','dodson','dodson','dodson','dodson','dodson',
                               'dodson','dodson','dodson','dodson',
                              ]
        recordedanimals = animal1_fixedorders 
        animal2_fixedorders = ['selene','selene','selene','selene','selene','selene',
                               'kanga', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga',
                               'kanga', 'kanga', 'kanga', 'kanga',
                              ]

        animal1_filenames = ["Dodson","Dodson","Dodson","Dodson","Dodson","Dodson",
                             "Dodson","Dodson","Dodson","Dodson","Dodson","Dodson",
                             "Dodson","Dodson","Dodson","Dodson",
                            ]
        animal2_filenames = ['Selene','Selene','Selene','Selene','Selene','Selene',
                             'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga',
                             'Kanga', 'Kanga', 'Kanga', 'Kanga',
                            ]

    
# dannon kanga
if 1:
    if do_DLPFC:
        neural_record_conditions = [
                        '20240508_Kanga_SR', '20240509_Kanga_MC', '20240513_Kanga_MC',
                        '20240514_Kanga_SR', '20240523_Kanga_MC', '20240524_Kanga_SR',

                        '20240606_Kanga_MC', '20240613_Kanga_MC_DannonAuto',
                        '20240614_Kanga_MC_DannonAuto', '20240617_Kanga_MC_DannonAuto',
                        '20240618_Kanga_MC_KangaAuto', '20240619_Kanga_MC_KangaAuto',

                        '20240620_Kanga_MC_KangaAuto', '20240621_1_Kanga_NoVis',
                        '20240624_Kanga_NoVis', '20240626_Kanga_NoVis',

                        '20240808_Kanga_MC_withGinger', '20240809_Kanga_MC_withGinger',
                        '20240812_Kanga_MC_withGinger', '20240813_Kanga_MC_withKoala',
                        '20240814_Kanga_MC_withKoala', '20240815_Kanga_MC_withKoala',

                        '20240819_Kanga_MC_withVermelho', '20240821_Kanga_MC_withVermelho',
                        '20240822_Kanga_MC_withVermelho',

                        '20250415_Kanga_MC_withDodson', '20250416_Kanga_SR_withDodson',
                        '20250417_Kanga_MC_withDodson', '20250418_Kanga_SR_withDodson',
                        '20250421_Kanga_SR_withDodson', '20250422_Kanga_MC_withDodson',

                        '20250422_Kanga_SR_withDodson', '20250423_Kanga_MC_withDodson',
                        '20250423_Kanga_SR_withDodson',

                        '20250424_Kanga_NV_withDodson', '20250424_Kanga_MC_withDodson',
                        '20250424_Kanga_SR_withDodson', '20250425_Kanga_NV_withDodson',
                        '20250425_Kanga_SR_withDodson',

                        '20250428_Kanga_NV_withDodson', '20250428_Kanga_MC_withDodson',
                        '20250428_Kanga_SR_withDodson', '20250429_Kanga_NV_withDodson',
                        '20250429_Kanga_MC_withDodson', '20250429_Kanga_SR_withDodson',

                        '20250430_Kanga_NV_withDodson', '20250430_Kanga_MC_withDodson',
                        '20250430_Kanga_SR_withDodson',
                    ]
        dates_list = [
                        "20240508", "20240509", "20240513", "20240514", "20240523", "20240524",
                        "20240606", "20240613", "20240614", "20240617", "20240618", "20240619",
                        "20240620", "20240621_1", "20240624", "20240626",

                        "20240808", "20240809", "20240812", "20240813", "20240814", "20240815",
                        "20240819", "20240821", "20240822",

                        "20250415", "20250416", "20250417", "20250418", "20250421", "20250422",
                        "20250422_SR",

                        '20250423', '20250423_SR', '20250424', '20250424_MC', '20250424_SR',
                        '20250425', '20250425_SR',

                        '20250428_NV', '20250428_MC', '20250428_SR',
                        '20250429_NV', '20250429_MC', '20250429_SR',

                        '20250430_NV', '20250430_MC', '20250430_SR',
                    ]
        videodates_list = dates_list
        task_conditions = [
                        'SR', 'MC', 'MC', 'SR', 'MC', 'SR',
                        'MC', 'MC_DannonAuto', 'MC_DannonAuto', 'MC_DannonAuto',
                        'MC_KangaAuto', 'MC_KangaAuto',

                        'MC_KangaAuto', 'NV', 'NV', 'NV',

                        'MC_withGinger', 'MC_withGinger', 'MC_withGinger',
                        'MC_withKoala', 'MC_withKoala', 'MC_withKoala',

                        'MC_withVermelho', 'MC_withVermelho', 'MC_withVermelho',

                        'MC_withDodson', 'SR_withDodson', 'MC_withDodson',
                        'SR_withDodson', 'SR_withDodson', 'MC_withDodson',

                        'SR_withDodson', 'MC_withDodson', 'SR_withDodson',

                        'NV_withDodson', 'MC_withDodson', 'SR_withDodson',
                        'NV_withDodson', 'SR_withDodson',

                        'NV_withDodson', 'MC_withDodson', 'SR_withDodson',
                        'NV_withDodson', 'MC_withDodson', 'SR_withDodson',

                        'NV_withDodson', 'MC_withDodson', 'SR_withDodson',
                    ]
        session_start_times = [
                        0.00, 36.0, 69.5, 0.00, 62.0, 0.00,
                        89.0, 0.00, 0.00, 0.00, 165.8, 96.0,
            
                        0.00, 0.00, 0.00, 48.0,
                        59.2, 49.5, 40.0, 50.0, 0.00, 69.8,
            
                        85.0, 212.9, 68.5,
                        363, 0.00, 79.0, 162.6, 231.9, 109,
            
                        0.00,
                        0.00, 0.00, 0.00, 0.00, 0.00,

                        0.00, 93.0,

                        0.00, 0.00, 0.00, 0.00, 0.00,
                        0.00,

                        0.00, 274.4, 0.00,
                    ]
        
        kilosortvers = list((np.ones(np.shape(dates_list))*4).astype(int))
        
        trig_channelnames = ['Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0',
                             'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0',
                             'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0',
                             'Dev1/ai0','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai0','Dev1/ai0',
                             'Dev1/ai0','Dev1/ai0','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9',
                             'Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9',
                              ]
        
        animal1_fixedorders = ['dannon','dannon','dannon','dannon','dannon','dannon','dannon','dannon',
                               'dannon','dannon','dannon','dannon','dannon','dannon','dannon','dannon',
                               'ginger','ginger','ginger','koala','koala','koala','vermelho','vermelho',
                               'vermelho','dodson','dodson','dodson','dodson','dodson','dodson','dodson',
                               'dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson',
                               'dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson',
                              ]
        animal2_fixedorders = ['kanga','kanga','kanga','kanga','kanga','kanga','kanga','kanga',
                               'kanga','kanga','kanga','kanga','kanga','kanga','kanga','kanga',
                               'kanga','kanga','kanga','kanga','kanga','kanga','kanga','kanga',
                               'kanga','kanga','kanga','kanga','kanga','kanga','kanga','kanga',
                               'kanga','kanga','kanga','kanga','kanga','kanga','kanga','kanga',
                               'kanga','kanga','kanga','kanga','kanga','kanga','kanga','kanga',
                              ]
        recordedanimals = animal2_fixedorders

        animal1_filenames = ["Dannon","Dannon","Dannon","Dannon","Dannon","Dannon","Dannon","Dannon",
                             "Dannon","Dannon","Dannon","Dannon","Dannon","Dannon","Dannon","Dannon",
                             "Ginger","Ginger","Ginger", "Kanga", "Kanga", "Kanga", "Kanga", "Kanga",
                              "Kanga","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson",
                             "Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson",
                             "Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson",
                             
                            ]
        animal2_filenames = ["Kanga","Kanga","Kanga","Kanga","Kanga","Kanga","Kanga","Kanga",
                             "Kanga","Kanga","Kanga","Kanga","Kanga","Kanga","Kanga","Kanga",
                             "Kanga","Kanga","Kanga","Koala","Koala","Koala","Vermelho","Vermelho",
                             "Vermelho","Kanga","Kanga","Kanga","Kanga","Kanga","Kanga","Kanga",
                             "Kanga","Kanga","Kanga","Kanga","Kanga","Kanga","Kanga","Kanga",
                             "Kanga","Kanga","Kanga","Kanga","Kanga","Kanga","Kanga","Kanga",
                            ]
        
    elif do_OFC:
        neural_record_conditions = [
                        '20260309_Kanga_OFC_31turns_1sMCwithDodson', '20260309_Kanga_OFC_31turns_SRwithDodson',
                        '20260310_Kanga_OFC_31turns_1sMCwithDodson', '20260310_Kanga_OFC_31turns_SRwithDodson',
                        '20260311_Kanga_OFC_31turns_1sMCwithDodson', '20260311_Kanga_OFC_31turns_SRwithDodson',
                        '20260312_Kanga_OFC_31turns_1sMCwithDodson', '20260312_Kanga_OFC_31turns_SRwithDodson',
                        '20260313_Kanga_OFC_31turns_1sMCwithDodson', '20260313_Kanga_OFC_31turns_SRwithDodson',
                    ]
        task_conditions = [
                        'MC', 'SR', 'MC', 'SR', 'MC', 'SR',
                        'MC', 'SR', 'MC', 'SR',
                    ]
        dates_list = [
                        '20260309', '20260309_SR', '20260310', '20260310_SR', '20260311', '20260311_SR',
                        '20260312', '20260312_SR', '20260313', '20260313_SR',
                    ]
        videodates_list = [
                        '20260309', '20260309_SR', '20260310', '20260310_SR', '20260311', '20260311_SR',
                        '20260312', '20260312_SR', '20260313', '20260313_SR',
                    ] 
        
        session_start_times = [
                         0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 
                         0.00, 0.00, 0.00, 0.00, 
                    ]
        
        kilosortvers = list((np.ones(np.shape(dates_list))*4).astype(int))
        
        trig_channelnames = [ 'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0', 
                              'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0',
                              ]
        animal1_fixedorders = [
                               'dodson','dodson','dodson','dodson','dodson','dodson',
                               'dodson','dodson','dodson','dodson',
                              ]
        animal2_fixedorders = [
                               'kanga', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga',
                               'kanga', 'kanga', 'kanga', 'kanga',
                              ]
        recordedanimals = animal2_fixedorders 

        animal1_filenames = [
                             "Dodson","Dodson","Dodson","Dodson","Dodson","Dodson",
                             "Dodson","Dodson","Dodson","Dodson",
                            ]
        animal2_filenames = [
                             'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga',
                             'Kanga', 'Kanga', 'Kanga', 'Kanga',
                            ]

    
# a test case
if 0: # kanga example
    neural_record_conditions = ['20250415_Kanga_MC_withDodson']
    dates_list = ["20250415"]
    videodates_list = dates_list
    task_conditions = ['MC_withDodson']
    session_start_times = [363] # in second
    kilosortvers = [4]
    trig_channelnames = ['Dev1/ai9']
    animal1_fixedorders = ['dodson']
    animal2_fixedorders = ['kanga']
    recordedanimals = animal2_fixedorders
    animal1_filenames = ["Dodson"]
    animal2_filenames = ["Kanga"]
if 0: # dodson example 
    neural_record_conditions = ['20250415_Dodson_MC_withKanga']
    dates_list = ["20250415"]
    videodates_list = dates_list
    task_conditions = ['MC_withKanga']
    session_start_times = [363] # in second
    kilosortvers = [4]
    trig_channelnames = ['Dev1/ai0']
    animal1_fixedorders = ['dodson']
    recordedanimals = animal1_fixedorders
    animal2_fixedorders = ['kanga']
    animal1_filenames = ["Dodson"]
    animal2_filenames = ["Kanga"]
    
ndates = np.shape(dates_list)[0]

session_start_frames = session_start_times * fps # fps is 30Hz

# totalsess_time = 600

# video tracking results info
animalnames_videotrack = ['dodson','scorch'] # does not really mean dodson and scorch, instead, indicate animal1 and animal2
bodypartnames_videotrack = ['rightTuft','whiteBlaze','leftTuft','rightEye','leftEye','mouth']


# which camera to analyzed
cameraID = 'camera-2'
cameraID_short = 'cam2'

considerlevertube = 1
considertubeonly = 0

# location of levers and tubes for camera 2
# # camera 1
# lever_locs_camI = {'dodson':np.array([645,600]),'scorch':np.array([425,435])}
# tube_locs_camI  = {'dodson':np.array([1350,630]),'scorch':np.array([555,345])}
# # camera 2
# # location of the estimiated middle of the box
lever_locs_camI = {'dodson':np.array([1325,615]),'scorch':np.array([560,615])}
# # location of the estimated lever
# lever_locs_camI = {'dodson':np.array([1335,715]),'scorch':np.array([550,715])}
tube_locs_camI  = {'dodson':np.array([1550,515]),'scorch':np.array([350,515])}
# # old
# # lever_locs_camI = {'dodson':np.array([1335,715]),'scorch':np.array([550,715])}
# # tube_locs_camI  = {'dodson':np.array([1650,490]),'scorch':np.array([250,490])}
# # camera 3
# lever_locs_camI = {'dodson':np.array([1580,440]),'scorch':np.array([1296,540])}
# tube_locs_camI  = {'dodson':np.array([1470,375]),'scorch':np.array([805,475])}


if np.shape(session_start_times)[0] != np.shape(dates_list)[0]:
    exit()

    
# define hddm data summarizing data set    
hddm_datas_all_dates = dict.fromkeys(dates_list, [])
hddm_datas_flexibleTW_all_dates = dict.fromkeys(dates_list, [])
hddm_datas_flexibleTW_newRTdefinition_all_dates = dict.fromkeys(dates_list, [])


# where to save the summarizing data
data_saved_folder = '/gpfs/radev/pi/nandy/jadi_gibbs_data/VideoTracker_SocialInter/3d_recontruction_analysis_self_and_coop_task_data_saved/'
# data_saved_folder = '/Users/weikangshi/Downloads/MishaCode_temp/'
# data_saved_folder = ''

# neural data folder
neural_data_folder = '/gpfs/radev/pi/nandy/jadi_gibbs_data/Marmoset_neural_recording/'

    

In [ ]:
print(np.shape(neural_record_conditions))
print(np.shape(task_conditions))
print(np.shape(dates_list))
print(np.shape(videodates_list)) 
print(np.shape(session_start_times))

print(np.shape(kilosortvers))

print(np.shape(trig_channelnames))
print(np.shape(animal1_fixedorders)) 
print(np.shape(recordedanimals))
print(np.shape(animal2_fixedorders))

print(np.shape(animal1_filenames))
print(np.shape(animal2_filenames))  

In [ ]:
# basic behavior analysis (define time stamps for each bhv events, etc)

try:
    if redo_anystep:
        dummy
    
    # dummy 
    
    #
    print('loading all data')
    
    # load saved data
    data_saved_subfolder = data_saved_folder+'data_saved_singlecam_wholebody_neural_and_hddm'+savefile_sufix+'/'+cameraID+'/'+animal1_fixedorders[0]+animal2_fixedorders[0]+'/'
    
    with open(data_saved_subfolder+'/hddm_datas_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
        hddm_datas_all_dates = pickle.load(f)
   
    with open(data_saved_subfolder+'/hddm_datas_flexibleTW_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
        hddm_datas_flexibleTW_all_dates = pickle.load(f)        
        
    with open(data_saved_subfolder+'/hddm_datas_flexibleTW_newRTdefinition_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
        hddm_datas_flexibleTW_newRTdefinition_all_dates = pickle.load(f)   
        
    print('all data from all dates are loaded')

except:

    print('analyze all dates')

    for idate in np.arange(0,ndates,1):
    
        date_tgt = dates_list[idate]
        videodate_tgt = videodates_list[idate]
        
        neural_record_condition = neural_record_conditions[idate]
        
        session_start_time = session_start_times[idate]
        
        kilosortver = kilosortvers[idate]

        trig_channelname = trig_channelnames[idate]
        
        animal1_filename = animal1_filenames[idate]
        animal2_filename = animal2_filenames[idate]
        
        animal1_fixedorder = [animal1_fixedorders[idate]]
        animal2_fixedorder = [animal2_fixedorders[idate]]
        
        recordedanimal = recordedanimals[idate]

        # folder and file path
        camera12_analyzed_path = "/gpfs/radev/pi/nandy/jadi_gibbs_data/VideoTracker_SocialInter/test_video_cooperative_task_3d/"+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_camera12/"
        camera23_analyzed_path = "/gpfs/radev/pi/nandy/jadi_gibbs_data/VideoTracker_SocialInter/test_video_cooperative_task_3d/"+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_camera23/"
        
        # 
        try: 
            singlecam_ana_type = "DLC_dlcrnetms5_marmoset_tracking_with_middle_camera_withHeadchamberFeb28shuffle1_167500"
            bodyparts_camI_camIJ = camera12_analyzed_path+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_"+cameraID+singlecam_ana_type+"_el_filtered.h5"
            if not os.path.exists(bodyparts_camI_camIJ):
                singlecam_ana_type = "DLC_dlcrnetms5_marmoset_tracking_with_middle_camera_withHeadchamberFeb28shuffle1_80000"
                bodyparts_camI_camIJ = camera12_analyzed_path+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_"+cameraID+singlecam_ana_type+"_el_filtered.h5"
            if not os.path.exists(bodyparts_camI_camIJ):
                singlecam_ana_type = "DLC_dlcrnetms5_marmoset_tracking_with_middle_cameraSep1shuffle1_150000"
                bodyparts_camI_camIJ = camera12_analyzed_path+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_"+cameraID+singlecam_ana_type+"_el_filtered.h5"                
            # get the bodypart data from files
            bodyparts_locs_camI = body_part_locs_singlecam(bodyparts_camI_camIJ,singlecam_ana_type,animalnames_videotrack,bodypartnames_videotrack,videodate_tgt)
            video_file_original = camera12_analyzed_path+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_"+cameraID+".mp4"
        except:
            singlecam_ana_type = "DLC_dlcrnetms5_marmoset_tracking_with_middle_camera_withHeadchamberFeb28shuffle1_167500"
            bodyparts_camI_camIJ = camera23_analyzed_path+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_"+cameraID+singlecam_ana_type+"_el_filtered.h5"
            if not os.path.exists(bodyparts_camI_camIJ):
                singlecam_ana_type = "DLC_dlcrnetms5_marmoset_tracking_with_middle_camera_withHeadchamberFeb28shuffle1_80000"
                bodyparts_camI_camIJ = camera23_analyzed_path+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_"+cameraID+singlecam_ana_type+"_el_filtered.h5"
            if not os.path.exists(bodyparts_camI_camIJ):
                singlecam_ana_type = "DLC_dlcrnetms5_marmoset_tracking_with_middle_cameraSep1shuffle1_150000"
                bodyparts_camI_camIJ = camera23_analyzed_path+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_"+cameraID+singlecam_ana_type+"_el_filtered.h5"
            
            # get the bodypart data from files
            bodyparts_locs_camI = body_part_locs_singlecam(bodyparts_camI_camIJ,singlecam_ana_type,animalnames_videotrack,bodypartnames_videotrack,videodate_tgt)
            video_file_original = camera23_analyzed_path+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_"+cameraID+".mp4"        
        
        
        # load behavioral results
        try:
            bhv_data_path = "/gpfs/radev/pi/nandy/jadi_gibbs_data/VideoTracker_SocialInter/marmoset_tracking_bhv_data_from_task_code/"+date_tgt+"_"+animal1_filename+"_"+animal2_filename+"/"
            trial_record_json = glob.glob(bhv_data_path +date_tgt+"_"+animal2_filename+"_"+animal1_filename+"_TrialRecord_" + "*.json")
            bhv_data_json = glob.glob(bhv_data_path + date_tgt+"_"+animal2_filename+"_"+animal1_filename+"_bhv_data_" + "*.json")
            session_info_json = glob.glob(bhv_data_path + date_tgt+"_"+animal2_filename+"_"+animal1_filename+"_session_info_" + "*.json")
            ni_data_json = glob.glob(bhv_data_path + date_tgt+"_"+animal2_filename+"_"+animal1_filename+"_ni_data_" + "*.json")
            #
            trial_record = pd.read_json(trial_record_json[0])
            bhv_data = pd.read_json(bhv_data_json[0])
            session_info = pd.read_json(session_info_json[0])
            # 
            with open(ni_data_json[0]) as f:
                for line in f:
                    ni_data=json.loads(line)   
        except:
            bhv_data_path = "/gpfs/radev/pi/nandy/jadi_gibbs_data/VideoTracker_SocialInter/marmoset_tracking_bhv_data_from_task_code/"+date_tgt+"_"+animal1_filename+"_"+animal2_filename+"/"
            trial_record_json = glob.glob(bhv_data_path + date_tgt+"_"+animal1_filename+"_"+animal2_filename+"_TrialRecord_" + "*.json")
            bhv_data_json = glob.glob(bhv_data_path + date_tgt+"_"+animal1_filename+"_"+animal2_filename+"_bhv_data_" + "*.json")
            session_info_json = glob.glob(bhv_data_path + date_tgt+"_"+animal1_filename+"_"+animal2_filename+"_session_info_" + "*.json")
            ni_data_json = glob.glob(bhv_data_path + date_tgt+"_"+animal1_filename+"_"+animal2_filename+"_ni_data_" + "*.json")
            #
            trial_record = pd.read_json(trial_record_json[0])
            bhv_data = pd.read_json(bhv_data_json[0])
            session_info = pd.read_json(session_info_json[0])
            #
            with open(ni_data_json[0]) as f:
                for line in f:
                    ni_data=json.loads(line)

        # get animal info from the session information
        animal1 = session_info['lever1_animal'][0].lower()
        animal2 = session_info['lever2_animal'][0].lower()

        
        # get task type and cooperation threshold
        try:
            coop_thres = session_info["pulltime_thres"][0]
            tasktype = session_info["task_type"][0]
        except:
            coop_thres = 0
            tasktype = 1
    
            
        # clean up the trial_record
        warnings.filterwarnings('ignore')
        trial_record_clean = pd.DataFrame(columns=trial_record.columns)
        # for itrial in np.arange(0,np.max(trial_record['trial_number']),1):
        for itrial in trial_record['trial_number']:
            # trial_record_clean.loc[itrial] = trial_record[trial_record['trial_number']==itrial+1].iloc[[0]]
            trial_record_clean = trial_record_clean.append(trial_record[trial_record['trial_number']==itrial].iloc[[0]])
        trial_record_clean = trial_record_clean.reset_index(drop = True)

        # change bhv_data time to the absolute time
        time_points_new = pd.DataFrame(np.zeros(np.shape(bhv_data)[0]),columns=["time_points_new"])
        # for itrial in np.arange(0,np.max(trial_record_clean['trial_number']),1):
        for itrial in np.arange(0,np.shape(trial_record_clean)[0],1):
            # ind = bhv_data["trial_number"]==itrial+1
            ind = bhv_data["trial_number"]==trial_record_clean['trial_number'][itrial]
            new_time_itrial = bhv_data[ind]["time_points"] + trial_record_clean["trial_starttime"].iloc[itrial]
            time_points_new["time_points_new"][ind] = new_time_itrial
        bhv_data["time_points"] = time_points_new["time_points_new"]
        bhv_data = bhv_data[bhv_data["time_points"] != 0]

        
        
        # load behavioral event results
        try:
            # dummy
            print('load social gaze with '+cameraID+' only of '+date_tgt)
            with open(data_saved_folder+"bhv_events_singlecam_wholebody/"+animal1_fixedorder[0]+animal2_fixedorder[0]+"/"+cameraID+'/'+date_tgt+'/output_look_ornot.pkl', 'rb') as f:
                output_look_ornot = pickle.load(f)
            with open(data_saved_folder+"bhv_events_singlecam_wholebody/"+animal1_fixedorder[0]+animal2_fixedorder[0]+"/"+cameraID+'/'+date_tgt+'/output_allvectors.pkl', 'rb') as f:
                output_allvectors = pickle.load(f)
            with open(data_saved_folder+"bhv_events_singlecam_wholebody/"+animal1_fixedorder[0]+animal2_fixedorder[0]+"/"+cameraID+'/'+date_tgt+'/output_allangles.pkl', 'rb') as f:
                output_allangles = pickle.load(f)  
            with open(data_saved_folder+"bhv_events_singlecam_wholebody/"+animal1_fixedorder[0]+animal2_fixedorder[0]+"/"+cameraID+'/'+date_tgt+'/output_key_locations.pkl', 'rb') as f:
                output_key_locations = pickle.load(f)
        except:   
            print('analyze social gaze with '+cameraID+' only of '+date_tgt)
            # get social gaze information 
            output_look_ornot, output_allvectors, output_allangles = find_socialgaze_timepoint_singlecam_wholebody(bodyparts_locs_camI,lever_locs_camI,tube_locs_camI,
                                                                                                                   considerlevertube,considertubeonly,sqr_thres_tubelever,
                                                                                                                   sqr_thres_face,sqr_thres_body)
            output_key_locations = find_socialgaze_timepoint_singlecam_wholebody_2(bodyparts_locs_camI,lever_locs_camI,tube_locs_camI,considerlevertube)
            
            # save data
            current_dir = data_saved_folder+'/bhv_events_singlecam_wholebody/'+animal1_fixedorder[0]+animal2_fixedorder[0]
            add_date_dir = os.path.join(current_dir,cameraID+'/'+date_tgt)
            if not os.path.exists(add_date_dir):
                os.makedirs(add_date_dir)
            #
            with open(data_saved_folder+"bhv_events_singlecam_wholebody/"+animal1_fixedorder[0]+animal2_fixedorder[0]+"/"+cameraID+'/'+date_tgt+'/output_look_ornot.pkl', 'wb') as f:
                pickle.dump(output_look_ornot, f)
            with open(data_saved_folder+"bhv_events_singlecam_wholebody/"+animal1_fixedorder[0]+animal2_fixedorder[0]+"/"+cameraID+'/'+date_tgt+'/output_allvectors.pkl', 'wb') as f:
                pickle.dump(output_allvectors, f)
            with open(data_saved_folder+"bhv_events_singlecam_wholebody/"+animal1_fixedorder[0]+animal2_fixedorder[0]+"/"+cameraID+'/'+date_tgt+'/output_allangles.pkl', 'wb') as f:
                pickle.dump(output_allangles, f)
            with open(data_saved_folder+"bhv_events_singlecam_wholebody/"+animal1_fixedorder[0]+animal2_fixedorder[0]+"/"+cameraID+'/'+date_tgt+'/output_key_locations.pkl', 'wb') as f:
                pickle.dump(output_key_locations, f)
                

        look_at_other_or_not_merge = output_look_ornot['look_at_other_or_not_merge']
        look_at_tube_or_not_merge = output_look_ornot['look_at_tube_or_not_merge']
        look_at_lever_or_not_merge = output_look_ornot['look_at_lever_or_not_merge']
        look_at_otherlever_or_not_merge = output_look_ornot['look_at_otherlever_or_not_merge']
        look_at_otherface_or_not_merge = output_look_ornot['look_at_otherface_or_not_merge']
        
        # change the unit to second and align to the start of the session
        session_start_time = session_start_times[idate]
        look_at_other_or_not_merge['time_in_second'] = np.arange(0,np.shape(look_at_other_or_not_merge['dodson'])[0],1)/fps - session_start_time
        look_at_lever_or_not_merge['time_in_second'] = np.arange(0,np.shape(look_at_lever_or_not_merge['dodson'])[0],1)/fps - session_start_time
        look_at_tube_or_not_merge['time_in_second'] = np.arange(0,np.shape(look_at_tube_or_not_merge['dodson'])[0],1)/fps - session_start_time 
        look_at_otherlever_or_not_merge['time_in_second'] = np.arange(0,np.shape(look_at_otherlever_or_not_merge['dodson'])[0],1)/fps - session_start_time
        look_at_otherface_or_not_merge['time_in_second'] = np.arange(0,np.shape(look_at_otherface_or_not_merge['dodson'])[0],1)/fps - session_start_time

        
        # find time point of behavioral events
        output_time_points_socialgaze ,output_time_points_levertube = bhv_events_timepoint_singlecam(bhv_data,look_at_other_or_not_merge,look_at_lever_or_not_merge,look_at_tube_or_not_merge)
        time_point_pull1 = output_time_points_socialgaze['time_point_pull1']
        time_point_pull2 = output_time_points_socialgaze['time_point_pull2']
        oneway_gaze1 = output_time_points_socialgaze['oneway_gaze1']
        oneway_gaze2 = output_time_points_socialgaze['oneway_gaze2']
        mutual_gaze1 = output_time_points_socialgaze['mutual_gaze1']
        mutual_gaze2 = output_time_points_socialgaze['mutual_gaze2']
        # 
        # mostly just for the sessions in which MC and SR are in the same session 
        firstpulltime = np.nanmin([np.nanmin(time_point_pull1),np.nanmin(time_point_pull2)])
        oneway_gaze1 = oneway_gaze1[oneway_gaze1>(firstpulltime-15)] # 15s before the first pull (animal1 or 2) count as the active period
        oneway_gaze2 = oneway_gaze2[oneway_gaze2>(firstpulltime-15)]
        mutual_gaze1 = mutual_gaze1[mutual_gaze1>(firstpulltime-15)]
        mutual_gaze2 = mutual_gaze2[mutual_gaze2>(firstpulltime-15)]  
        #    
        # newly added condition: only consider gaze during the active pulling time (15s after the last pull)    
        lastpulltime = np.nanmax([np.nanmax(time_point_pull1),np.nanmax(time_point_pull2)])
        oneway_gaze1 = oneway_gaze1[oneway_gaze1<(lastpulltime+15)]    
        oneway_gaze2 = oneway_gaze2[oneway_gaze2<(lastpulltime+15)]
        mutual_gaze1 = mutual_gaze1[mutual_gaze1<(lastpulltime+15)]
        mutual_gaze2 = mutual_gaze2[mutual_gaze2<(lastpulltime+15)] 
            
        # define successful pulls and failed pulls
        # a new definition of successful and failed pulls
        # separate successful and failed pulls
        # step 1 all pull and juice
        time_point_pull1 = bhv_data["time_points"][bhv_data["behavior_events"]==1]
        time_point_pull2 = bhv_data["time_points"][bhv_data["behavior_events"]==2]
        time_point_juice1 = bhv_data["time_points"][bhv_data["behavior_events"]==3]
        time_point_juice2 = bhv_data["time_points"][bhv_data["behavior_events"]==4]
        # step 2:
        # pull 1
        # Find the last pull before each juice
        successful_pull1 = [time_point_pull1[time_point_pull1 < juice].max() for juice in time_point_juice1]
        # Convert to Pandas Series
        successful_pull1 = pd.Series(successful_pull1, index=time_point_juice1.index)
        # Find failed pulls (pulls that are not successful)
        failed_pull1 = time_point_pull1[~time_point_pull1.isin(successful_pull1)]
        # pull 2
        # Find the last pull before each juice
        successful_pull2 = [time_point_pull2[time_point_pull2 < juice].max() for juice in time_point_juice2]
        # Convert to Pandas Series
        successful_pull2 = pd.Series(successful_pull2, index=time_point_juice2.index)
        # Find failed pulls (pulls that are not successful)
        failed_pull2 = time_point_pull2[~time_point_pull2.isin(successful_pull2)]
        #
        # step 3:
        time_point_pull1_succ = np.round(successful_pull1,1)
        time_point_pull2_succ = np.round(successful_pull2,1)
        time_point_pull1_fail = np.round(failed_pull1,1)
        time_point_pull2_fail = np.round(failed_pull2,1)
        # 
        time_point_pulls_succfail = { "pull1_succ":time_point_pull1_succ,
                                      "pull2_succ":time_point_pull2_succ,
                                      "pull1_fail":time_point_pull1_fail,
                                      "pull2_fail":time_point_pull2_fail,
                                    }
        
        # 
        # based on time point pull and juice, define some features for each pull action
        pull_infos = get_pull_infos(animal1, animal2, time_point_pull1, time_point_pull2, 
                                    time_point_juice1, time_point_juice2)
        
        # new total session time (instead of 600s) - total time of the video recording
        totalsess_time = np.ceil(np.shape(output_look_ornot['look_at_lever_or_not_merge']['dodson'])[0]/30) 
        
        #
        # organize variables that are required by the HDDM functions
        # load the data first, if not process and then save the data 
        print('prepare the data for HDDM')
        try:
            # dummy
            current_dir = data_saved_folder+'/bhv_events_singlecam_wholebody_with_hddm_model/'+animal1_fixedorder[0]+animal2_fixedorder[0]
            add_date_dir = os.path.join(current_dir,cameraID+'/'+date_tgt)
            with open(add_date_dir+'/hddm_data.pkl', 'rb') as f:
                hddm_data = pickle.load(f)
            with open(add_date_dir+'/hddm_data_flexibleTW.pkl', 'rb') as f:
                hddm_data_flexibleTW = pickle.load(f)
            with open(add_date_dir+'/hddm_data_flexibleTW_newRT.pkl', 'rb') as f:
                hddm_data_flexibleTW_newRT = pickle.load(f)
        except:
            print('no HDDM data, analyze it and save it')
            #
            gausKernelsize = 4
            #
            # clean the data
            time_point_pull1 = np.array(time_point_pull1)+session_start_time
            time_point_pull1 = time_point_pull1[time_point_pull1<totalsess_time]
            time_point_pull2 = np.array(time_point_pull2)+session_start_time
            time_point_pull2 = time_point_pull2[time_point_pull2<totalsess_time]
            #
            time_point_juice1 = np.array(time_point_juice1)+session_start_time
            time_point_juice1 = time_point_juice1[time_point_juice1<totalsess_time]
            time_point_juice2 = np.array(time_point_juice2)+session_start_time
            time_point_juice2 = time_point_juice2[time_point_juice2<totalsess_time]
            #
            oneway_gaze1 = np.sort(np.hstack((oneway_gaze1,mutual_gaze1)))
            animal1_gaze = oneway_gaze1
            animal1_gaze = np.sort(np.unique(animal1_gaze))+session_start_time
            #
            oneway_gaze2 = np.sort(np.hstack((oneway_gaze2,mutual_gaze2)))
            animal2_gaze = oneway_gaze2
            animal2_gaze = np.sort(np.unique(animal2_gaze))+session_start_time
            #
            # organize the data into a time series
            pull1_data = np.zeros([int(totalsess_time*fps),])
            pull1_data[np.round(time_point_pull1*fps).astype(int)]=1
            #
            pull2_data = np.zeros([int(totalsess_time*fps),])
            pull2_data[np.round(time_point_pull2*fps).astype(int)]=1
            #
            juice1_data = np.zeros([int(totalsess_time*fps),])
            juice1_data[np.round(time_point_juice1*fps).astype(int)]=1
            #
            juice2_data = np.zeros([int(totalsess_time*fps),])
            juice2_data[np.round(time_point_juice2*fps).astype(int)]=1
            #
            gaze1_data = np.zeros([int(totalsess_time*fps),])
            gaze1_data[np.round(animal1_gaze*fps).astype(int)]=1
            gaze1_data = scipy.ndimage.gaussian_filter1d(gaze1_data,gausKernelsize)
            #
            gaze2_data = np.zeros([int(totalsess_time*fps),])
            gaze2_data[np.round(animal2_gaze*fps).astype(int)]=1
            gaze2_data = scipy.ndimage.gaussian_filter1d(gaze2_data,gausKernelsize)
            #
            facemass1 = output_key_locations['facemass_loc_all_merge']['dodson'].transpose()
            facemass1 = np.hstack((facemass1,[[np.nan],[np.nan]]))
            at1_min_at0 = (facemass1[:,1:]-facemass1[:,:-1])
            speed1_data = np.sqrt(np.einsum('ij,ij->j', at1_min_at0, at1_min_at0))*fps 
            speed1_data = scipy.ndimage.gaussian_filter1d(speed1_data,gausKernelsize)
            #
            facemass2 = output_key_locations['facemass_loc_all_merge']['scorch'].transpose()
            facemass2 = np.hstack((facemass2,[[np.nan],[np.nan]]))
            at1_min_at0 = (facemass2[:,1:]-facemass2[:,:-1])
            speed2_data = np.sqrt(np.einsum('ij,ij->j', at1_min_at0, at1_min_at0))*fps 
            speed2_data = scipy.ndimage.gaussian_filter1d(speed2_data,gausKernelsize)
            #
            # get the data that can be used for the HDDM pipeline
            # also plot the correlation 
            time_window_start_s=-4 # same setting as in the other analysis pipeline
            time_window_end_s=0
            coop_window = 1
            resolution = 1/fps
            #
            try:
                if np.isin(animal1,animal1_fixedorder):
                    hddm_data = analyze_pull_aligned_data(pull1_data, pull2_data, gaze1_data, gaze2_data, 
                                                         speed1_data, speed2_data, resolution, time_window_start_s, 
                                                         time_window_end_s, coop_window)

                    hddm_data_flexibleTW = analyze_pull_aligned_data_flexibleTW(pull1_data, pull2_data, 
                                                                               gaze1_data, gaze2_data, 
                                                                               speed1_data, speed2_data, resolution, 
                                                                               time_window_start_s, time_window_end_s, 
                                                                               coop_window)

                    hddm_data_flexibleTW_newRT = analyze_pull_aligned_data_flexibleTW_newRTdefinition(pull1_data, pull2_data, 
                                                                               juice1_data, juice2_data,
                                                                               gaze1_data, gaze2_data, 
                                                                               speed1_data, speed2_data, resolution, 
                                                                               time_window_start_s, time_window_end_s, 
                                                                               coop_window,fps)
                else:
                    hddm_data = analyze_pull_aligned_data(pull2_data, pull1_data, gaze2_data, gaze1_data, 
                                                         speed2_data, speed1_data, resolution, time_window_start_s, 
                                                         time_window_end_s, coop_window)

                    hddm_data_flexibleTW = analyze_pull_aligned_data_flexibleTW(pull2_data, pull1_data, 
                                                                               gaze2_data, gaze1_data, 
                                                                               speed2_data, speed1_data, resolution, 
                                                                               time_window_start_s, time_window_end_s, 
                                                                               coop_window)

                    hddm_data_flexibleTW_newRT = analyze_pull_aligned_data_flexibleTW_newRTdefinition(pull2_data, pull1_data, 
                                                                               juice2_data, juice1_data,
                                                                               gaze2_data, gaze1_data, 
                                                                               speed2_data, speed1_data, resolution, 
                                                                               time_window_start_s, time_window_end_s, 
                                                                               coop_window,fps)
            except:
                print('ereor')
                
                hddm_data = np.nan
                hddm_data_flexibleTW = np.nan
                hddm_data_flexibleTW_newRT = np.nan
                
            
            #
            # save data
            current_dir = data_saved_folder+'/bhv_events_singlecam_wholebody_with_hddm_model/'+animal1_fixedorder[0]+animal2_fixedorder[0]
            add_date_dir = os.path.join(current_dir,cameraID+'/'+date_tgt)
            if not os.path.exists(add_date_dir):
                os.makedirs(add_date_dir)
            #
            with open(add_date_dir+'/hddm_data.pkl', 'wb') as f:
                pickle.dump(hddm_data, f)
            with open(add_date_dir+'/hddm_data_flexibleTW.pkl', 'wb') as f:
                pickle.dump(hddm_data_flexibleTW, f)
            with open(add_date_dir+'/hddm_data_flexibleTW_newRT.pkl', 'wb') as f:
                pickle.dump(hddm_data_flexibleTW_newRT, f)
      
        #    
        hddm_datas_all_dates[date_tgt] = hddm_data
        hddm_datas_flexibleTW_all_dates[date_tgt] = hddm_data_flexibleTW
        hddm_datas_flexibleTW_newRTdefinition_all_dates[date_tgt] = hddm_data_flexibleTW_newRT
        
        
            
        # # #     
        # load and organize neural related data   
        # # # 
            
        # session starting time compared with the neural recording
        session_start_time_niboard_offset = ni_data['session_t0_offset'] # in the unit of second
        try:
            neural_start_time_niboard_offset = ni_data['trigger_ts'][0]['elapsed_time'] # in the unit of second
        except: # for the multi-animal recording setup
            neural_start_time_niboard_offset = next(
                entry['timepoints'][0]['elapsed_time']
                for entry in ni_data['trigger_ts']
                if entry['channel_name'] == f"{trig_channelname}")
        neural_start_time_session_start_offset = neural_start_time_niboard_offset-session_start_time_niboard_offset
    
    
        # load channel maps
        channel_map_file = '/home/ws523/kilisort_spikesorting/Channel-Maps/Neuronexus_whitematter_2x32.mat'
        # channel_map_file = '/home/ws523/kilisort_spikesorting/Channel-Maps/Neuronexus_whitematter_2x32_kilosort4_new.mat'
        channel_map_data = scipy.io.loadmat(channel_map_file)     
        
        # 
        # # load spike sorting results
        if 0:
            print('load spike data for '+neural_record_condition)
            if kilosortver == 2:
                spike_time_file = neural_data_folder+neural_record_condition+'/Kilosort/spike_times.npy'
                spike_time_data = np.load(spike_time_file)
            elif kilosortver == 4:
                spike_time_file = neural_data_folder+neural_record_condition+'/kilosort4_6500HzNotch/spike_times.npy'
                spike_time_data = np.load(spike_time_file)
            # 
            # align the FR recording time stamps
            spike_time_data = spike_time_data + fs_spikes*neural_start_time_session_start_offset
            # down-sample the spike recording resolution to 30Hz
            spike_time_data = spike_time_data/fs_spikes*fps
            # spike_time_data = np.round(spike_time_data)
            #
            if kilosortver == 2:
                spike_clusters_file = neural_data_folder+neural_record_condition+'/Kilosort/spike_clusters.npy'
                spike_clusters_data = np.load(spike_clusters_file)
                spike_channels_data = np.copy(spike_clusters_data)
            elif kilosortver == 4:
                spike_clusters_file = neural_data_folder+neural_record_condition+'/kilosort4_6500HzNotch/spike_clusters.npy'
                spike_clusters_data = np.load(spike_clusters_file)
                spike_channels_data = np.copy(spike_clusters_data)
            #
            if kilosortver == 2:
                channel_maps_file = neural_data_folder+neural_record_condition+'/Kilosort/channel_map.npy'
                channel_maps_data = np.load(channel_maps_file)
            elif kilosortver == 4:
                channel_maps_file = neural_data_folder+neural_record_condition+'/kilosort4_6500HzNotch/channel_map.npy'
                channel_maps_data = np.load(channel_maps_file)
            #
            if kilosortver == 2:
                channel_pos_file = neural_data_folder+neural_record_condition+'/Kilosort/channel_positions.npy'
                channel_pos_data = np.load(channel_pos_file)
            elif kilosortver == 4:
                channel_pos_file = neural_data_folder+neural_record_condition+'/kilosort4_6500HzNotch/channel_positions.npy'
                channel_pos_data = np.load(channel_pos_file)
            #
            if kilosortver == 2:
                clusters_info_file = neural_data_folder+neural_record_condition+'/Kilosort/cluster_info.tsv'
                clusters_info_data = pd.read_csv(clusters_info_file,sep="\t")
            elif kilosortver == 4:
                clusters_info_file = neural_data_folder+neural_record_condition+'/kilosort4_6500HzNotch/cluster_info.tsv'
                clusters_info_data = pd.read_csv(clusters_info_file,sep="\t")
            #
            # only get the spikes that are manually checked
            try:
                good_clusters = clusters_info_data[(clusters_info_data.group=='good')|(clusters_info_data.group=='mua')]['cluster_id'].values
            except:
                good_clusters = clusters_info_data[(clusters_info_data.group=='good')|(clusters_info_data.group=='mua')]['id'].values
            #
            clusters_info_data = clusters_info_data[~pd.isnull(clusters_info_data.group)]
            #
            spike_time_data = spike_time_data[np.isin(spike_clusters_data,good_clusters)]
            spike_channels_data = spike_channels_data[np.isin(spike_clusters_data,good_clusters)]
            spike_clusters_data = spike_clusters_data[np.isin(spike_clusters_data,good_clusters)]
            
            #
            nclusters = np.shape(clusters_info_data)[0]
            #
            for icluster in np.arange(0,nclusters,1):
                try:
                    cluster_id = clusters_info_data['id'].iloc[icluster]
                except:
                    cluster_id = clusters_info_data['cluster_id'].iloc[icluster]
                spike_channels_data[np.isin(spike_clusters_data,cluster_id)] = clusters_info_data['ch'].iloc[icluster]   
            # 
            # get the channel to depth information, change 2 shanks to 1 shank 
            try:
                channel_depth=np.hstack([channel_pos_data[channel_pos_data[:,0]==0,1]*2,channel_pos_data[channel_pos_data[:,0]==1,1]*2+1])
                # channel_depth=np.hstack([channel_pos_data[channel_pos_data[:,0]==0,1],channel_pos_data[channel_pos_data[:,0]==31.2,1]])            
                # channel_to_depth = np.vstack([channel_maps_data.T[0],channel_depth])
                channel_to_depth = np.vstack([channel_maps_data.T,channel_depth])
            except:
                channel_depth=np.hstack([channel_pos_data[channel_pos_data[:,0]==0,1],channel_pos_data[channel_pos_data[:,0]==31.2,1]])            
                # channel_to_depth = np.vstack([channel_maps_data.T[0],channel_depth])
                channel_to_depth = np.vstack([channel_maps_data.T,channel_depth])
                channel_to_depth[1] = channel_to_depth[1]/30-64 # make the y axis consistent
            #
           
            
            # calculate the firing rate
            # FR_kernel = 0.20 # in the unit of second
            FR_kernel = 1/30 # in the unit of second # 1/30 same resolution as the video recording
            # FR_kernel is sent to to be this if want to explore it's relationship with continuous trackng data
            
            totalsess_time_forFR = np.floor(np.shape(output_look_ornot['look_at_lever_or_not_merge']['dodson'])[0]/30)  # to match the total time of the video recording
            #
            _,FR_timepoint_allch,FR_allch,FR_zscore_allch = spike_analysis_FR_calculation(fps, FR_kernel, totalsess_time_forFR,
                                                                                          spike_clusters_data, spike_time_data)
            # _,FR_timepoint_allch,FR_allch,FR_zscore_allch = spike_analysis_FR_calculation(fps,FR_kernel,totalsess_time_forFR,
            #                                                                              spike_channels_data, spike_time_data)                     
            
            #
            # Run PCA analysis
            FR_zscore_allch_np_merged = np.array(pd.DataFrame(FR_zscore_allch).T)
            FR_zscore_allch_np_merged = FR_zscore_allch_np_merged[~np.isnan(np.sum(FR_zscore_allch_np_merged,axis=1)),:]
            # # run PCA on the entire session
            pca = PCA(n_components=3)
            FR_zscore_allch_PCs = pca.fit_transform(FR_zscore_allch_np_merged.T)
           
            
 

    # save data
    if 0:
        data_saved_subfolder = data_saved_folder+'data_saved_singlecam_wholebody_neural_and_hddm'+savefile_sufix+'/'+cameraID+'/'+animal1_fixedorders[0]+animal2_fixedorders[0]+'/'
        if not os.path.exists(data_saved_subfolder):
            os.makedirs(data_saved_subfolder)

        with open(data_saved_subfolder+'/hddm_datas_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
            pickle.dump(hddm_datas_all_dates, f) 
        with open(data_saved_subfolder+'/hddm_datas_flexibleTW_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
            pickle.dump(hddm_datas_flexibleTW_all_dates, f) 
        with open(data_saved_subfolder+'/hddm_datas_flexibleTW_newRTdefinition_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
            pickle.dump(hddm_datas_flexibleTW_newRTdefinition_all_dates, f) 
            
    
    # only save a subset 
    if 0:
        data_saved_subfolder = data_saved_folder+'data_saved_singlecam_wholebody_neural_and_hddm'+savefile_sufix+'/'+cameraID+'/'+animal1_fixedorders[0]+animal2_fixedorders[0]+'/'
        if not os.path.exists(data_saved_subfolder):
            os.makedirs(data_saved_subfolder)

        with open(data_saved_subfolder+'/hddm_datas_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
            pickle.dump(hddm_datas_all_dates, f) 
        with open(data_saved_subfolder+'/hddm_datas_flexibleTW_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
            pickle.dump(hddm_datas_flexibleTW_all_dates, f) 
        with open(data_saved_subfolder+'/hddm_datas_flexibleTW_newRTdefinition_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
            pickle.dump(hddm_datas_flexibleTW_newRTdefinition_all_dates, f) 
     
   
    

In [ ]:
singlecam_ana_type

### cartoon illustrator to show the variables of v, a, t, and z. 

In [ ]:
if 0:
    import matplotlib.pyplot as plt
    import matplotlib.patches as patches
    import numpy as np
    from scipy.stats import gaussian_kde

    # Create a figure and axes for the cartoon
    fig, ax = plt.subplots(figsize=(10, 8))

    # Set the background to a light color and remove the plot frame
    ax.set_facecolor("#f0f4f8")
    ax.axis('off')

    # --- Define the key variables for the model ---
    # Boundary separation 'a'
    boundary_a = 5
    # Starting point 'z'
    starting_z = 0
    # Drift rate 'v' - the slope of the line
    drift_rate_v = 1.0
    # Non-decision time 't'
    non_decision_t = 2
    # Noise term (diffusion coefficient 'c')
    diffusion_c = 1.0

    # --- Draw the model's components ---

    # The decision boundary (a single upper boundary for this model)
    ax.plot([0, 15], [boundary_a, boundary_a], 'k--', linewidth=2, zorder=1)
    ax.plot([0, 15], [0, 0], 'k-', linewidth=2, zorder=1) # Ground line at 0

    # The starting point
    ax.plot([non_decision_t], [starting_z], 'ro', markersize=10, zorder=3)

    # Add a text label and a cartoon clock for non-decision time
    ax.text(non_decision_t / 2, -0.5, 'Non-decision time (t)', horizontalalignment='center', fontsize=12, color='gray')
    ax.text(non_decision_t / 2, -1.2, '⏱️', horizontalalignment='center', fontsize=20)
    ax.axvline(x=non_decision_t, color='gray', linestyle=':', linewidth=2, zorder=1)

    # --- Drift process: both a straight line and a random walk simulation ---
    # Time for mean drift to hit the boundary
    time_to_hit = (boundary_a - starting_z) / drift_rate_v
    end_x_drift = non_decision_t + time_to_hit

    # Straight drift line (representing the mean drift)
    # This now consists of a flat line for non-decision time and a sloped line for decision time
    x_drift_non_dec = [0, non_decision_t]
    y_drift_non_dec = [starting_z, starting_z]
    ax.plot(x_drift_non_dec, y_drift_non_dec, 'b--', linewidth=2, zorder=2)

    x_drift_dec = [non_decision_t, end_x_drift]
    y_drift_dec = [starting_z, starting_z + (end_x_drift - non_decision_t) * drift_rate_v]
    ax.plot(x_drift_dec, y_drift_dec, 'b-', linewidth=3, zorder=2, label='Mean Drift (v)')
    ax.text(end_x_drift - 1.5, y_drift_dec[1] - 0.5, 'Evidence Accumulation (v)', color='b', fontsize=12, horizontalalignment='center')
    ax.arrow(x_drift_dec[0], y_drift_dec[0], 0.5, 0.5 * drift_rate_v, head_width=0.3, head_length=0.3, fc='b', ec='b', zorder=2)

    # Random walk simulation for a single trial
    # Create a single random walk trace for the entire period
    num_steps_total = 1000
    dt_total = 15 / num_steps_total
    time_points_total = np.linspace(0, 15, num_steps_total)

    # Create a drift rate array that is 0 during non-decision time and v during decision time
    drift_rate_array = np.zeros_like(time_points_total)
    drift_rate_array[time_points_total >= non_decision_t] = drift_rate_v

    # Generate random steps and calculate the cumulative sum
    random_steps = np.random.normal(loc=0, scale=diffusion_c, size=num_steps_total) * np.sqrt(dt_total)
    evidence_accumulation_walk = np.cumsum(drift_rate_array * dt_total + random_steps)

    # Find the first point where the random walk crosses the boundary
    hit_index = np.where(evidence_accumulation_walk >= boundary_a)[0]
    if hit_index.size > 0:
        hit_index = hit_index[0]
        # Plot only up to the boundary crossing
        ax.plot(time_points_total[:hit_index+1], evidence_accumulation_walk[:hit_index+1], 'g-', alpha=0.7, linewidth=2, label='Single Random Walk')
    else:
        # If it never hits, plot the whole path
        ax.plot(time_points_total, evidence_accumulation_walk, 'g-', alpha=0.7, linewidth=2, label='Single Random Walk')

    ax.legend(loc='lower right')

    # --- Simulate multiple trials to get a response time distribution ---
    num_trials = 5000
    dt_sim = 0.01
    hit_times = []
    for _ in range(num_trials):
        evidence = starting_z
        time = 0
        # Combined non-decision and decision phase
        while evidence < boundary_a and time < 15: # Increased time limit for the simulation
            drift_term = drift_rate_v * dt_sim if time >= non_decision_t else 0
            evidence += drift_term + np.random.normal(loc=0, scale=diffusion_c) * np.sqrt(dt_sim)
            time += dt_sim
        if evidence >= boundary_a:
            hit_times.append(time)

    # --- Add the response time distribution plot on top of the boundary ---
    if hit_times:
        kde = gaussian_kde(hit_times)
        kde_x = np.linspace(min(hit_times), max(hit_times), 100)
        kde_y = kde(kde_x)

        # Scale the KDE plot to fit on the boundary
        kde_y_scaled = (kde_y / kde_y.max()) * 1.5 + boundary_a

        ax.plot(kde_x, kde_y_scaled, color='skyblue', linewidth=2)
        ax.fill_between(kde_x, boundary_a, kde_y_scaled, color='skyblue', alpha=0.5)

        ax.text(np.mean(hit_times), boundary_a + 2, 'Response Time Distribution', horizontalalignment='center', fontsize=12, fontweight='bold', color='skyblue')

    # --- Add text annotations and cartoon elements ---
    ax.text(10.2, boundary_a, 'Boundary (a)', verticalalignment='center', fontsize=14, fontweight='bold')
    ax.text(10.2, 0, 'Start point (z)', verticalalignment='center', fontsize=14, fontweight='bold')
    ax.text(non_decision_t - 0.2, starting_z, 'z', verticalalignment='center', horizontalalignment='right', fontsize=14, fontweight='bold', color='red')
    ax.text(2, -0.5, 't', verticalalignment='top', horizontalalignment='center', fontsize=14, fontweight='bold', color='gray')

    # --- Display the DDM formulas using LaTeX ---
    # The formula shows the stochastic process that the model is built on.
    ax.text(8.5, 0.5, r'$dx = v \cdot dt + c \cdot dW$', fontsize=16, color='black', ha='center',
            bbox=dict(boxstyle='round,pad=0.5', fc='white', ec='gray', lw=1))

    # Add the formula for mean response time
    ax.text(8.5, 1.5, r'$E(RT) \approx t + \frac{a - z}{v}$', fontsize=16, color='black', ha='center',
            bbox=dict(boxstyle='round,pad=0.5', fc='white', ec='gray', lw=1))

    # --- Cartoon elements for influences on 'v' ---
    box_rect = patches.FancyBboxPatch((0.5, 6.5), 9, 2.5, boxstyle="round,pad=0.3", fc="#e6f2ff", ec="#99c2ff", linewidth=2)
    ax.add_patch(box_rect)
    ax.text(5, 8.5, 'Influences on Drift Rate (v)', horizontalalignment='center', fontsize=16, fontweight='bold')

    # Self Action
    ax.text(2.5, 7.5, 'Self Action:', horizontalalignment='center', fontsize=12, fontweight='bold')
    ax.text(2.5, 7.0, '• Self speed mean/std', horizontalalignment='center', fontsize=11)
    ax.text(2.5, 6.5, '🏃💨', horizontalalignment='center', fontsize=20)

    # Partner's Action
    ax.text(7.5, 7.5, "Partner's Action:", horizontalalignment='center', fontsize=12, fontweight='bold')
    ax.text(7.5, 7.0, '• Action PC1 mean/std', horizontalalignment='center', fontsize=11)
    ax.text(7.5, 6.5, '🤝🤖', horizontalalignment='center', fontsize=20)

    # Set plot limits and aspect ratio
    ax.set_xlim(0, 15) # Adjusted x-axis to accommodate longer response times
    ax.set_ylim(-1.5, 9.5) # Adjusted y-axis to accommodate new label
    ax.set_aspect('equal', adjustable='box')

    plt.title('Cartoon of the One Boundary Drift-Diffusion Model', fontsize=18, fontweight='bold')

    plt.savefig('cartoon_of_the_ddm.pdf')

### fit the HDDM model for all the data, copy this part and following to the Misha later when HDDM pipeline is installed successfully

In [ ]:
# 
# animal_tgt = 'kanga' # 'kanga','kanga_partner','dodson','dodson_partner'
animal_tgt = 'dodson'

#
# conditions_to_ana = np.unique(task_conditions)
###
# For Kanga
# conditions_to_ana = ['MC', 'MC_withDodson','MC_withGinger', 'MC_withKoala', 'MC_withVermelho', ] # all MC
# conditions_to_ana = [ 'MC_withVermelho']
# conditions_to_ana = ['SR', 'SR_withDodson', ] # all SR
# conditions_to_ana = ['MC', 'MC_withDodson', 'MC_withVermelho', ] # MC with male
# conditions_to_ana = ['MC_withGinger', 'MC_withKoala', ] # MC with female
# conditions_to_ana = ['MC', ] # MC with familiar male
# conditions_to_ana = ['MC_withGinger', ] # MC with familiar female
# conditions_to_ana = ['MC_withDodson', 'MC_withVermelho', ] # MC with unfamiliar male
# conditions_to_ana = ['MC_withKoala', ] # MC with unfamiliar female
# conditions_to_ana = ['MC_DannonAuto'] # partner AL
# conditions_to_ana = ['MC_KangaAuto'] # self AL
# conditions_to_ana = ['NV','NV_withDodson'] # NV
# conditions_to_ana = ['MC', 'MC_withDodson','MC_withGinger', 'MC_withKoala', 'MC_withVermelho', 
#                      'SR', 'SR_withDodson',]
###
# For Dodson
conditions_to_ana = ['MC', 'MC_withGingerNew', 'MC_withKanga', 'MC_withKoala', ] # all MC
# conditions_to_ana = ['MC', 'MC_withGingerNew', 'MC_withKanga', ] # all MC
# conditions_to_ana = ['MC', 'MC_withKanga', 'MC_withKoala', ] # all MC
# conditions_to_ana = ['SR', 'SR_withGingerNew', 'SR_withKanga', 'SR_withKoala', ] # all SR
# conditions_to_ana = ['MC', 'MC_withKanga', 'MC_withKoala', ] # all MC, no gingerNew
# conditions_to_ana = ['SR', 'SR_withKanga', 'SR_withKoala', ] # all SR,  no gingerNew
# conditions_to_ana = ['MC', 'MC_withGingerNew', 'MC_withKanga', 'MC_withKoala', ] # MC with female
# conditions_to_ana = ['MC', 'MC_withGingerNew', ] # MC with familiar female
# conditions_to_ana = ['MC_withKanga', 'MC_withKoala', ] # MC with unfamiliar female
# conditions_to_ana = ['MC_KoalaAuto_withKoala'] # partner AL
# conditions_to_ana = ['MC_DodsonAuto_withKoala'] # self AL
# conditions_to_ana = ['NV_withKanga'] # NV
# conditions_to_ana = ['MC', 'MC_withGingerNew', 'MC_withKanga', 'MC_withKoala', 
#                      'SR', 'SR_withGingerNew', 'SR_withKanga', 'SR_withKoala', ]


# cond_toplot_type = 'allSR'
cond_toplot_type = 'allMC'
# cond_toplot_type =  'MC_withVermelho'

# make sure to only look certain condition, and keep things consistent
ind_tgt = list(np.isin(task_conditions,conditions_to_ana))

animal1_fixedorders_toana = list(np.array(animal1_fixedorders)[ind_tgt])
animal2_fixedorders_toana = list(np.array(animal2_fixedorders)[ind_tgt])
task_conditions_toana = list(np.array(task_conditions)[ind_tgt])

dates_to_ana = list(np.array(dates_list)[ind_tgt])
ndates_to_ana = np.shape(dates_to_ana)[0]

In [ ]:
# do the hddm fitting for all sessions separately
# not in use because the reaction time definition is outdated
if 0:

    dofixTW = 0 # 0: not do the fixed TW, instead use the flexible TW (from one pull to another); 1: do the fixed tw (-4s to 0s)
    doflexTW_newRT = 1 # 1: do the flexible TW with new definition of RT
    #
    if dofixTW:
        fixTW_sufix = '_fixedTW'
    elif not dofixTW:
        if not doflexTW_newRT:
            fixTW_sufix = ''
        elif doflexTW_newRT:
            fixTW_sufix = '_flexTW_newRT'
    
    # HDDM settings - the basic setting
    samples=250 # 2000
    burn=100
    thin=1
    
    # keep the regression r and p between gaze_accum and diffusion slope v
    corr_gazeaccum_driftv = np.ones((ndates_to_ana,))*np.nan
    p_gazeaccum_driftv = np.ones((ndates_to_ana,))*np.nan

    for idate in np.arange(0,ndates_to_ana,1):
    # for idate in np.arange(3,4,1):

        date_toana = dates_to_ana[idate]
        
        task_condition_toana = task_conditions_toana[idate]
        
        animal1_fixedorder = [animal1_fixedorders_toana[idate]]
        animal2_fixedorder = [animal2_fixedorders_toana[idate]]

        try:
            # try to load the data first
            try:
                current_dir = data_saved_folder+'bhv_events_singlecam_wholebody_with_hddm_model/'+animal1_fixedorder[0]+animal2_fixedorder[0]
                add_date_dir = os.path.join(current_dir,cameraID+'/'+date_toana)
                #
                with open(add_date_dir+'/hddm_model_fitted_traces'+fixTW_sufix+'.pkl', 'rb') as f:
                    hddm_model_fitted_traces = pickle.load(f)
                with open(add_date_dir+'/hddm_model_fitted_nogaze_traces'+fixTW_sufix+'.pkl', 'rb') as f:
                    hddm_model_fitted_nogaze_traces = pickle.load(f)
                
                with open(add_date_dir+'/hddm_model_fitted_stats'+fixTW_sufix+'.pkl', 'rb') as f:
                    hddm_model_fitted_stats = pickle.load(f)
                with open(add_date_dir+'/hddm_model_fitted_nogaze_stats'+fixTW_sufix+'.pkl', 'rb') as f:
                    hddm_model_fitted_nogaze_stats = pickle.load(f)
                    
                with open(add_date_dir+'/hddm_model_fitted_dic'+fixTW_sufix+'.pkl', 'rb') as f:
                    hddm_model_fitted_dic = pickle.load(f)
                with open(add_date_dir+'/hddm_model_fitted_nogaze_dic'+fixTW_sufix+'.pkl', 'rb') as f:
                    hddm_model_fitted_nogaze_dic = pickle.load(f)
                    
                if dofixTW:
                    with open(add_date_dir+'/hddm_data.pkl', 'rb') as f:
                        hddm_data_tgt_idate = pickle.load(f)
                elif not dofixTW:
                    if not doflexTW_newRT:
                        with open(add_date_dir+'/hddm_data_flexibleTW.pkl', 'rb') as f:
                            hddm_data_tgt_idate = pickle.load(f)
                    elif doflexTW_newRT:
                        with open(add_date_dir+'/hddm_data_flexibleTW_newRT.pkl', 'rb') as f:
                            hddm_data_tgt_idate = pickle.load(f)
                #
                hddm_data_tgt_idate = hddm_data_tgt_idate['hddm_data_'+animal_id_toana]
                                                         
                        
                print(animal_tgt+' '+date_toana+' hddm fitting results loaded')

            # do the hddm model fitting
            except:

                print(animal_tgt+' '+date_toana+' running hddm fitting')

                if np.isin(animal_tgt,animal1_fixedorders[idate]):
                    animal_id_toana = 'animal1'
                elif np.isin(animal_tgt,animal2_fixedorders[idate]):
                    animal_id_toana = 'animal2'

                # # use the fixed -4s to 0s time window
                if dofixTW:
                    hddm_data = hddm_datas_all_dates[date_toana]
                # use the flexible time window setting
                elif not dofixTW:
                    hddm_data = hddm_datas_flexibleTW_all_dates[date_toana]
                    
                hddm_data_tgt_idate = hddm_data['hddm_data_'+animal_id_toana]

                    
                # run the hddm nodel - to fix the problem, use a lazy methods
                try:
                    hddm_model_fitted, hddm_model_fitted_nogaze = run_hddm_modeling(hddm_data_tgt_idate, 
                                                                                animal_id_toana, samples, burn, thin)
                except:
                    samples=500 # 2000
                    burn=100
                    thin=2  
                    hddm_model_fitted, hddm_model_fitted_nogaze = run_hddm_modeling(hddm_data_tgt_idate, 
                                                                                animal_id_toana, samples, burn, thin)

                #
                hddm_model_fitted_traces = hddm_model_fitted.get_traces()
                hddm_model_fitted_stats  = hddm_model_fitted.gen_stats()
                hddm_model_fitted_dic = hddm_model_fitted.dic_info
                #
                hddm_model_fitted_nogaze_traces = hddm_model_fitted_nogaze.get_traces()
                hddm_model_fitted_nogaze_stats  = hddm_model_fitted_nogaze.gen_stats()
                hddm_model_fitted_nogaze_dic = hddm_model_fitted_nogaze.dic_info

                # save data
                if 1:
                    current_dir = data_saved_folder+'bhv_events_singlecam_wholebody_with_hddm_model/'+animal1_fixedorder[0]+animal2_fixedorder[0]
                    add_date_dir = os.path.join(current_dir,cameraID+'/'+date_toana)
                    if not os.path.exists(add_date_dir):
                        os.makedirs(add_date_dir)
                    #
                    with open(add_date_dir+'/hddm_model_fitted_traces'+fixTW_sufix+'.pkl', 'wb') as f:
                        pickle.dump(hddm_model_fitted_traces, f)
                    with open(add_date_dir+'/hddm_model_fitted_nogaze_traces'+fixTW_sufix+'.pkl', 'wb') as f:
                        pickle.dump(hddm_model_fitted_nogaze_traces, f)
                    
                    with open(add_date_dir+'/hddm_model_fitted_stats'+fixTW_sufix+'.pkl', 'wb') as f:
                        pickle.dump(hddm_model_fitted_stats, f)
                    with open(add_date_dir+'/hddm_model_fitted_nogaze_stats'+fixTW_sufix+'.pkl', 'wb') as f:
                        pickle.dump(hddm_model_fitted_nogaze_stats, f)
                        
                    with open(add_date_dir+'/hddm_model_fitted_dic'+fixTW_sufix+'.pkl', 'wb') as f:
                        pickle.dump(hddm_model_fitted_dic, f)
                    with open(add_date_dir+'/hddm_model_fitted_nogaze_dic'+fixTW_sufix+'.pkl', 'wb') as f:
                        pickle.dump(hddm_model_fitted_nogaze_dic, f)
                        
                        
            # do some plotting
            if 0:
                df_combined = hddm_data_tgt_idate
                #
                # traces = hddm_model_fitted_traces
                traces = hddm_model_fitted_nogaze_traces
                #
                # Get the mean value for the intercept
                v_intercept_mean = traces['v_Intercept'].mean()
                #
                # Get the mean value for the coefficient of each predictor
                # v_gaze_coef_mean = traces['v_self_gaze_auc'].mean()
                v_speed_coef_mean = traces['v_partner_mean_speed'].mean()
                v_failed_pulls_before_reward_mean = traces['v_failed_pulls_before_reward'].mean()
                v_time_since_last_reward_mean = traces['v_time_since_last_reward'].mean()
                #
                print(f"Mean v_Intercept: {v_intercept_mean:.3f}")
                # print(f"Mean v_self_gaze_auc Coef: {v_gaze_coef_mean:.3f}")
                print(f"Mean v_partner_mean_speed Coef: {v_speed_coef_mean:.3f}")
                print(f"Mean v_failed_pulls_before_reward Coef: {v_failed_pulls_before_reward_mean:.3f}")
                print(f"Mean v_time_since_last_reward Coef: {v_time_since_last_reward_mean:.3f}")
                #
                # --- STEP 2: Apply the regression equation to your dataframe ---
                # This calculates the predicted 'v' for each trial based on its unique covariate values.
                df_with_v = df_combined.copy() # Work with a copy
                #
                df_with_v['predicted_v'] = (v_intercept_mean +
                                              # (v_gaze_coef_mean * df_with_v['self_gaze_auc']) +
                                              (v_speed_coef_mean * df_with_v['partner_mean_speed'])+
                                              (v_failed_pulls_before_reward_mean * df_with_v['failed_pulls_before_reward'])+
                                              (v_time_since_last_reward_mean * df_with_v['time_since_last_reward']))

                #!!! Add some criteria for plotting
                # ind_good1 = (df_with_v['self_gaze_auc']>=0) & (df_with_v['self_gaze_auc']<=1)
                # ind_good2 = df_with_v['predicted_v']<3
                # df_with_v = df_with_v[ind_good1 & ind_good2]
                #
                fig, r_vals, p_vals = do_hddm_model_fitted_plot(df_with_v)
                #
                corr_gazeaccum_driftv[idate] = r_vals
                p_gazeaccum_driftv[idate] = p_vals

        except:
            continue
    

In [ ]:
# combine the target sessions together, then run the hddm fitting
# not in use because the reaction time definition is outdated

if 0:

    dofixTW = 0 # 0: not do the fixed TW, instead use the flexible TW (from one pull to another); 1: do the fixed tw (-4s to 0s)
    doflexTW_newRT = 1 # 1: do the flexible TW with new definition of RT
    #
    if dofixTW:
        fixTW_sufix = '_fixedTW'
    elif not dofixTW:
        if not doflexTW_newRT:
            fixTW_sufix = ''
        elif doflexTW_newRT:
            fixTW_sufix = '_flexTW_newRT'
        
    # only run the noselfgaze hddm fitting
    doNogazeOnly = 0
    
    # use more variables for v a and z
    doExaustModel = 0
    if doExaustModel:
        exaustModel_sufix = '_exaustModel'
    elif not doExaustModel:
        exaustModel_sufix = ''
        
        
    # try to load the data first
    try:
        
        # dummpy
        
        current_dir = data_saved_folder+'data_saved_singlecam_wholebody_neural_and_hddm'+savefile_sufix+'/'+cameraID+'/'+animal_tgt
        add_date_dir = os.path.join(current_dir+'/hddm_model_fitted_combinedsession')
        #
        if not doNogazeOnly:
            with open(add_date_dir+'/hddm_model_fitted_traces_'+cond_toplot_type+fixTW_sufix+exaustModel_sufix+'.pkl', 'rb') as f:
                hddm_model_fitted_traces = pickle.load(f)
            
            with open(add_date_dir+'/hddm_model_fitted_stats_'+cond_toplot_type+fixTW_sufix+exaustModel_sufix+'.pkl', 'rb') as f:
                hddm_model_fitted_stats = pickle.load(f)
            
            with open(add_date_dir+'/hddm_model_fitted_dic_'+cond_toplot_type+fixTW_sufix+exaustModel_sufix+'.pkl', 'rb') as f:
                hddm_model_fitted_dic = pickle.load(f)
        #
        with open(add_date_dir+'/hddm_model_fitted_nogaze_traces_'+cond_toplot_type+fixTW_sufix+exaustModel_sufix+'.pkl', 'rb') as f:
            hddm_model_fitted_nogaze_traces = pickle.load(f)
        
        with open(add_date_dir+'/hddm_model_fitted_nogaze_stats_'+cond_toplot_type+fixTW_sufix+exaustModel_sufix+'.pkl', 'rb') as f:
            hddm_model_fitted_nogaze_stats = pickle.load(f)
        
        with open(add_date_dir+'/hddm_model_fitted_nogaze_dic_'+cond_toplot_type+fixTW_sufix+exaustModel_sufix+'.pkl', 'rb') as f:
            hddm_model_fitted_nogaze_dic = pickle.load(f)
            
        with open(add_date_dir+'/hddm_data_tgt_alldates_'+cond_toplot_type+fixTW_sufix+exaustModel_sufix+'.pkl', 'rb') as f:
            hddm_data_tgt_alldates = pickle.load(f)

        print(animal_tgt+' '+cond_toplot_type+' hddm fitting results loaded')

    # do the hddm model fitting    
    except:
        print(animal_tgt+' '+cond_toplot_type+' running hddm fitting')
        
        for idate in np.arange(0,ndates_to_ana,1):
        # for idate in np.arange(3,4,1):

            date_toana = dates_to_ana[idate]
            
            task_condition_toana = task_conditions_toana[idate]

            animal1_fixedorder = [animal1_fixedorders_toana[idate]]
            animal2_fixedorder = [animal2_fixedorders_toana[idate]]
    

            if np.isin(animal_tgt,animal1_fixedorders[idate]):
                animal_id_toana = 'animal1'
            elif np.isin(animal_tgt,animal2_fixedorders[idate]):
                animal_id_toana = 'animal2'

            # # use the fixed -4s to 0s time window
            if dofixTW:
                hddm_data = hddm_datas_all_dates[date_toana]
            # us the flexible time window setting
            elif not dofixTW:
                if not doflexTW_newRT:
                    hddm_data = hddm_datas_flexibleTW_all_dates[date_toana]
                # use the new RT definition
                if doflexTW_newRT:
                    hddm_data = hddm_datas_flexibleTW_newRTdefinition_all_dates[date_toana]

            hddm_data_tgt_idate = hddm_data['hddm_data_'+animal_id_toana]
            
            # add a date column and task condition column
            hddm_data_tgt_idate['date'] = date_toana
            hddm_data_tgt_idate['condition'] = task_condition_toana       
            
            #
            if idate == 0:
                hddm_data_tgt_alldates = hddm_data_tgt_idate
            else:
                hddm_data_tgt_alldates = pd.concat([hddm_data_tgt_alldates, hddm_data_tgt_idate])
              
            
        # # add some semi-artbitrary cretieras
        # hddm_data_tgt_alldates = hddm_data_tgt_alldates[hddm_data_tgt_alldates['prev_trial_outcome']==1]
        # hddm_data_tgt_alldates = hddm_data_tgt_alldates[hddm_data_tgt_alldates['partner_mean_speed']<=300]
        #
        lower_q = 0.05
        upper_q = 0.95
        rt_lower, rt_upper = hddm_data_tgt_alldates['rt'].quantile([lower_q, upper_q])
        hddm_data_tgt_alldates = hddm_data_tgt_alldates[hddm_data_tgt_alldates['rt']>rt_lower]
        hddm_data_tgt_alldates = hddm_data_tgt_alldates[hddm_data_tgt_alldates['rt']<rt_upper]
        #
        tslr_lower, tslr_upper = hddm_data_tgt_alldates['time_since_last_reward'].quantile([lower_q, upper_q])
        hddm_data_tgt_alldates = hddm_data_tgt_alldates[hddm_data_tgt_alldates['time_since_last_reward']>tslr_lower]
        hddm_data_tgt_alldates = hddm_data_tgt_alldates[hddm_data_tgt_alldates['time_since_last_reward']<tslr_upper]
            
        # 
        # remove nan row first
        hddm_data_tgt_alldates = hddm_data_tgt_alldates.dropna()
        
        # 
        # normalize all the predictors
        from scipy.stats import zscore
        #
        # List of predictor columns to normalize (exclude 'rt' and identifiers)
        predictor_columns = [
            'self_gaze_auc',
            'partner_mean_speed',
            'self_mean_speed',
            'partner_speed_std',
            'self_speed_std',
            'failed_pulls_before_reward',
            'time_since_last_reward'
        ]
        # Rename originals and replace with z-scored values
        for col in predictor_columns:
            hddm_data_tgt_alldates[f"{col}_origin"] = hddm_data_tgt_alldates[col]
            hddm_data_tgt_alldates[col] = zscore(hddm_data_tgt_alldates[col])  
            
            
        # run the hddm nodel
        if not doExaustModel:
            # HDDM settings - the basic setting
            # samples=500 # 2000
            # burn=100
            # thin=2
            samples=1000 # 2000
            burn=100
            thin=2
            #
            if not doNogazeOnly:
                hddm_model_fitted, hddm_model_fitted_nogaze = run_hddm_modeling(hddm_data_tgt_alldates, animal_id_toana,
                                                                            samples, burn, thin, doNogazeOnly)
            elif doNogazeOnly:
                _, hddm_model_fitted_nogaze = run_hddm_modeling(hddm_data_tgt_alldates, animal_id_toana,
                                                                            samples, burn, thin, doNogazeOnly)
                
        elif doExaustModel:
            # samples=2000 # 2000
            # burn=1000
            # thin=4
            samples=200 # 2000
            burn=100
            thin=2
            #
            if not doNogazeOnly:
                hddm_model_fitted, hddm_model_fitted_nogaze = run_hddm_modeling_exaustModel(hddm_data_tgt_alldates, animal_id_toana, 
                                                                            samples, burn, thin, doNogazeOnly)
            elif doNogazeOnly:
                _, hddm_model_fitted_nogaze = run_hddm_modeling_exaustModel(hddm_data_tgt_alldates, animal_id_toana,
                                                                            samples, burn, thin, doNogazeOnly)

        #
        if not doNogazeOnly:
            hddm_model_fitted_traces = hddm_model_fitted.get_traces()
            hddm_model_fitted_stats  = hddm_model_fitted.gen_stats()
            hddm_model_fitted_dic = hddm_model_fitted.dic_info
        #
        hddm_model_fitted_nogaze_traces = hddm_model_fitted_nogaze.get_traces()
        hddm_model_fitted_nogaze_stats  = hddm_model_fitted_nogaze.gen_stats()
        hddm_model_fitted_nogaze_dic = hddm_model_fitted_nogaze.dic_info

        # save data
        if 1:
            current_dir = data_saved_folder+'data_saved_singlecam_wholebody_neural_and_hddm'+savefile_sufix+'/'+cameraID+'/'+animal_tgt
            add_date_dir = os.path.join(current_dir+'/hddm_model_fitted_combinedsession')
            if not os.path.exists(add_date_dir):
                os.makedirs(add_date_dir)
            #
            if not doNogazeOnly:
                with open(add_date_dir+'/hddm_model_fitted_traces_'+cond_toplot_type+fixTW_sufix+exaustModel_sufix+'.pkl', 'wb') as f:
                    pickle.dump(hddm_model_fitted_traces, f)
            
                with open(add_date_dir+'/hddm_model_fitted_stats_'+cond_toplot_type+fixTW_sufix+exaustModel_sufix+'.pkl', 'wb') as f:
                    pickle.dump(hddm_model_fitted_stats, f)
               
                with open(add_date_dir+'/hddm_model_fitted_dic_'+cond_toplot_type+fixTW_sufix+exaustModel_sufix+'.pkl', 'wb') as f:
                    pickle.dump(hddm_model_fitted_dic, f)
            
            #
            with open(add_date_dir+'/hddm_model_fitted_nogaze_traces_'+cond_toplot_type+fixTW_sufix+exaustModel_sufix+'.pkl', 'wb') as f:
                pickle.dump(hddm_model_fitted_nogaze_traces, f)
            
            with open(add_date_dir+'/hddm_model_fitted_nogaze_stats_'+cond_toplot_type+fixTW_sufix+exaustModel_sufix+'.pkl', 'wb') as f:
                pickle.dump(hddm_model_fitted_nogaze_stats, f)  
            
            with open(add_date_dir+'/hddm_model_fitted_nogaze_dic_'+cond_toplot_type+fixTW_sufix+exaustModel_sufix+'.pkl', 'wb') as f:
                pickle.dump(hddm_model_fitted_nogaze_dic, f)

            with open(add_date_dir+'/hddm_data_tgt_alldates_'+cond_toplot_type+fixTW_sufix+exaustModel_sufix+'.pkl', 'wb') as f:
                pickle.dump(hddm_data_tgt_alldates, f)
      
    
    #
    # do some plotting
    if 1:
        df_combined = hddm_data_tgt_alldates  # your combined trial-level dataframe
        traces = hddm_model_fitted_nogaze_traces  # or use `hddm_model_fitted_traces` if needed
        stats = hddm_model_fitted_nogaze_stats

        # --- STEP 1: Use all posterior samples to compute predicted v for each trial ---
        n_samples = len(traces)
        n_trials = len(df_combined)

        # Allocate space to hold predicted v for each sample x trial
        predicted_v_samples = np.zeros((n_samples, n_trials))

        # Pre-fetch covariate values as numpy arrays
        x1 = df_combined['partner_mean_speed'].values
        x2 = df_combined['partner_speed_std'].values
        x3 = df_combined['self_mean_speed'].values
        x4 = df_combined['self_speed_std'].values
        # x3 = df_combined['failed_pulls_before_reward'].values
        # x3 = df_combined['time_since_last_reward'].values

        for i in range(n_samples):
            intercept = traces['v_Intercept'].values[i]
            coef_x1 = traces['v_partner_mean_speed'].values[i]
            coef_x2 = traces['v_partner_speed_std'].values[i]
            coef_x3 = traces['v_self_mean_speed'].values[i]
            coef_x4 = traces['v_self_speed_std'].values[i]
            # coef_x3 = traces['v_failed_pulls_before_reward'].values[i]
            # coef_x3 = traces['v_time_since_last_reward'].values[i]

            # Apply regression formula for all trials
            predicted_v_samples[i, :] = (
                # intercept + coef_x1 * x1 + coef_x2 * x2 + coef_x3 * x3
                # intercept + coef_x1 * x1 
                # intercept + coef_x1 * x1 + coef_x3 * x3
                intercept + coef_x1 * x1 + coef_x2 * x2 + coef_x3 * x3 + coef_x4 * x4
                
                
            )

        # --- STEP 2: Compute posterior mean (and optionally CI) of predicted v ---
        df_with_v = df_combined.copy()
        df_with_v['predicted_v'] = predicted_v_samples.mean(axis=0)

        # Optional: also store credible intervals (e.g., 95% CI)
        df_with_v['predicted_v_lower'] = np.percentile(predicted_v_samples, 2.5, axis=0)
        df_with_v['predicted_v_upper'] = np.percentile(predicted_v_samples, 97.5, axis=0)        
        
        
        # df_with_v['self_gaze_auc'] = df_with_v['self_gaze_auc'] * df_with_v['rt']
        
        #
        #!!! Add some criteria for plotting
        #
        # remove dodson with gingerNew and Koala
        # ind_bad = (df_with_v['condition']=='MC_withGingerNew') | (df_with_v['condition']=='MC_withKoala')
        # df_with_v = df_with_v[~ind_bad]
        
        
        # Define quantile thresholds
        lower_q = 0.05
        upper_q = 0.95
        # Compute quantile-based limits
        gaze_lower, gaze_upper = df_with_v['self_gaze_auc'].quantile([lower_q, upper_q])
        v_lower, v_upper = df_with_v['predicted_v'].quantile([lower_q, upper_q])
        # Filter out outliers
        ind_good = (
            # (df_with_v['self_gaze_auc'] >= gaze_lower) & (df_with_v['self_gaze_auc'] <= gaze_upper) &
            (df_with_v['predicted_v'] >= v_lower) & (df_with_v['predicted_v'] <= v_upper)
        )
        #
        # df_with_v = df_with_v[ind_good]
        
        fig, r_vals, p_vals = do_hddm_model_fitted_plot(df_with_v)
        
        # savefig
        savefig = 1
        if savefig:
            current_dir = data_saved_folder+'fig_for_basic_neural_analysis_allsessions_basicEvents_PCA_Pullfocused_continuousBhv_partnerDistVaris_with_hddm_model/'+animal_tgt
            add_fig_dir = os.path.join(current_dir,cameraID+'/fig_for_hddm_model_fitted_combinedsession')
            #
            if not os.path.exists(add_fig_dir):
                os.makedirs(add_fig_dir)
            
            fig.savefig(add_fig_dir+'/gaze_accum_vs_driftslopev_'+cond_toplot_type+fixTW_sufix+exaustModel_sufix+'.pdf', 
                        format='pdf', bbox_inches='tight')

            
            
            
        
        # do the plotting separating different conditions (different pairs)
        conds_toplot = np.unique(df_with_v['condition'])
        nconds_toplot = np.shape(conds_toplot)[0]
        
        for icond_toplot in np.arange(0,nconds_toplot,1):
            
            ind_toplot = df_with_v['condition']==conds_toplot[icond_toplot]
            
            fig_i, r_vals_i, p_vals_i = do_hddm_model_fitted_plot(df_with_v[ind_toplot])
            
            ax = fig_i.axes[0]
            ax.set_title(conds_toplot[icond_toplot]+
                        ' Deming Regression: Self-Gaze vs Predicted Drift Rate (v)', fontsize=14)
            
            # savefig
            savefig = 1
            if savefig:
                current_dir = data_saved_folder+'fig_for_basic_neural_analysis_allsessions_basicEvents_PCA_Pullfocused_continuousBhv_partnerDistVaris_with_hddm_model'+savefile_sufix+'/'+animal_tgt
                add_fig_dir = os.path.join(current_dir,cameraID+'/fig_for_hddm_model_fitted_combinedsession')
                #
                if not os.path.exists(add_fig_dir):
                    os.makedirs(add_fig_dir)

                fig_i.savefig(add_fig_dir+'/gaze_accum_vs_driftslopev_'+conds_toplot[icond_toplot]+fixTW_sufix+exaustModel_sufix+'.pdf', 
                            format='pdf', bbox_inches='tight')
        
    

In [ ]:
# run the hddm fitting using the data set that contained average neuron activity (across neurons for each trial)
# the data set is calculated in Misha - basic_neural_analysis_allsessions_basicEvents_PCA_makeBhvNeuronVideos_PullStartToPull_section_continuousBhv

if 0:
    
    # In this data set all the rt is calculated using the new definition, but the pull onset is defined slightly different
    #
    # if use the bhv PCA
    dobhvPCA = 1
    #
    doparnterleverspeed = 0 # this one only works when dobhvPCA is 0, it means use partner to lever speed that any speed
    #
    if not dobhvPCA:
        # if use onset of the first increase after min
        doOnsetAfterMin = 1
        if not doOnsetAfterMin:
            doOnsetAfterMin_suffix = ''
        elif doOnsetAfterMin:
            if not doparnterleverspeed:
                doOnsetAfterMin_suffix = 'PullOnsetAfterMin_'
            elif doparnterleverspeed:
                doOnsetAfterMin_suffix = 'PullOnsetAfterMin_PartnerLeverSpeed_'
    #
    elif dobhvPCA:
        # if use onset of the first increase after min
        doOnsetAfterMin = 1
        doBasedonSpeed = 1
        if not doOnsetAfterMin:
            doOnsetAfterMin_suffix = 'bhvPCA_'
        elif doOnsetAfterMin:
            if not doBasedonSpeed:
                doOnsetAfterMin_suffix = 'bhvPCA_PullOnsetAfterMin_'
            elif doBasedonSpeed:
                doOnsetAfterMin_suffix = 'bhvPCA_PullOnsetAfterMinBasedonSpeed_'

                
    # if use a hmm based method to find the trial start
    doHMMmethod = 0
    if doHMMmethod:
        doOnsetAfterMin_suffix = 'HMMmethods_'
        
        
    # only run the noselfgaze hddm fitting
    doNogazeOnly = 0
    
    # use more variables for v a and z
    doExaustModel = 0
    if doExaustModel:
        exaustModel_sufix = '_exaustModel'
    elif not doExaustModel:
        exaustModel_sufix = ''
        
    # load the dataset first
    current_dir = data_saved_folder+'data_saved_singlecam_wholebody_neural_and_hddm'+savefile_sufix+'/'+cameraID+'/'+animal_tgt
    add_date_dir = os.path.join(current_dir+'/hddm_model_fitted_combinedsession_withNeurons')
    
    if cond_toplot_type == 'allMC':
        if not (doOnsetAfterMin_suffix == 'bhvPCA_PullOnsetAfterMinBasedonSpeed_') |\
               (doOnsetAfterMin_suffix == 'PullOnsetAfterMin_PartnerLeverSpeed_'):
            with open(add_date_dir+'/hddm_datas_flexibleTW_newRTdefinition_'+doOnsetAfterMin_suffix+'withFRs_all_dates_'+
                      animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
                hddm_data_tgt_alldates = pickle.load(f)
        #
        elif (doOnsetAfterMin_suffix == 'bhvPCA_PullOnsetAfterMinBasedonSpeed_') |\
             (doOnsetAfterMin_suffix == 'PullOnsetAfterMin_PartnerLeverSpeed_'):
            with open(add_date_dir+'/hddm_datas_flexibleTW_newRTdefinition_'+'PullOnsetAfterMin_'+'withFRs_all_dates_'+
                      animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
                hddm_data_tgt_alldates = pickle.load(f)
                
    else:
        if not (doOnsetAfterMin_suffix == 'bhvPCA_PullOnsetAfterMinBasedonSpeed_') |\
               (doOnsetAfterMin_suffix == 'PullOnsetAfterMin_PartnerLeverSpeed_'):
            with open(add_date_dir+'/hddm_datas_flexibleTW_newRTdefinition_'+doOnsetAfterMin_suffix+'withFRs_all_dates_'+
                      animal1_fixedorders[0]+animal2_fixedorders[0]+'_'+cond_toplot_type+'.pkl', 'rb') as f:
                hddm_data_tgt_alldates = pickle.load(f)
        #
        elif (doOnsetAfterMin_suffix == 'bhvPCA_PullOnsetAfterMinBasedonSpeed_') |\
             (doOnsetAfterMin_suffix == 'PullOnsetAfterMin_PartnerLeverSpeed_'):
            with open(add_date_dir+'/hddm_datas_flexibleTW_newRTdefinition_'+'PullOnsetAfterMin_'+'withFRs_all_dates_'+
                      animal1_fixedorders[0]+animal2_fixedorders[0]+'_'+cond_toplot_type+'.pkl', 'rb') as f:
                hddm_data_tgt_alldates = pickle.load(f)
        

    # make the hddm_data structure consistent with other code
    ind_goodcond = np.isin(hddm_data_tgt_alldates['condition'],conditions_to_ana)
    hddm_data_tgt_alldates = hddm_data_tgt_alldates[ind_goodcond]

    # mostly just for allSR
    if cond_toplot_type == 'allSR':
        hddm_data_tgt_alldates['failed_pulls_before_reward']= 0

    
    if not dobhvPCA:
        if not doparnterleverspeed:
            rename_dict = {
                'act_animal':'subj_idx',
                'dates': 'date',
                'pull_rt': 'rt',
                'previous_pull_outcome': 'prev_trial_outcome',
                'socialgaze_auc': 'self_gaze_auc',
                'mass_move_speed_mean': 'self_mean_speed',
                'mass_move_speed_std': 'self_speed_std',
                'other_mass_move_speed_mean': 'partner_mean_speed',
                'other_mass_move_speed_std': 'partner_speed_std',
                'num_preceding_failpull': 'failed_pulls_before_reward',
                'time_from_last_reward': 'time_since_last_reward',
                }
        elif doparnterleverspeed:
            rename_dict = {
                'act_animal':'subj_idx',
                'dates': 'date',
                'pull_rt': 'rt',
                'previous_pull_outcome': 'prev_trial_outcome',
                'socialgaze_auc': 'self_gaze_auc',
                'mass_move_speed_mean': 'self_mean_speed',
                'mass_move_speed_std': 'self_speed_std',
                'other_lever_speed_mean': 'partner_mean_speed',
                'other_lever_speed_std': 'partner_speed_std',
                'num_preceding_failpull': 'failed_pulls_before_reward',
                'time_from_last_reward': 'time_since_last_reward',
                }
    elif dobhvPCA:
        rename_dict = {
            'act_animal':'subj_idx',
            'dates': 'date',
            'pull_rt': 'rt',
            'previous_pull_outcome': 'prev_trial_outcome',
            'socialgaze_auc': 'self_gaze_auc',
            'self_PC1_mean': 'self_mean_speed',
            'self_PC1_std': 'self_speed_std',
            'other_PC1_mean': 'partner_mean_speed',
            'other_PC1_std': 'partner_speed_std',
            'num_preceding_failpull': 'failed_pulls_before_reward',
            'time_from_last_reward': 'time_since_last_reward',
            }
        
    # Apply the renaming
    hddm_data_tgt_alldates = hddm_data_tgt_alldates.rename(columns=rename_dict)
    hddm_data_tgt_alldates['response'] = 1

    # 
    # remove nan row first
    hddm_data_tgt_alldates = hddm_data_tgt_alldates.dropna()
    
    #
    # # add some semi-artbitrary cretieras
    # hddm_data_tgt_alldates = hddm_data_tgt_alldates[hddm_data_tgt_alldates['prev_trial_outcome']==1]
    # hddm_data_tgt_alldates = hddm_data_tgt_alldates[hddm_data_tgt_alldates['partner_mean_speed']<=300]
    #
    lower_q = 0.05
    upper_q = 0.95
    rt_lower, rt_upper = hddm_data_tgt_alldates['rt'].quantile([lower_q, upper_q])
    hddm_data_tgt_alldates = hddm_data_tgt_alldates[hddm_data_tgt_alldates['rt']>rt_lower]
    hddm_data_tgt_alldates = hddm_data_tgt_alldates[hddm_data_tgt_alldates['rt']<rt_upper]
    #
    tslr_lower, tslr_upper = hddm_data_tgt_alldates['time_since_last_reward'].quantile([lower_q, upper_q])
    hddm_data_tgt_alldates = hddm_data_tgt_alldates[hddm_data_tgt_alldates['time_since_last_reward']>tslr_lower]
    hddm_data_tgt_alldates = hddm_data_tgt_alldates[hddm_data_tgt_alldates['time_since_last_reward']<tslr_upper]

    # 
    # normalize all the predictors
    from scipy.stats import zscore
    #
    # List of predictor columns to normalize (exclude 'rt' and identifiers)
    predictor_columns = [
        'self_gaze_auc',
        'partner_mean_speed',
        'self_mean_speed',
        'partner_speed_std',
        'self_speed_std',
        'failed_pulls_before_reward',
        'time_since_last_reward'
    ]
    # Rename originals and replace with z-scored values
    for col in predictor_columns:
        try:
            hddm_data_tgt_alldates[f"{col}_origin"] = hddm_data_tgt_alldates[col]
            hddm_data_tgt_alldates[col] = zscore(hddm_data_tgt_alldates[col])  
        except:
            continue
        
    # try to load the analyzed
    try:
        
        # dummpy
        #
        if not doNogazeOnly:
            with open(add_date_dir+'/hddm_model_fitted_traces_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
                hddm_model_fitted_traces = pickle.load(f)
            
            with open(add_date_dir+'/hddm_model_fitted_stats_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
                hddm_model_fitted_stats = pickle.load(f)
            
            with open(add_date_dir+'/hddm_model_fitted_dic_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
                hddm_model_fitted_dic = pickle.load(f)
        #
        with open(add_date_dir+'/hddm_model_fitted_nogaze_traces_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
            hddm_model_fitted_nogaze_traces = pickle.load(f)
        
        with open(add_date_dir+'/hddm_model_fitted_nogaze_stats_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
            hddm_model_fitted_nogaze_stats = pickle.load(f)
        
        with open(add_date_dir+'/hddm_model_fitted_nogaze_dic_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
            hddm_model_fitted_nogaze_dic = pickle.load(f)
            
        print(animal_tgt+' '+cond_toplot_type+' hddm fitting results loaded')
        
    except:
        
        print(animal_tgt+' '+cond_toplot_type+' running hddm fitting')
            
        # run the hddm nodel
        if not doExaustModel:
            # HDDM settings - the basic setting
            # samples=500 # 2000
            # burn=100
            # thin=2
            samples=200 # 2000
            burn=100
            thin=2
            #
            if not doNogazeOnly:
                hddm_model_fitted, hddm_model_fitted_nogaze = run_hddm_modeling(hddm_data_tgt_alldates, animal_tgt,
                                                                            samples, burn, thin, doNogazeOnly)
            elif doNogazeOnly:
                _, hddm_model_fitted_nogaze = run_hddm_modeling(hddm_data_tgt_alldates, animal_tgt,
                                                                            samples, burn, thin, doNogazeOnly)
                
        elif doExaustModel:
            # samples=2000 # 2000
            # burn=1000
            # thin=4
            samples=200 # 2000
            burn=100
            thin=2
            #
            if not doNogazeOnly:
                hddm_model_fitted, hddm_model_fitted_nogaze = run_hddm_modeling_exaustModel(hddm_data_tgt_alldates, animal_tgt, 
                                                                            samples, burn, thin, doNogazeOnly)
            elif doNogazeOnly:
                _, hddm_model_fitted_nogaze = run_hddm_modeling_exaustModel(hddm_data_tgt_alldates, animal_tgt,
                                                                            samples, burn, thin, doNogazeOnly)

        #
        if not doNogazeOnly:
            hddm_model_fitted_traces = hddm_model_fitted.get_traces()
            hddm_model_fitted_stats  = hddm_model_fitted.gen_stats()
            hddm_model_fitted_dic = hddm_model_fitted.dic_info
        #
        hddm_model_fitted_nogaze_traces = hddm_model_fitted_nogaze.get_traces()
        hddm_model_fitted_nogaze_stats  = hddm_model_fitted_nogaze.gen_stats()
        hddm_model_fitted_nogaze_dic = hddm_model_fitted_nogaze.dic_info

        # save data
        if 1:
            current_dir = data_saved_folder+'data_saved_singlecam_wholebody_neural_and_hddm'+savefile_sufix+'/'+cameraID+'/'+animal_tgt
            add_date_dir = os.path.join(current_dir+'/hddm_model_fitted_combinedsession_withNeurons')
            if not os.path.exists(add_date_dir):
                os.makedirs(add_date_dir)
            #
            if not doNogazeOnly:
                with open(add_date_dir+'/hddm_model_fitted_traces_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'wb') as f:
                    pickle.dump(hddm_model_fitted_traces, f)
            
                with open(add_date_dir+'/hddm_model_fitted_stats_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'wb') as f:
                    pickle.dump(hddm_model_fitted_stats, f)
               
                with open(add_date_dir+'/hddm_model_fitted_dic_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'wb') as f:
                    pickle.dump(hddm_model_fitted_dic, f)
            
            #
            with open(add_date_dir+'/hddm_model_fitted_nogaze_traces_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'wb') as f:
                pickle.dump(hddm_model_fitted_nogaze_traces, f)
            
            with open(add_date_dir+'/hddm_model_fitted_nogaze_stats_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'wb') as f:
                pickle.dump(hddm_model_fitted_nogaze_stats, f)  
            
            with open(add_date_dir+'/hddm_model_fitted_nogaze_dic_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'wb') as f:
                pickle.dump(hddm_model_fitted_nogaze_dic, f)
            
          
        #
    # do some plotting
    if 1:
        df_combined = hddm_data_tgt_alldates  # your combined trial-level dataframe
        traces = hddm_model_fitted_nogaze_traces  # or use `hddm_model_fitted_traces` if needed
        stats = hddm_model_fitted_nogaze_stats
        # traces = hddm_model_fitted_traces  # or use `hddm_model_fitted_traces` if needed
        # stats = hddm_model_fitted_stats

        # --- STEP 1: Use all posterior samples to compute predicted v for each trial ---
        n_samples = len(traces)
        n_trials = len(df_combined)

        # Allocate space to hold predicted v for each sample x trial
        predicted_v_samples = np.zeros((n_samples, n_trials))

        # Pre-fetch covariate values as numpy arrays
        x1 = df_combined['partner_mean_speed'].values
        x2 = df_combined['partner_speed_std'].values
        x3 = df_combined['self_mean_speed'].values
        x4 = df_combined['self_speed_std'].values
        # x3 = df_combined['failed_pulls_before_reward'].values
        # x3 = df_combined['time_since_last_reward'].values
        # x7 = df_combined['self_gaze_auc'].values

        for i in range(n_samples):
            intercept = traces['v_Intercept'].values[i]
            coef_x1 = traces['v_partner_mean_speed'].values[i]
            coef_x2 = traces['v_partner_speed_std'].values[i]
            coef_x3 = traces['v_self_mean_speed'].values[i]
            coef_x4 = traces['v_self_speed_std'].values[i]
            # coef_x3 = traces['v_failed_pulls_before_reward'].values[i]
            # coef_x3 = traces['v_time_since_last_reward'].values[i]
            # coef_x7 = traces['v_self_gaze_auc'].values[i]

            # Apply regression formula for all trials
            predicted_v_samples[i, :] = (
                # intercept + coef_x1 * x1 + coef_x2 * x2 + coef_x3 * x3
                # intercept + coef_x1 * x1 
                # intercept + coef_x1 * x1 + coef_x3 * x3
                intercept + coef_x1 * x1 + coef_x2 * x2 + coef_x3 * x3 + coef_x4 * x4
                # intercept + coef_x1 * x1 + coef_x2 * x2 + coef_x3 * x3 + coef_x4 * x4 + coef_x7 * x7
            )

        # --- STEP 2: Compute posterior mean (and optionally CI) of predicted v ---
        df_with_v = df_combined.copy()
        df_with_v['predicted_v'] = predicted_v_samples.mean(axis=0)

        # Optional: also store credible intervals (e.g., 95% CI)
        df_with_v['predicted_v_lower'] = np.percentile(predicted_v_samples, 2.5, axis=0)
        df_with_v['predicted_v_upper'] = np.percentile(predicted_v_samples, 97.5, axis=0)        
        
        
        # df_with_v['self_gaze_auc'] = df_with_v['self_gaze_auc'] * df_with_v['rt']
        
        #
        #!!! Add some criteria for plotting
        #
        # remove dodson with gingerNew and Koala
        # ind_bad = (df_with_v['condition']=='MC_withGingerNew') | (df_with_v['condition']=='MC_withKoala')
        # df_with_v = df_with_v[~ind_bad]
        
        
        # Define quantile thresholds
        lower_q = 0.05
        upper_q = 0.95
        # Compute quantile-based limits
        gaze_lower, gaze_upper = df_with_v['self_gaze_auc'].quantile([lower_q, upper_q])
        v_lower, v_upper = df_with_v['predicted_v'].quantile([lower_q, upper_q])
        # Filter out outliers
        ind_good = (
            # (df_with_v['self_gaze_auc'] >= gaze_lower) & (df_with_v['self_gaze_auc'] <= gaze_upper) &
            (df_with_v['predicted_v'] >= v_lower) & (df_with_v['predicted_v'] <= v_upper)
        )
        #
        # df_with_v = df_with_v[ind_good]
        
        fig, r_vals, p_vals = do_hddm_model_fitted_plot(df_with_v)
        
        # savefig
        savefig = 1
        if savefig:
            current_dir = data_saved_folder+'fig_for_basic_neural_analysis_allsessions_basicEvents_PCA_Pullfocused_continuousBhv_partnerDistVaris_with_hddm_model/'+animal_tgt
            add_fig_dir = os.path.join(current_dir,cameraID+'/fig_for_hddm_model_fitted_combinedsession_withNeurons')
            #
            if not os.path.exists(add_fig_dir):
                os.makedirs(add_fig_dir)
            
            fig.savefig(add_fig_dir+'/gaze_accum_vs_driftslopev_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pdf', 
                        format='pdf', bbox_inches='tight')

            
            
            
        
        # do the plotting separating different conditions (different pairs)
        conds_toplot = np.unique(df_with_v['condition'])
        nconds_toplot = np.shape(conds_toplot)[0]
        
        for icond_toplot in np.arange(0,nconds_toplot,1):
            
            ind_toplot = df_with_v['condition']==conds_toplot[icond_toplot]
            
            fig_i, r_vals_i, p_vals_i = do_hddm_model_fitted_plot(df_with_v[ind_toplot])
            
            ax = fig_i.axes[0]
            ax.set_title(conds_toplot[icond_toplot]+
                        ' Deming Regression: Self-Gaze vs Predicted Drift Rate (v)', fontsize=14)
            
            # savefig
            savefig = 1
            if savefig:
                current_dir = data_saved_folder+'fig_for_basic_neural_analysis_allsessions_basicEvents_PCA_Pullfocused_continuousBhv_partnerDistVaris_with_hddm_model'+savefile_sufix+'/'+animal_tgt
                add_fig_dir = os.path.join(current_dir,cameraID+'/fig_for_hddm_model_fitted_combinedsession_withNeurons')
                #
                if not os.path.exists(add_fig_dir):
                    os.makedirs(add_fig_dir)

                fig_i.savefig(add_fig_dir+'/gaze_accum_vs_driftslopev_'+conds_toplot[icond_toplot]+doOnsetAfterMin_suffix+exaustModel_sufix+'.pdf', 
                            format='pdf', bbox_inches='tight')

            
            
    
    
    

In [ ]:
# run the hddm fitting using the data set that contained average neuron activity (across neurons for each trial)
# the data set is calculated in Misha - basic_neural_analysis_allsessions_basicEvents_PCA_makeBhvNeuronVideos_PullStartToPull_section_continuousBhv

# this one is to fit the ddm in each sesssion separately

if 0:
    
    import seaborn as sns
    from scipy.stats import ttest_1samp

    # In this data set all the rt is calculated using the new definition, but the pull onset is defined slightly different
    #
    # In this data set all the rt is calculated using the new definition, but the pull onset is defined slightly different
    #
    # if use the bhv PCA
    dobhvPCA = 1
    #
    doparnterleverspeed = 0 # this one only works when dobhvPCA is 0, it means use partner to lever speed that any speed
    #
    if not dobhvPCA:
        # if use onset of the first increase after min
        doOnsetAfterMin = 1
        if not doOnsetAfterMin:
            doOnsetAfterMin_suffix = ''
        elif doOnsetAfterMin:
            if not doparnterleverspeed:
                doOnsetAfterMin_suffix = 'PullOnsetAfterMin_'
            elif doparnterleverspeed:
                doOnsetAfterMin_suffix = 'PullOnsetAfterMin_PartnerLeverSpeed_'
    #
    elif dobhvPCA:
        # if use onset of the first increase after min
        doOnsetAfterMin = 1
        doBasedonSpeed = 1
        if not doOnsetAfterMin:
            doOnsetAfterMin_suffix = 'bhvPCA_'
        elif doOnsetAfterMin:
            if not doBasedonSpeed:
                doOnsetAfterMin_suffix = 'bhvPCA_PullOnsetAfterMin_'
            elif doBasedonSpeed:
                doOnsetAfterMin_suffix = 'bhvPCA_PullOnsetAfterMinBasedonSpeed_'
                
                
    # if use a hmm based method to find the trial start
    doHMMmethod = 0
    if doHMMmethod:
        doOnsetAfterMin_suffix = 'HMMmethods_'
        
        
    # only run the noselfgaze hddm fitting
    doNogazeOnly = 0
        
    
    # use more variables for v a and z
    doExaustModel = 0
    if doExaustModel:
        exaustModel_sufix = '_exaustModel'
    elif not doExaustModel:
        exaustModel_sufix = ''
        
    # load the dataset first
    current_dir = data_saved_folder+'data_saved_singlecam_wholebody_neural_and_hddm'+savefile_sufix+'/'+cameraID+'/'+animal_tgt
    add_date_dir = os.path.join(current_dir+'/hddm_model_fitted_combinedsession_withNeurons')
    
    if cond_toplot_type == 'allMC':
        if not (doOnsetAfterMin_suffix == 'bhvPCA_PullOnsetAfterMinBasedonSpeed_') |\
               (doOnsetAfterMin_suffix == 'PullOnsetAfterMin_PartnerLeverSpeed_'):
            with open(add_date_dir+'/hddm_datas_flexibleTW_newRTdefinition_'+doOnsetAfterMin_suffix+'withFRs_all_dates_'+
                      animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
                hddm_data_tgt_alldates = pickle.load(f)
        #
        elif (doOnsetAfterMin_suffix == 'bhvPCA_PullOnsetAfterMinBasedonSpeed_') |\
             (doOnsetAfterMin_suffix == 'PullOnsetAfterMin_PartnerLeverSpeed_'):
            with open(add_date_dir+'/hddm_datas_flexibleTW_newRTdefinition_'+'PullOnsetAfterMin_'+'withFRs_all_dates_'+
                      animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
                hddm_data_tgt_alldates = pickle.load(f)
    #
    else:
        if not (doOnsetAfterMin_suffix == 'bhvPCA_PullOnsetAfterMinBasedonSpeed_') |\
               (doOnsetAfterMin_suffix == 'PullOnsetAfterMin_PartnerLeverSpeed_'):
            with open(add_date_dir+'/hddm_datas_flexibleTW_newRTdefinition_'+doOnsetAfterMin_suffix+'withFRs_all_dates_'+
                      animal1_fixedorders[0]+animal2_fixedorders[0]+'_'+cond_toplot_type+'.pkl', 'rb') as f:
                hddm_data_tgt_alldates = pickle.load(f)
        #
        elif (doOnsetAfterMin_suffix == 'bhvPCA_PullOnsetAfterMinBasedonSpeed_') |\
             (doOnsetAfterMin_suffix == 'PullOnsetAfterMin_PartnerLeverSpeed_'):
            with open(add_date_dir+'/hddm_datas_flexibleTW_newRTdefinition_'+'PullOnsetAfterMin_'+'withFRs_all_dates_'+
                      animal1_fixedorders[0]+animal2_fixedorders[0]+'_'+cond_toplot_type+'.pkl', 'rb') as f:
                hddm_data_tgt_alldates = pickle.load(f)
    

    # make the hddm_data structure consistent with other code
    ind_goodcond = np.isin(hddm_data_tgt_alldates['condition'],conditions_to_ana)
    hddm_data_tgt_alldates = hddm_data_tgt_alldates[ind_goodcond]
    
    if not dobhvPCA:
        if not doparnterleverspeed:
            rename_dict = {
                'act_animal':'subj_idx',
                'dates': 'date',
                'pull_rt': 'rt',
                'previous_pull_outcome': 'prev_trial_outcome',
                'socialgaze_auc': 'self_gaze_auc',
                'mass_move_speed_mean': 'self_mean_speed',
                'mass_move_speed_std': 'self_speed_std',
                'other_mass_move_speed_mean': 'partner_mean_speed',
                'other_mass_move_speed_std': 'partner_speed_std',
                'num_preceding_failpull': 'failed_pulls_before_reward',
                'time_from_last_reward': 'time_since_last_reward',
                }
        elif doparnterleverspeed:
            rename_dict = {
                'act_animal':'subj_idx',
                'dates': 'date',
                'pull_rt': 'rt',
                'previous_pull_outcome': 'prev_trial_outcome',
                'socialgaze_auc': 'self_gaze_auc',
                'mass_move_speed_mean': 'self_mean_speed',
                'mass_move_speed_std': 'self_speed_std',
                'other_lever_speed_mean': 'partner_mean_speed',
                'other_lever_speed_std': 'partner_speed_std',
                'num_preceding_failpull': 'failed_pulls_before_reward',
                'time_from_last_reward': 'time_since_last_reward',
                }
    elif dobhvPCA:
        rename_dict = {
            'act_animal':'subj_idx',
            'dates': 'date',
            'pull_rt': 'rt',
            'previous_pull_outcome': 'prev_trial_outcome',
            'socialgaze_auc': 'self_gaze_auc',
            # 'self_PC1_mean': 'self_mean_speed',
            # 'self_PC1_std': 'self_speed_std',
            'mass_move_speed_mean': 'self_mean_speed',
            'mass_move_speed_std': 'self_speed_std',
            'other_PC1_mean': 'partner_mean_speed',
            'other_PC1_std': 'partner_speed_std',
            'num_preceding_failpull': 'failed_pulls_before_reward',
            'time_from_last_reward': 'time_since_last_reward',
            }
        
    # Apply the renaming
    hddm_data_tgt_alldates = hddm_data_tgt_alldates.rename(columns=rename_dict)
    hddm_data_tgt_alldates['response'] = 1

    # mostly just for allSR
    if cond_toplot_type == 'allSR':
        hddm_data_tgt_alldates['failed_pulls_before_reward']= 0
    
    # 
    # remove nan row first
    hddm_data_tgt_alldates = hddm_data_tgt_alldates.dropna()

    
    # 
    # normalize all the predictors
    from scipy.stats import zscore
    #
    # List of predictor columns to normalize (exclude 'rt' and identifiers)
    predictor_columns = [
        'self_gaze_auc',
        'partner_mean_speed',
        'self_mean_speed',
        'partner_speed_std',
        'self_speed_std',
        # 'failed_pulls_before_reward',
        'time_since_last_reward'
    ]
    # Rename originals and replace with z-scored values
    for col in predictor_columns:
        hddm_data_tgt_alldates[f"{col}_origin"] = hddm_data_tgt_alldates[col]
        hddm_data_tgt_alldates[col] = zscore(hddm_data_tgt_alldates[col])  
      
    #
    # initialize a dataframe for summarizing all sessions results
    summary_list = []
    
    
    # run the fitting for each session
    for session_toana, hddm_data_tgt_idate in hddm_data_tgt_alldates.groupby('date'):
    
        ind_ = np.isin(dates_list,session_toana)
        task_cond = task_conditions[np.where(ind_)[0][0]]
        
        # Inside the loop:
        # 'session_toana' is the date for the current group
        # 'hddm_data_tgt_idate' is a DataFrame containing only the rows for that date
        
        #
        # # add some semi-artbitrary cretieras
        # hddm_data_tgt_alldates = hddm_data_tgt_idate[hddm_data_tgt_alldates['prev_trial_outcome']==1]
        # hddm_data_tgt_alldates = hddm_data_tgt_idate[hddm_data_tgt_alldates['partner_mean_speed']<=300]
        #
        lower_q = 0.05
        upper_q = 0.95
        rt_lower, rt_upper = hddm_data_tgt_idate['rt'].quantile([lower_q, upper_q])
        # hddm_data_tgt_idate = hddm_data_tgt_idate[hddm_data_tgt_idate['rt']>rt_lower]
        # hddm_data_tgt_idate = hddm_data_tgt_idate[hddm_data_tgt_idate['rt']<rt_upper]
        #
        tslr_lower, tslr_upper = hddm_data_tgt_idate['time_since_last_reward'].quantile([lower_q, upper_q])
        # hddm_data_tgt_idate = hddm_data_tgt_idate[hddm_data_tgt_idate['time_since_last_reward']>tslr_lower]
        # hddm_data_tgt_idate = hddm_data_tgt_idate[hddm_data_tgt_idate['time_since_last_reward']<tslr_upper]
    
        # try to load the analyzed
        try:

            # dummpy
            
            current_dir = data_saved_folder+'data_saved_singlecam_wholebody_neural_and_hddm'+savefile_sufix+'/'+cameraID+'/'+animal_tgt
            add_date_dir = os.path.join(current_dir+'/hddm_model_fitted_allsession_withNeurons/'+session_toana)

            #
            if not doNogazeOnly:
                with open(add_date_dir+'/hddm_model_fitted_traces_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
                    hddm_model_fitted_traces = pickle.load(f)

                with open(add_date_dir+'/hddm_model_fitted_stats_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
                    hddm_model_fitted_stats = pickle.load(f)

                with open(add_date_dir+'/hddm_model_fitted_dic_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
                    hddm_model_fitted_dic = pickle.load(f)
            #
            with open(add_date_dir+'/hddm_model_fitted_nogaze_traces_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
                hddm_model_fitted_nogaze_traces = pickle.load(f)

            with open(add_date_dir+'/hddm_model_fitted_nogaze_stats_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
                hddm_model_fitted_nogaze_stats = pickle.load(f)

            with open(add_date_dir+'/hddm_model_fitted_nogaze_dic_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
                hddm_model_fitted_nogaze_dic = pickle.load(f)
                
            

            print(animal_tgt+' '+cond_toplot_type+' hddm fitting results loaded')

        except:

            print(animal_tgt+' '+cond_toplot_type+' running hddm fitting')
            
            try:
                # run the hddm nodel
                if not doExaustModel:
                    # HDDM settings - the basic setting
                    # samples=500 # 2000
                    # burn=100
                    # thin=2
                    samples=200 # 2000
                    burn=100
                    thin=2
                    #
                    if not doNogazeOnly:
                        hddm_model_fitted, hddm_model_fitted_nogaze = run_hddm_modeling(hddm_data_tgt_idate, animal_tgt,
                                                                                    samples, burn, thin, doNogazeOnly)
                    elif doNogazeOnly:
                        _, hddm_model_fitted_nogaze = run_hddm_modeling(hddm_data_tgt_idate, animal_tgt,
                                                                        samples, burn, thin, doNogazeOnly)

                elif doExaustModel:
                    # samples=2000 # 2000
                    # burn=1000
                    # thin=4
                    samples=200 # 2000
                    burn=100
                    thin=2
                    #
                    if not doNogazeOnly:
                        hddm_model_fitted, hddm_model_fitted_nogaze = run_hddm_modeling_exaustModel(hddm_data_tgt_idate, animal_tgt, 
                                                                                    samples, burn, thin, doNogazeOnly)
                    elif doNogazeOnly:
                        _, hddm_model_fitted_nogaze = run_hddm_modeling_exaustModel(hddm_data_tgt_idate, animal_tgt,
                                                                                    samples, burn, thin, doNogazeOnly)

            except:
                continue
                
            #
            if not doNogazeOnly:
                hddm_model_fitted_traces = hddm_model_fitted.get_traces()
                hddm_model_fitted_stats  = hddm_model_fitted.gen_stats()
                hddm_model_fitted_dic = hddm_model_fitted.dic_info
            #
            hddm_model_fitted_nogaze_traces = hddm_model_fitted_nogaze.get_traces()
            hddm_model_fitted_nogaze_stats  = hddm_model_fitted_nogaze.gen_stats()
            hddm_model_fitted_nogaze_dic = hddm_model_fitted_nogaze.dic_info

            # save data
            if 1:
                current_dir = data_saved_folder+'data_saved_singlecam_wholebody_neural_and_hddm'+savefile_sufix+'/'+cameraID+'/'+animal_tgt
                add_date_dir = os.path.join(current_dir+'/hddm_model_fitted_allsession_withNeurons/'+session_toana)
                if not os.path.exists(add_date_dir):
                    os.makedirs(add_date_dir)
                #
                if not doNogazeOnly:
                    with open(add_date_dir+'/hddm_model_fitted_traces_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'wb') as f:
                        pickle.dump(hddm_model_fitted_traces, f)

                    with open(add_date_dir+'/hddm_model_fitted_stats_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'wb') as f:
                        pickle.dump(hddm_model_fitted_stats, f)

                    with open(add_date_dir+'/hddm_model_fitted_dic_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'wb') as f:
                        pickle.dump(hddm_model_fitted_dic, f)

                #
                with open(add_date_dir+'/hddm_model_fitted_nogaze_traces_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'wb') as f:
                    pickle.dump(hddm_model_fitted_nogaze_traces, f)

                with open(add_date_dir+'/hddm_model_fitted_nogaze_stats_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'wb') as f:
                    pickle.dump(hddm_model_fitted_nogaze_stats, f)  

                with open(add_date_dir+'/hddm_model_fitted_nogaze_dic_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'wb') as f:
                    pickle.dump(hddm_model_fitted_nogaze_dic, f)
            
          
        #
        # reconstruct V
        #
        df_combined = hddm_data_tgt_idate # your combined trial-level dataframe
        traces = hddm_model_fitted_nogaze_traces  # or use `hddm_model_fitted_traces` if needed
        stats = hddm_model_fitted_nogaze_stats
        # traces = hddm_model_fitted_traces  # or use `hddm_model_fitted_traces` if needed
        # stats = hddm_model_fitted_stats

        # --- STEP 1: Use all posterior samples to compute predicted v for each trial ---
        n_samples = len(traces)
        n_trials = len(df_combined)

        # Allocate space to hold predicted v for each sample x trial
        predicted_v_samples = np.zeros((n_samples, n_trials))

        # Pre-fetch covariate values as numpy arrays
        x1 = df_combined['partner_mean_speed'].values
        x2 = df_combined['partner_speed_std'].values
        x3 = df_combined['self_mean_speed'].values
        x4 = df_combined['self_speed_std'].values
        # x3 = df_combined['failed_pulls_before_reward'].values
        # x3 = df_combined['time_since_last_reward'].values
        # x7 = df_combined['self_gaze_auc'].values

        for i in range(n_samples):
            intercept = traces['v_Intercept'].values[i]
            coef_x1 = traces['v_partner_mean_speed'].values[i]
            coef_x2 = traces['v_partner_speed_std'].values[i]
            coef_x3 = traces['v_self_mean_speed'].values[i]
            coef_x4 = traces['v_self_speed_std'].values[i]
            # coef_x3 = traces['v_failed_pulls_before_reward'].values[i]
            # coef_x3 = traces['v_time_since_last_reward'].values[i]
            # coef_x7 = traces['v_self_gaze_auc'].values[i]

            # Apply regression formula for all trials
            predicted_v_samples[i, :] = (
                # intercept + coef_x1 * x1 + coef_x2 * x2 + coef_x3 * x3
                # intercept + coef_x1 * x1 
                # intercept + coef_x1 * x1 + coef_x3 * x3
                intercept + coef_x1 * x1 + coef_x2 * x2 + coef_x3 * x3 + coef_x4 * x4
                # intercept + coef_x1 * x1 + coef_x2 * x2 + coef_x3 * x3 + coef_x4 * x4 + coef_x7 * x7
            )

        # --- STEP 2: Compute posterior mean (and optionally CI) of predicted v ---
        df_with_v = df_combined.copy()
        df_with_v['predicted_v'] = predicted_v_samples.mean(axis=0)

        # Optional: also store credible intervals (e.g., 95% CI)
        df_with_v['predicted_v_lower'] = np.percentile(predicted_v_samples, 2.5, axis=0)
        df_with_v['predicted_v_upper'] = np.percentile(predicted_v_samples, 97.5, axis=0) 
            
        
            
        #
        # save the summaring data across date   
        #
        # --- Process the model data ---
        # --- Task 1: Process the HDDM model coefficients (same as before) ---
        stats['significance'] = (stats['2.5q'] * stats['97.5q'] > 0).astype(int)
        session_summary = stats.reset_index().rename(columns={'index': 'coefficient'})
        session_summary['date'] = session_toana
        session_summary['condition'] = task_cond
        session_summary = session_summary[['date', 'condition', 'coefficient', 'mean', 'std', 'significance']]
        summary_list.append(session_summary)


        # --- Task 2: Calculate and add the v-rt correlation ---
        # Calculate the Pearson correlation coefficient and the p-value.
        corr_coef, p_value = st.pearsonr(df_with_v['predicted_v'], df_with_v['rt'])
        #
        # Create a small DataFrame for the correlation result.
        # We use the 'mean' column for the coefficient and 'significance' for the p-value
        # to keep the data structure consistent.
        corr_summary = pd.DataFrame([{
            'date': session_toana,
            'condition':task_cond,
            'coefficient': 'predited_v_rt_correlation',
            'mean': corr_coef,
            'std': np.nan,  # Standard deviation isn't applicable here
            'significance': (p_value<0.05).astype(int)
        }])
        #
        # Add the correlation result to the list.
        summary_list.append(corr_summary)
        
        # with mean firing rate
        corr_coef, p_value = st.pearsonr(df_with_v['predicted_v'], df_with_v['fr_mean'])
        #
        # Create a small DataFrame for the correlation result.
        # We use the 'mean' column for the coefficient and 'significance' for the p-value
        # to keep the data structure consistent.
        corr_summary = pd.DataFrame([{
            'date': session_toana,
            'condition':task_cond,
            'coefficient': 'predicted_v_fr_mean_correlation',
            'mean': corr_coef,
            'std': np.nan,  # Standard deviation isn't applicable here
            'significance': (p_value<0.05).astype(int)
        }])
        #
        # Add the correlation result to the list.
        summary_list.append(corr_summary)
        
         # with slope firing rate
        corr_coef, p_value = st.pearsonr(df_with_v['predicted_v'], df_with_v['fr_slope'])
        #
        # Create a small DataFrame for the correlation result.
        # We use the 'mean' column for the coefficient and 'significance' for the p-value
        # to keep the data structure consistent.
        corr_summary = pd.DataFrame([{
            'date': session_toana,
            'condition':task_cond,
            'coefficient': 'predicted_v_fr_slope_correlation',
            'mean': corr_coef,
            'std': np.nan,  # Standard deviation isn't applicable here
            'significance': (p_value<0.05).astype(int)
        }])
        #
        # Add the correlation result to the list.
        summary_list.append(corr_summary)
    
    
    
    # Combine the list of DataFrames into the final summary DataFrame.
    hddm_model_fitted_alldates_summary = pd.concat(summary_list, ignore_index=True)
    
    ## 
    # do some plotting
    ##
    plot_data = hddm_model_fitted_alldates_summary.copy()
    
    # ---- NEW: Filter out the coefficients you don't want to see ----
    coeffs_to_exclude = ['z', 'a', 'v_Intercept', 'a_Intercept','a_time_since_last_reward',
                         'predicted_v_fr_mean_correlation','predicted_v_fr_slope_correlation']
    #
    do_v_and_FR = 0
    if do_v_and_FR:
        coeffs_to_exclude = ['t', 'z', 'v_Intercept', 'a_Intercept','a_time_since_last_reward',
                             'v_self_gaze_auc', 'v_partner_mean_speed','v_partner_speed_std',
                             'v_self_mean_speed','v_self_speed_std',]
    #
    plot_data = plot_data[~plot_data['coefficient'].isin(coeffs_to_exclude)]

    plot_data['mean'] = pd.to_numeric(plot_data['mean'], errors='coerce')
    
    # ---- Remove outliers for each coefficient using the IQR method ----
    # For each coefficient, calculate the bounds
    Q1 = plot_data.groupby('coefficient')['mean'].transform(lambda x: x.quantile(0.25))
    Q3 = plot_data.groupby('coefficient')['mean'].transform(lambda x: x.quantile(0.75))
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    #
    # Filter out the outliers
    plot_data_no_outliers = plot_data[(plot_data['mean'] >= lower_bound) & (plot_data['mean'] <= upper_bound)]
    # plot_data_no_outliers = plot_data
    #
    plot_data_no_outliers = plot_data_no_outliers.sort_values(by='coefficient')

    
    
    # --- Plotting Code (using the outlier-free data) ---
    plt.figure(figsize=(14, 8))

    # Create the swarm plot
    # We use 'hue' to color points based on the 'significance' column for that date.
    ax = sns.swarmplot(
        x='coefficient',
        y='mean',
        data=plot_data_no_outliers,
        hue='significance',
        palette={0: 'gray', 1: 'red'},  # 0=Not Significant (gray), 1=Significant (red)
        size=8
    )
    # Hide the legend as the colors are intuitive
    ax.get_legend().remove()


    # --- Add Overall Significance Asterisks ---
    unique_coeffs = plot_data_no_outliers['coefficient'].unique()
    for i, coef in enumerate(unique_coeffs):
        
        coef_data_all_df = plot_data_no_outliers[plot_data_no_outliers['coefficient'] == coef]
        
        if 0: # this part is using the lmm
            # Ensure there's enough data to run the model
            if len(coef_data_all_df) > 1:
                p_value = 1.0 # Default to non-significant

                # Use a try/except block to handle cases where the LMM might not converge
                try:
                    # --- START: LMM Replacement ---
                    # Define the LMM: model the 'mean' with an intercept, grouped by 'condition'
                    model = smf.mixedlm("mean ~ 1", coef_data_all_df, groups=coef_data_all_df['condition'])

                    # Fit the model (disp=False prevents printing convergence messages in the loop)
                    result = model.fit(disp=False)

                    # Get the p-value for the intercept (our test against 0)
                    p_value = result.pvalues['Intercept']
                    print(p_value)
                    
                    # --- END: LMM Replacement ---

                except Exception as e:
                    # If the model fails, print a warning and keep p_value at 1.0
                    print(f"LMM failed for coefficient '{coef}'. Error: {e}")


                # The rest of your plotting logic remains the same
                if p_value < 0.05:
                    # Use the 'mean' column from the subsetted dataframe for positioning
                    y_pos = coef_data_all_df['mean'].max()
                    offset = coef_data_all_df['mean'].std() * 0.1
                    plt.text(i, y_pos + offset, '*', ha='center', va='bottom', fontsize=24, color='black')

        
        if 1: # this part is using the simple ttest/wilcoxon
            coef_data = plot_data_no_outliers[plot_data_no_outliers['coefficient'] == coef]['mean']

            if len(coef_data) > 1:
                # t_stat, p_value = ttest_1samp(coef_data, 0, nan_policy='omit')
                w_stat, p_value = st.wilcoxon(coef_data, nan_policy='omit')
                if p_value < 0.05:
                    y_pos = coef_data.max()
                    offset = plot_data_no_outliers['mean'].std() * 0.1
                    plt.text(i, y_pos + offset, '*', ha='center', va='bottom', fontsize=24, color='black')

    plt.axhline(y=0, linestyle='--', color='black', linewidth=1.5)            
    
    # --- Finalize and Save Plot ---
    plt.title('Model Coefficients Across All Dates', fontsize=18)
    plt.ylabel('Coefficient Value (Mean)', fontsize=16)
    plt.xlabel('Model Coefficient', fontsize=16)
    plt.xticks(rotation=45, ha='right', fontsize = 16)
    plt.tight_layout()

     # savefig
    savefig = 1
    if savefig:
        current_dir = data_saved_folder+'fig_for_basic_neural_analysis_allsessions_basicEvents_PCA_Pullfocused_continuousBhv_partnerDistVaris_with_hddm_model'+savefile_sufix+'/'+animal_tgt
        add_fig_dir = os.path.join(current_dir,cameraID+'/fig_for_hddm_model_fitted_allsession_withNeurons')
        #
        if not os.path.exists(add_fig_dir):
            os.makedirs(add_fig_dir)
        if not do_v_and_FR:
            plt.savefig(add_fig_dir+'/predicted_v_and_other_bhv_variables_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pdf', 
                        format='pdf', bbox_inches='tight')
        elif do_v_and_FR:
            plt.savefig(add_fig_dir+'/predicted_v_and_FR_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pdf', 
                        format='pdf', bbox_inches='tight')


In [ ]:
# doOnsetAfterMin_suffix

In [ ]:
#hddm_data_tgt_idate

In [ ]:
# run the hddm fitting using the data set that contained average neuron activity (across neurons for each trial)
# the data set is calculated in Misha - basic_neural_analysis_allsessions_basicEvents_PCA_makeBhvNeuronVideos_PullStartToPull_section_continuousBhv

# this one is to fit the ddm in each sesssion separately
# this one use gaze period to filter the partner's variables

from tqdm import tqdm # A helpful library for progress bars


if 1:
    
    import seaborn as sns
    from scipy.stats import ttest_1samp

    # In this data set all the rt is calculated using the new definition, but the pull onset is defined slightly different
    #
    # if use the bhv PCA
    dobhvPCA = 1
    #
    doparnterleverspeed = 0 # this one only works when dobhvPCA is 0, it means use partner to lever speed that any speed
    #
    if not dobhvPCA:
        # if use onset of the first increase after min
        doOnsetAfterMin = 1
        if not doOnsetAfterMin:
            doOnsetAfterMin_suffix = ''
        elif doOnsetAfterMin:
            if not doparnterleverspeed:
                doOnsetAfterMin_suffix = 'PullOnsetAfterMin_'
            elif doparnterleverspeed:
                doOnsetAfterMin_suffix = 'PullOnsetAfterMin_PartnerLeverSpeed_'
    #
    elif dobhvPCA:
        # if use onset of the first increase after min
        doOnsetAfterMin = 1
        doBasedonSpeed = 1
        if not doOnsetAfterMin:
            doOnsetAfterMin_suffix = 'bhvPCA_'
        elif doOnsetAfterMin:
            if not doBasedonSpeed:
                doOnsetAfterMin_suffix = 'bhvPCA_PullOnsetAfterMin_'
            elif doBasedonSpeed:
                doOnsetAfterMin_suffix = 'bhvPCA_PullOnsetAfterMinBasedonSpeed_'
                

    # if use a hmm based method to find the trial start
    doHMMmethod = 0
    if doHMMmethod:
        doOnsetAfterMin_suffix = 'HMMmethods_'
        
    exaustModel_sufix = ''
        
    # load the dataset first
    current_dir = data_saved_folder+'data_saved_singlecam_wholebody_neural_and_hddm'+savefile_sufix+'/'+cameraID+'/'+animal_tgt
    add_date_dir = os.path.join(current_dir+'/hddm_model_fitted_combinedsession_withNeurons')
    
    if cond_toplot_type == 'allMC':
        if not (doOnsetAfterMin_suffix == 'bhvPCA_PullOnsetAfterMinBasedonSpeed_') |\
               (doOnsetAfterMin_suffix == 'PullOnsetAfterMin_PartnerLeverSpeed_'):
            with open(add_date_dir+'/hddm_datas_flexibleTW_newRTdefinition_'+doOnsetAfterMin_suffix+'withFRs_all_dates_'+
                      animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
                hddm_data_tgt_alldates = pickle.load(f)
        #
        elif (doOnsetAfterMin_suffix == 'bhvPCA_PullOnsetAfterMinBasedonSpeed_') |\
             (doOnsetAfterMin_suffix == 'PullOnsetAfterMin_PartnerLeverSpeed_'):
            with open(add_date_dir+'/hddm_datas_flexibleTW_newRTdefinition_'+'PullOnsetAfterMin_'+'withFRs_all_dates_'+
                      animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
                hddm_data_tgt_alldates = pickle.load(f)
    #
    else:
        if not (doOnsetAfterMin_suffix == 'bhvPCA_PullOnsetAfterMinBasedonSpeed_') |\
               (doOnsetAfterMin_suffix == 'PullOnsetAfterMin_PartnerLeverSpeed_'):
            with open(add_date_dir+'/hddm_datas_flexibleTW_newRTdefinition_'+doOnsetAfterMin_suffix+'withFRs_all_dates_'+
                      animal1_fixedorders[0]+animal2_fixedorders[0]+'_'+cond_toplot_type+'.pkl', 'rb') as f:
                hddm_data_tgt_alldates = pickle.load(f)
        #
        elif (doOnsetAfterMin_suffix == 'bhvPCA_PullOnsetAfterMinBasedonSpeed_') |\
             (doOnsetAfterMin_suffix == 'PullOnsetAfterMin_PartnerLeverSpeed_'):
            with open(add_date_dir+'/hddm_datas_flexibleTW_newRTdefinition_'+'PullOnsetAfterMin_'+'withFRs_all_dates_'+
                      animal1_fixedorders[0]+animal2_fixedorders[0]+'_'+cond_toplot_type+'.pkl', 'rb') as f:
                hddm_data_tgt_alldates = pickle.load(f)
        
            

    # make the hddm_data structure consistent with other code
    ind_goodcond = np.isin(hddm_data_tgt_alldates['condition'],conditions_to_ana)
    hddm_data_tgt_alldates = hddm_data_tgt_alldates[ind_goodcond]

    if not dobhvPCA:
        if not doparnterleverspeed:
            rename_dict = {
                'act_animal':'subj_idx',
                'dates': 'date',
                'pull_rt': 'rt',
                'previous_pull_outcome': 'prev_trial_outcome',
                'socialgaze_auc': 'self_gaze_auc',
                'mass_move_speed_mean': 'self_mean_speed',
                'mass_move_speed_std': 'self_speed_std',
                'other_mass_move_speed_mean': 'partner_mean_speed',
                'other_mass_move_speed_std': 'partner_speed_std',
                'num_preceding_failpull': 'failed_pulls_before_reward',
                'time_from_last_reward': 'time_since_last_reward',
                }
        elif doparnterleverspeed:
            rename_dict = {
                'act_animal':'subj_idx',
                'dates': 'date',
                'pull_rt': 'rt',
                'previous_pull_outcome': 'prev_trial_outcome',
                'socialgaze_auc': 'self_gaze_auc',
                'mass_move_speed_mean': 'self_mean_speed',
                'mass_move_speed_std': 'self_speed_std',
                'other_lever_speed_mean': 'partner_mean_speed',
                'other_lever_speed_std': 'partner_speed_std',
                'num_preceding_failpull': 'failed_pulls_before_reward',
                'time_from_last_reward': 'time_since_last_reward',
                }
    elif dobhvPCA:
        rename_dict = {
            'act_animal':'subj_idx',
            'dates': 'date',
            'pull_rt': 'rt',
            'previous_pull_outcome': 'prev_trial_outcome',
            'socialgaze_auc': 'self_gaze_auc',
            # 'self_PC1_mean': 'self_mean_speed',
            # 'self_PC1_std': 'self_speed_std',
            'mass_move_speed_mean': 'self_mean_speed',
            'mass_move_speed_std': 'self_speed_std',
            'other_PC1_mean': 'partner_mean_speed',
            'other_PC1_std': 'partner_speed_std',
            'num_preceding_failpull': 'failed_pulls_before_reward',
            'time_from_last_reward': 'time_since_last_reward',
            }
        
    # Apply the renaming
    hddm_data_tgt_alldates = hddm_data_tgt_alldates.rename(columns=rename_dict)
    hddm_data_tgt_alldates['response'] = 1

    
    #
    # calculated the new partner_mean_speed and partner_speed_std with self_gaze filter
    #
    hddm_data_tgt_alldates['partner_speed_std_nofilter'] = hddm_data_tgt_alldates['partner_speed_std']
    hddm_data_tgt_alldates['partner_mean_speed_nofilter'] = hddm_data_tgt_alldates['partner_mean_speed']
    
    #
    ntrials = np.shape(hddm_data_tgt_alldates)[0]
    for itrial in np.arange(0,ntrials,1):
        gaze_prob_itrial = hddm_data_tgt_alldates['socialgaze_prob'].iloc[itrial]
        #
        if not dobhvPCA:
            if not doparnterleverspeed:
                partner_speed_itrial = hddm_data_tgt_alldates['other_mass_move_speed'].iloc[itrial]
            elif doparnterleverspeed:
                partner_speed_itrial = hddm_data_tgt_alldates['other_lever_speed'].iloc[itrial]
        elif dobhvPCA:
            partner_speed_itrial = hddm_data_tgt_alldates['other_PC1'].iloc[itrial]
        #
        # the shape of 'other_lever_speed' and other variables might be different
        partner_speed_shape = np.shape(partner_speed_itrial)[0]
        gaze_prob_shape = np.shape(gaze_prob_itrial)[0]
        if partner_speed_shape > gaze_prob_shape:
            partner_speed_itrial = partner_speed_itrial[np.arange(0,gaze_prob_shape,1)]
        elif partner_speed_shape < gaze_prob_shape:
            gaze_prob_itrial = gaze_prob_itrial[np.arange(0,partner_speed_shape,1)]

        # smooth the gaze duration a bit more
        gaze_prob_itrial = gaussian_filter1d(gaze_prob_itrial, sigma=8) # the final sigma would be sqrt(4*4+sigma*sigma)

        partner_speed_filtered = partner_speed_itrial[gaze_prob_itrial>0]

        partner_mean_speed_filtered = np.nanmean(partner_speed_filtered)
        partner_std_speed_filtered = np.nanstd(partner_speed_filtered)
        
        hddm_data_tgt_alldates['partner_mean_speed'].iloc[itrial] = partner_mean_speed_filtered
        hddm_data_tgt_alldates['partner_speed_std'].iloc[itrial] = partner_std_speed_filtered
    
    # mostly just for allSR
    if cond_toplot_type == 'allSR':
        hddm_data_tgt_alldates['failed_pulls_before_reward']= 0
    
    # 
    # remove nan row first
    hddm_data_tgt_alldates = hddm_data_tgt_alldates.dropna()
    
    
    # 
    # normalize all the predictors
    from scipy.stats import zscore
    #
    # List of predictor columns to normalize (exclude 'rt' and identifiers)
    predictor_columns = [
        'self_gaze_auc',
        'partner_mean_speed',
        'self_mean_speed',
        'partner_speed_std',
        'self_speed_std',
        # 'failed_pulls_before_reward',
        'time_since_last_reward'
    ]
    # Rename originals and replace with z-scored values
    for col in predictor_columns:
        hddm_data_tgt_alldates[f"{col}_origin"] = hddm_data_tgt_alldates[col]
        hddm_data_tgt_alldates[col] = zscore(hddm_data_tgt_alldates[col])  
        # hddm_data_tgt_alldates[col] = hddm_data_tgt_alldates[col]  
      
    #
    # initialize a dataframe for summarizing all sessions results
    summary_list = []
    
    
    # run the fitting for each session
    for session_toana, hddm_data_tgt_idate in hddm_data_tgt_alldates.groupby('date'):
    
        ind_ = np.isin(dates_list,session_toana)
        task_cond = task_conditions[np.where(ind_)[0][0]]
        
        # Inside the loop:
        # 'session_toana' is the date for the current group
        # 'hddm_data_tgt_idate' is a DataFrame containing only the rows for that date
    
        #
        # # add some semi-artbitrary cretieras
        # hddm_data_tgt_alldates = hddm_data_tgt_idate[hddm_data_tgt_alldates['prev_trial_outcome']==1]
        # hddm_data_tgt_alldates = hddm_data_tgt_idate[hddm_data_tgt_alldates['partner_mean_speed']<=300]
        #
        lower_q = 0.05
        upper_q = 0.95
        rt_lower, rt_upper = hddm_data_tgt_idate['rt'].quantile([lower_q, upper_q])
        # hddm_data_tgt_idate = hddm_data_tgt_idate[hddm_data_tgt_idate['rt']>rt_lower]
        # hddm_data_tgt_idate = hddm_data_tgt_idate[hddm_data_tgt_idate['rt']<rt_upper]
        #
        tslr_lower, tslr_upper = hddm_data_tgt_idate['time_since_last_reward'].quantile([lower_q, upper_q])
        # hddm_data_tgt_idate = hddm_data_tgt_idate[hddm_data_tgt_idate['time_since_last_reward']>tslr_lower]
        # hddm_data_tgt_idate = hddm_data_tgt_idate[hddm_data_tgt_idate['time_since_last_reward']<tslr_upper]
        
        
        # try to load the analyzed
        try:

            # dummpy
            
            current_dir = data_saved_folder+'data_saved_singlecam_wholebody_neural_and_hddm'+savefile_sufix+'/'+cameraID+'/'+animal_tgt
            add_date_dir = os.path.join(current_dir+'/hddm_model_fitted_allsession_withNeurons/'+session_toana)

            #
            with open(add_date_dir+'/hddm_model_fitted_nogaze_gazefilter_traces_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
                hddm_model_fitted_nogaze_gazefilter_traces = pickle.load(f)

            with open(add_date_dir+'/hddm_model_fitted_nogaze_gazefilter_stats_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
                hddm_model_fitted_nogaze_gazefilter_stats = pickle.load(f)

            with open(add_date_dir+'/hddm_model_fitted_nogaze_gazefilter_dic_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
                hddm_model_fitted_nogaze_gazefilter_dic = pickle.load(f)
                
            

            print(animal_tgt+' '+cond_toplot_type+' hddm fitting results loaded')

        except:

            print(animal_tgt+' '+cond_toplot_type+' running hddm fitting')
            
            try:

                # run the hddm nodel
                # HDDM settings - the basic setting
                # samples=500 # 2000
                # burn=100
                # thin=2
                samples=200 # 2000
                burn=100
                thin=2
                #
                doNogazeOnly = 1
                #
                if dobhvPCA:
                    _, hddm_model_fitted_nogaze_gazefilter = run_hddm_modeling_bhvPCA_gazefilter(hddm_data_tgt_idate, animal_tgt,
                                                                        samples, burn, thin, doNogazeOnly)
                elif not dobhvPCA:
                    _, hddm_model_fitted_nogaze_gazefilter = run_hddm_modeling_gazefilter(hddm_data_tgt_idate, animal_tgt,
                                                                        samples, burn, thin, doNogazeOnly)

                #
                hddm_model_fitted_nogaze_gazefilter_traces = hddm_model_fitted_nogaze_gazefilter.get_traces()
                hddm_model_fitted_nogaze_gazefilter_stats  = hddm_model_fitted_nogaze_gazefilter.gen_stats()
                hddm_model_fitted_nogaze_gazefilter_dic = hddm_model_fitted_nogaze_gazefilter.dic_info
            
            except:
                continue

                
            # save data
            if 1:
                current_dir = data_saved_folder+'data_saved_singlecam_wholebody_neural_and_hddm'+savefile_sufix+'/'+cameraID+'/'+animal_tgt
                add_date_dir = os.path.join(current_dir+'/hddm_model_fitted_allsession_withNeurons/'+session_toana)
                if not os.path.exists(add_date_dir):
                    os.makedirs(add_date_dir)
                #
                with open(add_date_dir+'/hddm_model_fitted_nogaze_gazefilter_traces_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'wb') as f:
                    pickle.dump(hddm_model_fitted_nogaze_gazefilter_traces, f)

                with open(add_date_dir+'/hddm_model_fitted_nogaze_gazefilter_stats_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'wb') as f:
                    pickle.dump(hddm_model_fitted_nogaze_gazefilter_stats, f)  

                with open(add_date_dir+'/hddm_model_fitted_nogaze_gazefilter_dic_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'wb') as f:
                    pickle.dump(hddm_model_fitted_nogaze_gazefilter_dic, f)
            
          
        #
        # reconstruct V
        #
        df_combined = hddm_data_tgt_idate # your combined trial-level dataframe
        traces = hddm_model_fitted_nogaze_gazefilter_traces  # or use `hddm_model_fitted_traces` if needed
        stats = hddm_model_fitted_nogaze_gazefilter_stats
        dics = hddm_model_fitted_nogaze_gazefilter_dic

        # add a fitting criteria
        if dics['pD'] < 0:
            print('bad fitting for '+session_toana)
            continue
        

        # --- STEP 1: Use all posterior samples to compute predicted v for each trial ---
        n_samples = len(traces)
        n_trials = len(df_combined)

        # Allocate space to hold predicted v for each sample x trial
        predicted_v_samples = np.zeros((n_samples, n_trials))

        # Pre-fetch covariate values as numpy arrays
        x1 = df_combined['partner_mean_speed'].values
        x2 = df_combined['partner_speed_std'].values
        x3 = df_combined['self_mean_speed'].values
        x4 = df_combined['self_speed_std'].values
        # x3 = df_combined['failed_pulls_before_reward'].values
        # x3 = df_combined['time_since_last_reward'].values
        # x7 = df_combined['self_gaze_auc'].values

        for i in range(n_samples):
            intercept = traces['v_Intercept'].values[i]
            coef_x1 = traces['v_partner_mean_speed'].values[i]
            coef_x2 = traces['v_partner_speed_std'].values[i]
            coef_x3 = traces['v_self_mean_speed'].values[i]
            coef_x4 = traces['v_self_speed_std'].values[i]
            # coef_x3 = traces['v_failed_pulls_before_reward'].values[i]
            # coef_x3 = traces['v_time_since_last_reward'].values[i]
            # coef_x7 = traces['v_self_gaze_auc'].values[i]

            # Apply regression formula for all trials
            predicted_v_samples[i, :] = (
                # intercept + coef_x1 * x1 + coef_x2 * x2 + coef_x3 * x3
                # intercept + coef_x1 * x1 
                # intercept + coef_x2 * x2 
                # intercept + coef_x1 * x1 + coef_x2 * x2 + coef_x3 * x3
                # intercept + coef_x1 * x1 + coef_x3 * x3
                intercept + coef_x1 * x1 + coef_x2 * x2 + coef_x3 * x3 + coef_x4 * x4
                # intercept + coef_x1 * x1 + coef_x2 * x2 + coef_x3 * x3 + coef_x4 * x4 + coef_x7 * x7
                # intercept + coef_x2 * x2 + coef_x4 * x4
            )

        # --- STEP 2: Compute posterior mean (and optionally CI) of predicted v ---
        df_with_v = df_combined.copy()
        df_with_v['predicted_v'] = predicted_v_samples.mean(axis=0)

        # Optional: also store credible intervals (e.g., 95% CI)
        df_with_v['predicted_v_lower'] = np.percentile(predicted_v_samples, 2.5, axis=0)
        df_with_v['predicted_v_upper'] = np.percentile(predicted_v_samples, 97.5, axis=0) 
            
            
        # # --- STEP 3: Get Posterior Means for Other Key Parameters ---
        # Get the posterior mean for the other key DDM parameters
        mean_a = stats.loc['a', 'mean']
        mean_t = stats.loc['t', 'mean']
        try:
            mean_z = stats.loc['z', 'mean']
        except KeyError:
            mean_z = 0.5

        print("Simulating RTs for each trial one-by-one...")

        # 1. Create an empty list to store the results
        predicted_rts = []
        predicted_responses = []

        # 2. Loop through each row of your dataframe
        # tqdm provides a nice progress bar for long loops
        for index, trial in tqdm(df_with_v.iterrows(), total=df_with_v.shape[0]):

            # Get the specific predicted drift rate for this single trial
            v_trial = trial['predicted_v']

            # Create the parameter set for just this one trial
            sim_params_trial = {
                'v': v_trial,
                'a': mean_a,
                't': mean_t,
                'z': mean_z
            }

            # Simulate just one RT with this trial's parameters
            sim_data = hddm.generate.gen_rts(size=1, 
                                             method='drift', # Keep 'drift' or try 'cdf'
                                             **sim_params_trial)

            # Append the single result to our lists
            predicted_rts.append(sim_data.rt.iloc[0])
            predicted_responses.append(sim_data.response.iloc[0])

        # 3. Add the completed lists as new columns to your dataframe
        df_with_v['predicted_rt'] = predicted_rts
        df_with_v['predicted_response'] = predicted_responses
            
        #####
        # add a sanity check plot
        #####
        if 1:
            # --- Your existing code to get the data ---
            xxx = df_with_v['rt']
            yyy = df_with_v['predicted_rt']

            # --- Enhanced Plot ---

            # Create the figure and axes
            plt.figure(figsize=(7, 7))

            # Create the scatter plot
            plt.plot(xxx, yyy, '.', alpha=0.5, label='Data')

            # --- Add the identity line (y=x) ---
            # Find the appropriate limits for the line
            lims = [
                np.min([plt.xlim(), plt.ylim()]),  # min of both axes
                np.max([plt.xlim(), plt.ylim()]),  # max of both axes
            ]
            # Plot the line
            plt.plot(lims, lims, 'r--', alpha=0.75, zorder=0, label='Identity Line')
            plt.xlim(lims)
            plt.ylim(lims)

            # --- Add labels and title ---
            plt.xlabel("Observed RT (s)", fontsize=12)
            plt.ylabel("Predicted RT (s)", fontsize=12)
            plt.title("Model Sanity Check: Observed vs. Predicted RT", fontsize=14)
            plt.legend()
            plt.grid(True, linestyle='--', alpha=0.6)

            # --- (Optional) Add correlation coefficient to the plot ---
            r, p = scipy.stats.pearsonr(xxx, yyy)
            # Place text in the top-left corner of the plot
            plt.text(0.05, 0.95, f'$r = {r:.2f}$', 
                     transform=plt.gca().transAxes, 
                     fontsize=12, 
                     verticalalignment='top')


            # Show the plot
            plt.show()
                   
        #
        # save the summaring data across date   
        #
        # --- Process the model data ---
        # --- Task 1: Process the HDDM model coefficients (same as before) ---
        stats['significance'] = (stats['2.5q'] * \
                                 stats['97.5q'] > 0).astype(int)
        session_summary = stats.reset_index().rename(columns={'index': 'coefficient'})
        session_summary['date'] = session_toana
        session_summary['condition'] = task_cond
        session_summary = session_summary[['date', 'condition', 'coefficient', 'mean', 'std', 'significance']]
        summary_list.append(session_summary)


        # --- Task 2: Calculate and add the v-rt correlation ---
        # Calculate the Pearson correlation coefficient and the p-value.
        corr_coef, p_value = st.pearsonr(df_with_v['predicted_v'], df_with_v['rt'])
        #
        # Create a small DataFrame for the correlation result.
        # We use the 'mean' column for the coefficient and 'significance' for the p-value
        # to keep the data structure consistent.
        corr_summary = pd.DataFrame([{
            'date': session_toana,
            'condition':task_cond,
            'coefficient': 'predicted_v_rt_correlation',
            'mean': corr_coef,
            'std': np.nan,  # Standard deviation isn't applicable here
            'significance': (p_value<0.05).astype(int)
        }])
        
        #
        # Add the correlation result to the list.
        summary_list.append(corr_summary)
        
        # with mean firing rate
        corr_coef, p_value = st.pearsonr(df_with_v['predicted_v'], df_with_v['fr_mean'])
        #
        # Create a small DataFrame for the correlation result.
        # We use the 'mean' column for the coefficient and 'significance' for the p-value
        # to keep the data structure consistent.
        corr_summary = pd.DataFrame([{
            'date': session_toana,
            'condition':task_cond,
            'coefficient': 'predicted_v_fr_mean_correlation',
            'mean': corr_coef,
            'std': np.nan,  # Standard deviation isn't applicable here
            'significance': (p_value<0.05).astype(int)
        }])
        #
        # Add the correlation result to the list.
        summary_list.append(corr_summary)
        
         # with slope firing rate
        corr_coef, p_value = st.pearsonr(df_with_v['predicted_v'], df_with_v['fr_slope'])
        #
        # Create a small DataFrame for the correlation result.
        # We use the 'mean' column for the coefficient and 'significance' for the p-value
        # to keep the data structure consistent.
        corr_summary = pd.DataFrame([{
            'date': session_toana,
            'condition':task_cond,
            'coefficient': 'predicted_v_fr_slope_correlation',
            'mean': corr_coef,
            'std': np.nan,  # Standard deviation isn't applicable here
            'significance': (p_value<0.05).astype(int)
        }])
        #
        
        # Add the correlation result to the list.
        summary_list.append(corr_summary)
    
    
    # Combine the list of DataFrames into the final summary DataFrame.
    hddm_model_fitted_alldates_summary = pd.concat(summary_list, ignore_index=True)
    
    ## 
    # do some plotting
    ##
    plot_data = hddm_model_fitted_alldates_summary.copy()
    plot_data.loc[plot_data['condition'] == 'MC_withGingerNew', 'condition'] = 'MC'
    
    # change the GingerNew to Ginger
    
    
    # ---- NEW: Filter out the coefficients you don't want to see ----
    coeffs_to_exclude = ['t', 'z', 'a', 'v_Intercept', 'a_Intercept','a_time_since_last_reward',
                         'predicted_v_fr_mean_correlation','predicted_v_fr_slope_correlation']
    #
    do_v_and_FR = 0
    if do_v_and_FR:
        coeffs_to_exclude = ['t', 'z', 'v_Intercept', 'a_Intercept','a_time_since_last_reward',
                             'v_self_gaze_auc', 'v_partner_mean_speed','v_partner_speed_std',
                             'v_self_mean_speed','v_self_speed_std',]
    #
    plot_data = plot_data[~plot_data['coefficient'].isin(coeffs_to_exclude)]

    plot_data['mean'] = pd.to_numeric(plot_data['mean'], errors='coerce')
    
    # ---- Remove outliers for each coefficient using the IQR method ----
    # For each coefficient, calculate the bounds
    Q1 = plot_data.groupby('coefficient')['mean'].transform(lambda x: x.quantile(0.05))
    Q3 = plot_data.groupby('coefficient')['mean'].transform(lambda x: x.quantile(0.95))
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    #
    # Filter out the outliers
    plot_data_no_outliers = plot_data[(plot_data['mean'] >= lower_bound) & (plot_data['mean'] <= upper_bound)]
    # plot_data_no_outliers = plot_data
    # plot_data_no_outliers = plot_data_no_outliers.sort_values(by='coefficient')

    # # remove some unwanted condition
    # ind_dontwant = (plot_data_no_outliers['condition']=='MC_withGingerNew')# |\
    #          (plot_data_no_outliers['condition']=='MC_withKoala')|\
    #         (plot_data_no_outliers['condition']=='MC_withVermelho')
    # ind_want = plot_data_no_outliers['condition']=='MC'
    # plot_data_no_outliers = plot_data_no_outliers[~ind_dontwant]
    # # plot_data_no_outliers = plot_data_no_outliers[ind_want]
    
    
    
    # --- Plotting Code (using the outlier-free data) ---
    plt.figure(figsize=(14, 8))

    # Create the swarm plot
    # We use 'hue' to color points based on the 'significance' column for that date.
    if 0:
        ax = sns.swarmplot(
            x='coefficient',
            y='mean',
            data=plot_data_no_outliers,
            hue='significance',
            palette={0: 'gray', 1: 'red'},  # 0=Not Significant (gray), 1=Significant (red)
            size=8
        )
    else:
        ax = sns.violinplot(
            x='coefficient',
            y='mean',
            data=plot_data_no_outliers,
            bw_adjust=1.5
        )
    # Hide the legend as the colors are intuitive
    # ax.get_legend().remove()


    # --- Add Overall Significance Asterisks ---
    unique_coeffs = plot_data_no_outliers['coefficient'].unique()
    for i, coef in enumerate(unique_coeffs):
        
        coef_data_all_df = plot_data_no_outliers[plot_data_no_outliers['coefficient'] == coef]
        
        # coef_data_all_df = coef_data_all_df.dropna()
        
        if 0: # this part is using the lmm
            # Ensure there's enough data to run the model
            if len(coef_data_all_df) > 1:
                p_value = 1.0 # Default to non-significant

                # print(coef_data_all_df.groupby('condition').size())

                # Use a try/except block to handle cases where the LMM might not converge
                try:
                    # --- START: LMM Replacement ---
                    # Define the LMM: model the 'mean' with an intercept, grouped by 'condition'
                    model = smf.mixedlm("mean ~ 1", coef_data_all_df, groups=coef_data_all_df['condition'])

                    # Fit the model (disp=False prevents printing convergence messages in the loop)
                    result = model.fit(disp=False)

                    # Get the p-value for the intercept (our test against 0)
                    p_value = result.pvalues['Intercept']
                    intercept_value = result.params['Intercept']

                    print(coef+f" Intercept: {intercept_value}")
                    print(coef+f" P-value: {p_value}")
                    
                    # --- END: LMM Replacement ---

                except Exception as e:
                    # If the model fails, print a warning and keep p_value at 1.0
                    print(f"LMM failed for coefficient '{coef}'. Error: {e}")


                # The rest of your plotting logic remains the same
                if p_value < 0.05:
                    # Use the 'mean' column from the subsetted dataframe for positioning
                    y_pos = coef_data_all_df['mean'].max()
                    offset = coef_data_all_df['mean'].std() * 0.1
                    plt.text(i, y_pos + offset, '*', ha='center', va='bottom', fontsize=24, color='black')

        
        if 1: # this part is using the simple ttest/wilcoxon
            coef_data = plot_data_no_outliers[plot_data_no_outliers['coefficient'] == coef]['mean']

            if len(coef_data) > 1:
                t_stat, p_value = ttest_1samp(coef_data, 0, nan_policy='omit')
                # w_stat, p_value = st.wilcoxon(coef_data, nan_policy='omit')
                print(p_value)
                if p_value < 0.05:
                    y_pos = coef_data.max()
                    offset = plot_data_no_outliers['mean'].std() * 0.1
                    plt.text(i, y_pos + offset, '*', ha='center', va='bottom', fontsize=24, color='black')

    plt.axhline(y=0, linestyle='--', color='black', linewidth=1.5)            
    
    # --- Finalize and Save Plot ---
    plt.title('Model Coefficients Across All Dates', fontsize=18)
    plt.ylabel('Coefficient Value (Mean)', fontsize=16)
    plt.xlabel('Model Coefficient', fontsize=16)
    plt.xticks(rotation=45, ha='right', fontsize = 16)
    plt.tight_layout()

    # savefig
    savefig = 0
    if savefig:
        current_dir = data_saved_folder+'fig_for_basic_neural_analysis_allsessions_basicEvents_PCA_Pullfocused_continuousBhv_partnerDistVaris_with_hddm_model'+savefile_sufix+'/'+animal_tgt
        add_fig_dir = os.path.join(current_dir,cameraID+'/fig_for_hddm_model_fitted_allsession_withNeurons')
        #
        if not os.path.exists(add_fig_dir):
            os.makedirs(add_fig_dir)
        if not do_v_and_FR:
            plt.savefig(add_fig_dir+'/predicted_v_and_other_bhv_variables_nogaze_gazefilter_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pdf', 
                        format='pdf', bbox_inches='tight')
        elif do_v_and_FR:
            plt.savefig(add_fig_dir+'/predicted_v_and_FR_nogaze_gazefilter_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pdf', 
                        format='pdf', bbox_inches='tight')


In [ ]:
# to answer reviewer1's question about the effect of partner ID on the drift diffusion process. 

# it's useful, turn off for now

if 1:
    
    from scipy.stats import wilcoxon

    coef_type = 'v_partner_speed_std'
    # plot_data_tgt = plot_data.copy()
    plot_data_tgt = plot_data_no_outliers.copy()
    
    plot_data_tgt = plot_data_tgt[plot_data_tgt['coefficient']==coef_type]

    print(np.shape(plot_data_tgt))
    
    # Partner identity = condition (keep as-is)
    plot_data_tgt['partner'] = plot_data_tgt['condition']

    # Order partners by mean effect
    partner_order = (
        plot_data_tgt.groupby('partner')['mean']
                     .mean()
                     .sort_values()
                     .index
                     .tolist()
    )
    plot_data_tgt['partner'] = pd.Categorical(
        plot_data_tgt['partner'],
        categories=partner_order,
        ordered=True
    )

    # Partner-level mean across sessions
    partner_mean = (
        plot_data_tgt.groupby('partner', observed=True)['mean']
                     .mean()
                     .reset_index()
    )

    # y positions
    y_pos = {p: i for i, p in enumerate(partner_order)}

    # -----------------
    # Wilcoxon test per partner vs 0 (session-level)
    # -----------------
    partner_stats = {}
    for partner in partner_order:
        vals = plot_data_tgt.loc[plot_data_tgt['partner'] == partner, 'mean'].values.astype(float)

        # default
        partner_stats[partner] = {'p': np.nan, 'star': ''}

        # validity checks
        if len(vals) < 2:
            continue
        if np.allclose(vals, vals[0]):  # all identical -> Wilcoxon undefined / uninformative
            continue
        if np.allclose(vals, 0):
            continue

        try:
            _, p = wilcoxon(vals, alternative='two-sided')
        except Exception:
            continue

        if p < 0.001:
            star = '***'
        elif p < 0.01:
            star = '**'
        elif p < 0.05:
            star = '*'
        else:
            star = 'ns'

        partner_stats[partner] = {'p': p, 'star': star}

    # -----------------
    # Plot (box + session points + partner mean + stars)
    # -----------------
    fig, ax = plt.subplots(figsize=(8.5, 0.6 * max(4, len(partner_order))))

    # box plot per partner (horizontal)
    sns.boxplot(
        data=plot_data_tgt,
        x='mean',
        y='partner',
        # order=partner_order,
        orient='h',
        showcaps=True,
        showfliers=False,
        width=0.5,
        boxprops=dict(facecolor='lightgray', edgecolor='black'),
        medianprops=dict(color='black'),
        whiskerprops=dict(color='black'),
        ax=ax
    )

    # individual session points
    sns.stripplot(
        data=plot_data_tgt,
        x='mean',
        y='partner',
        # order=partner_order,
        orient='h',
        jitter=0.18,
        size=6,
        alpha=0.75,
        color='black',
        ax=ax
    )

    # reference line at 0
    ax.axvline(0, color='black', linestyle='--', linewidth=1)

    # y labels with session counts
    labels = []
    for p in partner_order:
        n = int((plot_data_tgt['partner'] == p).sum())
        labels.append(f"{p} (n={n})")
    ax.set_yticklabels(labels)

    # add stars (place them just to the right of the max value)
    xmax = np.nanmax(plot_data_tgt['mean'].values.astype(float))
    xmin = np.nanmin(plot_data_tgt['mean'].values.astype(float))
    xrange = xmax - xmin if np.isfinite(xmax - xmin) and (xmax - xmin) > 0 else 1.0
    x_star = xmax + 0.06 * xrange

    for p in partner_order:
        star = partner_stats[p]['star']
        if star == '':
            continue
        ax.text(x_star, y_pos[p], star, va='center', ha='left', fontsize=14, fontweight='bold')

    # expand x-limits so stars are visible
    ax.set_xlim(xmin - 0.05 * xrange, x_star + 0.05 * xrange)

    ax.set_xlabel(f"{coef_type} (session mean)")
    ax.set_ylabel("Partner condition")
    ax.set_title("Partner-specific social-evidence effects on drift rate (per-partner Wilcoxon vs 0)")

    plt.tight_layout()
    
    # savefig
    savefig = 0
    if savefig:
        current_dir = data_saved_folder+'fig_for_basic_neural_analysis_allsessions_basicEvents_PCA_Pullfocused_continuousBhv_partnerDistVaris_with_hddm_model'+savefile_sufix+'/'+animal_tgt
        add_fig_dir = os.path.join(current_dir,cameraID+'/fig_for_hddm_model_fitted_allsession_withNeurons')
        #
        fig.savefig(add_fig_dir+'/beta4_separate_partnerID_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pdf', 
                    format='pdf', bbox_inches='tight')
        
        
        
        
    ### LMM 
    
    # Normalize condition strings for matching
    cond_lower = plot_data_tgt["condition"].str.lower()

    if animal_tgt == 'kanga':
        # -------------------------
        # 1) Add partner-level annotations: sex + familiarity (manual mapping)
        # -------------------------
        plot_data_tgt["partner_sex"] = np.nan
        plot_data_tgt.loc[cond_lower.isin(["mc_withdodson", "mc", "mc_withvermelho"]), "partner_sex"] = "male"
        plot_data_tgt.loc[cond_lower.isin(["mc_withkoala", "mc_withginger"]), "partner_sex"] = "female"
        plot_data_tgt["partner_sex"] = plot_data_tgt["partner_sex"].astype("category")

        plot_data_tgt["partner_familiar"] = "unfamiliar"
        plot_data_tgt.loc[cond_lower.isin(["mc", "mc_withginger"]), "partner_familiar"] = "familiar"
        plot_data_tgt["partner_familiar"] = plot_data_tgt["partner_familiar"].astype("category")
    
    elif animal_tgt == 'dodson':
        # -------------------------
        # 1) Add partner-level annotations: sex + familiarity (manual mapping)
        # -------------------------
        plot_data_tgt["partner_sex"] = np.nan
        plot_data_tgt.loc[cond_lower.isin(["mc_withdodson", "mc_withvermelho"]), "partner_sex"] = "male"
        plot_data_tgt.loc[cond_lower.isin(["mc_withkoala", "mc_withkanga","mc"]), "partner_sex"] = "female"
        plot_data_tgt["partner_sex"] = plot_data_tgt["partner_sex"].astype("category")

        plot_data_tgt["partner_familiar"] = "unfamiliar"
        plot_data_tgt.loc[cond_lower.isin(["mc"]), "partner_familiar"] = "familiar"
        plot_data_tgt["partner_familiar"] = plot_data_tgt["partner_familiar"].astype("category")

    # Partner ID (random effect)
    plot_data_tgt["partner_id"] = plot_data_tgt["condition"].astype("category")

    # Optional: date random intercept via variance components (recommended if multiple sessions/dates)
    plot_data_tgt["date"] = plot_data_tgt["date"].astype("category")

    # -------------------------
    # 2) Set reference levels so intercept is meaningful
    #    (Intercept = mean for male + familiar partners)
    # -------------------------
    # plot_data_tgt["partner_sex"] = plot_data_tgt["partner_sex"].cat.reorder_categories(
    #     ["male", "female"], ordered=True
    #  )
    plot_data_tgt["partner_familiar"] = plot_data_tgt["partner_familiar"].cat.reorder_categories(
        ["familiar", "unfamiliar"], ordered=True
    )

    # Quick sanity check
    print(plot_data_tgt[["condition", "partner_sex", "partner_familiar"]].drop_duplicates())

    # -------------------------
    # 3) LMM #1: baseline overall intercept controlling for partner_id
    #    Tests: is mean != 0 after accounting for partner identity?
    # -------------------------
    m1 = smf.mixedlm(
        "mean ~ 1",
        data=plot_data_tgt,
        groups=plot_data_tgt["partner_id"],
    )
    r1 = m1.fit(disp=False)

    print("\n=== LMM #1: mean ~ 1 + (1|partner_id) ===")
    print(r1.summary())
    
    # Fixed-effect intercept test vs 0 is in the summary (z, p-value).
    # You can also pull it directly:
    print("\nFixed intercept estimate (overall mean):", float(r1.params["Intercept"]))
    print("Intercept p-value:", float(r1.pvalues["Intercept"]))

    var_partner = float(r1.cov_re.iloc[0, 0])
    var_resid = float(r1.scale)
    icc = var_partner / (var_partner + var_resid) if (var_partner + var_resid) > 0 else np.nan
    print("partner random-intercept variance:", var_partner)
    print("residual variance:", var_resid)
    print("ICC:", icc)

    # -------------------------
    # 4) LMM #2: intercept + sex + familiarity, controlling partner_id (+ optional date VC)
    #    Tests: intercept (baseline male+familiar), sex effect, familiarity effect,
    #           while still accounting for partner_id random intercept.
    # -------------------------
    m2 = smf.mixedlm(
        "mean ~ C(partner_sex) + C(partner_familiar)",
        data=plot_data_tgt,
        groups=plot_data_tgt["partner_id"],
        vc_formula={"date": "0 + C(date)"}  # comment this line out if you don't want date random intercept
    )
    r2 = m2.fit(disp=False)

    print("\n=== LMM #2: mean ~ sex + familiarity + (1|partner_id) + (1|date) ===")
    print(r2.summary())

    # -------------------------
    # 5) Extract key numbers (optional)
    # -------------------------
    print("\n--- LMM #2 key estimates ---")
    print("Intercept (male+familiar baseline):", float(r2.params["Intercept"]))
    print("Intercept p-value:", float(r2.pvalues["Intercept"]))

    if "C(partner_sex)[T.female]" in r2.params:
        print("Sex effect (female vs male):", float(r2.params["C(partner_sex)[T.female]"]),
              "p=", float(r2.pvalues["C(partner_sex)[T.female]"]))
    if "C(partner_familiar)[T.unfamiliar]" in r2.params:
        print("Familiarity effect (unfamiliar vs familiar):", float(r2.params["C(partner_familiar)[T.unfamiliar]"]),
              "p=", float(r2.pvalues["C(partner_familiar)[T.unfamiliar]"]))

    # Partner variance / ICC (optional)
    # partner random-intercept variance (may be absent / collapsed)
    if hasattr(r2, "cov_re") and (r2.cov_re is not None) and (r2.cov_re.shape[0] > 0):
        var_partner = float(r2.cov_re.iloc[0, 0])
    else:
        var_partner = 0.0  # collapsed / not estimable
    var_resid = float(r2.scale)
    icc = var_partner / (var_partner + var_resid) if (var_partner + var_resid) > 0 else np.nan
    print("partner random-intercept variance:", var_partner)
    print("residual variance:", var_resid)
    print("ICC:", icc)

In [ ]:
hddm_data_tgt_alldates.keys()

In [ ]:
# for answering reviewer 1's question about social gaze bouts
# the analysis should be based on the dataframe that is already loaded - hddm_data_tgt_alldates

if 1: # It's useful, turn off for now
    
    # =========================
    # SETTINGS
    # =========================
    FPS = 30.0                  # camera frame rate
    BOUT_THRESHOLD = 0.0        # define bout as socialgaze_prob > threshold
    MIN_BOUT_DUR_S = 0.0        # set e.g. 0.1 to ignore ultra-short bouts

    # RT outlier removal (choose ONE method)
    RT_OUTLIER_METHOD = "quantile"   # "quantile" or "iqr"
    RT_Q_LOWER = 0.05                # used if method == "quantile"
    RT_Q_UPPER = 0.95                # used if method == "quantile"
    RT_IQR_K = 1.5                   # used if method == "iqr"

    # If True, remove RT outliers within each animal (subj_idx); else globally
    RT_FILTER_WITHIN_ANIMAL = True

    # =========================
    # HELPERS
    # =========================
    def _find_contiguous_true_runs(mask: np.ndarray):
        """Return list of (start_idx, end_idx) inclusive for contiguous True runs in boolean mask."""
        if mask.size == 0:
            return []
        mask = mask.astype(bool)

        d = np.diff(mask.astype(int))
        starts = np.where(d == 1)[0] + 1
        ends   = np.where(d == -1)[0]

        if mask[0]:
            starts = np.r_[0, starts]
        if mask[-1]:
            ends = np.r_[ends, mask.size - 1]

        return list(zip(starts, ends))

    def compute_gaze_bout_stats_from_prob(
        gaze_prob,
        fps: float = FPS,
        threshold: float = BOUT_THRESHOLD,
        min_bout_dur_s: float = MIN_BOUT_DUR_S
    ):
        """
        Compute per-trial gaze bout stats from smoothed gaze probability trace.

        Assumes the trace spans the trial window immediately preceding pull,
        so last sample is closest to pull.

        Returns:
          n_bouts
          mean_bout_dur_s
          min_bout_dur_s
          max_bout_dur_s
          median_bout_dur_s
          total_gaze_dur_s
          first_bout_start_s  (relative to pull; negative = before pull)
          last_bout_end_s     (relative to pull; negative = before pull)
          trial_window_s
        """
        if gaze_prob is None or (isinstance(gaze_prob, float) and np.isnan(gaze_prob)):
            return {
                "n_bouts": np.nan,
                "mean_bout_dur_s": np.nan,
                "min_bout_dur_s": np.nan,
                "max_bout_dur_s": np.nan,
                "median_bout_dur_s": np.nan,
                "total_gaze_dur_s": np.nan,
                "first_bout_start_s": np.nan,
                "last_bout_end_s": np.nan,
                "trial_window_s": np.nan,
            }

        gaze_prob = np.asarray(gaze_prob).astype(float)
        if gaze_prob.size == 0 or np.all(np.isnan(gaze_prob)):
            T = gaze_prob.size / fps if gaze_prob.size else 0.0
            return {
                "n_bouts": 0,
                "mean_bout_dur_s": 0.0,
                "min_bout_dur_s": 0.0,
                "max_bout_dur_s": 0.0,
                "median_bout_dur_s": 0.0,
                "total_gaze_dur_s": 0.0,
                "first_bout_start_s": np.nan,
                "last_bout_end_s": np.nan,
                "trial_window_s": T,
            }

        mask = gaze_prob > threshold
        runs = _find_contiguous_true_runs(mask)

        if min_bout_dur_s > 0:
            min_len = int(np.ceil(min_bout_dur_s * fps))
            runs = [(s, e) for (s, e) in runs if (e - s + 1) >= min_len]

        n_bouts = len(runs)

        bout_durs_s = [((e - s + 1) / fps) for (s, e) in runs]
        total_gaze_dur_s = float(np.sum(bout_durs_s)) if n_bouts > 0 else 0.0

        mean_bout_dur_s = float(np.mean(bout_durs_s)) if n_bouts > 0 else 0.0
        median_bout_dur_s = float(np.median(bout_durs_s)) if n_bouts > 0 else 0.0
        min_bout_dur_s_out = float(np.min(bout_durs_s)) if n_bouts > 0 else 0.0
        max_bout_dur_s_out = float(np.max(bout_durs_s)) if n_bouts > 0 else 0.0

        T = gaze_prob.size / fps

        if n_bouts > 0:
            first_start_idx = runs[0][0]
            last_end_idx = runs[-1][1]
            first_bout_start_s = (first_start_idx / fps) - T
            last_bout_end_s = (last_end_idx / fps) - T
        else:
            first_bout_start_s = np.nan
            last_bout_end_s = np.nan

        return {
            "n_bouts": int(n_bouts),
            "mean_bout_dur_s": mean_bout_dur_s,
            "min_bout_dur_s": min_bout_dur_s_out,
            "max_bout_dur_s": max_bout_dur_s_out,
            "median_bout_dur_s": median_bout_dur_s,
            "total_gaze_dur_s": total_gaze_dur_s,
            "first_bout_start_s": first_bout_start_s,
            "last_bout_end_s": last_bout_end_s,
            "trial_window_s": T,
        }

    def mean_min_max_str(x, precision=3):
        x = np.asarray(x, dtype=float)
        if np.all(np.isnan(x)):
            return "nan [nan, nan]"
        return (
            f"{np.nanmean(x):.{precision}f} "
            f"[{np.nanmin(x):.{precision}f}, {np.nanmax(x):.{precision}f}]"
        )

    def _rt_filter_mask(series: pd.Series, method: str):
        """Return boolean mask (True = keep) for RT outlier removal."""
        s = series.astype(float)
        s_nonan = s.dropna()
        if s_nonan.empty:
            return pd.Series(False, index=series.index)

        if method == "quantile":
            lo, hi = s_nonan.quantile([RT_Q_LOWER, RT_Q_UPPER])
            return (s >= lo) & (s <= hi)

        if method == "iqr":
            q1 = s_nonan.quantile(0.25)
            q3 = s_nonan.quantile(0.75)
            iqr = q3 - q1
            lo = q1 - RT_IQR_K * iqr
            hi = q3 + RT_IQR_K * iqr
            return (s >= lo) & (s <= hi)

        raise ValueError(f"Unknown RT_OUTLIER_METHOD: {method}")

    # =========================
    # MAIN
    # =========================
    # Expects hddm_data_tgt_alldates already exists and contains 'socialgaze_prob', 'rt', 'subj_idx'

    # Start from a copy so we don't mutate your original
    df_gaze = hddm_data_tgt_alldates.copy()

    # --- Remove RT outliers ---
    n_before = len(df_gaze)

    if RT_FILTER_WITHIN_ANIMAL:
        keep_mask = (
            df_gaze.groupby("subj_idx", group_keys=False)["rt"]
            .apply(lambda s: _rt_filter_mask(s, RT_OUTLIER_METHOD))
        )
    else:
        keep_mask = _rt_filter_mask(df_gaze["rt"], RT_OUTLIER_METHOD)

    df_gaze = df_gaze[keep_mask].copy()
    n_after = len(df_gaze)

    print(f"RT outlier removal ({RT_OUTLIER_METHOD}, within_animal={RT_FILTER_WITHIN_ANIMAL}): "
          f"kept {n_after}/{n_before} trials, removed {n_before - n_after}")

    # --- Compute per-trial bout stats from socialgaze_prob ---
    bout_stats = df_gaze["socialgaze_prob"].apply(
        lambda x: compute_gaze_bout_stats_from_prob(
            x,
            fps=FPS,
            threshold=BOUT_THRESHOLD,
            min_bout_dur_s=MIN_BOUT_DUR_S
        )
    )

    bout_stats_df = pd.DataFrame(list(bout_stats.values), index=df_gaze.index)
    df_gaze = pd.concat([df_gaze, bout_stats_df], axis=1)
    
    # --- Gaze fraction of trial time (total gaze / RT) ---
    df_gaze["gaze_fraction_of_trial"] = np.nan
    valid = np.isfinite(df_gaze["total_gaze_dur_s"].values) & np.isfinite(df_gaze["rt"].values) & (df_gaze["rt"].values > 0)
    df_gaze.loc[valid, "gaze_fraction_of_trial"] = df_gaze.loc[valid, "total_gaze_dur_s"] / df_gaze.loc[valid, "rt"]

    # =========================
    # TABLE Sx: per-animal summary (mean [min, max])
    # =========================
    summary_map = {
        "n_bouts": "Gaze bouts per trial",
        "mean_bout_dur_s": "Mean gaze bout duration (s)",
        "min_bout_dur_s": "Min gaze bout duration (s)",
        "max_bout_dur_s": "Max gaze bout duration (s)",
        "total_gaze_dur_s": "Total gaze duration per trial (s)",
        "first_bout_start_s": "First gaze bout start (s rel. pull)",
        "last_bout_end_s": "Last gaze bout end (s rel. pull)",
        "rt": "Response time / trial duration (s)",
        "gaze_fraction_of_trial":"Gaze fraction of trial (total gaze / RT)",
    }

    table_sx = (
        df_gaze
        .groupby("subj_idx")
        .agg({k: mean_min_max_str for k in summary_map.keys()})
        .rename(columns=summary_map)
    )

    print("\n=== Table Sx (mean [min, max]) by animal (subj_idx) ===")
    print(table_sx)

    # Optional: save for easy copy into a doc
    # table_sx.to_csv("Table_Sx_gaze_bout_summary_RTfiltered.csv", index=True)

In [ ]:
# run the hddm fitting using the data set that contained average neuron activity (across neurons for each trial)
# the data set is calculated in Misha - basic_neural_analysis_allsessions_basicEvents_PCA_makeBhvNeuronVideos_PullStartToPull_section_continuousBhv

# this one is to fit the ddm in each sesssion separately
# this one use gaze period to filter the partner's variables

# this one also consider z ~ prev_trial_outcome

if 1:
    
    import seaborn as sns
    from scipy.stats import ttest_1samp

    # In this data set all the rt is calculated using the new definition, but the pull onset is defined slightly different
    #
    # if use the bhv PCA
    dobhvPCA = 1
    #
    doparnterleverspeed = 0 # this one only works when dobhvPCA is 0, it means use partner to lever speed that any speed
    #
    if not dobhvPCA:
        # if use onset of the first increase after min
        doOnsetAfterMin = 1
        if not doOnsetAfterMin:
            doOnsetAfterMin_suffix = ''
        elif doOnsetAfterMin:
            if not doparnterleverspeed:
                doOnsetAfterMin_suffix = 'PullOnsetAfterMin_'
            elif doparnterleverspeed:
                doOnsetAfterMin_suffix = 'PullOnsetAfterMin_PartnerLeverSpeed_'
    #
    elif dobhvPCA:
        # if use onset of the first increase after min
        doOnsetAfterMin = 1
        doBasedonSpeed = 1
        if not doOnsetAfterMin:
            doOnsetAfterMin_suffix = 'bhvPCA_'
        elif doOnsetAfterMin:
            if not doBasedonSpeed:
                doOnsetAfterMin_suffix = 'bhvPCA_PullOnsetAfterMin_'
            elif doBasedonSpeed:
                doOnsetAfterMin_suffix = 'bhvPCA_PullOnsetAfterMinBasedonSpeed_'
                

    # if use a hmm based method to find the trial start
    doHMMmethod = 0
    if doHMMmethod:
        doOnsetAfterMin_suffix = 'HMMmethods_'
        
    exaustModel_sufix = ''
        
    # load the dataset first
    current_dir = data_saved_folder+'data_saved_singlecam_wholebody_neural_and_hddm'+savefile_sufix+'/'+cameraID+'/'+animal_tgt
    add_date_dir = os.path.join(current_dir+'/hddm_model_fitted_combinedsession_withNeurons')
    
    if cond_toplot_type == 'allMC':
        if not (doOnsetAfterMin_suffix == 'bhvPCA_PullOnsetAfterMinBasedonSpeed_') |\
               (doOnsetAfterMin_suffix == 'PullOnsetAfterMin_PartnerLeverSpeed_'):
            with open(add_date_dir+'/hddm_datas_flexibleTW_newRTdefinition_'+doOnsetAfterMin_suffix+'withFRs_all_dates_'+
                      animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
                hddm_data_tgt_alldates = pickle.load(f)
        #
        elif (doOnsetAfterMin_suffix == 'bhvPCA_PullOnsetAfterMinBasedonSpeed_') |\
             (doOnsetAfterMin_suffix == 'PullOnsetAfterMin_PartnerLeverSpeed_'):
            with open(add_date_dir+'/hddm_datas_flexibleTW_newRTdefinition_'+'PullOnsetAfterMin_'+'withFRs_all_dates_'+
                      animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
                hddm_data_tgt_alldates = pickle.load(f)
    #
    else:
        if not (doOnsetAfterMin_suffix == 'bhvPCA_PullOnsetAfterMinBasedonSpeed_') |\
               (doOnsetAfterMin_suffix == 'PullOnsetAfterMin_PartnerLeverSpeed_'):
            with open(add_date_dir+'/hddm_datas_flexibleTW_newRTdefinition_'+doOnsetAfterMin_suffix+'withFRs_all_dates_'+
                      animal1_fixedorders[0]+animal2_fixedorders[0]+'_'+cond_toplot_type+'.pkl', 'rb') as f:
                hddm_data_tgt_alldates = pickle.load(f)
        #
        elif (doOnsetAfterMin_suffix == 'bhvPCA_PullOnsetAfterMinBasedonSpeed_') |\
             (doOnsetAfterMin_suffix == 'PullOnsetAfterMin_PartnerLeverSpeed_'):
            with open(add_date_dir+'/hddm_datas_flexibleTW_newRTdefinition_'+'PullOnsetAfterMin_'+'withFRs_all_dates_'+
                      animal1_fixedorders[0]+animal2_fixedorders[0]+'_'+cond_toplot_type+'.pkl', 'rb') as f:
                hddm_data_tgt_alldates = pickle.load(f)
            

    # make the hddm_data structure consistent with other code
    ind_goodcond = np.isin(hddm_data_tgt_alldates['condition'],conditions_to_ana)
    hddm_data_tgt_alldates = hddm_data_tgt_alldates[ind_goodcond]
    
    

    if not dobhvPCA:
        if not doparnterleverspeed:
            rename_dict = {
                'act_animal':'subj_idx',
                'dates': 'date',
                'pull_rt': 'rt',
                'previous_pull_outcome': 'prev_trial_outcome',
                'socialgaze_auc': 'self_gaze_auc',
                'mass_move_speed_mean': 'self_mean_speed',
                'mass_move_speed_std': 'self_speed_std',
                'other_mass_move_speed_mean': 'partner_mean_speed',
                'other_mass_move_speed_std': 'partner_speed_std',
                'num_preceding_failpull': 'failed_pulls_before_reward',
                'time_from_last_reward': 'time_since_last_reward',
                'cumulative_reward': 'cumulative_reward',
                }
        elif doparnterleverspeed:
            rename_dict = {
                'act_animal':'subj_idx',
                'dates': 'date',
                'pull_rt': 'rt',
                'previous_pull_outcome': 'prev_trial_outcome',
                'socialgaze_auc': 'self_gaze_auc',
                'mass_move_speed_mean': 'self_mean_speed',
                'mass_move_speed_std': 'self_speed_std',
                'other_lever_speed_mean': 'partner_mean_speed',
                'other_lever_speed_std': 'partner_speed_std',
                'num_preceding_failpull': 'failed_pulls_before_reward',
                'time_from_last_reward': 'time_since_last_reward',
                'cumulative_reward': 'cumulative_reward',
                }
    elif dobhvPCA:
        rename_dict = {
            'act_animal':'subj_idx',
            'dates': 'date',
            'pull_rt': 'rt',
            'previous_pull_outcome': 'prev_trial_outcome',
            'socialgaze_auc': 'self_gaze_auc',
            # 'self_PC1_mean': 'self_mean_speed',
            # 'self_PC1_std': 'self_speed_std',
            'mass_move_speed_mean': 'self_mean_speed',
            'mass_move_speed_std': 'self_speed_std',
            'other_PC1_mean': 'partner_mean_speed',
            'other_PC1_std': 'partner_speed_std',
            'num_preceding_failpull': 'failed_pulls_before_reward',
            'time_from_last_reward': 'time_since_last_reward',
            'cumulative_reward': 'cumulative_reward',
            }
        
    # Apply the renaming
    hddm_data_tgt_alldates = hddm_data_tgt_alldates.rename(columns=rename_dict)
    hddm_data_tgt_alldates['response'] = 1
    
    # mostly just for allSR
    if cond_toplot_type == 'allSR':
        hddm_data_tgt_alldates['failed_pulls_before_reward']= 0

    
    #
    # calculated the new partner_mean_speed and partner_speed_std with self_gaze filter
    #
    hddm_data_tgt_alldates['partner_speed_std_nofilter'] = hddm_data_tgt_alldates['partner_speed_std']
    hddm_data_tgt_alldates['partner_mean_speed_nofilter'] = hddm_data_tgt_alldates['partner_mean_speed']
    
    #
    ntrials = np.shape(hddm_data_tgt_alldates)[0]
    for itrial in np.arange(0,ntrials,1):
        gaze_prob_itrial = hddm_data_tgt_alldates['socialgaze_prob'].iloc[itrial]
        #
        if not dobhvPCA:
            if not doparnterleverspeed:
                partner_speed_itrial = hddm_data_tgt_alldates['other_mass_move_speed'].iloc[itrial]
            elif doparnterleverspeed:
                partner_speed_itrial = hddm_data_tgt_alldates['other_lever_speed'].iloc[itrial]
        elif dobhvPCA:
            partner_speed_itrial = hddm_data_tgt_alldates['other_PC1'].iloc[itrial]
        #
        # the shape of 'other_lever_speed' and other variables might be different
        partner_speed_shape = np.shape(partner_speed_itrial)[0]
        gaze_prob_shape = np.shape(gaze_prob_itrial)[0]
        if partner_speed_shape > gaze_prob_shape:
            partner_speed_itrial = partner_speed_itrial[np.arange(0,gaze_prob_shape,1)]
        elif partner_speed_shape < gaze_prob_shape:
            gaze_prob_itrial = gaze_prob_itrial[np.arange(0,partner_speed_shape,1)]

        # smooth the gaze duration a bit more
        gaze_prob_itrial = gaussian_filter1d(gaze_prob_itrial, sigma=8) # the final sigma would be sqrt(4*4+sigma*sigma)

        partner_speed_filtered = partner_speed_itrial[gaze_prob_itrial>0]

        partner_mean_speed_filtered = np.nanmean(partner_speed_filtered)
        partner_std_speed_filtered = np.nanstd(partner_speed_filtered)
        
        hddm_data_tgt_alldates['partner_mean_speed'].iloc[itrial] = partner_mean_speed_filtered
        hddm_data_tgt_alldates['partner_speed_std'].iloc[itrial] = partner_std_speed_filtered
    
    
    # 
    # remove nan row first
    hddm_data_tgt_alldates = hddm_data_tgt_alldates.dropna()
    
    
    # 
    # normalize all the predictors
    from scipy.stats import zscore
    #
    # List of predictor columns to normalize (exclude 'rt' and identifiers)
    predictor_columns = [
        'self_gaze_auc',
        'partner_mean_speed',
        'self_mean_speed',
        'partner_speed_std',
        'self_speed_std',
        # 'failed_pulls_before_reward',
        'time_since_last_reward'
    ]
    # Rename originals and replace with z-scored values
    for col in predictor_columns:
        hddm_data_tgt_alldates[f"{col}_origin"] = hddm_data_tgt_alldates[col]
        hddm_data_tgt_alldates[col] = zscore(hddm_data_tgt_alldates[col])  
        # hddm_data_tgt_alldates[col] = hddm_data_tgt_alldates[col]  
      
    #
    # initialize a dataframe for summarizing all sessions results
    summary_list = []
    
    
    # run the fitting for each session
    for session_toana, hddm_data_tgt_idate in hddm_data_tgt_alldates.groupby('date'):
    
        hddm_data_tgt_idate['cumulative_reward'] = hddm_data_tgt_idate['prev_trial_outcome'].cumsum()
    
        ind_ = np.isin(dates_list,session_toana)
        task_cond = task_conditions[np.where(ind_)[0][0]]
        
        # Inside the loop:
        # 'session_toana' is the date for the current group
        # 'hddm_data_tgt_idate' is a DataFrame containing only the rows for that date
    
        #
        # # add some semi-artbitrary cretieras
        # hddm_data_tgt_alldates = hddm_data_tgt_idate[hddm_data_tgt_alldates['prev_trial_outcome']==1]
        # hddm_data_tgt_alldates = hddm_data_tgt_idate[hddm_data_tgt_alldates['partner_mean_speed']<=300]
        #
        lower_q = 0.05
        upper_q = 0.95
        rt_lower, rt_upper = hddm_data_tgt_idate['rt'].quantile([lower_q, upper_q])
        # hddm_data_tgt_idate = hddm_data_tgt_idate[hddm_data_tgt_idate['rt']>rt_lower]
        # hddm_data_tgt_idate = hddm_data_tgt_idate[hddm_data_tgt_idate['rt']<rt_upper]
        #
        tslr_lower, tslr_upper = hddm_data_tgt_idate['time_since_last_reward'].quantile([lower_q, upper_q])
        # hddm_data_tgt_idate = hddm_data_tgt_idate[hddm_data_tgt_idate['time_since_last_reward']>tslr_lower]
        # hddm_data_tgt_idate = hddm_data_tgt_idate[hddm_data_tgt_idate['time_since_last_reward']<tslr_upper]
        
        
        # try to load the analyzed
        try:

            # dummpy
            
            current_dir = data_saved_folder+'data_saved_singlecam_wholebody_neural_and_hddm'+savefile_sufix+'/'+cameraID+'/'+animal_tgt
            add_date_dir = os.path.join(current_dir+'/hddm_model_fitted_allsession_withNeurons/'+session_toana)

            #
            with open(add_date_dir+'/hddm_model_fitted_nogaze_gazefilter_ZHistOut_traces_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
                hddm_model_fitted_nogaze_gazefilter_ZHistOut_traces = pickle.load(f)

            with open(add_date_dir+'/hddm_model_fitted_nogaze_gazefilter_ZHistOut_stats_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
                hddm_model_fitted_nogaze_gazefilter_ZHistOut_stats = pickle.load(f)

            with open(add_date_dir+'/hddm_model_fitted_nogaze_gazefilter_ZHistOut_dic_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
                hddm_model_fitted_nogaze_gazefilter_ZHistOut_dic = pickle.load(f)
                
            

            print(animal_tgt+' '+cond_toplot_type+' hddm fitting results loaded')

        except:

            print(animal_tgt+' '+cond_toplot_type+' running hddm fitting')
            
            try:

                # run the hddm nodel
                # HDDM settings - the basic setting
                # samples=500 # 2000
                # burn=100
                # thin=2
                samples=200 # 2000
                burn=100
                thin=2
                #
                doNogazeOnly = 1
                #
                if dobhvPCA:
                    _, hddm_model_fitted_nogaze_gazefilter_ZHistOut = run_hddm_modeling_bhvPCA_gazefilter_ZHistOut(hddm_data_tgt_idate, animal_tgt,
                                                                        samples, burn, thin, doNogazeOnly)
                elif not dobhvPCA:
                    _, hddm_model_fitted_nogaze_gazefilter_ZHistOut = run_hddm_modeling_gazefilter_ZHistOut(hddm_data_tgt_idate, animal_tgt,
                                                                        samples, burn, thin, doNogazeOnly)

                #
                hddm_model_fitted_nogaze_gazefilter_ZHistOut_traces = hddm_model_fitted_nogaze_gazefilter_ZHistOut.get_traces()
                hddm_model_fitted_nogaze_gazefilter_ZHistOut_stats  = hddm_model_fitted_nogaze_gazefilter_ZHistOut.gen_stats()
                hddm_model_fitted_nogaze_gazefilter_ZHistOut_dic = hddm_model_fitted_nogaze_gazefilter_ZHistOut.dic_info
            
            except:
                continue

                
            # save data
            if 1:
                current_dir = data_saved_folder+'data_saved_singlecam_wholebody_neural_and_hddm'+savefile_sufix+'/'+cameraID+'/'+animal_tgt
                add_date_dir = os.path.join(current_dir+'/hddm_model_fitted_allsession_withNeurons/'+session_toana)
                if not os.path.exists(add_date_dir):
                    os.makedirs(add_date_dir)
                #
                with open(add_date_dir+'/hddm_model_fitted_nogaze_gazefilter_ZHistOut_traces_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'wb') as f:
                    pickle.dump(hddm_model_fitted_nogaze_gazefilter_ZHistOut_traces, f)

                with open(add_date_dir+'/hddm_model_fitted_nogaze_gazefilter_ZHistOut_stats_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'wb') as f:
                    pickle.dump(hddm_model_fitted_nogaze_gazefilter_ZHistOut_stats, f)  

                with open(add_date_dir+'/hddm_model_fitted_nogaze_gazefilter_ZHistOut_dic_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'wb') as f:
                    pickle.dump(hddm_model_fitted_nogaze_gazefilter_ZHistOut_dic, f)
            
          
        #
        # reconstruct V
        #
        df_combined = hddm_data_tgt_idate # your combined trial-level dataframe
        traces = hddm_model_fitted_nogaze_gazefilter_ZHistOut_traces  # or use `hddm_model_fitted_traces` if needed
        stats = hddm_model_fitted_nogaze_gazefilter_ZHistOut_stats
        dics = hddm_model_fitted_nogaze_gazefilter_ZHistOut_dic
        
        # add a fitting criteria
        if dics['pD'] < 0:
            print('bad fitting for '+session_toana)
            continue


        # --- STEP 1: Use all posterior samples to compute predicted v for each trial ---
        n_samples = len(traces)
        n_trials = len(df_combined)

        # Allocate space to hold predicted v for each sample x trial
        predicted_v_samples = np.zeros((n_samples, n_trials))

        # Pre-fetch covariate values as numpy arrays
        # x1 = df_combined['partner_mean_speed'].values
        x2 = df_combined['partner_speed_std'].values
        x3 = df_combined['self_mean_speed'].values
        x4 = df_combined['self_speed_std'].values
        # x3 = df_combined['failed_pulls_before_reward'].values
        # x3 = df_combined['time_since_last_reward'].values
        # x7 = df_combined['self_gaze_auc'].values

        for i in range(n_samples):
            intercept = traces['v_Intercept'].values[i]
            # coef_x1 = traces['v_partner_mean_speed'].values[i]
            coef_x2 = traces['v_partner_speed_std'].values[i]
            coef_x3 = traces['v_self_mean_speed'].values[i]
            coef_x4 = traces['v_self_speed_std'].values[i]
            # coef_x3 = traces['v_failed_pulls_before_reward'].values[i]
            # coef_x3 = traces['v_time_since_last_reward'].values[i]
            # coef_x7 = traces['v_self_gaze_auc'].values[i]

            # Apply regression formula for all trials
            predicted_v_samples[i, :] = (
                # intercept + coef_x1 * x1 + coef_x2 * x2 + coef_x3 * x3
                # intercept + coef_x1 * x1 
                # intercept + coef_x2 * x2 
                # intercept + coef_x1 * x1 + coef_x2 * x2 + coef_x3 * x3
                # intercept + coef_x1 * x1 + coef_x3 * x3
                # intercept + coef_x1 * x1 + coef_x2 * x2 + coef_x3 * x3 + coef_x4 * x4
                intercept + coef_x2 * x2 + coef_x3 * x3 + coef_x4 * x4
                # intercept + coef_x1 * x1 + coef_x2 * x2 + coef_x3 * x3 + coef_x4 * x4 + coef_x7 * x7
                # intercept + coef_x2 * x2 + coef_x4 * x4
            )

        # --- STEP 2: Compute posterior mean (and optionally CI) of predicted v ---
        df_with_v = df_combined.copy()
        df_with_v['predicted_v'] = predicted_v_samples.mean(axis=0)

        # Optional: also store credible intervals (e.g., 95% CI)
        df_with_v['predicted_v_lower'] = np.percentile(predicted_v_samples, 2.5, axis=0)
        df_with_v['predicted_v_upper'] = np.percentile(predicted_v_samples, 97.5, axis=0) 
            
                   
        #
        # save the summaring data across date   
        #
        # --- Process the model data ---
        # --- Task 1: Process the HDDM model coefficients (same as before) ---
        stats['significance'] = (stats['2.5q'] * \
                                 stats['97.5q'] > 0).astype(int)
        session_summary = stats.reset_index().rename(columns={'index': 'coefficient'})
        session_summary['date'] = session_toana
        session_summary['condition'] = task_cond
        session_summary = session_summary[['date', 'condition', 'coefficient', 'mean', 'std', 'significance']]
        summary_list.append(session_summary)


        # --- Task 2: Calculate and add the v-rt correlation ---
        # Calculate the Pearson correlation coefficient and the p-value.
        corr_coef, p_value = st.pearsonr(df_with_v['predicted_v'], df_with_v['rt'])
        #
        # Create a small DataFrame for the correlation result.
        # We use the 'mean' column for the coefficient and 'significance' for the p-value
        # to keep the data structure consistent.
        corr_summary = pd.DataFrame([{
            'date': session_toana,
            'condition':task_cond,
            'coefficient': 'predicted_v_rt_correlation',
            'mean': corr_coef,
            'std': np.nan,  # Standard deviation isn't applicable here
            'significance': (p_value<0.05).astype(int)
        }])
        #
        # Add the correlation result to the list.
        summary_list.append(corr_summary)
        
        # with mean firing rate
        corr_coef, p_value = st.pearsonr(df_with_v['predicted_v'], df_with_v['fr_mean'])
        #
        # Create a small DataFrame for the correlation result.
        # We use the 'mean' column for the coefficient and 'significance' for the p-value
        # to keep the data structure consistent.
        corr_summary = pd.DataFrame([{
            'date': session_toana,
            'condition':task_cond,
            'coefficient': 'predicted_v_fr_mean_correlation',
            'mean': corr_coef,
            'std': np.nan,  # Standard deviation isn't applicable here
            'significance': (p_value<0.05).astype(int)
        }])
        #
        # Add the correlation result to the list.
        summary_list.append(corr_summary)
        
         # with slope firing rate
        corr_coef, p_value = st.pearsonr(df_with_v['predicted_v'], df_with_v['fr_slope'])
        #
        # Create a small DataFrame for the correlation result.
        # We use the 'mean' column for the coefficient and 'significance' for the p-value
        # to keep the data structure consistent.
        corr_summary = pd.DataFrame([{
            'date': session_toana,
            'condition':task_cond,
            'coefficient': 'predicted_v_fr_slope_correlation',
            'mean': corr_coef,
            'std': np.nan,  # Standard deviation isn't applicable here
            'significance': (p_value<0.05).astype(int)
        }])
        #
        # Add the correlation result to the list.
        summary_list.append(corr_summary)
    
    
    # Combine the list of DataFrames into the final summary DataFrame.
    hddm_model_fitted_alldates_summary = pd.concat(summary_list, ignore_index=True)
    
    ## 
    # do some plotting
    ##
    plot_data = hddm_model_fitted_alldates_summary.copy()
    
    # ---- NEW: Filter out the coefficients you don't want to see ----
    coeffs_to_exclude = ['t', 'z', 'a', 'v_Intercept', 'a_Intercept','a_time_since_last_reward',
                         'predicted_v_fr_mean_correlation','predicted_v_fr_slope_correlation']
    #
    do_v_and_FR = 0
    if do_v_and_FR:
        coeffs_to_exclude = ['t', 'z', 'v_Intercept', 'a_Intercept','a_time_since_last_reward',
                             'v_self_gaze_auc', 'v_partner_mean_speed','v_partner_speed_std',
                             'v_self_mean_speed','v_self_speed_std',]
    #
    plot_data = plot_data[~plot_data['coefficient'].isin(coeffs_to_exclude)]

    plot_data['mean'] = pd.to_numeric(plot_data['mean'], errors='coerce')
    
    # ---- Remove outliers for each coefficient using the IQR method ----
    # For each coefficient, calculate the bounds
    Q1 = plot_data.groupby('coefficient')['mean'].transform(lambda x: x.quantile(0.25))
    Q3 = plot_data.groupby('coefficient')['mean'].transform(lambda x: x.quantile(0.75))
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    #
    # Filter out the outliers
    plot_data_no_outliers = plot_data[(plot_data['mean'] >= lower_bound) & (plot_data['mean'] <= upper_bound)]
    # plot_data_no_outliers = plot_data
    plot_data_no_outliers = plot_data_no_outliers.sort_values(by='coefficient')

    # # remove some unwanted condition
    # ind_dontwant = (plot_data_no_outliers['condition']=='MC_withGingerNew')# |\
    #          (plot_data_no_outliers['condition']=='MC_withKoala')|\
    #         (plot_data_no_outliers['condition']=='MC_withVermelho')
    # ind_want = plot_data_no_outliers['condition']=='MC'
    # plot_data_no_outliers = plot_data_no_outliers[~ind_dontwant]
    # # plot_data_no_outliers = plot_data_no_outliers[ind_want]
    
    
    
    # --- Plotting Code (using the outlier-free data) ---
    plt.figure(figsize=(14, 8))

    # Create the swarm plot
    # We use 'hue' to color points based on the 'significance' column for that date.
    if 0:
        ax = sns.swarmplot(
            x='coefficient',
            y='mean',
            data=plot_data_no_outliers,
            hue='significance',
            palette={0: 'gray', 1: 'red'},  # 0=Not Significant (gray), 1=Significant (red)
            size=8
        )
    else:
        ax = sns.violinplot(
            x='coefficient',
            y='mean',
            data=plot_data_no_outliers,
            bw_adjust=1.5
        )
    # Hide the legend as the colors are intuitive
    # ax.get_legend().remove()


    # --- Add Overall Significance Asterisks ---
    unique_coeffs = plot_data_no_outliers['coefficient'].unique()
    for i, coef in enumerate(unique_coeffs):
        
        coef_data_all_df = plot_data_no_outliers[plot_data_no_outliers['coefficient'] == coef]
        
        if 0: # this part is using the lmm
            # Ensure there's enough data to run the model
            if len(coef_data_all_df) > 1:
                p_value = 1.0 # Default to non-significant

                # print(coef_data_all_df.groupby('condition').size())

                # Use a try/except block to handle cases where the LMM might not converge
                try:
                    # --- START: LMM Replacement ---
                    # Define the LMM: model the 'mean' with an intercept, grouped by 'condition'
                    model = smf.mixedlm("mean ~ 1", coef_data_all_df, groups=coef_data_all_df['condition'])

                    # Fit the model (disp=False prevents printing convergence messages in the loop)
                    result = model.fit(disp=False)

                    # Get the p-value for the intercept (our test against 0)
                    p_value = result.pvalues['Intercept']
                    intercept_value = result.params['Intercept']

                    print(coef+f" Intercept: {intercept_value}")
                    print(coef+f" P-value: {p_value}")
                    
                    # --- END: LMM Replacement ---

                except Exception as e:
                    # If the model fails, print a warning and keep p_value at 1.0
                    print(f"LMM failed for coefficient '{coef}'. Error: {e}")


                # The rest of your plotting logic remains the same
                if p_value < 0.05:
                    # Use the 'mean' column from the subsetted dataframe for positioning
                    y_pos = coef_data_all_df['mean'].max()
                    offset = coef_data_all_df['mean'].std() * 0.1
                    plt.text(i, y_pos + offset, '*', ha='center', va='bottom', fontsize=24, color='black')

        
        if 1: # this part is using the simple ttest/wilcoxon
            coef_data = plot_data_no_outliers[plot_data_no_outliers['coefficient'] == coef]['mean']

            if len(coef_data) > 1:
                t_stat, p_value = ttest_1samp(coef_data, 0, nan_policy='omit')
                # w_stat, p_value = st.wilcoxon(coef_data, nan_policy='omit')
                print(p_value)
                if p_value < 0.05:
                    y_pos = coef_data.max()
                    offset = plot_data_no_outliers['mean'].std() * 0.1
                    plt.text(i, y_pos + offset, '*', ha='center', va='bottom', fontsize=24, color='black')

    plt.axhline(y=0, linestyle='--', color='black', linewidth=1.5)            
    
    # --- Finalize and Save Plot ---
    plt.title('Model Coefficients Across All Dates', fontsize=18)
    plt.ylabel('Coefficient Value (Mean)', fontsize=16)
    plt.xlabel('Model Coefficient', fontsize=16)
    plt.xticks(rotation=45, ha='right', fontsize = 16)
    plt.tight_layout()

    # savefig
    savefig = 0
    if savefig:
        current_dir = data_saved_folder+'fig_for_basic_neural_analysis_allsessions_basicEvents_PCA_Pullfocused_continuousBhv_partnerDistVaris_with_hddm_model'+savefile_sufix+'/'+animal_tgt
        add_fig_dir = os.path.join(current_dir,cameraID+'/fig_for_hddm_model_fitted_allsession_withNeurons')
        #
        if not os.path.exists(add_fig_dir):
            os.makedirs(add_fig_dir)
        if not do_v_and_FR:
            plt.savefig(add_fig_dir+'/predicted_v_and_other_bhv_variables_nogaze_gazefilter_ZHistOut_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pdf', 
                        format='pdf', bbox_inches='tight')
        elif do_v_and_FR:
            plt.savefig(add_fig_dir+'/predicted_v_and_FR_nogaze_gazefilter_ZHistOut_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pdf', 
                        format='pdf', bbox_inches='tight')


In [ ]:
doOnsetAfterMin_suffix

In [ ]:
# run the hddm fitting using the data set that contained average neuron activity (across neurons for each trial)
# the data set is calculated in Misha - basic_neural_analysis_allsessions_basicEvents_PCA_makeBhvNeuronVideos_PullStartToPull_section_continuousBhv

# this one is to fit the ddm in each sesssion separately
# this one use gaze period to filter the partner's variables

# this one also consider z ~ prev_trial_outcome + cummulative reward

if 0:
    
    import seaborn as sns
    from scipy.stats import ttest_1samp

    # In this data set all the rt is calculated using the new definition, but the pull onset is defined slightly different
    #
    # if use the bhv PCA
    dobhvPCA = 1
    #
    doparnterleverspeed = 0 # this one only works when dobhvPCA is 0, it means use partner to lever speed that any speed
    #
    if not dobhvPCA:
        # if use onset of the first increase after min
        doOnsetAfterMin = 1
        if not doOnsetAfterMin:
            doOnsetAfterMin_suffix = ''
        elif doOnsetAfterMin:
            if not doparnterleverspeed:
                doOnsetAfterMin_suffix = 'PullOnsetAfterMin_'
            elif doparnterleverspeed:
                doOnsetAfterMin_suffix = 'PullOnsetAfterMin_PartnerLeverSpeed_'
    #
    elif dobhvPCA:
        # if use onset of the first increase after min
        doOnsetAfterMin = 1
        doBasedonSpeed = 1
        if not doOnsetAfterMin:
            doOnsetAfterMin_suffix = 'bhvPCA_'
        elif doOnsetAfterMin:
            if not doBasedonSpeed:
                doOnsetAfterMin_suffix = 'bhvPCA_PullOnsetAfterMin_'
            elif doBasedonSpeed:
                doOnsetAfterMin_suffix = 'bhvPCA_PullOnsetAfterMinBasedonSpeed_'
                

    # if use a hmm based method to find the trial start
    doHMMmethod = 0
    if doHMMmethod:
        doOnsetAfterMin_suffix = 'HMMmethods_'
        
    exaustModel_sufix = ''
        
    # load the dataset first
    current_dir = data_saved_folder+'data_saved_singlecam_wholebody_neural_and_hddm'+savefile_sufix+'/'+cameraID+'/'+animal_tgt
    add_date_dir = os.path.join(current_dir+'/hddm_model_fitted_combinedsession_withNeurons')
    
    if cond_toplot_type == 'allMC':
        if not (doOnsetAfterMin_suffix == 'bhvPCA_PullOnsetAfterMinBasedonSpeed_') |\
               (doOnsetAfterMin_suffix == 'PullOnsetAfterMin_PartnerLeverSpeed_'):
            with open(add_date_dir+'/hddm_datas_flexibleTW_newRTdefinition_'+doOnsetAfterMin_suffix+'withFRs_all_dates_'+
                      animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
                hddm_data_tgt_alldates = pickle.load(f)
        #
        elif (doOnsetAfterMin_suffix == 'bhvPCA_PullOnsetAfterMinBasedonSpeed_') |\
             (doOnsetAfterMin_suffix == 'PullOnsetAfterMin_PartnerLeverSpeed_'):
            with open(add_date_dir+'/hddm_datas_flexibleTW_newRTdefinition_'+'PullOnsetAfterMin_'+'withFRs_all_dates_'+
                      animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
                hddm_data_tgt_alldates = pickle.load(f)
    #
    else:
        if not (doOnsetAfterMin_suffix == 'bhvPCA_PullOnsetAfterMinBasedonSpeed_') |\
               (doOnsetAfterMin_suffix == 'PullOnsetAfterMin_PartnerLeverSpeed_'):
            with open(add_date_dir+'/hddm_datas_flexibleTW_newRTdefinition_'+doOnsetAfterMin_suffix+'withFRs_all_dates_'+
                      animal1_fixedorders[0]+animal2_fixedorders[0]+'_'+cond_toplot_type+'.pkl', 'rb') as f:
                hddm_data_tgt_alldates = pickle.load(f)
        #
        elif (doOnsetAfterMin_suffix == 'bhvPCA_PullOnsetAfterMinBasedonSpeed_') |\
             (doOnsetAfterMin_suffix == 'PullOnsetAfterMin_PartnerLeverSpeed_'):
            with open(add_date_dir+'/hddm_datas_flexibleTW_newRTdefinition_'+'PullOnsetAfterMin_'+'withFRs_all_dates_'+
                      animal1_fixedorders[0]+animal2_fixedorders[0]+'_'+cond_toplot_type+'.pkl', 'rb') as f:
                hddm_data_tgt_alldates = pickle.load(f)
            

    # make the hddm_data structure consistent with other code
    ind_goodcond = np.isin(hddm_data_tgt_alldates['condition'],conditions_to_ana)
    hddm_data_tgt_alldates = hddm_data_tgt_alldates[ind_goodcond]
    
    

    if not dobhvPCA:
        if not doparnterleverspeed:
            rename_dict = {
                'act_animal':'subj_idx',
                'dates': 'date',
                'pull_rt': 'rt',
                'previous_pull_outcome': 'prev_trial_outcome',
                'socialgaze_auc': 'self_gaze_auc',
                'mass_move_speed_mean': 'self_mean_speed',
                'mass_move_speed_std': 'self_speed_std',
                'other_mass_move_speed_mean': 'partner_mean_speed',
                'other_mass_move_speed_std': 'partner_speed_std',
                'num_preceding_failpull': 'failed_pulls_before_reward',
                'time_from_last_reward': 'time_since_last_reward',
                'cumulative_reward': 'cumulative_reward',
                }
        elif doparnterleverspeed:
            rename_dict = {
                'act_animal':'subj_idx',
                'dates': 'date',
                'pull_rt': 'rt',
                'previous_pull_outcome': 'prev_trial_outcome',
                'socialgaze_auc': 'self_gaze_auc',
                'mass_move_speed_mean': 'self_mean_speed',
                'mass_move_speed_std': 'self_speed_std',
                'other_lever_speed_mean': 'partner_mean_speed',
                'other_lever_speed_std': 'partner_speed_std',
                'num_preceding_failpull': 'failed_pulls_before_reward',
                'time_from_last_reward': 'time_since_last_reward',
                'cumulative_reward': 'cumulative_reward',
                }
    elif dobhvPCA:
        rename_dict = {
            'act_animal':'subj_idx',
            'dates': 'date',
            'pull_rt': 'rt',
            'previous_pull_outcome': 'prev_trial_outcome',
            'socialgaze_auc': 'self_gaze_auc',
            # 'self_PC1_mean': 'self_mean_speed',
            # 'self_PC1_std': 'self_speed_std',
            'mass_move_speed_mean': 'self_mean_speed',
            'mass_move_speed_std': 'self_speed_std',
            'other_PC1_mean': 'partner_mean_speed',
            'other_PC1_std': 'partner_speed_std',
            'num_preceding_failpull': 'failed_pulls_before_reward',
            'time_from_last_reward': 'time_since_last_reward',
            'cumulative_reward': 'cumulative_reward',
            }
        
    # Apply the renaming
    hddm_data_tgt_alldates = hddm_data_tgt_alldates.rename(columns=rename_dict)
    hddm_data_tgt_alldates['response'] = 1
    
    # mostly just for allSR
    if cond_toplot_type == 'allSR':
        hddm_data_tgt_alldates['failed_pulls_before_reward']= 0

    
    #
    # calculated the new partner_mean_speed and partner_speed_std with self_gaze filter
    #
    hddm_data_tgt_alldates['partner_speed_std_nofilter'] = hddm_data_tgt_alldates['partner_speed_std']
    hddm_data_tgt_alldates['partner_mean_speed_nofilter'] = hddm_data_tgt_alldates['partner_mean_speed']
    
    #
    ntrials = np.shape(hddm_data_tgt_alldates)[0]
    for itrial in np.arange(0,ntrials,1):
        gaze_prob_itrial = hddm_data_tgt_alldates['socialgaze_prob'].iloc[itrial]
        #
        if not dobhvPCA:
            if not doparnterleverspeed:
                partner_speed_itrial = hddm_data_tgt_alldates['other_mass_move_speed'].iloc[itrial]
            elif doparnterleverspeed:
                partner_speed_itrial = hddm_data_tgt_alldates['other_lever_speed'].iloc[itrial]
        elif dobhvPCA:
            partner_speed_itrial = hddm_data_tgt_alldates['other_PC1'].iloc[itrial]
        #
        # the shape of 'other_lever_speed' and other variables might be different
        partner_speed_shape = np.shape(partner_speed_itrial)[0]
        gaze_prob_shape = np.shape(gaze_prob_itrial)[0]
        if partner_speed_shape > gaze_prob_shape:
            partner_speed_itrial = partner_speed_itrial[np.arange(0,gaze_prob_shape,1)]
        elif partner_speed_shape < gaze_prob_shape:
            gaze_prob_itrial = gaze_prob_itrial[np.arange(0,partner_speed_shape,1)]

        # smooth the gaze duration a bit more
        gaze_prob_itrial = gaussian_filter1d(gaze_prob_itrial, sigma=8) # the final sigma would be sqrt(4*4+sigma*sigma)

        partner_speed_filtered = partner_speed_itrial[gaze_prob_itrial>0]

        partner_mean_speed_filtered = np.nanmean(partner_speed_filtered)
        partner_std_speed_filtered = np.nanstd(partner_speed_filtered)
        
        hddm_data_tgt_alldates['partner_mean_speed'].iloc[itrial] = partner_mean_speed_filtered
        hddm_data_tgt_alldates['partner_speed_std'].iloc[itrial] = partner_std_speed_filtered
    
    
    # 
    # remove nan row first
    hddm_data_tgt_alldates = hddm_data_tgt_alldates.dropna()
    
    
    # 
    # normalize all the predictors
    from scipy.stats import zscore
    #
    # List of predictor columns to normalize (exclude 'rt' and identifiers)
    predictor_columns = [
        'self_gaze_auc',
        'partner_mean_speed',
        'self_mean_speed',
        'partner_speed_std',
        'self_speed_std',
        # 'failed_pulls_before_reward',
        'time_since_last_reward'
    ]
    # Rename originals and replace with z-scored values
    for col in predictor_columns:
        hddm_data_tgt_alldates[f"{col}_origin"] = hddm_data_tgt_alldates[col]
        hddm_data_tgt_alldates[col] = zscore(hddm_data_tgt_alldates[col])  
        # hddm_data_tgt_alldates[col] = hddm_data_tgt_alldates[col]  
      
    #
    # initialize a dataframe for summarizing all sessions results
    summary_list = []
    
    
    # run the fitting for each session
    for session_toana, hddm_data_tgt_idate in hddm_data_tgt_alldates.groupby('date'):
    
        hddm_data_tgt_idate['cumulative_reward'] = hddm_data_tgt_idate['prev_trial_outcome'].cumsum()
    
        ind_ = np.isin(dates_list,session_toana)
        task_cond = task_conditions[np.where(ind_)[0][0]]
        
        # Inside the loop:
        # 'session_toana' is the date for the current group
        # 'hddm_data_tgt_idate' is a DataFrame containing only the rows for that date
    
        #
        # # add some semi-artbitrary cretieras
        # hddm_data_tgt_alldates = hddm_data_tgt_idate[hddm_data_tgt_alldates['prev_trial_outcome']==1]
        # hddm_data_tgt_alldates = hddm_data_tgt_idate[hddm_data_tgt_alldates['partner_mean_speed']<=300]
        #
        lower_q = 0.05
        upper_q = 0.95
        rt_lower, rt_upper = hddm_data_tgt_idate['rt'].quantile([lower_q, upper_q])
        # hddm_data_tgt_idate = hddm_data_tgt_idate[hddm_data_tgt_idate['rt']>rt_lower]
        # hddm_data_tgt_idate = hddm_data_tgt_idate[hddm_data_tgt_idate['rt']<rt_upper]
        #
        tslr_lower, tslr_upper = hddm_data_tgt_idate['time_since_last_reward'].quantile([lower_q, upper_q])
        # hddm_data_tgt_idate = hddm_data_tgt_idate[hddm_data_tgt_idate['time_since_last_reward']>tslr_lower]
        # hddm_data_tgt_idate = hddm_data_tgt_idate[hddm_data_tgt_idate['time_since_last_reward']<tslr_upper]
        
        
        # try to load the analyzed
        try:

            # dummpy
            
            current_dir = data_saved_folder+'data_saved_singlecam_wholebody_neural_and_hddm'+savefile_sufix+'/'+cameraID+'/'+animal_tgt
            add_date_dir = os.path.join(current_dir+'/hddm_model_fitted_allsession_withNeurons/'+session_toana)

            #
            with open(add_date_dir+'/hddm_model_fitted_nogaze_gazefilter_ZHistOutCumRew_traces_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
                hddm_model_fitted_nogaze_gazefilter_ZHistOutCumRew_traces = pickle.load(f)

            with open(add_date_dir+'/hddm_model_fitted_nogaze_gazefilter_ZHistOutCumRew_stats_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
                hddm_model_fitted_nogaze_gazefilter_ZHistOutCumRew_stats = pickle.load(f)

            with open(add_date_dir+'/hddm_model_fitted_nogaze_gazefilter_ZHistOutCumRew_dic_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
                hddm_model_fitted_nogaze_gazefilter_ZHistOutCumRew_dic = pickle.load(f)
                
            

            print(animal_tgt+' '+cond_toplot_type+' hddm fitting results loaded')

        except:

            print(animal_tgt+' '+cond_toplot_type+' running hddm fitting')
            
            try:

                # run the hddm nodel
                # HDDM settings - the basic setting
                # samples=500 # 2000
                # burn=100
                # thin=2
                samples=200 # 2000
                burn=100
                thin=2
                #
                doNogazeOnly = 1
                #
                if dobhvPCA:
                    _, hddm_model_fitted_nogaze_gazefilter_ZHistOutCumRew = run_hddm_modeling_bhvPCA_gazefilter_ZHistOutCumRew(hddm_data_tgt_idate, animal_tgt,
                                                                        samples, burn, thin, doNogazeOnly)
                elif not dobhvPCA:
                    _, hddm_model_fitted_nogaze_gazefilter_ZHistOutCumRew = run_hddm_modeling_gazefilter_ZHistOutCumRew(hddm_data_tgt_idate, animal_tgt,
                                                                        samples, burn, thin, doNogazeOnly)

                #
                hddm_model_fitted_nogaze_gazefilter_ZHistOutCumRew_traces = hddm_model_fitted_nogaze_gazefilter_ZHistOutCumRew.get_traces()
                hddm_model_fitted_nogaze_gazefilter_ZHistOutCumRew_stats  = hddm_model_fitted_nogaze_gazefilter_ZHistOutCumRew.gen_stats()
                hddm_model_fitted_nogaze_gazefilter_ZHistOutCumRew_dic = hddm_model_fitted_nogaze_gazefilter_ZHistOutCumRew.dic_info
            
            except:
                continue

                
            # save data
            if 1:
                current_dir = data_saved_folder+'data_saved_singlecam_wholebody_neural_and_hddm'+savefile_sufix+'/'+cameraID+'/'+animal_tgt
                add_date_dir = os.path.join(current_dir+'/hddm_model_fitted_allsession_withNeurons/'+session_toana)
                if not os.path.exists(add_date_dir):
                    os.makedirs(add_date_dir)
                #
                with open(add_date_dir+'/hddm_model_fitted_nogaze_gazefilter_ZHistOutCumRew_traces_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'wb') as f:
                    pickle.dump(hddm_model_fitted_nogaze_gazefilter_ZHistOutCumRew_traces, f)

                with open(add_date_dir+'/hddm_model_fitted_nogaze_gazefilter_ZHistOutCumRew_stats_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'wb') as f:
                    pickle.dump(hddm_model_fitted_nogaze_gazefilter_ZHistOutCumRew_stats, f)  

                with open(add_date_dir+'/hddm_model_fitted_nogaze_gazefilter_ZHistOutCumRew_dic_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'wb') as f:
                    pickle.dump(hddm_model_fitted_nogaze_gazefilter_ZHistOutCumRew_dic, f)
            
          
        #
        # reconstruct V
        #
        df_combined = hddm_data_tgt_idate # your combined trial-level dataframe
        traces = hddm_model_fitted_nogaze_gazefilter_ZHistOutCumRew_traces  # or use `hddm_model_fitted_traces` if needed
        stats = hddm_model_fitted_nogaze_gazefilter_ZHistOutCumRew_stats
        dics = hddm_model_fitted_nogaze_gazefilter_ZHistOutCumRew_dic
        
        # add a fitting criteria
        if dics['pD'] < 0:
            print('bad fitting for '+session_toana)
            continue


        # --- STEP 1: Use all posterior samples to compute predicted v for each trial ---
        n_samples = len(traces)
        n_trials = len(df_combined)

        # Allocate space to hold predicted v for each sample x trial
        predicted_v_samples = np.zeros((n_samples, n_trials))

        # Pre-fetch covariate values as numpy arrays
        # x1 = df_combined['partner_mean_speed'].values
        x2 = df_combined['partner_speed_std'].values
        x3 = df_combined['self_mean_speed'].values
        x4 = df_combined['self_speed_std'].values
        # x3 = df_combined['failed_pulls_before_reward'].values
        # x3 = df_combined['time_since_last_reward'].values
        # x7 = df_combined['self_gaze_auc'].values

        for i in range(n_samples):
            intercept = traces['v_Intercept'].values[i]
            # coef_x1 = traces['v_partner_mean_speed'].values[i]
            coef_x2 = traces['v_partner_speed_std'].values[i]
            coef_x3 = traces['v_self_mean_speed'].values[i]
            coef_x4 = traces['v_self_speed_std'].values[i]
            # coef_x3 = traces['v_failed_pulls_before_reward'].values[i]
            # coef_x3 = traces['v_time_since_last_reward'].values[i]
            # coef_x7 = traces['v_self_gaze_auc'].values[i]

            # Apply regression formula for all trials
            predicted_v_samples[i, :] = (
                # intercept + coef_x1 * x1 + coef_x2 * x2 + coef_x3 * x3
                # intercept + coef_x1 * x1 
                # intercept + coef_x2 * x2 
                # intercept + coef_x1 * x1 + coef_x2 * x2 + coef_x3 * x3
                # intercept + coef_x1 * x1 + coef_x3 * x3
                # intercept + coef_x1 * x1 + coef_x2 * x2 + coef_x3 * x3 + coef_x4 * x4
                intercept + coef_x2 * x2 + coef_x3 * x3 + coef_x4 * x4
                # intercept + coef_x1 * x1 + coef_x2 * x2 + coef_x3 * x3 + coef_x4 * x4 + coef_x7 * x7
                # intercept + coef_x2 * x2 + coef_x4 * x4
            )

        # --- STEP 2: Compute posterior mean (and optionally CI) of predicted v ---
        df_with_v = df_combined.copy()
        df_with_v['predicted_v'] = predicted_v_samples.mean(axis=0)

        # Optional: also store credible intervals (e.g., 95% CI)
        df_with_v['predicted_v_lower'] = np.percentile(predicted_v_samples, 2.5, axis=0)
        df_with_v['predicted_v_upper'] = np.percentile(predicted_v_samples, 97.5, axis=0) 
            
                   
        #
        # save the summaring data across date   
        #
        # --- Process the model data ---
        # --- Task 1: Process the HDDM model coefficients (same as before) ---
        stats['significance'] = (stats['2.5q'] * \
                                 stats['97.5q'] > 0).astype(int)
        session_summary = stats.reset_index().rename(columns={'index': 'coefficient'})
        session_summary['date'] = session_toana
        session_summary['condition'] = task_cond
        session_summary = session_summary[['date', 'condition', 'coefficient', 'mean', 'std', 'significance']]
        summary_list.append(session_summary)


        # --- Task 2: Calculate and add the v-rt correlation ---
        # Calculate the Pearson correlation coefficient and the p-value.
        corr_coef, p_value = st.pearsonr(df_with_v['predicted_v'], df_with_v['rt'])
        #
        # Create a small DataFrame for the correlation result.
        # We use the 'mean' column for the coefficient and 'significance' for the p-value
        # to keep the data structure consistent.
        corr_summary = pd.DataFrame([{
            'date': session_toana,
            'condition':task_cond,
            'coefficient': 'predicted_v_rt_correlation',
            'mean': corr_coef,
            'std': np.nan,  # Standard deviation isn't applicable here
            'significance': (p_value<0.05).astype(int)
        }])
        #
        # Add the correlation result to the list.
        summary_list.append(corr_summary)
        
        # with mean firing rate
        corr_coef, p_value = st.pearsonr(df_with_v['predicted_v'], df_with_v['fr_mean'])
        #
        # Create a small DataFrame for the correlation result.
        # We use the 'mean' column for the coefficient and 'significance' for the p-value
        # to keep the data structure consistent.
        corr_summary = pd.DataFrame([{
            'date': session_toana,
            'condition':task_cond,
            'coefficient': 'predicted_v_fr_mean_correlation',
            'mean': corr_coef,
            'std': np.nan,  # Standard deviation isn't applicable here
            'significance': (p_value<0.05).astype(int)
        }])
        #
        # Add the correlation result to the list.
        summary_list.append(corr_summary)
        
         # with slope firing rate
        corr_coef, p_value = st.pearsonr(df_with_v['predicted_v'], df_with_v['fr_slope'])
        #
        # Create a small DataFrame for the correlation result.
        # We use the 'mean' column for the coefficient and 'significance' for the p-value
        # to keep the data structure consistent.
        corr_summary = pd.DataFrame([{
            'date': session_toana,
            'condition':task_cond,
            'coefficient': 'predicted_v_fr_slope_correlation',
            'mean': corr_coef,
            'std': np.nan,  # Standard deviation isn't applicable here
            'significance': (p_value<0.05).astype(int)
        }])
        #
        # Add the correlation result to the list.
        summary_list.append(corr_summary)
    
    
    # Combine the list of DataFrames into the final summary DataFrame.
    hddm_model_fitted_alldates_summary = pd.concat(summary_list, ignore_index=True)
    
    ## 
    # do some plotting
    ##
    plot_data = hddm_model_fitted_alldates_summary.copy()
    
    # ---- NEW: Filter out the coefficients you don't want to see ----
    coeffs_to_exclude = ['t', 'z', 'z_Intercept', 'a', 'v_Intercept', 'a_Intercept','a_time_since_last_reward',
                         'predicted_v_fr_mean_correlation','predicted_v_fr_slope_correlation',
                         'v_self_mean_speed', 'v_self_speed_std'
                        ]
    #
    do_v_and_FR = 0
    if do_v_and_FR:
        coeffs_to_exclude = ['t', 'z', 'z_Intercept',  'v_Intercept', 'a_Intercept','a_time_since_last_reward',
                             'v_self_gaze_auc', 'v_partner_mean_speed','v_partner_speed_std',
                             'v_self_mean_speed','v_self_speed_std',]
    #
    plot_data = plot_data[~plot_data['coefficient'].isin(coeffs_to_exclude)]

    plot_data['mean'] = pd.to_numeric(plot_data['mean'], errors='coerce')
    
    # ---- Remove outliers for each coefficient using the IQR method ----
    # For each coefficient, calculate the bounds
    Q1 = plot_data.groupby('coefficient')['mean'].transform(lambda x: x.quantile(0.25))
    Q3 = plot_data.groupby('coefficient')['mean'].transform(lambda x: x.quantile(0.75))
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    #
    # Filter out the outliers
    plot_data_no_outliers = plot_data[(plot_data['mean'] >= lower_bound) & (plot_data['mean'] <= upper_bound)]
    # plot_data_no_outliers = plot_data
    plot_data_no_outliers = plot_data_no_outliers.sort_values(by='coefficient')

    # # remove some unwanted condition
    # ind_dontwant = (plot_data_no_outliers['condition']=='MC_withGingerNew')# |\
    #          (plot_data_no_outliers['condition']=='MC_withKoala')|\
    #         (plot_data_no_outliers['condition']=='MC_withVermelho')
    # ind_want = plot_data_no_outliers['condition']=='MC'
    # plot_data_no_outliers = plot_data_no_outliers[~ind_dontwant]
    # # plot_data_no_outliers = plot_data_no_outliers[ind_want]
    
    
    
    # --- Plotting Code (using the outlier-free data) ---
    plt.figure(figsize=(14, 8))

    # Create the swarm plot
    # We use 'hue' to color points based on the 'significance' column for that date.
    if 0:
        ax = sns.swarmplot(
            x='coefficient',
            y='mean',
            data=plot_data_no_outliers,
            hue='significance',
            palette={0: 'gray', 1: 'red'},  # 0=Not Significant (gray), 1=Significant (red)
            size=8
        )
    else:
        ax = sns.violinplot(
            x='coefficient',
            y='mean',
            data=plot_data_no_outliers,
            bw_adjust=1.5
        )
    # Hide the legend as the colors are intuitive
    # ax.get_legend().remove()


    # --- Add Overall Significance Asterisks ---
    unique_coeffs = plot_data_no_outliers['coefficient'].unique()
    for i, coef in enumerate(unique_coeffs):
        
        coef_data_all_df = plot_data_no_outliers[plot_data_no_outliers['coefficient'] == coef]
        
        if 0: # this part is using the lmm
            # Ensure there's enough data to run the model
            if len(coef_data_all_df) > 1:
                p_value = 1.0 # Default to non-significant

                # print(coef_data_all_df.groupby('condition').size())

                # Use a try/except block to handle cases where the LMM might not converge
                try:
                    # --- START: LMM Replacement ---
                    # Define the LMM: model the 'mean' with an intercept, grouped by 'condition'
                    model = smf.mixedlm("mean ~ 1", coef_data_all_df, groups=coef_data_all_df['condition'])

                    # Fit the model (disp=False prevents printing convergence messages in the loop)
                    result = model.fit(disp=False)

                    # Get the p-value for the intercept (our test against 0)
                    p_value = result.pvalues['Intercept']
                    intercept_value = result.params['Intercept']

                    print(coef+f" Intercept: {intercept_value}")
                    print(coef+f" P-value: {p_value}")
                    
                    # --- END: LMM Replacement ---

                except Exception as e:
                    # If the model fails, print a warning and keep p_value at 1.0
                    print(f"LMM failed for coefficient '{coef}'. Error: {e}")


                # The rest of your plotting logic remains the same
                if p_value < 0.05:
                    # Use the 'mean' column from the subsetted dataframe for positioning
                    y_pos = coef_data_all_df['mean'].max()
                    offset = coef_data_all_df['mean'].std() * 0.1
                    plt.text(i, y_pos + offset, '*', ha='center', va='bottom', fontsize=24, color='black')

        
        if 1: # this part is using the simple ttest/wilcoxon
            coef_data = plot_data_no_outliers[plot_data_no_outliers['coefficient'] == coef]['mean']

            if len(coef_data) > 1:
                t_stat, p_value = ttest_1samp(coef_data, 0, nan_policy='omit')
                # w_stat, p_value = st.wilcoxon(coef_data, nan_policy='omit')
                print(p_value)
                if p_value < 0.05:
                    y_pos = coef_data.max()
                    offset = plot_data_no_outliers['mean'].std() * 0.1
                    plt.text(i, y_pos + offset, '*', ha='center', va='bottom', fontsize=24, color='black')

    plt.axhline(y=0, linestyle='--', color='black', linewidth=1.5)            
    
    # --- Finalize and Save Plot ---
    plt.title('Model Coefficients Across All Dates', fontsize=18)
    plt.ylabel('Coefficient Value (Mean)', fontsize=16)
    plt.xlabel('Model Coefficient', fontsize=16)
    plt.xticks(rotation=45, ha='right', fontsize = 16)
    plt.tight_layout()

    # savefig
    savefig = 1
    if savefig:
        current_dir = data_saved_folder+'fig_for_basic_neural_analysis_allsessions_basicEvents_PCA_Pullfocused_continuousBhv_partnerDistVaris_with_hddm_model'+savefile_sufix+'/'+animal_tgt
        add_fig_dir = os.path.join(current_dir,cameraID+'/fig_for_hddm_model_fitted_allsession_withNeurons')
        #
        if not os.path.exists(add_fig_dir):
            os.makedirs(add_fig_dir)
        if not do_v_and_FR:
            plt.savefig(add_fig_dir+'/predicted_v_and_other_bhv_variables_nogaze_gazefilter_ZHistOutCumRew_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pdf', 
                        format='pdf', bbox_inches='tight')
        elif do_v_and_FR:
            plt.savefig(add_fig_dir+'/predicted_v_and_FR_nogaze_gazefilter_ZHistOutCumRew_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pdf', 
                        format='pdf', bbox_inches='tight')


In [ ]:
# plot_data_no_outliers[plot_data_no_outliers['coefficient']=='z_cumulative_reward']

In [ ]:
# run the hddm fitting using the data set that contained average neuron activity (across neurons for each trial)
# the data set is calculated in Misha - basic_neural_analysis_allsessions_basicEvents_PCA_makeBhvNeuronVideos_PullStartToPull_section_continuousBhv

# this one is to fit the ddm in each sesssion separately
# this one consider the gaze accumulation level as the interaction term with partner movement variables 

if 0:
    
    import seaborn as sns
    from scipy.stats import ttest_1samp

    # In this data set all the rt is calculated using the new definition, but the pull onset is defined slightly different
    #
    # if use the bhv PCA
    dobhvPCA = 1
    #
    doparnterleverspeed = 0 # this one only works when dobhvPCA is 0, it means use partner to lever speed that any speed
    #
    if not dobhvPCA:
        # if use onset of the first increase after min
        doOnsetAfterMin = 1
        if not doOnsetAfterMin:
            doOnsetAfterMin_suffix = ''
        elif doOnsetAfterMin:
            if not doparnterleverspeed:
                doOnsetAfterMin_suffix = 'PullOnsetAfterMin_'
            elif doparnterleverspeed:
                doOnsetAfterMin_suffix = 'PullOnsetAfterMin_PartnerLeverSpeed_'
    #
    elif dobhvPCA:
        # if use onset of the first increase after min
        doOnsetAfterMin = 1
        doBasedonSpeed = 1
        if not doOnsetAfterMin:
            doOnsetAfterMin_suffix = 'bhvPCA_'
        elif doOnsetAfterMin:
            if not doBasedonSpeed:
                doOnsetAfterMin_suffix = 'bhvPCA_PullOnsetAfterMin_'
            elif doBasedonSpeed:
                doOnsetAfterMin_suffix = 'bhvPCA_PullOnsetAfterMinBasedonSpeed_'
                

    # if use a hmm based method to find the trial start
    doHMMmethod = 0
    if doHMMmethod:
        doOnsetAfterMin_suffix = 'HMMmethods_'
        
    exaustModel_sufix = ''
        
    # load the dataset first
    current_dir = data_saved_folder+'data_saved_singlecam_wholebody_neural_and_hddm'+savefile_sufix+'/'+cameraID+'/'+animal_tgt
    add_date_dir = os.path.join(current_dir+'/hddm_model_fitted_combinedsession_withNeurons')
    
    if cond_toplot_type == 'allMC':
        if not (doOnsetAfterMin_suffix == 'bhvPCA_PullOnsetAfterMinBasedonSpeed_') |\
               (doOnsetAfterMin_suffix == 'PullOnsetAfterMin_PartnerLeverSpeed_'):
            with open(add_date_dir+'/hddm_datas_flexibleTW_newRTdefinition_'+doOnsetAfterMin_suffix+'withFRs_all_dates_'+
                      animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
                hddm_data_tgt_alldates = pickle.load(f)
        #
        elif (doOnsetAfterMin_suffix == 'bhvPCA_PullOnsetAfterMinBasedonSpeed_') |\
             (doOnsetAfterMin_suffix == 'PullOnsetAfterMin_PartnerLeverSpeed_'):
            with open(add_date_dir+'/hddm_datas_flexibleTW_newRTdefinition_'+'PullOnsetAfterMin_'+'withFRs_all_dates_'+
                      animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
                hddm_data_tgt_alldates = pickle.load(f)
    #
    else:
        if not (doOnsetAfterMin_suffix == 'bhvPCA_PullOnsetAfterMinBasedonSpeed_') |\
               (doOnsetAfterMin_suffix == 'PullOnsetAfterMin_PartnerLeverSpeed_'):
            with open(add_date_dir+'/hddm_datas_flexibleTW_newRTdefinition_'+doOnsetAfterMin_suffix+'withFRs_all_dates_'+
                      animal1_fixedorders[0]+animal2_fixedorders[0]+'_'+cond_toplot_type+'.pkl', 'rb') as f:
                hddm_data_tgt_alldates = pickle.load(f)
        #
        elif (doOnsetAfterMin_suffix == 'bhvPCA_PullOnsetAfterMinBasedonSpeed_') |\
             (doOnsetAfterMin_suffix == 'PullOnsetAfterMin_PartnerLeverSpeed_'):
            with open(add_date_dir+'/hddm_datas_flexibleTW_newRTdefinition_'+'PullOnsetAfterMin_'+'withFRs_all_dates_'+
                      animal1_fixedorders[0]+animal2_fixedorders[0]+'_'+cond_toplot_type+'.pkl', 'rb') as f:
                hddm_data_tgt_alldates = pickle.load(f)
            

    # make the hddm_data structure consistent with other code
    ind_goodcond = np.isin(hddm_data_tgt_alldates['condition'],conditions_to_ana)
    hddm_data_tgt_alldates = hddm_data_tgt_alldates[ind_goodcond]
    
    # mostly just for allSR
    if cond_toplot_type == 'allSR':
        hddm_data_tgt_alldates['failed_pulls_before_reward']= 0

    if not dobhvPCA:
        if not doparnterleverspeed:
            rename_dict = {
                'act_animal':'subj_idx',
                'dates': 'date',
                'pull_rt': 'rt',
                'previous_pull_outcome': 'prev_trial_outcome',
                'socialgaze_auc': 'self_gaze_auc',
                'mass_move_speed_mean': 'self_mean_speed',
                'mass_move_speed_std': 'self_speed_std',
                'other_mass_move_speed_mean': 'partner_mean_speed',
                'other_mass_move_speed_std': 'partner_speed_std',
                'num_preceding_failpull': 'failed_pulls_before_reward',
                'time_from_last_reward': 'time_since_last_reward',
                }
        elif doparnterleverspeed:
            rename_dict = {
                'act_animal':'subj_idx',
                'dates': 'date',
                'pull_rt': 'rt',
                'previous_pull_outcome': 'prev_trial_outcome',
                'socialgaze_auc': 'self_gaze_auc',
                'mass_move_speed_mean': 'self_mean_speed',
                'mass_move_speed_std': 'self_speed_std',
                'other_lever_speed_mean': 'partner_mean_speed',
                'other_lever_speed_std': 'partner_speed_std',
                'num_preceding_failpull': 'failed_pulls_before_reward',
                'time_from_last_reward': 'time_since_last_reward',
                }
    elif dobhvPCA:
        rename_dict = {
            'act_animal':'subj_idx',
            'dates': 'date',
            'pull_rt': 'rt',
            'previous_pull_outcome': 'prev_trial_outcome',
            'socialgaze_auc': 'self_gaze_auc',
            # 'self_PC1_mean': 'self_mean_speed',
            # 'self_PC1_std': 'self_speed_std',
            'mass_move_speed_mean': 'self_mean_speed',
            'mass_move_speed_std': 'self_speed_std',
            'other_PC1_mean': 'partner_mean_speed',
            'other_PC1_std': 'partner_speed_std',
            'num_preceding_failpull': 'failed_pulls_before_reward',
            'time_from_last_reward': 'time_since_last_reward',
            }
        
    # Apply the renaming
    hddm_data_tgt_alldates = hddm_data_tgt_alldates.rename(columns=rename_dict)
    hddm_data_tgt_alldates['response'] = 1
    
    # 
    # remove nan row first
    hddm_data_tgt_alldates = hddm_data_tgt_alldates.dropna()
    
    # 
    # normalize all the predictors
    from scipy.stats import zscore
    #
    # List of predictor columns to normalize (exclude 'rt' and identifiers)
    predictor_columns = [
        'self_gaze_auc',
        'partner_mean_speed',
        'self_mean_speed',
        'partner_speed_std',
        'self_speed_std',
        'failed_pulls_before_reward',
        'time_since_last_reward'
    ]
    # Rename originals and replace with z-scored values
    for col in predictor_columns:
        hddm_data_tgt_alldates[f"{col}_origin"] = hddm_data_tgt_alldates[col]
        hddm_data_tgt_alldates[col] = zscore(hddm_data_tgt_alldates[col])  
        # hddm_data_tgt_alldates[col] = hddm_data_tgt_alldates[col]  
      
    #
    # initialize a dataframe for summarizing all sessions results
    summary_list = []
    
    
    # run the fitting for each session
    for session_toana, hddm_data_tgt_idate in hddm_data_tgt_alldates.groupby('date'):
    
        ind_ = np.isin(dates_list,session_toana)
        task_cond = task_conditions[np.where(ind_)[0][0]]
        
        # Inside the loop:
        # 'session_toana' is the date for the current group
        # 'hddm_data_tgt_idate' is a DataFrame containing only the rows for that date
    
        #
        # # add some semi-artbitrary cretieras
        # hddm_data_tgt_alldates = hddm_data_tgt_idate[hddm_data_tgt_alldates['prev_trial_outcome']==1]
        # hddm_data_tgt_alldates = hddm_data_tgt_idate[hddm_data_tgt_alldates['partner_mean_speed']<=300]
        #
        lower_q = 0.05
        upper_q = 0.95
        rt_lower, rt_upper = hddm_data_tgt_idate['rt'].quantile([lower_q, upper_q])
        # hddm_data_tgt_idate = hddm_data_tgt_idate[hddm_data_tgt_idate['rt']>rt_lower]
        # hddm_data_tgt_idate = hddm_data_tgt_idate[hddm_data_tgt_idate['rt']<rt_upper]
        #
        tslr_lower, tslr_upper = hddm_data_tgt_idate['time_since_last_reward'].quantile([lower_q, upper_q])
        # hddm_data_tgt_idate = hddm_data_tgt_idate[hddm_data_tgt_idate['time_since_last_reward']>tslr_lower]
        # hddm_data_tgt_idate = hddm_data_tgt_idate[hddm_data_tgt_idate['time_since_last_reward']<tslr_upper]
        
        
        # try to load the analyzed
        try:

            # dummpy
            
            current_dir = data_saved_folder+'data_saved_singlecam_wholebody_neural_and_hddm'+savefile_sufix+'/'+cameraID+'/'+animal_tgt
            add_date_dir = os.path.join(current_dir+'/hddm_model_fitted_allsession_withNeurons/'+session_toana)

            #
            with open(add_date_dir+'/hddm_model_fitted_nogaze_gazeInteract_traces_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
                hddm_model_fitted_nogaze_gazeInteract_traces = pickle.load(f)

            with open(add_date_dir+'/hddm_model_fitted_nogaze_gazeInteract_stats_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
                hddm_model_fitted_nogaze_gazeInteract_stats = pickle.load(f)

            with open(add_date_dir+'/hddm_model_fitted_nogaze_gazeInteract_dic_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
                hddm_model_fitted_nogaze_gazeInteract_dic = pickle.load(f)
                
            

            print(animal_tgt+' '+cond_toplot_type+' hddm fitting results loaded')

        except:

            print(animal_tgt+' '+cond_toplot_type+' running hddm fitting')
            
            try:

                # run the hddm nodel
                # HDDM settings - the basic setting
                # samples=500 # 2000
                # burn=100
                # thin=2
                samples=200 # 2000
                burn=100
                thin=2
                #
                doNogazeOnly = 1
                #
                if dobhvPCA:
                    _, hddm_model_fitted_nogaze_gazeInteract = run_hddm_modeling_bhvPCA_gazeInteract(hddm_data_tgt_idate, animal_tgt,
                                                                        samples, burn, thin, doNogazeOnly)
                elif not dobhvPCA:
                    _, hddm_model_fitted_nogaze_gazeInteract = run_hddm_modeling_gazeInteract(hddm_data_tgt_idate, animal_tgt,
                                                                        samples, burn, thin, doNogazeOnly)

                #
                hddm_model_fitted_nogaze_gazeInteract_traces = hddm_model_fitted_nogaze_gazeInteract.get_traces()
                hddm_model_fitted_nogaze_gazeInteract_stats  = hddm_model_fitted_nogaze_gazeInteract.gen_stats()
                hddm_model_fitted_nogaze_gazeInteract_dic = hddm_model_fitted_nogaze_gazeInteract.dic_info
            
            except:
                continue

                
            # save data
            if 0:
                current_dir = data_saved_folder+'data_saved_singlecam_wholebody_neural_and_hddm'+savefile_sufix+'/'+cameraID+'/'+animal_tgt
                add_date_dir = os.path.join(current_dir+'/hddm_model_fitted_allsession_withNeurons/'+session_toana)
                if not os.path.exists(add_date_dir):
                    os.makedirs(add_date_dir)
                #
                with open(add_date_dir+'/hddm_model_fitted_nogaze_gazeInteract_traces_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'wb') as f:
                    pickle.dump(hddm_model_fitted_nogaze_gazeInteract_traces, f)

                with open(add_date_dir+'/hddm_model_fitted_nogaze_gazeInteract_stats_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'wb') as f:
                    pickle.dump(hddm_model_fitted_nogaze_gazeInteract_stats, f)  

                with open(add_date_dir+'/hddm_model_fitted_nogaze_gazeInteract_dic_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'wb') as f:
                    pickle.dump(hddm_model_fitted_nogaze_gazeInteract_dic, f)
            
          
        #
        # reconstruct V
        #
        df_combined = hddm_data_tgt_idate # your combined trial-level dataframe
        traces = hddm_model_fitted_nogaze_gazeInteract_traces  # or use `hddm_model_fitted_traces` if needed
        stats = hddm_model_fitted_nogaze_gazeInteract_stats


        # --- STEP 1: Use all posterior samples to compute predicted v for each trial ---
        n_samples = len(traces)
        n_trials = len(df_combined)

        # Allocate space to hold predicted v for each sample x trial
        predicted_v_samples = np.zeros((n_samples, n_trials))

        # Pre-fetch covariate values as numpy arrays
        x1 = df_combined['partner_mean_speed'].values
        x2 = df_combined['partner_speed_std'].values
        x3 = df_combined['self_mean_speed'].values
        # x4 = df_combined['self_speed_std'].values
        # x3 = df_combined['failed_pulls_before_reward'].values
        # x3 = df_combined['time_since_last_reward'].values
        x7 = df_combined['self_gaze_auc'].values
        

        for i in range(n_samples):
            intercept = traces['v_Intercept'].values[i]
            coef_x1 = traces['v_partner_mean_speed'].values[i]
            coef_x2 = traces['v_partner_speed_std'].values[i]
            coef_x3 = traces['v_self_mean_speed'].values[i]
            # coef_x4 = traces['v_self_speed_std'].values[i]
            # coef_x3 = traces['v_failed_pulls_before_reward'].values[i]
            # coef_x3 = traces['v_time_since_last_reward'].values[i]
            coef_x7 = traces['v_self_gaze_auc'].values[i]
            coef_x71 = traces['v_self_gaze_auc:partner_mean_speed'].values[i]
            coef_x72 = traces['v_self_gaze_auc:partner_speed_std'].values[i]
            
            

            # Apply regression formula for all trials
            predicted_v_samples[i, :] = (
                # intercept + coef_x1 * x1 + coef_x2 * x2 + coef_x3 * x3
                # intercept + coef_x1 * x1 
                # intercept + coef_x2 * x2 
                # intercept + coef_x1 * x1 + coef_x2 * x2
                # intercept + coef_x1 * x1 + coef_x3 * x3
                # intercept + coef_x1 * x1 + coef_x2 * x2 + coef_x3 * x3 + coef_x4 * x4
                intercept + coef_x1 * x1 + coef_x2 * x2 + coef_x3 * x3  + coef_x7 * x7 +
                            coef_x71 * x7 * x1 + coef_x71 * x7 *x2
                # intercept + coef_x2 * x2 + coef_x4 * x4
            )

        # --- STEP 2: Compute posterior mean (and optionally CI) of predicted v ---
        df_with_v = df_combined.copy()
        df_with_v['predicted_v'] = predicted_v_samples.mean(axis=0)

        # Optional: also store credible intervals (e.g., 95% CI)
        df_with_v['predicted_v_lower'] = np.percentile(predicted_v_samples, 2.5, axis=0)
        df_with_v['predicted_v_upper'] = np.percentile(predicted_v_samples, 97.5, axis=0) 
            
        
            
        #
        # save the summaring data across date   
        #
        # --- Process the model data ---
        # --- Task 1: Process the HDDM model coefficients (same as before) ---
        stats['significance'] = (stats['2.5q'] * \
                                 stats['97.5q'] > 0).astype(int)
        session_summary = stats.reset_index().rename(columns={'index': 'coefficient'})
        session_summary['date'] = session_toana
        session_summary['condition'] = task_cond
        session_summary = session_summary[['date', 'condition', 'coefficient', 'mean', 'std', 'significance']]
        summary_list.append(session_summary)


        # --- Task 2: Calculate and add the v-rt correlation ---
        # Calculate the Pearson correlation coefficient and the p-value.
        corr_coef, p_value = st.pearsonr(df_with_v['predicted_v'], df_with_v['rt'])
        #
        # Create a small DataFrame for the correlation result.
        # We use the 'mean' column for the coefficient and 'significance' for the p-value
        # to keep the data structure consistent.
        corr_summary = pd.DataFrame([{
            'date': session_toana,
            'condition':task_cond,
            'coefficient': 'predicted_v_rt_correlation',
            'mean': corr_coef,
            'std': np.nan,  # Standard deviation isn't applicable here
            'significance': (p_value<0.05).astype(int)
        }])
        #
        # Add the correlation result to the list.
        summary_list.append(corr_summary)
        
        # with mean firing rate
        corr_coef, p_value = st.pearsonr(df_with_v['predicted_v'], df_with_v['fr_mean'])
        #
        # Create a small DataFrame for the correlation result.
        # We use the 'mean' column for the coefficient and 'significance' for the p-value
        # to keep the data structure consistent.
        corr_summary = pd.DataFrame([{
            'date': session_toana,
            'condition':task_cond,
            'coefficient': 'predicted_v_fr_mean_correlation',
            'mean': corr_coef,
            'std': np.nan,  # Standard deviation isn't applicable here
            'significance': (p_value<0.05).astype(int)
        }])
        #
        # Add the correlation result to the list.
        summary_list.append(corr_summary)
        
         # with slope firing rate
        corr_coef, p_value = st.pearsonr(df_with_v['predicted_v'], df_with_v['fr_slope'])
        #
        # Create a small DataFrame for the correlation result.
        # We use the 'mean' column for the coefficient and 'significance' for the p-value
        # to keep the data structure consistent.
        corr_summary = pd.DataFrame([{
            'date': session_toana,
            'condition':task_cond,
            'coefficient': 'predicted_v_fr_slope_correlation',
            'mean': corr_coef,
            'std': np.nan,  # Standard deviation isn't applicable here
            'significance': (p_value<0.05).astype(int)
        }])
        #
        # Add the correlation result to the list.
        summary_list.append(corr_summary)
    
    
    
    # Combine the list of DataFrames into the final summary DataFrame.
    hddm_model_fitted_alldates_summary = pd.concat(summary_list, ignore_index=True)
    
    ## 
    # do some plotting
    ##
    plot_data = hddm_model_fitted_alldates_summary.copy()
    
    # ---- NEW: Filter out the coefficients you don't want to see ----
    coeffs_to_exclude = ['t', 'z', 'a', 'v_Intercept', 'a_Intercept','a_time_since_last_reward',
                         'predicted_v_fr_mean_correlation','predicted_v_fr_slope_correlation']
    #
    do_v_and_FR = 0
    if do_v_and_FR:
        coeffs_to_exclude = ['t', 'z', 'v_Intercept', 'a_Intercept','a_time_since_last_reward',
                             'v_self_gaze_auc', 'v_partner_mean_speed','v_partner_speed_std',
                             'v_self_mean_speed','v_self_speed_std',]
    #
    plot_data = plot_data[~plot_data['coefficient'].isin(coeffs_to_exclude)]

    plot_data['mean'] = pd.to_numeric(plot_data['mean'], errors='coerce')
    
    # ---- Remove outliers for each coefficient using the IQR method ----
    # For each coefficient, calculate the bounds
    Q1 = plot_data.groupby('coefficient')['mean'].transform(lambda x: x.quantile(0.25))
    Q3 = plot_data.groupby('coefficient')['mean'].transform(lambda x: x.quantile(0.75))
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    #
    # Filter out the outliers
    plot_data_no_outliers = plot_data[(plot_data['mean'] >= lower_bound) & (plot_data['mean'] <= upper_bound)]
    # plot_data_no_outliers = plot_data
    plot_data_no_outliers = plot_data_no_outliers.sort_values(by='coefficient')

    # # remove some unwanted condition
    # ind_dontwant = (plot_data_no_outliers['condition']=='MC_withGingerNew')|\
    #         (plot_data_no_outliers['condition']=='MC_withKoala')|\
    #         (plot_data_no_outliers['condition']=='MC_withVermelho')
    # ind_want = plot_data_no_outliers['condition']=='MC'
    # plot_data_no_outliers = plot_data_no_outliers[~ind_dontwant]
    # # plot_data_no_outliers = plot_data_no_outliers[ind_want]
    
    
    
    # --- Plotting Code (using the outlier-free data) ---
    plt.figure(figsize=(14, 8))

    # Create the swarm plot
    # We use 'hue' to color points based on the 'significance' column for that date.
    ax = sns.swarmplot(
        x='coefficient',
        y='mean',
        data=plot_data_no_outliers,
        hue='significance',
        palette={0: 'gray', 1: 'red'},  # 0=Not Significant (gray), 1=Significant (red)
        size=8
    )
    # Hide the legend as the colors are intuitive
    ax.get_legend().remove()


    # --- Add Overall Significance Asterisks ---
    unique_coeffs = plot_data_no_outliers['coefficient'].unique()
    for i, coef in enumerate(unique_coeffs):
        
        coef_data_all_df = plot_data_no_outliers[plot_data_no_outliers['coefficient'] == coef]
        
        if 0: # this part is using the lmm
            # Ensure there's enough data to run the model
            if len(coef_data_all_df) > 1:
                p_value = 1.0 # Default to non-significant

                # print(coef_data_all_df.groupby('condition').size())

                # Use a try/except block to handle cases where the LMM might not converge
                try:
                    # --- START: LMM Replacement ---
                    # Define the LMM: model the 'mean' with an intercept, grouped by 'condition'
                    model = smf.mixedlm("mean ~ 1", coef_data_all_df, groups=coef_data_all_df['condition'])

                    # Fit the model (disp=False prevents printing convergence messages in the loop)
                    result = model.fit(disp=False)

                    # Get the p-value for the intercept (our test against 0)
                    p_value = result.pvalues['Intercept']
                    print(p_value)
                    
                    # --- END: LMM Replacement ---

                except Exception as e:
                    # If the model fails, print a warning and keep p_value at 1.0
                    print(f"LMM failed for coefficient '{coef}'. Error: {e}")


                # The rest of your plotting logic remains the same
                if p_value < 0.05:
                    # Use the 'mean' column from the subsetted dataframe for positioning
                    y_pos = coef_data_all_df['mean'].max()
                    offset = coef_data_all_df['mean'].std() * 0.1
                    plt.text(i, y_pos + offset, '*', ha='center', va='bottom', fontsize=24, color='black')

        
        if 1: # this part is using the simple ttest/wilcoxon
            coef_data = plot_data_no_outliers[plot_data_no_outliers['coefficient'] == coef]['mean']

            if len(coef_data) > 1:
                t_stat, p_value = ttest_1samp(coef_data, 0, nan_policy='omit')
                # w_stat, p_value = st.wilcoxon(coef_data, nan_policy='omit')
                print(p_value)
                if p_value < 0.05:
                    y_pos = coef_data.max()
                    offset = plot_data_no_outliers['mean'].std() * 0.1
                    plt.text(i, y_pos + offset, '*', ha='center', va='bottom', fontsize=24, color='black')

    plt.axhline(y=0, linestyle='--', color='black', linewidth=1.5)            
    
    # --- Finalize and Save Plot ---
    plt.title('Model Coefficients Across All Dates', fontsize=18)
    plt.ylabel('Coefficient Value (Mean)', fontsize=16)
    plt.xlabel('Model Coefficient', fontsize=16)
    plt.xticks(rotation=45, ha='right', fontsize = 16)
    plt.tight_layout()

     # savefig
    savefig = 1
    if savefig:
        current_dir = data_saved_folder+'fig_for_basic_neural_analysis_allsessions_basicEvents_PCA_Pullfocused_continuousBhv_partnerDistVaris_with_hddm_model'+savefile_sufix+'/'+animal_tgt
        add_fig_dir = os.path.join(current_dir,cameraID+'/fig_for_hddm_model_fitted_allsession_withNeurons')
        #
        if not os.path.exists(add_fig_dir):
            os.makedirs(add_fig_dir)
        if not do_v_and_FR:
            plt.savefig(add_fig_dir+'/predicted_v_and_other_bhv_variables_nogaze_gazeInteract_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pdf', 
                        format='pdf', bbox_inches='tight')
        elif do_v_and_FR:
            plt.savefig(add_fig_dir+'/predicted_v_and_FR_nogaze_gazeInteract_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pdf', 
                        format='pdf', bbox_inches='tight')


In [ ]:
np.mean(hddm_model_fitted_alldates_summary[hddm_model_fitted_alldates_summary['coefficient']=='t']['mean'])

In [ ]:
doOnsetAfterMin_suffix

In [ ]:
# do a summarizing model comparison plot

if 0: # useful code, turn off for now
    
    # load the dataset first
    current_dir = data_saved_folder+'data_saved_singlecam_wholebody_neural_and_hddm'+savefile_sufix+'/'+cameraID+'/'+animal_tgt
    add_date_dir = os.path.join(current_dir+'/hddm_model_fitted_combinedsession_withNeurons')
    
    if cond_toplot_type == 'allMC':
        if not (doOnsetAfterMin_suffix == 'bhvPCA_PullOnsetAfterMinBasedonSpeed_') |\
               (doOnsetAfterMin_suffix == 'PullOnsetAfterMin_PartnerLeverSpeed_'):
            with open(add_date_dir+'/hddm_datas_flexibleTW_newRTdefinition_'+doOnsetAfterMin_suffix+'withFRs_all_dates_'+
                      animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
                hddm_data_tgt_alldates = pickle.load(f)
        #
        elif (doOnsetAfterMin_suffix == 'bhvPCA_PullOnsetAfterMinBasedonSpeed_') |\
             (doOnsetAfterMin_suffix == 'PullOnsetAfterMin_PartnerLeverSpeed_'):
            with open(add_date_dir+'/hddm_datas_flexibleTW_newRTdefinition_'+'PullOnsetAfterMin_'+'withFRs_all_dates_'+
                      animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
                hddm_data_tgt_alldates = pickle.load(f)
    #
    else:
        if not (doOnsetAfterMin_suffix == 'bhvPCA_PullOnsetAfterMinBasedonSpeed_') |\
               (doOnsetAfterMin_suffix == 'PullOnsetAfterMin_PartnerLeverSpeed_'):
            with open(add_date_dir+'/hddm_datas_flexibleTW_newRTdefinition_'+doOnsetAfterMin_suffix+'withFRs_all_dates_'+
                      animal1_fixedorders[0]+animal2_fixedorders[0]+'_'+cond_toplot_type+'.pkl', 'rb') as f:
                hddm_data_tgt_alldates = pickle.load(f)
        #
        elif (doOnsetAfterMin_suffix == 'bhvPCA_PullOnsetAfterMinBasedonSpeed_') |\
             (doOnsetAfterMin_suffix == 'PullOnsetAfterMin_PartnerLeverSpeed_'):
            with open(add_date_dir+'/hddm_datas_flexibleTW_newRTdefinition_'+'PullOnsetAfterMin_'+'withFRs_all_dates_'+
                      animal1_fixedorders[0]+animal2_fixedorders[0]+'_'+cond_toplot_type+'.pkl', 'rb') as f:
                hddm_data_tgt_alldates = pickle.load(f)
            

    # make the hddm_data structure consistent with other code
    ind_goodcond = np.isin(hddm_data_tgt_alldates['condition'],conditions_to_ana)
    hddm_data_tgt_alldates = hddm_data_tgt_alldates[ind_goodcond]
    
    # 
    # remove nan row first
    hddm_data_tgt_alldates = hddm_data_tgt_alldates.dropna()
    
    #
    # initialize a dataframe for summarizing all sessions results
    DIC_session_summary = []
    
    # run the fitting for each session
    for session_toana, hddm_data_tgt_idate in hddm_data_tgt_alldates.groupby('dates'):
        
        session_summary = dict.fromkeys(['date','condition','nogaze','withgaze','gazefilter','gazefilter_ZHistOut',
                                         'gazeInteract',
                                         'nogazeDim','withgazeDim','gazefilterDim','gazefilterZHistOutDim',
                                         'gazeInteractDim'])
        
        # print(session_toana)
        
        ind_ = np.isin(dates_list,session_toana)
        task_cond = task_conditions[np.where(ind_)[0][0]]
        
        current_dir = data_saved_folder+'data_saved_singlecam_wholebody_neural_and_hddm'+savefile_sufix+'/'+cameraID+'/'+animal_tgt
        add_date_dir = os.path.join(current_dir+'/hddm_model_fitted_allsession_withNeurons/'+session_toana)
        #
        with open(add_date_dir+'/hddm_model_fitted_nogaze_dic_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
            hddm_model_fitted_nogaze_dic = pickle.load(f)
        with open(add_date_dir+'/hddm_model_fitted_nogaze_stats_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
            hddm_model_fitted_nogaze_stats = pickle.load(f)
            
        with open(add_date_dir+'/hddm_model_fitted_dic_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
            hddm_model_fitted_dic = pickle.load(f)
        with open(add_date_dir+'/hddm_model_fitted_stats_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
            hddm_model_fitted_stats = pickle.load(f)
            
        with open(add_date_dir+'/hddm_model_fitted_nogaze_gazefilter_dic_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
            hddm_model_fitted_nogaze_gazefilter_dic = pickle.load(f)
        with open(add_date_dir+'/hddm_model_fitted_nogaze_gazefilter_stats_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
            hddm_model_fitted_nogaze_gazefilter_stats = pickle.load(f)
            
        with open(add_date_dir+'/hddm_model_fitted_nogaze_gazefilter_ZHistOut_dic_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
            hddm_model_fitted_nogaze_gazefilter_ZHistOut_dic = pickle.load(f)
        with open(add_date_dir+'/hddm_model_fitted_nogaze_gazefilter_ZHistOut_stats_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
            hddm_model_fitted_nogaze_gazefilter_ZHistOut_stats = pickle.load(f)
        
        with open(add_date_dir+'/hddm_model_fitted_nogaze_gazeInteract_dic_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
            hddm_model_fitted_nogaze_gazeInteract_dic = pickle.load(f)
        with open(add_date_dir+'/hddm_model_fitted_nogaze_gazeInteract_stats_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
            hddm_model_fitted_nogaze_gazeInteract_stats = pickle.load(f)
    
        #
        # put data together
        session_summary['date'] = session_toana
        session_summary['condition'] = task_cond
        session_summary['nogaze'] = hddm_model_fitted_nogaze_dic['DIC']
        session_summary['withgaze'] = hddm_model_fitted_dic['DIC']
        session_summary['gazefilter'] = hddm_model_fitted_nogaze_gazefilter_dic['DIC']
        session_summary['gazefilter_ZHistOut'] = hddm_model_fitted_nogaze_gazefilter_ZHistOut_dic['DIC']
        session_summary['gazeInteract'] = hddm_model_fitted_nogaze_gazeInteract_dic['DIC']
        #
        session_summary['nogazeDim'] = np.shape(hddm_model_fitted_nogaze_stats)[0]
        session_summary['withgazeDim'] = np.shape(hddm_model_fitted_stats)[0]
        session_summary['gazefilterDim'] = np.shape(hddm_model_fitted_nogaze_gazefilter_stats)[0]
        session_summary['gazefilterZHistOutDim'] = np.shape(hddm_model_fitted_nogaze_gazefilter_ZHistOut_stats)[0]
        session_summary['gazeInteractDim'] = np.shape(hddm_model_fitted_nogaze_gazeInteract_stats)[0]
        #
        DIC_session_summary.append(session_summary)

    DIC_session_summary = pd.DataFrame.from_dict(DIC_session_summary)

    # 
    # do the plot
    #
    df = DIC_session_summary.copy()
    #
    # Calculate the difference scores relative to the 'gazefilter' model
    df['nogaze_diff'] = df['nogaze'] - df['gazefilter']
    df['withgaze_diff'] = df['withgaze'] - df['gazefilter']
    df['gazeInteract_diff'] = df['gazeInteract'] - df['gazefilter']
    df['gazefilter_diff'] = df['gazefilter'] - df['gazefilter']
    df['gazefilter_ZHistOut_diff'] = df['gazefilter_ZHistOut'] - df['gazefilter']
    
    # --- Optional Outlier Removal Step ---
    # uncomment this section to remove outliers
    if 1:
        def remove_outliers_iqr(series):
            Q1 = series.quantile(0.25)
            Q3 = series.quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR
            return series[(series >= lower_bound) & (series <= upper_bound)]

        df['nogaze_diff'] = remove_outliers_iqr(df['nogaze_diff'])
        df['withgaze_diff'] = remove_outliers_iqr(df['withgaze_diff'])
        df['gazefilter_ZHistOut_diff'] = remove_outliers_iqr(df['gazefilter_ZHistOut_diff'])
        df['gazeInteract_diff'] = remove_outliers_iqr(df['gazeInteract_diff'])
    # ------------------------------------

    # Create a DataFrame with the difference values in a long format, suitable for seaborn
    df_long_diff = pd.melt(df, id_vars=['date'],
                    value_vars=['nogaze_diff', 'withgaze_diff', 'gazeInteract_diff', 
                                'gazefilter_diff','gazefilter_ZHistOut_diff'],
                    var_name='model_difference', value_name='DIC_difference')

    # Create the violin plot
    plt.figure(figsize=(10, 6))
    # A horizontal line at 0 for visual reference
    plt.axhline(y=0, color='red', linestyle='--', linewidth=1.5, alpha=0.7)
    sns.violinplot(x='model_difference', y='DIC_difference', data=df_long_diff, inner='quartile')

    # Calculate medians and Wilcoxon p-values for annotation
    if 0: # as indicated, use wilcoxon
        medians = df_long_diff.groupby('model_difference')['DIC_difference'].median()
        wilcoxon_pvalues = {
            'nogaze_diff': st.wilcoxon(df['nogaze_diff'].dropna()).pvalue,
            'withgaze_diff': st.wilcoxon(df['withgaze_diff'].dropna()).pvalue,
            'gazeInteract_diff': st.wilcoxon(df['gazeInteract_diff'].dropna()).pvalue,
            'gazefilter_ZHistOut_diff': st.wilcoxon(df['gazefilter_ZHistOut_diff'].dropna()).pvalue
        }
    else: # use ttest
        medians = df_long_diff.groupby('model_difference')['DIC_difference'].mean()
        wilcoxon_pvalues = {
            'nogaze_diff': st.ttest_1samp(df['nogaze_diff'].dropna(),0).pvalue,
            'withgaze_diff': st.ttest_1samp(df['withgaze_diff'].dropna(),0).pvalue,
            'gazeInteract_diff': st.ttest_1samp(df['gazeInteract_diff'].dropna(),0).pvalue,
            'gazefilter_ZHistOut_diff': st.ttest_1samp(df['gazefilter_ZHistOut_diff'].dropna(),0).pvalue
        }
        

    # Add annotations for median and Wilcoxon p-value to each violin plot
    plt.text(0, medians['nogaze_diff'] - 5, f"Median: {medians['nogaze_diff']:.2f}\np={wilcoxon_pvalues['nogaze_diff']:.3f}", 
             ha='center', va='bottom', fontsize=10, color='black', fontweight='bold')
    plt.text(1, medians['withgaze_diff'] - 5, f"Median: {medians['withgaze_diff']:.2f}\np={wilcoxon_pvalues['withgaze_diff']:.3f}", 
             ha='center', va='bottom', fontsize=10, color='black', fontweight='bold')
    plt.text(2, medians['gazeInteract_diff'] - 5, f"Median: {medians['gazeInteract_diff']:.2f}\np={wilcoxon_pvalues['gazeInteract_diff']:.3f}", 
             ha='center', va='bottom', fontsize=10, color='black', fontweight='bold')
    plt.text(4, medians['gazefilter_ZHistOut_diff'] - 5, f"Median: {medians['gazefilter_ZHistOut_diff']:.2f}\np={wilcoxon_pvalues['gazefilter_ZHistOut_diff']:.3f}", 
             ha='center', va='bottom', fontsize=10, color='black', fontweight='bold')
    plt.title('Distribution of DIC Differences Relative to gazefilter Model')
    plt.ylabel('DIC Difference (Model - gazefilter)')
    plt.xlabel('Model Comparison')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    
    # savefig
    savefig = 0
    if savefig:
        current_dir = data_saved_folder+'fig_for_basic_neural_analysis_allsessions_basicEvents_PCA_Pullfocused_continuousBhv_partnerDistVaris_with_hddm_model'+savefile_sufix+'/'+animal_tgt
        add_fig_dir = os.path.join(current_dir,cameraID+'/fig_for_hddm_model_fitted_allsession_withNeurons')
        #
        if not os.path.exists(add_fig_dir):
            os.makedirs(add_fig_dir)
        
        plt.savefig(add_fig_dir+'/'+animal_tgt+'_Distribution_of_DIC_Differences_Relative_to_gazefilter_Model_'+
                    cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pdf', 
                    format='pdf', bbox_inches='tight')

        
    # print(DIC_session_summary)

In [ ]:
# do a summarizing model comparison plot

if 1:  # useful code, turn off for now
    
    doOnsetAfterMin_suffix_PC1 = 'bhvPCA_PullOnsetAfterMinBasedonSpeed_'
    doOnsetAfterMin_suffix_spd = 'PullOnsetAfterMin_'
    
    
    # load the dataset first
    current_dir = data_saved_folder+'data_saved_singlecam_wholebody_neural_and_hddm'+savefile_sufix+'/'+cameraID+'/'+animal_tgt
    add_date_dir = os.path.join(current_dir+'/hddm_model_fitted_combinedsession_withNeurons')
    # 
    if cond_toplot_type == 'allMC':
        if not (doOnsetAfterMin_suffix == 'bhvPCA_PullOnsetAfterMinBasedonSpeed_') |\
               (doOnsetAfterMin_suffix == 'PullOnsetAfterMin_PartnerLeverSpeed_'):
            with open(add_date_dir+'/hddm_datas_flexibleTW_newRTdefinition_'+doOnsetAfterMin_suffix+'withFRs_all_dates_'+
                      animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
                hddm_data_tgt_alldates = pickle.load(f)
        #
        elif (doOnsetAfterMin_suffix == 'bhvPCA_PullOnsetAfterMinBasedonSpeed_') |\
             (doOnsetAfterMin_suffix == 'PullOnsetAfterMin_PartnerLeverSpeed_'):
            with open(add_date_dir+'/hddm_datas_flexibleTW_newRTdefinition_'+'PullOnsetAfterMin_'+'withFRs_all_dates_'+
                      animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
                hddm_data_tgt_alldates = pickle.load(f)
    #
    else:
        if not (doOnsetAfterMin_suffix == 'bhvPCA_PullOnsetAfterMinBasedonSpeed_') |\
               (doOnsetAfterMin_suffix == 'PullOnsetAfterMin_PartnerLeverSpeed_'):
            with open(add_date_dir+'/hddm_datas_flexibleTW_newRTdefinition_'+doOnsetAfterMin_suffix+'withFRs_all_dates_'+
                      animal1_fixedorders[0]+animal2_fixedorders[0]+'_'+cond_toplot_type+'.pkl', 'rb') as f:
                hddm_data_tgt_alldates = pickle.load(f)
        #
        elif (doOnsetAfterMin_suffix == 'bhvPCA_PullOnsetAfterMinBasedonSpeed_') |\
             (doOnsetAfterMin_suffix == 'PullOnsetAfterMin_PartnerLeverSpeed_'):
            with open(add_date_dir+'/hddm_datas_flexibleTW_newRTdefinition_'+'PullOnsetAfterMin_'+'withFRs_all_dates_'+
                      animal1_fixedorders[0]+animal2_fixedorders[0]+'_'+cond_toplot_type+'.pkl', 'rb') as f:
                hddm_data_tgt_alldates = pickle.load(f)
            

    # make the hddm_data structure consistent with other code
    ind_goodcond = np.isin(hddm_data_tgt_alldates['condition'],conditions_to_ana)
    hddm_data_tgt_alldates = hddm_data_tgt_alldates[ind_goodcond]
    
    # 
    # remove nan row first
    hddm_data_tgt_alldates = hddm_data_tgt_alldates.dropna()
    
    #
    # initialize a dataframe for summarizing all sessions results
    DIC_session_summary = []
    
    # run the fitting for each session
    for session_toana, hddm_data_tgt_idate in hddm_data_tgt_alldates.groupby('dates'):
        
        session_summary = dict.fromkeys(['date','condition',])
        
        # print(session_toana)
        
        ind_ = np.isin(dates_list,session_toana)
        task_cond = task_conditions[np.where(ind_)[0][0]]
        
        current_dir = data_saved_folder+'data_saved_singlecam_wholebody_neural_and_hddm'+savefile_sufix+'/'+cameraID+'/'+animal_tgt
        add_date_dir = os.path.join(current_dir+'/hddm_model_fitted_allsession_withNeurons/'+session_toana)
        #
            
        with open(add_date_dir+'/hddm_model_fitted_nogaze_gazefilter_dic_'+cond_toplot_type+doOnsetAfterMin_suffix_PC1+\
                  exaustModel_sufix+'.pkl', 'rb') as f:
            hddm_model_fitted_nogaze_gazefilter_PC1_dic = pickle.load(f)
        with open(add_date_dir+'/hddm_model_fitted_nogaze_gazefilter_stats_'+cond_toplot_type+doOnsetAfterMin_suffix_PC1+\
                  exaustModel_sufix+'.pkl', 'rb') as f:
            hddm_model_fitted_nogaze_gazefilter_PC1_stats = pickle.load(f)
            
        with open(add_date_dir+'/hddm_model_fitted_nogaze_gazefilter_dic_'+cond_toplot_type+doOnsetAfterMin_suffix_spd+\
                  exaustModel_sufix+'.pkl', 'rb') as f:
            hddm_model_fitted_nogaze_gazefilter_spd_dic = pickle.load(f)
        with open(add_date_dir+'/hddm_model_fitted_nogaze_gazefilter_stats_'+cond_toplot_type+doOnsetAfterMin_suffix_spd+\
                  exaustModel_sufix+'.pkl', 'rb') as f:
            hddm_model_fitted_nogaze_gazefilter_spd_stats = pickle.load(f)
            
        
    
        #
        # put data together
        session_summary['date'] = session_toana
        session_summary['condition'] = task_cond
        #
        session_summary['gazefilter_PC1'] = hddm_model_fitted_nogaze_gazefilter_PC1_dic['DIC']
        session_summary['gazefilter_spd'] = hddm_model_fitted_nogaze_gazefilter_spd_dic['DIC']
        #
        session_summary['gazefilter_PC1_PD'] = hddm_model_fitted_nogaze_gazefilter_PC1_dic['pD']
        session_summary['gazefilter_spd_PD'] = hddm_model_fitted_nogaze_gazefilter_spd_dic['pD']
        #
        session_summary['gazefilter_PC1Dim'] = np.shape(hddm_model_fitted_nogaze_gazefilter_PC1_stats)[0]
        session_summary['gazefilter_spdDim'] = np.shape(hddm_model_fitted_nogaze_gazefilter_spd_stats)[0]

        #
        DIC_session_summary.append(session_summary)

    DIC_session_summary = pd.DataFrame.from_dict(DIC_session_summary)
     
    # 
    # Remove pd < 0
    if 1:
        ind_bad = (DIC_session_summary['gazefilter_PC1_PD']<0) | \
                  (DIC_session_summary['gazefilter_spd_PD']<0)
        #
        DIC_session_summary = DIC_session_summary[~ind_bad]
    
    # 
    # do the plot
    #
    df = DIC_session_summary.copy()
    #
    # Calculate the difference scores relative to the 'gazefilter' model
    df['gazefilter_PC1_spd_diff'] = df['gazefilter_spd'] - df['gazefilter_PC1']
    
    # --- Optional Outlier Removal Step ---
    # uncomment this section to remove outliers
    if 1:
        def remove_outliers_iqr(series):
            Q1 = series.quantile(0.25)
            Q3 = series.quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR
            return series[(series >= lower_bound) & (series <= upper_bound)]

        df['gazefilter_PC1_spd_diff'] = remove_outliers_iqr(df['gazefilter_PC1_spd_diff'])
    # ------------------------------------
    
    # LMM
    # --- MODEL FITTING ---
    # Fit the linear model that includes 'condition' as a factor
    confound_model = smf.ols('gazefilter_PC1_spd_diff ~ C(condition, Sum)', data=df).fit()
    # Print the summary
    print(confound_model.summary())
    

    # Create a DataFrame with the difference values in a long format, suitable for seaborn
    df_long_diff = pd.melt(df, id_vars=['date'],
                    value_vars=['gazefilter_PC1_spd_diff'],
                    var_name='model_difference', value_name='DIC_difference')

    # Create the violin plot
    plt.figure(figsize=(10, 6))
    # A horizontal line at 0 for visual reference
    plt.axhline(y=0, color='red', linestyle='--', linewidth=1.5, alpha=0.7)
    sns.violinplot(x='model_difference', y='DIC_difference', data=df_long_diff, inner='quartile')

    # Calculate medians and Wilcoxon p-values for annotation
    if 0: # as indicated, use wilcoxon
        medians = df_long_diff.groupby('model_difference')['DIC_difference'].median()
        wilcoxon_pvalues = {
            'gazefilter_PC1_spd_diff': st.wilcoxon(df['gazefilter_PC1_spd_diff'].dropna()).pvalue
        }
    else: # use ttest
        medians = df_long_diff.groupby('model_difference')['DIC_difference'].mean()
        wilcoxon_pvalues = {
            'gazefilter_PC1_spd_diff': st.ttest_1samp(df['gazefilter_PC1_spd_diff'].dropna(),0).pvalue
        }
        

    # Add annotations for median and Wilcoxon p-value to each violin plot
    plt.text(0, medians['gazefilter_PC1_spd_diff'] - 5, 
             f"Median: {medians['gazefilter_PC1_spd_diff']:.2f}\np={wilcoxon_pvalues['gazefilter_PC1_spd_diff']:.3f}", 
             ha='center', va='bottom', fontsize=10, color='black', fontweight='bold')
    
    plt.title('Distribution of DIC Differences Relative to gazefilter Model')
    plt.ylabel('DIC Difference (Model - gazefilter)')
    plt.xlabel('Model Comparison')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    
    # savefig
    savefig = 0
    if savefig:
        current_dir = data_saved_folder+'fig_for_basic_neural_analysis_allsessions_basicEvents_PCA_Pullfocused_continuousBhv_partnerDistVaris_with_hddm_model'+savefile_sufix+'/'+animal_tgt
        add_fig_dir = os.path.join(current_dir,cameraID+'/fig_for_hddm_model_fitted_allsession_withNeurons')
        #
        if not os.path.exists(add_fig_dir):
            os.makedirs(add_fig_dir)
        
        plt.savefig(add_fig_dir+'/'+animal_tgt+'_Distribution_of_DIC_Differences_between_partnerPC1_vs_Speed_'+
                    cond_toplot_type+exaustModel_sufix+'.pdf', 
                    format='pdf', bbox_inches='tight')
    
    

In [ ]:
# do a summarizing model comparison plot

if 1:
    
    # load the dataset first
    current_dir = data_saved_folder+'data_saved_singlecam_wholebody_neural_and_hddm'+savefile_sufix+'/'+cameraID+'/'+animal_tgt
    add_date_dir = os.path.join(current_dir+'/hddm_model_fitted_combinedsession_withNeurons')
    
    if cond_toplot_type == 'allMC':
        if not (doOnsetAfterMin_suffix == 'bhvPCA_PullOnsetAfterMinBasedonSpeed_') |\
               (doOnsetAfterMin_suffix == 'PullOnsetAfterMin_PartnerLeverSpeed_'):
            with open(add_date_dir+'/hddm_datas_flexibleTW_newRTdefinition_'+doOnsetAfterMin_suffix+'withFRs_all_dates_'+
                      animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
                hddm_data_tgt_alldates = pickle.load(f)
        #
        elif (doOnsetAfterMin_suffix == 'bhvPCA_PullOnsetAfterMinBasedonSpeed_') |\
             (doOnsetAfterMin_suffix == 'PullOnsetAfterMin_PartnerLeverSpeed_'):
            with open(add_date_dir+'/hddm_datas_flexibleTW_newRTdefinition_'+'PullOnsetAfterMin_'+'withFRs_all_dates_'+
                      animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
                hddm_data_tgt_alldates = pickle.load(f)
    #
    else:
        if not (doOnsetAfterMin_suffix == 'bhvPCA_PullOnsetAfterMinBasedonSpeed_') |\
               (doOnsetAfterMin_suffix == 'PullOnsetAfterMin_PartnerLeverSpeed_'):
            with open(add_date_dir+'/hddm_datas_flexibleTW_newRTdefinition_'+doOnsetAfterMin_suffix+'withFRs_all_dates_'+
                      animal1_fixedorders[0]+animal2_fixedorders[0]+'_'+cond_toplot_type+'.pkl', 'rb') as f:
                hddm_data_tgt_alldates = pickle.load(f)
        #
        elif (doOnsetAfterMin_suffix == 'bhvPCA_PullOnsetAfterMinBasedonSpeed_') |\
             (doOnsetAfterMin_suffix == 'PullOnsetAfterMin_PartnerLeverSpeed_'):
            with open(add_date_dir+'/hddm_datas_flexibleTW_newRTdefinition_'+'PullOnsetAfterMin_'+'withFRs_all_dates_'+
                      animal1_fixedorders[0]+animal2_fixedorders[0]+'_'+cond_toplot_type+'.pkl', 'rb') as f:
                hddm_data_tgt_alldates = pickle.load(f)
            

    # make the hddm_data structure consistent with other code
    ind_goodcond = np.isin(hddm_data_tgt_alldates['condition'],conditions_to_ana)
    hddm_data_tgt_alldates = hddm_data_tgt_alldates[ind_goodcond]
    
    # 
    # remove nan row first
    hddm_data_tgt_alldates = hddm_data_tgt_alldates.dropna()
    
    #
    # initialize a dataframe for summarizing all sessions results
    DIC_session_summary = []
    
    # run the fitting for each session
    for session_toana, hddm_data_tgt_idate in hddm_data_tgt_alldates.groupby('dates'):
        
        session_summary = dict.fromkeys(['date','condition','nogaze','withgaze','gazefilter','gazefilter_ZHistOut',
                                         'gazeInteract',
                                         'nogazeDim','withgazeDim','gazefilterDim','gazefilterZHistOutDim',
                                         'gazeInteractDim'])
        
        # print(session_toana)
        
        ind_ = np.isin(dates_list,session_toana)
        task_cond = task_conditions[np.where(ind_)[0][0]]
        
        current_dir = data_saved_folder+'data_saved_singlecam_wholebody_neural_and_hddm'+savefile_sufix+'/'+cameraID+'/'+animal_tgt
        add_date_dir = os.path.join(current_dir+'/hddm_model_fitted_allsession_withNeurons/'+session_toana)
        #
        with open(add_date_dir+'/hddm_model_fitted_nogaze_dic_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
            hddm_model_fitted_nogaze_dic = pickle.load(f)
        with open(add_date_dir+'/hddm_model_fitted_nogaze_stats_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
            hddm_model_fitted_nogaze_stats = pickle.load(f)
            
        with open(add_date_dir+'/hddm_model_fitted_dic_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
            hddm_model_fitted_dic = pickle.load(f)
        with open(add_date_dir+'/hddm_model_fitted_stats_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
            hddm_model_fitted_stats = pickle.load(f)
            
        with open(add_date_dir+'/hddm_model_fitted_nogaze_gazefilter_dic_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
            hddm_model_fitted_nogaze_gazefilter_dic = pickle.load(f)
        with open(add_date_dir+'/hddm_model_fitted_nogaze_gazefilter_stats_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
            hddm_model_fitted_nogaze_gazefilter_stats = pickle.load(f)
            
        with open(add_date_dir+'/hddm_model_fitted_nogaze_gazefilter_ZHistOut_dic_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
            hddm_model_fitted_nogaze_gazefilter_ZHistOut_dic = pickle.load(f)
        with open(add_date_dir+'/hddm_model_fitted_nogaze_gazefilter_ZHistOut_stats_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
            hddm_model_fitted_nogaze_gazefilter_ZHistOut_stats = pickle.load(f)
        
        with open(add_date_dir+'/hddm_model_fitted_nogaze_gazeInteract_dic_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
            hddm_model_fitted_nogaze_gazeInteract_dic = pickle.load(f)
        with open(add_date_dir+'/hddm_model_fitted_nogaze_gazeInteract_stats_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
            hddm_model_fitted_nogaze_gazeInteract_stats = pickle.load(f)
    
        #
        # put data together
        session_summary['date'] = session_toana
        session_summary['condition'] = task_cond
        #
        session_summary['nogaze'] = hddm_model_fitted_nogaze_dic['DIC']
        session_summary['withgaze'] = hddm_model_fitted_dic['DIC']
        session_summary['gazefilter'] = hddm_model_fitted_nogaze_gazefilter_dic['DIC']
        session_summary['gazefilter_ZHistOut'] = hddm_model_fitted_nogaze_gazefilter_ZHistOut_dic['DIC']
        session_summary['gazeInteract'] = hddm_model_fitted_nogaze_gazeInteract_dic['DIC']
        #
        session_summary['nogazePD'] = hddm_model_fitted_nogaze_dic['pD']
        session_summary['withgazePD'] = hddm_model_fitted_dic['pD']
        session_summary['gazefilterPD'] = hddm_model_fitted_nogaze_gazefilter_dic['pD']
        session_summary['gazefilter_ZHistOutPD'] = hddm_model_fitted_nogaze_gazefilter_ZHistOut_dic['pD']
        session_summary['gazeInteractPD'] = hddm_model_fitted_nogaze_gazeInteract_dic['pD']
        #
        session_summary['nogazeDim'] = np.shape(hddm_model_fitted_nogaze_stats)[0]
        session_summary['withgazeDim'] = np.shape(hddm_model_fitted_stats)[0]
        session_summary['gazefilterDim'] = np.shape(hddm_model_fitted_nogaze_gazefilter_stats)[0]
        session_summary['gazefilterZHistOutDim'] = np.shape(hddm_model_fitted_nogaze_gazefilter_ZHistOut_stats)[0]
        session_summary['gazeInteractDim'] = np.shape(hddm_model_fitted_nogaze_gazeInteract_stats)[0]
        #
        DIC_session_summary.append(session_summary)

    DIC_session_summary = pd.DataFrame.from_dict(DIC_session_summary)

    # 
    # Remove pd < 0
    if 1:
        ind_bad = (DIC_session_summary['nogazePD']<0) | \
                  (DIC_session_summary['withgazePD']<0) | \
                  (DIC_session_summary['gazefilterPD']<0) | \
                  (DIC_session_summary['gazefilter_ZHistOutPD']<0) | \
                  (DIC_session_summary['gazeInteractPD']<0)
        #
        DIC_session_summary = DIC_session_summary[~ind_bad]
    
    
    # 
    # do the plot
    #
    df = DIC_session_summary.copy()
    #
    # Calculate the difference scores relative to the 'gazefilter' model
    df['nogaze_diff'] = df['nogaze'] - df['gazefilter']
    df['withgaze_diff'] = df['withgaze'] - df['gazefilter']
    df['gazeInteract_diff'] = df['gazeInteract'] - df['gazefilter']
    df['gazefilter_diff'] = df['gazefilter'] - df['gazefilter']
    df['gazefilter_ZHistOut_diff'] = df['gazefilter_ZHistOut'] - df['gazefilter']
    
    # --- Optional Outlier Removal Step ---
    # uncomment this section to remove outliers
    if 1:
        def remove_outliers_iqr(series):
            Q1 = series.quantile(0.25)
            Q3 = series.quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR
            return series[(series >= lower_bound) & (series <= upper_bound)]

        df['nogaze_diff'] = remove_outliers_iqr(df['nogaze_diff'])
        df['withgaze_diff'] = remove_outliers_iqr(df['withgaze_diff'])
        df['gazefilter_ZHistOut_diff'] = remove_outliers_iqr(df['gazefilter_ZHistOut_diff'])
        df['gazeInteract_diff'] = remove_outliers_iqr(df['gazeInteract_diff'])
    # ------------------------------------

    # Create a DataFrame with the difference values in a long format, suitable for seaborn
    df_long_diff = pd.melt(df, id_vars=['date'],
                    value_vars=['nogaze_diff', 'withgaze_diff', 'gazeInteract_diff', 
                                'gazefilter_diff','gazefilter_ZHistOut_diff'],
                    var_name='model_difference', value_name='DIC_difference')

    # Create the violin plot
    plt.figure(figsize=(10, 6))
    # A horizontal line at 0 for visual reference
    plt.axhline(y=0, color='red', linestyle='--', linewidth=1.5, alpha=0.7)
    sns.violinplot(x='model_difference', y='DIC_difference', data=df_long_diff, inner='quartile')

    # Calculate medians and Wilcoxon p-values for annotation
    if 0: # as indicated, use wilcoxon
        medians = df_long_diff.groupby('model_difference')['DIC_difference'].median()
        wilcoxon_pvalues = {
            'nogaze_diff': st.wilcoxon(df['nogaze_diff'].dropna()).pvalue,
            'withgaze_diff': st.wilcoxon(df['withgaze_diff'].dropna()).pvalue,
            'gazeInteract_diff': st.wilcoxon(df['gazeInteract_diff'].dropna()).pvalue,
            'gazefilter_ZHistOut_diff': st.wilcoxon(df['gazefilter_ZHistOut_diff'].dropna()).pvalue
        }
    else: # use ttest
        medians = df_long_diff.groupby('model_difference')['DIC_difference'].mean()
        wilcoxon_pvalues = {
            'nogaze_diff': st.ttest_1samp(df['nogaze_diff'].dropna(),0).pvalue,
            'withgaze_diff': st.ttest_1samp(df['withgaze_diff'].dropna(),0).pvalue,
            'gazeInteract_diff': st.ttest_1samp(df['gazeInteract_diff'].dropna(),0).pvalue,
            'gazefilter_ZHistOut_diff': st.ttest_1samp(df['gazefilter_ZHistOut_diff'].dropna(),0).pvalue
        }
        

    # Add annotations for median and Wilcoxon p-value to each violin plot
    plt.text(0, medians['nogaze_diff'] - 5, f"Median: {medians['nogaze_diff']:.2f}\np={wilcoxon_pvalues['nogaze_diff']:.3f}", 
             ha='center', va='bottom', fontsize=10, color='black', fontweight='bold')
    plt.text(1, medians['withgaze_diff'] - 5, f"Median: {medians['withgaze_diff']:.2f}\np={wilcoxon_pvalues['withgaze_diff']:.3f}", 
             ha='center', va='bottom', fontsize=10, color='black', fontweight='bold')
    plt.text(2, medians['gazeInteract_diff'] - 5, f"Median: {medians['gazeInteract_diff']:.2f}\np={wilcoxon_pvalues['gazeInteract_diff']:.3f}", 
             ha='center', va='bottom', fontsize=10, color='black', fontweight='bold')
    plt.text(4, medians['gazefilter_ZHistOut_diff'] - 5, f"Median: {medians['gazefilter_ZHistOut_diff']:.2f}\np={wilcoxon_pvalues['gazefilter_ZHistOut_diff']:.3f}", 
             ha='center', va='bottom', fontsize=10, color='black', fontweight='bold')
    plt.title('Distribution of DIC Differences Relative to gazefilter Model')
    plt.ylabel('DIC Difference (Model - gazefilter)')
    plt.xlabel('Model Comparison')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    
    # savefig
    savefig = 0
    if savefig:
        current_dir = data_saved_folder+'fig_for_basic_neural_analysis_allsessions_basicEvents_PCA_Pullfocused_continuousBhv_partnerDistVaris_with_hddm_model'+savefile_sufix+'/'+animal_tgt
        add_fig_dir = os.path.join(current_dir,cameraID+'/fig_for_hddm_model_fitted_allsession_withNeurons')
        #
        if not os.path.exists(add_fig_dir):
            os.makedirs(add_fig_dir)
        
        plt.savefig(add_fig_dir+'/'+animal_tgt+'_Distribution_of_DIC_Differences_Relative_to_gazefilter_Model_'+
                    cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pdf', 
                    format='pdf', bbox_inches='tight')

        
    # print(DIC_session_summary)

### plot in the very similar structure, but focus more on the single neuron

In [ ]:
# NOTE!!!
# run the hddm fitting using the data set that contained average neuron activity (across neurons for each trial)
# the data set is calculated in Misha - basic_neural_analysis_allsessions_basicEvents_PCA_makeBhvNeuronVideos_PullStartToPull_section_continuousBhv

# this one is to fit the ddm in each sesssion separately
# this one use gaze period to filter the partner's variables

# then combine it with the dataset that look at individual neurons' firing rate

if 1:
    import warnings
    warnings.filterwarnings('ignore')

    import seaborn as sns
    from scipy.stats import ttest_1samp

    # In this data set all the rt is calculated using the new definition, but the pull onset is defined slightly different
    #
    # if use the bhv PCA
    dobhvPCA = 1
    #
    doparnterleverspeed = 0 # this one only works when dobhvPCA is 0, it means use partner to lever speed that any speed
    #
    if not dobhvPCA:
        # if use onset of the first increase after min
        doOnsetAfterMin = 1
        if not doOnsetAfterMin:
            doOnsetAfterMin_suffix = ''
        elif doOnsetAfterMin:
            if not doparnterleverspeed:
                doOnsetAfterMin_suffix = 'PullOnsetAfterMin_'
            elif doparnterleverspeed:
                doOnsetAfterMin_suffix = 'PullOnsetAfterMin_PartnerLeverSpeed_'
    #
    elif dobhvPCA:
        # if use onset of the first increase after min
        doOnsetAfterMin = 1
        doBasedonSpeed = 1
        if not doOnsetAfterMin:
            doOnsetAfterMin_suffix = 'bhvPCA_'
        elif doOnsetAfterMin:
            if not doBasedonSpeed:
                doOnsetAfterMin_suffix = 'bhvPCA_PullOnsetAfterMin_'
            elif doBasedonSpeed:
                doOnsetAfterMin_suffix = 'bhvPCA_PullOnsetAfterMinBasedonSpeed_'
                

    # if use a hmm based method to find the trial start
    doHMMmethod = 0
    if doHMMmethod:
        doOnsetAfterMin_suffix = 'HMMmethods_'
        
    exaustModel_sufix = ''
        
    # load the dataset first
    current_dir = data_saved_folder+'data_saved_singlecam_wholebody_neural_and_hddm'+savefile_sufix+'/'+cameraID+'/'+animal_tgt
    add_date_dir = os.path.join(current_dir+'/hddm_model_fitted_combinedsession_withNeurons')
    
    
    if cond_toplot_type == 'allMC':
        if not (doOnsetAfterMin_suffix == 'bhvPCA_PullOnsetAfterMinBasedonSpeed_') |\
               (doOnsetAfterMin_suffix == 'PullOnsetAfterMin_PartnerLeverSpeed_'):
            with open(add_date_dir+'/hddm_datas_flexibleTW_newRTdefinition_'+doOnsetAfterMin_suffix+'withFRs_all_dates_'+
                      animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
                hddm_data_tgt_alldates = pickle.load(f)
        #
        elif (doOnsetAfterMin_suffix == 'bhvPCA_PullOnsetAfterMinBasedonSpeed_') |\
             (doOnsetAfterMin_suffix == 'PullOnsetAfterMin_PartnerLeverSpeed_'):
            with open(add_date_dir+'/hddm_datas_flexibleTW_newRTdefinition_'+'PullOnsetAfterMin_'+'withFRs_all_dates_'+
                      animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
                hddm_data_tgt_alldates = pickle.load(f)
        #
        with open(add_date_dir+'/hddm_RawFullDatas_flexibleTW_newRTdefinition_'+'PullOnsetAfterMin_'+'withFRs_all_dates_'+
                      animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
            hddm_RawFullDatas_tgt_alldates = pickle.load(f)
    #
    else:
        if not (doOnsetAfterMin_suffix == 'bhvPCA_PullOnsetAfterMinBasedonSpeed_') |\
               (doOnsetAfterMin_suffix == 'PullOnsetAfterMin_PartnerLeverSpeed_'):
            with open(add_date_dir+'/hddm_datas_flexibleTW_newRTdefinition_'+doOnsetAfterMin_suffix+'withFRs_all_dates_'+
                      animal1_fixedorders[0]+animal2_fixedorders[0]+'_'+cond_toplot_type+'.pkl', 'rb') as f:
                hddm_data_tgt_alldates = pickle.load(f)
        #
        elif (doOnsetAfterMin_suffix == 'bhvPCA_PullOnsetAfterMinBasedonSpeed_') |\
             (doOnsetAfterMin_suffix == 'PullOnsetAfterMin_PartnerLeverSpeed_'):
            with open(add_date_dir+'/hddm_datas_flexibleTW_newRTdefinition_'+'PullOnsetAfterMin_'+'withFRs_all_dates_'+
                      animal1_fixedorders[0]+animal2_fixedorders[0]+'_'+cond_toplot_type+'.pkl', 'rb') as f:
                hddm_data_tgt_alldates = pickle.load(f)
        #
        with open(add_date_dir+'/hddm_RawFullDatas_flexibleTW_newRTdefinition_'+'PullOnsetAfterMin_'+'withFRs_all_dates_'+
                      animal1_fixedorders[0]+animal2_fixedorders[0]+'_'+cond_toplot_type+'.pkl', 'rb') as f:
            hddm_RawFullDatas_tgt_alldates = pickle.load(f)
            

    # make the hddm_data structure consistent with other code
    ind_goodcond = np.isin(hddm_data_tgt_alldates['condition'],conditions_to_ana)
    hddm_data_tgt_alldates = hddm_data_tgt_alldates[ind_goodcond]

    if not dobhvPCA:
        if not doparnterleverspeed:
            rename_dict = {
                'act_animal':'subj_idx',
                'dates': 'date',
                'pull_rt': 'rt',
                'previous_pull_outcome': 'prev_trial_outcome',
                'socialgaze_auc': 'self_gaze_auc',
                'mass_move_speed_mean': 'self_mean_speed',
                'mass_move_speed_std': 'self_speed_std',
                'other_mass_move_speed_mean': 'partner_mean_speed',
                'other_mass_move_speed_std': 'partner_speed_std',
                'num_preceding_failpull': 'failed_pulls_before_reward',
                'time_from_last_reward': 'time_since_last_reward',
                }
        elif doparnterleverspeed:
            rename_dict = {
                'act_animal':'subj_idx',
                'dates': 'date',
                'pull_rt': 'rt',
                'previous_pull_outcome': 'prev_trial_outcome',
                'socialgaze_auc': 'self_gaze_auc',
                'mass_move_speed_mean': 'self_mean_speed',
                'mass_move_speed_std': 'self_speed_std',
                'other_lever_speed_mean': 'partner_mean_speed',
                'other_lever_speed_std': 'partner_speed_std',
                'num_preceding_failpull': 'failed_pulls_before_reward',
                'time_from_last_reward': 'time_since_last_reward',
                }
    elif dobhvPCA:
        rename_dict = {
            'act_animal':'subj_idx',
            'dates': 'date',
            'pull_rt': 'rt',
            'previous_pull_outcome': 'prev_trial_outcome',
            'socialgaze_auc': 'self_gaze_auc',
            # 'self_PC1_mean': 'self_mean_speed',
            # 'self_PC1_std': 'self_speed_std',
            'mass_move_speed_mean': 'self_mean_speed',
            'mass_move_speed_std': 'self_speed_std',
            'other_PC1_mean': 'partner_mean_speed',
            'other_PC1_std': 'partner_speed_std',
            'num_preceding_failpull': 'failed_pulls_before_reward',
            'time_from_last_reward': 'time_since_last_reward',
            }
        
    # Apply the renaming
    hddm_data_tgt_alldates = hddm_data_tgt_alldates.rename(columns=rename_dict)
    hddm_data_tgt_alldates['response'] = 1

    
    #
    # calculated the new partner_mean_speed and partner_speed_std with self_gaze filter
    #
    hddm_data_tgt_alldates['partner_speed_std_nofilter'] = hddm_data_tgt_alldates['partner_speed_std']
    hddm_data_tgt_alldates['partner_mean_speed_nofilter'] = hddm_data_tgt_alldates['partner_mean_speed']
    
    #
    ntrials = np.shape(hddm_data_tgt_alldates)[0]
    for itrial in np.arange(0,ntrials,1):
        gaze_prob_itrial = hddm_data_tgt_alldates['socialgaze_prob'].iloc[itrial]
        #
        if not dobhvPCA:
            if not doparnterleverspeed:
                partner_speed_itrial = hddm_data_tgt_alldates['other_mass_move_speed'].iloc[itrial]
            elif doparnterleverspeed:
                partner_speed_itrial = hddm_data_tgt_alldates['other_lever_speed'].iloc[itrial]
        elif dobhvPCA:
            partner_speed_itrial = hddm_data_tgt_alldates['other_PC1'].iloc[itrial]
        #
        # the shape of 'other_lever_speed' and other variables might be different
        partner_speed_shape = np.shape(partner_speed_itrial)[0]
        gaze_prob_shape = np.shape(gaze_prob_itrial)[0]
        if partner_speed_shape > gaze_prob_shape:
            partner_speed_itrial = partner_speed_itrial[np.arange(0,gaze_prob_shape,1)]
        elif partner_speed_shape < gaze_prob_shape:
            gaze_prob_itrial = gaze_prob_itrial[np.arange(0,partner_speed_shape,1)]

        # smooth the gaze duration a bit more
        gaze_prob_itrial = gaussian_filter1d(gaze_prob_itrial, sigma=8) # the final sigma would be sqrt(4*4+sigma*sigma)

        partner_speed_filtered = partner_speed_itrial[gaze_prob_itrial>0]

        partner_mean_speed_filtered = np.nanmean(partner_speed_filtered)
        partner_std_speed_filtered = np.nanstd(partner_speed_filtered)
        
        hddm_data_tgt_alldates['partner_mean_speed'].iloc[itrial] = partner_mean_speed_filtered
        hddm_data_tgt_alldates['partner_speed_std'].iloc[itrial] = partner_std_speed_filtered
        
    # 
    # remove nan row first for the glm data set
    hddm_data_tgt_alldates = hddm_data_tgt_alldates.dropna()
    
    # 
    # normalize all the predictors
    from scipy.stats import zscore
    #
    # List of predictor columns to normalize (exclude 'rt' and identifiers)
    predictor_columns = [
        'self_gaze_auc',
        'partner_mean_speed',
        'self_mean_speed',
        'partner_speed_std',
        'self_speed_std',
        'failed_pulls_before_reward',
        'time_since_last_reward'
    ]
    # Rename originals and replace with z-scored values
    for col in predictor_columns:
        hddm_data_tgt_alldates[f"{col}_origin"] = hddm_data_tgt_alldates[col]
        hddm_data_tgt_alldates[col] = zscore(hddm_data_tgt_alldates[col])  
        # hddm_data_tgt_alldates[col] = hddm_data_tgt_alldates[col]  
      
    #
    # initialize a dataframe for summarizing all sessions results
    summary_list_for_each_neuron = []
    
    
    # run the fitting for each session
    for session_toana, hddm_data_tgt_idate in hddm_data_tgt_alldates.groupby('date'):
    
        ind_ = np.isin(dates_list,session_toana)
        task_cond = task_conditions[np.where(ind_)[0][0]]
        
        # Inside the loop:
        # 'session_toana' is the date for the current group
        # 'hddm_data_tgt_idate' is a DataFrame containing only the rows for that date
    
        #
        # # add some semi-artbitrary cretieras
        # hddm_data_tgt_alldates = hddm_data_tgt_idate[hddm_data_tgt_alldates['prev_trial_outcome']==1]
        # hddm_data_tgt_alldates = hddm_data_tgt_idate[hddm_data_tgt_alldates['partner_mean_speed']<=300]
        #
        lower_q = 0.05
        upper_q = 0.95
        rt_lower, rt_upper = hddm_data_tgt_idate['rt'].quantile([lower_q, upper_q])
        hddm_data_tgt_idate = hddm_data_tgt_idate[hddm_data_tgt_idate['rt']>rt_lower]
        hddm_data_tgt_idate = hddm_data_tgt_idate[hddm_data_tgt_idate['rt']<rt_upper]
        #
        tslr_lower, tslr_upper = hddm_data_tgt_idate['time_since_last_reward'].quantile([lower_q, upper_q])
        hddm_data_tgt_idate = hddm_data_tgt_idate[hddm_data_tgt_idate['time_since_last_reward']>tslr_lower]
        hddm_data_tgt_idate = hddm_data_tgt_idate[hddm_data_tgt_idate['time_since_last_reward']<tslr_upper]
        
        
        # try to load the analyzed
        try:

            # dummpy
            
            current_dir = data_saved_folder+'data_saved_singlecam_wholebody_neural_and_hddm'+savefile_sufix+'/'+cameraID+'/'+animal_tgt
            add_date_dir = os.path.join(current_dir+'/hddm_model_fitted_allsession_withNeurons/'+session_toana)

            #
            with open(add_date_dir+'/hddm_model_fitted_nogaze_gazefilter_traces_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
                hddm_model_fitted_nogaze_gazefilter_traces = pickle.load(f)

            with open(add_date_dir+'/hddm_model_fitted_nogaze_gazefilter_stats_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
                hddm_model_fitted_nogaze_gazefilter_stats = pickle.load(f)

            with open(add_date_dir+'/hddm_model_fitted_nogaze_gazefilter_dic_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
                hddm_model_fitted_nogaze_gazefilter_dic = pickle.load(f)
                
            

            print(animal_tgt+' '+cond_toplot_type+' hddm fitting results loaded')

        except:

            print(animal_tgt+' '+cond_toplot_type+' running hddm fitting')
            
            try:

                # run the hddm nodel
                # HDDM settings - the basic setting
                # samples=500 # 2000
                # burn=100
                # thin=2
                samples=200 # 2000
                burn=100
                thin=2
                #
                doNogazeOnly = 1
                #
                if dobhvPCA:
                    _, hddm_model_fitted_nogaze_gazefilter = run_hddm_modeling_bhvPCA_gazefilter(hddm_data_tgt_idate, animal_tgt,
                                                                        samples, burn, thin, doNogazeOnly)
                elif not dobhvPCA:
                    _, hddm_model_fitted_nogaze_gazefilter = run_hddm_modeling_gazefilter(hddm_data_tgt_idate, animal_tgt,
                                                                        samples, burn, thin, doNogazeOnly)

                #
                hddm_model_fitted_nogaze_gazefilter_traces = hddm_model_fitted_nogaze_gazefilter.get_traces()
                hddm_model_fitted_nogaze_gazefilter_stats  = hddm_model_fitted_nogaze_gazefilter.gen_stats()
                hddm_model_fitted_nogaze_gazefilter_dic = hddm_model_fitted_nogaze_gazefilter.dic_info
            
            except:
                continue

                
            # save data
            if 1:
                current_dir = data_saved_folder+'data_saved_singlecam_wholebody_neural_and_hddm'+savefile_sufix+'/'+cameraID+'/'+animal_tgt
                add_date_dir = os.path.join(current_dir+'/hddm_model_fitted_allsession_withNeurons/'+session_toana)
                if not os.path.exists(add_date_dir):
                    os.makedirs(add_date_dir)
                #
                with open(add_date_dir+'/hddm_model_fitted_nogaze_gazefilter_traces_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'wb') as f:
                    pickle.dump(hddm_model_fitted_nogaze_gazefilter_traces, f)

                with open(add_date_dir+'/hddm_model_fitted_nogaze_gazefilter_stats_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'wb') as f:
                    pickle.dump(hddm_model_fitted_nogaze_gazefilter_stats, f)  

                with open(add_date_dir+'/hddm_model_fitted_nogaze_gazefilter_dic_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'wb') as f:
                    pickle.dump(hddm_model_fitted_nogaze_gazefilter_dic, f)
            
          
        #
        # reconstruct V
        #
        df_combined = hddm_data_tgt_idate # your combined trial-level dataframe
        traces = hddm_model_fitted_nogaze_gazefilter_traces  # or use `hddm_model_fitted_traces` if needed
        stats = hddm_model_fitted_nogaze_gazefilter_stats


        # --- STEP 1: Use all posterior samples to compute predicted v for each trial ---
        n_samples = len(traces)
        n_trials = len(df_combined)

        # Allocate space to hold predicted v for each sample x trial
        predicted_v_samples = np.zeros((n_samples, n_trials))

        # Pre-fetch covariate values as numpy arrays
        x1 = df_combined['partner_mean_speed'].values
        x2 = df_combined['partner_speed_std'].values
        x3 = df_combined['self_mean_speed'].values
        x4 = df_combined['self_speed_std'].values
        # x3 = df_combined['failed_pulls_before_reward'].values
        # x3 = df_combined['time_since_last_reward'].values
        # x7 = df_combined['self_gaze_auc'].values

        for i in range(n_samples):
            intercept = traces['v_Intercept'].values[i]
            coef_x1 = traces['v_partner_mean_speed'].values[i]
            coef_x2 = traces['v_partner_speed_std'].values[i]
            coef_x3 = traces['v_self_mean_speed'].values[i]
            coef_x4 = traces['v_self_speed_std'].values[i]
            # coef_x3 = traces['v_failed_pulls_before_reward'].values[i]
            # coef_x3 = traces['v_time_since_last_reward'].values[i]
            # coef_x7 = traces['v_self_gaze_auc'].values[i]

            # Apply regression formula for all trials
            predicted_v_samples[i, :] = (
                # intercept + coef_x1 * x1 + coef_x2 * x2 + coef_x3 * x3
                # intercept + coef_x1 * x1 
                # intercept + coef_x2 * x2 
                # intercept + coef_x1 * x1 + coef_x2 * x2 + coef_x3 * x3
                # intercept + coef_x1 * x1 + coef_x3 * x3
                intercept + coef_x1 * x1 + coef_x2 * x2 + coef_x3 * x3 + coef_x4 * x4
                # intercept + coef_x1 * x1 + coef_x2 * x2 + coef_x3 * x3 + coef_x4 * x4 + coef_x7 * x7
                # intercept + coef_x2 * x2 + coef_x4 * x4
            )

        # --- STEP 2: Compute posterior mean (and optionally CI) of predicted v ---
        df_with_v = df_combined.copy()
        df_with_v['predicted_v'] = predicted_v_samples.mean(axis=0)

        # Optional: also store credible intervals (e.g., 95% CI)
        df_with_v['predicted_v_lower'] = np.percentile(predicted_v_samples, 2.5, axis=0)
        df_with_v['predicted_v_upper'] = np.percentile(predicted_v_samples, 97.5, axis=0) 
        
        
        #
        # put data in the summarizing dataframe
        hddm_RawFullDatas_tgt_idate = hddm_RawFullDatas_tgt_alldates[hddm_RawFullDatas_tgt_alldates['dates']\
                                                                     == session_toana]
        neurons_idate = np.unique(hddm_RawFullDatas_tgt_idate['clusterID'])
        nneurons = np.shape(neurons_idate)[0]
        
        for ineuron in np.arange(0,nneurons,1):
            ineuron_idate = neurons_idate[ineuron]
            hddm_RawFullDatas_tgt_idate_ineuron = hddm_RawFullDatas_tgt_idate[hddm_RawFullDatas_tgt_idate['clusterID']\
                                                                              == ineuron_idate]
            #
            # mean fr
            # hddm_RawFullDatas_tgt_idate_ineuron['mean_fr'] = hddm_RawFullDatas_tgt_idate_ineuron['FR_ievent']\
            #                                                  .apply(np.mean)
            # hddm_RawFullDatas_tgt_idate_ineuron['mean_fr'] = hddm_RawFullDatas_tgt_idate_ineuron['FR_ievent']\
            #                                                     .apply(lambda x: np.mean(x[:int(0.85*fps)]))
            if animal_tgt == 'dodson':
                hddm_RawFullDatas_tgt_idate_ineuron['mean_fr'] = hddm_RawFullDatas_tgt_idate_ineuron['FR_ievent']\
                                                                .apply(lambda x: np.mean(x[:int(1*fps)]))
            else:
                hddm_RawFullDatas_tgt_idate_ineuron['mean_fr'] = hddm_RawFullDatas_tgt_idate_ineuron['FR_ievent']\
                                                                .apply(lambda x: np.mean(x[:int(1*fps)]))
            
            #
            # fr slope
            # === Fixed slope before 0.85s from pull ===
            pull_margin_frames = int(0.85 * fps)
            def calculate_slope(fr_trace):
                from scipy.stats import linregress
                #
                if isinstance(fr_trace, (list, np.ndarray)) and len(fr_trace) > pull_margin_frames:
                    y = fr_trace[:len(fr_trace) - pull_margin_frames]
                    x = np.arange(len(y))
                    try:
                        slope, *_ = linregress(x, y)
                        return abs(slope)
                    except:
                        return np.nan
                return np.nan
            
            #
            hddm_RawFullDatas_tgt_idate_ineuron['slope_fr'] = hddm_RawFullDatas_tgt_idate_ineuron['FR_ievent']\
                                                              .apply(calculate_slope)

            # merge the ddm fitting result and key bhv metric to it
            columns_to_merge = ['bhv_id', 'rt', 'partner_speed_std', 'predicted_v',
                                'failed_pulls_before_reward', 'prev_trial_outcome', 'time_since_last_reward']
            #
            merged_df = pd.merge(
                                    left=hddm_RawFullDatas_tgt_idate_ineuron,
                                    right=df_with_v[columns_to_merge],
                                    how='left',
                                    on='bhv_id'
                                )
            
            
            #---------------------
            #
            # correlation between neuron measures and bhv measures
            #
            # predictor_cols = ['rt', 'partner_speed_std', 'predicted_v', 
            #                   'failed_pulls_before_reward', 'prev_trial_outcome', 'time_since_last_reward']
            predictor_cols = ['prev_trial_outcome', ]
            dependent_cols = ['mean_fr', 'slope_fr'] 

            # Check the correlations between your predictor variables
            if ineuron == 0:
                print(merged_df[['rt', 'partner_speed_std', 'predicted_v']].corr())

            # --- New: Perform regression between mean_fr and slope_fr for this neuron ---
            beta_fr_corr, pval_fr_corr = np.nan, np.nan # Default to NaN
            clean_df_corr = merged_df.dropna(subset=['mean_fr', 'slope_fr'])
            if len(clean_df_corr) > 1:
                corr_model = smf.ols('slope_fr ~ mean_fr', data=clean_df_corr).fit()
                beta_fr_corr = corr_model.params.get('mean_fr')
                pval_fr_corr = corr_model.pvalues.get('mean_fr')

            # --- Run the regression for each dependent variable ---
            for dep_var in dependent_cols:

                # --- 1. Initialize the result dictionary for this neuron and DV ---
                result = {
                    'date': merged_df['dates'].iloc[0], # Corrected 'dates' to 'date'
                    'condition': merged_df['condition'].iloc[0],
                    'clusterID': merged_df['clusterID'].iloc[0],
                    'dependent_variable': dep_var,
                    'beta_rt': np.nan, 'pval_rt': np.nan,
                    'beta_partner_speed_std': np.nan, 'pval_partner_speed_std': np.nan,
                    'beta_predicted_v': np.nan, 'pval_predicted_v': np.nan,
                    'beta_meanfr_vs_slopefr': beta_fr_corr, # Add the pre-calculated beta
                    'pval_meanfr_vs_slopefr': pval_fr_corr  # Add the pre-calculated p-value
                }

                # --- 2. Loop through each predictor to run a separate single regression ---
                for predictor in predictor_cols:

                    # Define the simple formula (e.g., 'mean_fr ~ rt')
                    formula = f"{dep_var} ~ {predictor}"

                    # Clean data for this specific predictor and dependent variable
                    clean_df = merged_df.dropna(subset=[predictor, dep_var])

                    # Check for sufficient data (need at least 2 points for a simple regression)
                    if len(clean_df) > 1:
                        model = smf.ols(formula, data=clean_df).fit()

                        # Populate the result dict with the beta and p-value from this model
                        result[f'beta_{predictor}'] = model.params.get(predictor)
                        result[f'pval_{predictor}'] = model.pvalues.get(predictor)

                # --- 3. Append the fully populated result to the master list ---
                summary_list_for_each_neuron.append(result)
     
    #
    summary_list_for_each_neuron_df = pd.DataFrame(summary_list_for_each_neuron)
    
    #------------------
    # do the plotting
    from scipy import stats # Make sure stats is imported from scipy
    #
    # --- 2. Reshape the DataFrame from Wide to Long Format ---
    # This is the most important step for plotting
    long_df_list = []
    predictors = predictor_cols
    
    for pred in predictors:
        temp_df = summary_list_for_each_neuron_df[['dependent_variable', f'beta_{pred}', f'pval_{pred}']].copy()
        temp_df.rename(columns={f'beta_{pred}': 'beta_value', f'pval_{pred}': 'p_value'}, inplace=True)
        temp_df['predictor'] = pred
        long_df_list.append(temp_df)
    plot_df = pd.concat(long_df_list, ignore_index=True)

    # --- 3. Create the Two-Panel Figure ---
    sns.set_style("ticks")
    # Set sharey=False to allow independent y-axes
    fig, axes = plt.subplots(1, 2, figsize=(14, 7), sharey=False)
    fig.suptitle('Regression Coefficients for Neural Activity', fontsize=16)

    # --- Panel 1: Plot and Annotate for mean_fr ---
    axes[0].set_title('Dependent Variable: mean_fr')
    axes[0].tick_params(axis='x', labelrotation=90)
    df_mean_fr = plot_df[plot_df['dependent_variable'] == 'mean_fr']
    sns.swarmplot(ax=axes[0], data=df_mean_fr, x='predictor', y='beta_value', color='gray', s=5)
    sns.swarmplot(ax=axes[0], data=df_mean_fr[df_mean_fr['p_value'] < 0.05], x='predictor', y='beta_value', color='red', s=5)

    # --- Panel 2: Plot and Annotate for slope_fr ---
    axes[1].set_title('Dependent Variable: slope_fr')
    axes[1].tick_params(axis='x', labelrotation=90)
    # Filter for 'slope_fr'
    df_slope_fr = plot_df[plot_df['dependent_variable'] == 'slope_fr']
    sns.swarmplot(ax=axes[1], data=df_slope_fr, x='predictor', y='beta_value', color='gray', s=5)
    sns.swarmplot(ax=axes[1], data=df_slope_fr[df_slope_fr['p_value'] < 0.05], x='predictor', y='beta_value', color='red', s=5)

    # --- 4. Loop through axes and predictors to add annotations ---
    for i, ax in enumerate(axes):
        # Determine which data to use based on the panel index
        panel_df = df_mean_fr if i == 0 else df_slope_fr

        ymin, ymax = ax.get_ylim()

        for j, pred in enumerate(predictors):
            betas = panel_df[panel_df['predictor'] == pred]['beta_value'].dropna()

            if len(betas) > 1:
                mean_beta = betas.mean()
                t_stat, p_val = stats.ttest_1samp(betas, 0)

                p_text = f"p < 0.001" if p_val < 0.001 else f"p = {p_val:.3f}"
                annotation_text = f"mean β = {mean_beta:.2f}\n({p_text})"

                ax.text(j, ymax * 0.95, annotation_text, ha='center', va='top', fontsize=9, bbox=dict(facecolor='white', alpha=0.5, edgecolor='none'))

    # --- 5. Final Touches ---
    for ax in axes:
        ax.axhline(0, color='black', linestyle='--', linewidth=1)
        ax.set_xlabel('Predictor Variable')
        # Set the y-axis label for both panels
        ax.set_ylabel('Beta Coefficient (β)')
    sns.despine()
    plt.tight_layout(rect=[0, 0, 1, 0.96])
        
    # -------------------
    #
    # second figure, venn diagram    
    from matplotlib_venn import venn3
    # --- 1. Separate the results for each dependent variable ---
    df_mean_fr = summary_list_for_each_neuron_df[summary_list_for_each_neuron_df['dependent_variable'] == 'mean_fr']
    df_slope_fr = summary_list_for_each_neuron_df[summary_list_for_each_neuron_df['dependent_variable'] == 'slope_fr']
    #
    # --- 2. For each dependent variable, get the set of significant neuron IDs for each predictor ---
    # A unique neuron is identified by the combination of 'date' and 'clusterID'.
    # Sets for the 'mean_fr' model
    df_filtered_mean_rt = df_mean_fr[df_mean_fr['pval_rt'] < 0.05]
    mean_fr_rt_neurons = set(zip(df_filtered_mean_rt['date'], df_filtered_mean_rt['clusterID']))
    #
    df_filtered_mean_partner = df_mean_fr[df_mean_fr['pval_partner_speed_std'] < 0.05]
    mean_fr_partner_neurons = set(zip(df_filtered_mean_partner['date'], df_filtered_mean_partner['clusterID']))
    #
    df_filtered_mean_v = df_mean_fr[df_mean_fr['pval_predicted_v'] < 0.05]
    mean_fr_v_neurons = set(zip(df_filtered_mean_v['date'], df_filtered_mean_v['clusterID']))
    # Sets for the 'slope_fr' model
    df_filtered_slope_rt = df_slope_fr[df_slope_fr['pval_rt'] < 0.05]
    slope_fr_rt_neurons = set(zip(df_filtered_slope_rt['date'], df_filtered_slope_rt['clusterID']))
    #
    df_filtered_slope_partner = df_slope_fr[df_slope_fr['pval_partner_speed_std'] < 0.05]
    slope_fr_partner_neurons = set(zip(df_filtered_slope_partner['date'], df_filtered_slope_partner['clusterID']))
    #
    df_filtered_slope_v = df_slope_fr[df_slope_fr['pval_predicted_v'] < 0.05]
    slope_fr_v_neurons = set(zip(df_filtered_slope_v['date'], df_filtered_slope_v['clusterID']))
    #
    # --- 3. Create the new figure (fig2) with two panels for the Venn diagrams ---
    fig2, axes = plt.subplots(1, 2, figsize=(14, 7))
    fig2.suptitle('Overlap of Neurons with Significant Predictors (p < 0.05)', fontsize=16)
    # --- Panel 1: Venn Diagram for mean_fr ---
    ax1 = axes[0]
    venn3(
        subsets=[mean_fr_rt_neurons, mean_fr_partner_neurons, mean_fr_v_neurons],
        set_labels=('rt', 'partner_speed_std', 'predicted_v'),
        ax=ax1
    )
    ax1.set_title('Dependent Variable: mean_fr')
    # --- Panel 2: Venn Diagram for slope_fr ---
    ax2 = axes[1]
    venn3(
        subsets=[slope_fr_rt_neurons, slope_fr_partner_neurons, slope_fr_v_neurons],
        set_labels=('rt', 'partner_speed_std', 'predicted_v'),
        ax=ax2
    )
    ax2.set_title('Dependent Variable: slope_fr')
    
            
    # ---------------
    # 3rd plot, correlation between mean firing and firing rate slope
    # --- 2. Prepare the Data for Plotting ---
    # Since the beta is the same for both 'mean_fr' and 'slope_fr' rows for each neuron,
    # we can drop duplicates to get one unique value per neuron.
    plot_df = summary_list_for_each_neuron_df.copy()
    #
    # --- 3. Create the Single-Panel Figure ---
    sns.set_style("ticks")
    fig3, ax = plt.subplots(1, 1, figsize=(6, 7))
    fig3.suptitle('Relationship Between Mean Firing Rate and Firing Rate Slope', fontsize=14)
    #
    # Plot all points in gray
    sns.swarmplot(ax=ax, data=plot_df, y='beta_meanfr_vs_slopefr', color='gray', s=7)
    # Plot significant points (p < 0.05) in red on top
    sns.swarmplot(ax=ax, data=plot_df[plot_df['pval_meanfr_vs_slopefr'] < 0.05], y='beta_meanfr_vs_slopefr', color='red', s=7)
    #
    # --- 4. Add Annotation ---
    betas = plot_df['beta_meanfr_vs_slopefr'].dropna()
    if len(betas) > 1:
        mean_beta = betas.mean()
        t_stat, p_val = stats.ttest_1samp(betas, 0)

        p_text = f"p < 0.001" if p_val < 0.001 else f"p = {p_val:.3f}"
        annotation_text = f"mean β = {mean_beta:.2f}\n(t-test vs 0: {p_text})"

        ymin, ymax = ax.get_ylim()
        ax.text(0, ymax * 0.95, annotation_text, ha='center', va='top', fontsize=10, 
                bbox=dict(facecolor='white', alpha=0.5, edgecolor='none'))
    #
    # --- 5. Final Touches ---
    ax.axhline(0, color='black', linestyle='--', linewidth=1.5)
    ax.set_ylabel('Beta Coefficient (slope_fr ~ mean_fr)')
    ax.set_xticks([]) # Remove x-axis ticks
    ax.set_xlabel('All Neurons')
    sns.despine()
    plt.tight_layout(rect=[0, 0, 1, 0.95])
        
        
    #
    # savefig
    savefig = 0
    if savefig:
        current_dir = data_saved_folder+'fig_for_basic_neural_analysis_allsessions_basicEvents_PCA_Pullfocused_continuousBhv_partnerDistVaris_with_hddm_model'+savefile_sufix+'/'+animal_tgt
        add_fig_dir = os.path.join(current_dir,cameraID+'/fig_for_hddm_model_fitted_allsession_withNeurons')
        #
        
        fig.savefig(add_fig_dir+'/fr_correlation_with_predicted_v_bhv_variables_nogaze_gazefilter_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pdf', 
                        format='pdf', bbox_inches='tight')
        fig2.savefig(add_fig_dir+'/venn_fr_correlation_with_predicted_v_bhv_variables_nogaze_gazefilter_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pdf', 
                        format='pdf', bbox_inches='tight')
        
        fig3.savefig(add_fig_dir+'/fr_mean_slope_correlation_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pdf', 
                        format='pdf', bbox_inches='tight')

        
        
    # get the id for the neurons that showed significant correlation with predicted v
    slope_fr_v_encoding_neuronIDs = slope_fr_v_neurons.copy()
    
    

In [ ]:
df = summary_list_for_each_neuron_df.copy()

# Unique neurons defined by (date, clusterID)
unique_neurons = df[['date', 'clusterID']].drop_duplicates()

# Neuron count per session (date)
neurons_per_date = (
    unique_neurons
    .groupby('date')
    .size()
    .rename('n_neurons')
)

# Summary statistics
min_neurons = neurons_per_date.min()
max_neurons = neurons_per_date.max()
mean_neurons = neurons_per_date.mean()
std_neurons = neurons_per_date.std()

# Total number of neurons across all sessions
total_neurons = unique_neurons.shape[0]

print("Neurons per session:")
print(neurons_per_date)

print("\nSummary:")
print(f"Range of neurons per session: {min_neurons}–{max_neurons}")
print(f"Mean ± SD neurons per session: {mean_neurons:.1f} ± {std_neurons:.1f}")
print(f"Total number of neurons: {total_neurons}")

In [ ]:
# NOTE!!!
# run the hddm fitting using the data set that contained average neuron activity (across neurons for each trial)
# the data set is calculated in Misha - basic_neural_analysis_allsessions_basicEvents_PCA_makeBhvNeuronVideos_PullStartToPull_section_continuousBhv

# this one is to fit the ddm in each sesssion separately

# then combine it with the dataset that look at individual neurons' firing rate

if 1:
    import warnings
    warnings.filterwarnings('ignore')

    import seaborn as sns
    from scipy.stats import ttest_1samp

    # In this data set all the rt is calculated using the new definition, but the pull onset is defined slightly different
    #
    # if use the bhv PCA
    dobhvPCA = 1
    #
    doparnterleverspeed = 0 # this one only works when dobhvPCA is 0, it means use partner to lever speed that any speed
    #
    if not dobhvPCA:
        # if use onset of the first increase after min
        doOnsetAfterMin = 1
        if not doOnsetAfterMin:
            doOnsetAfterMin_suffix = ''
        elif doOnsetAfterMin:
            if not doparnterleverspeed:
                doOnsetAfterMin_suffix = 'PullOnsetAfterMin_'
            elif doparnterleverspeed:
                doOnsetAfterMin_suffix = 'PullOnsetAfterMin_PartnerLeverSpeed_'
    #
    elif dobhvPCA:
        # if use onset of the first increase after min
        doOnsetAfterMin = 1
        doBasedonSpeed = 1
        if not doOnsetAfterMin:
            doOnsetAfterMin_suffix = 'bhvPCA_'
        elif doOnsetAfterMin:
            if not doBasedonSpeed:
                doOnsetAfterMin_suffix = 'bhvPCA_PullOnsetAfterMin_'
            elif doBasedonSpeed:
                doOnsetAfterMin_suffix = 'bhvPCA_PullOnsetAfterMinBasedonSpeed_'
                

    # if use a hmm based method to find the trial start
    doHMMmethod = 0
    if doHMMmethod:
        doOnsetAfterMin_suffix = 'HMMmethods_'
        
    exaustModel_sufix = ''
        
    # load the dataset first
    current_dir = data_saved_folder+'data_saved_singlecam_wholebody_neural_and_hddm'+savefile_sufix+'/'+cameraID+'/'+animal_tgt
    add_date_dir = os.path.join(current_dir+'/hddm_model_fitted_combinedsession_withNeurons')
    
    if cond_toplot_type == 'allMC':
        if not (doOnsetAfterMin_suffix == 'bhvPCA_PullOnsetAfterMinBasedonSpeed_') |\
               (doOnsetAfterMin_suffix == 'PullOnsetAfterMin_PartnerLeverSpeed_'):
            with open(add_date_dir+'/hddm_datas_flexibleTW_newRTdefinition_'+doOnsetAfterMin_suffix+'withFRs_all_dates_'+
                      animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
                hddm_data_tgt_alldates = pickle.load(f)
        #
        elif (doOnsetAfterMin_suffix == 'bhvPCA_PullOnsetAfterMinBasedonSpeed_') |\
             (doOnsetAfterMin_suffix == 'PullOnsetAfterMin_PartnerLeverSpeed_'):
            with open(add_date_dir+'/hddm_datas_flexibleTW_newRTdefinition_'+'PullOnsetAfterMin_'+'withFRs_all_dates_'+
                      animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
                hddm_data_tgt_alldates = pickle.load(f)
        #
        with open(add_date_dir+'/hddm_RawFullDatas_flexibleTW_newRTdefinition_'+'PullOnsetAfterMin_'+'withFRs_all_dates_'+
                      animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
            hddm_RawFullDatas_tgt_alldates = pickle.load(f)
    #
    else:
        if not (doOnsetAfterMin_suffix == 'bhvPCA_PullOnsetAfterMinBasedonSpeed_') |\
               (doOnsetAfterMin_suffix == 'PullOnsetAfterMin_PartnerLeverSpeed_'):
            with open(add_date_dir+'/hddm_datas_flexibleTW_newRTdefinition_'+doOnsetAfterMin_suffix+'withFRs_all_dates_'+
                      animal1_fixedorders[0]+animal2_fixedorders[0]+'_'+cond_toplot_type+'.pkl', 'rb') as f:
                hddm_data_tgt_alldates = pickle.load(f)
        #
        elif (doOnsetAfterMin_suffix == 'bhvPCA_PullOnsetAfterMinBasedonSpeed_') |\
             (doOnsetAfterMin_suffix == 'PullOnsetAfterMin_PartnerLeverSpeed_'):
            with open(add_date_dir+'/hddm_datas_flexibleTW_newRTdefinition_'+'PullOnsetAfterMin_'+'withFRs_all_dates_'+
                      animal1_fixedorders[0]+animal2_fixedorders[0]+'_'+cond_toplot_type+'.pkl', 'rb') as f:
                hddm_data_tgt_alldates = pickle.load(f)
        #
        with open(add_date_dir+'/hddm_RawFullDatas_flexibleTW_newRTdefinition_'+'PullOnsetAfterMin_'+'withFRs_all_dates_'+
                      animal1_fixedorders[0]+animal2_fixedorders[0]+'_'+cond_toplot_type+'.pkl', 'rb') as f:
            hddm_RawFullDatas_tgt_alldates = pickle.load(f)
            

    # make the hddm_data structure consistent with other code
    ind_goodcond = np.isin(hddm_data_tgt_alldates['condition'],conditions_to_ana)
    hddm_data_tgt_alldates = hddm_data_tgt_alldates[ind_goodcond]

    if not dobhvPCA:
        if not doparnterleverspeed:
            rename_dict = {
                'act_animal':'subj_idx',
                'dates': 'date',
                'pull_rt': 'rt',
                'previous_pull_outcome': 'prev_trial_outcome',
                'socialgaze_auc': 'self_gaze_auc',
                'mass_move_speed_mean': 'self_mean_speed',
                'mass_move_speed_std': 'self_speed_std',
                'other_mass_move_speed_mean': 'partner_mean_speed',
                'other_mass_move_speed_std': 'partner_speed_std',
                'num_preceding_failpull': 'failed_pulls_before_reward',
                'time_from_last_reward': 'time_since_last_reward',
                }
        elif doparnterleverspeed:
            rename_dict = {
                'act_animal':'subj_idx',
                'dates': 'date',
                'pull_rt': 'rt',
                'previous_pull_outcome': 'prev_trial_outcome',
                'socialgaze_auc': 'self_gaze_auc',
                'mass_move_speed_mean': 'self_mean_speed',
                'mass_move_speed_std': 'self_speed_std',
                'other_lever_speed_mean': 'partner_mean_speed',
                'other_lever_speed_std': 'partner_speed_std',
                'num_preceding_failpull': 'failed_pulls_before_reward',
                'time_from_last_reward': 'time_since_last_reward',
                }
    elif dobhvPCA:
        rename_dict = {
            'act_animal':'subj_idx',
            'dates': 'date',
            'pull_rt': 'rt',
            'previous_pull_outcome': 'prev_trial_outcome',
            'socialgaze_auc': 'self_gaze_auc',
            # 'self_PC1_mean': 'self_mean_speed',
            # 'self_PC1_std': 'self_speed_std',
            'mass_move_speed_mean': 'self_mean_speed',
            'mass_move_speed_std': 'self_speed_std',
            'other_PC1_mean': 'partner_mean_speed',
            'other_PC1_std': 'partner_speed_std',
            'num_preceding_failpull': 'failed_pulls_before_reward',
            'time_from_last_reward': 'time_since_last_reward',
            }
        
    # Apply the renaming
    hddm_data_tgt_alldates = hddm_data_tgt_alldates.rename(columns=rename_dict)
    hddm_data_tgt_alldates['response'] = 1

    
    #
    # calculated the new partner_mean_speed and partner_speed_std with self_gaze filter
    #
    hddm_data_tgt_alldates['partner_speed_std_nofilter'] = hddm_data_tgt_alldates['partner_speed_std']
    hddm_data_tgt_alldates['partner_mean_speed_nofilter'] = hddm_data_tgt_alldates['partner_mean_speed']
    
    #
    ntrials = np.shape(hddm_data_tgt_alldates)[0]
    for itrial in np.arange(0,ntrials,1):
        gaze_prob_itrial = hddm_data_tgt_alldates['socialgaze_prob'].iloc[itrial]
        #
        if not dobhvPCA:
            if not doparnterleverspeed:
                partner_speed_itrial = hddm_data_tgt_alldates['other_mass_move_speed'].iloc[itrial]
            elif doparnterleverspeed:
                partner_speed_itrial = hddm_data_tgt_alldates['other_lever_speed'].iloc[itrial]
        elif dobhvPCA:
            partner_speed_itrial = hddm_data_tgt_alldates['other_PC1'].iloc[itrial]
        #
        # the shape of 'other_lever_speed' and other variables might be different
        partner_speed_shape = np.shape(partner_speed_itrial)[0]
        gaze_prob_shape = np.shape(gaze_prob_itrial)[0]
        if partner_speed_shape > gaze_prob_shape:
            partner_speed_itrial = partner_speed_itrial[np.arange(0,gaze_prob_shape,1)]
        elif partner_speed_shape < gaze_prob_shape:
            gaze_prob_itrial = gaze_prob_itrial[np.arange(0,partner_speed_shape,1)]

        # smooth the gaze duration a bit more
        gaze_prob_itrial = gaussian_filter1d(gaze_prob_itrial, sigma=8) # the final sigma would be sqrt(4*4+sigma*sigma)

        partner_speed_filtered = partner_speed_itrial[gaze_prob_itrial>0]

        partner_mean_speed_filtered = np.nanmean(partner_speed_filtered)
        partner_std_speed_filtered = np.nanstd(partner_speed_filtered)
        
        hddm_data_tgt_alldates['partner_mean_speed'].iloc[itrial] = partner_mean_speed_filtered
        hddm_data_tgt_alldates['partner_speed_std'].iloc[itrial] = partner_std_speed_filtered
        
    # 
    # remove nan row first for the glm data set
    hddm_data_tgt_alldates = hddm_data_tgt_alldates.dropna()
    
    # 
    # normalize all the predictors
    from scipy.stats import zscore
    #
    # List of predictor columns to normalize (exclude 'rt' and identifiers)
    predictor_columns = [
        'self_gaze_auc',
        'partner_mean_speed',
        'self_mean_speed',
        'partner_speed_std',
        'self_speed_std',
        'failed_pulls_before_reward',
        'time_since_last_reward'
    ]
    # Rename originals and replace with z-scored values
    for col in predictor_columns:
        hddm_data_tgt_alldates[f"{col}_origin"] = hddm_data_tgt_alldates[col]
        hddm_data_tgt_alldates[col] = zscore(hddm_data_tgt_alldates[col])  
        # hddm_data_tgt_alldates[col] = hddm_data_tgt_alldates[col]  
      
    #
    # initialize a dataframe for summarizing all sessions results
    summary_list_for_each_neuron = []
    
    
    # run the fitting for each session
    for session_toana, hddm_data_tgt_idate in hddm_data_tgt_alldates.groupby('date'):
    
        ind_ = np.isin(dates_list,session_toana)
        task_cond = task_conditions[np.where(ind_)[0][0]]
        
        # Inside the loop:
        # 'session_toana' is the date for the current group
        # 'hddm_data_tgt_idate' is a DataFrame containing only the rows for that date
    
        #
        # # add some semi-artbitrary cretieras
        # hddm_data_tgt_alldates = hddm_data_tgt_idate[hddm_data_tgt_alldates['prev_trial_outcome']==1]
        # hddm_data_tgt_alldates = hddm_data_tgt_idate[hddm_data_tgt_alldates['partner_mean_speed']<=300]
        #
        lower_q = 0.05
        upper_q = 0.95
        rt_lower, rt_upper = hddm_data_tgt_idate['rt'].quantile([lower_q, upper_q])
        hddm_data_tgt_idate = hddm_data_tgt_idate[hddm_data_tgt_idate['rt']>rt_lower]
        hddm_data_tgt_idate = hddm_data_tgt_idate[hddm_data_tgt_idate['rt']<rt_upper]
        #
        tslr_lower, tslr_upper = hddm_data_tgt_idate['time_since_last_reward'].quantile([lower_q, upper_q])
        hddm_data_tgt_idate = hddm_data_tgt_idate[hddm_data_tgt_idate['time_since_last_reward']>tslr_lower]
        hddm_data_tgt_idate = hddm_data_tgt_idate[hddm_data_tgt_idate['time_since_last_reward']<tslr_upper]
        
        
        # try to load the analyzed
        try:

            # dummpy
            
            doNogazeOnly = 1
            
            current_dir = data_saved_folder+'data_saved_singlecam_wholebody_neural_and_hddm'+savefile_sufix+'/'+cameraID+'/'+animal_tgt
            add_date_dir = os.path.join(current_dir+'/hddm_model_fitted_allsession_withNeurons/'+session_toana)

            #
            if not doNogazeOnly:
                with open(add_date_dir+'/hddm_model_fitted_traces_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
                    hddm_model_fitted_traces = pickle.load(f)

                with open(add_date_dir+'/hddm_model_fitted_stats_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
                    hddm_model_fitted_stats = pickle.load(f)

                with open(add_date_dir+'/hddm_model_fitted_dic_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
                    hddm_model_fitted_dic = pickle.load(f)
            #
            with open(add_date_dir+'/hddm_model_fitted_nogaze_traces_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
                hddm_model_fitted_nogaze_traces = pickle.load(f)

            with open(add_date_dir+'/hddm_model_fitted_nogaze_stats_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
                hddm_model_fitted_nogaze_stats = pickle.load(f)

            with open(add_date_dir+'/hddm_model_fitted_nogaze_dic_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
                hddm_model_fitted_nogaze_dic = pickle.load(f)
                
            

            print(animal_tgt+' '+cond_toplot_type+' hddm fitting results loaded')

        except:

            print(animal_tgt+' '+cond_toplot_type+' running hddm fitting')
            
            try:
                # run the hddm nodel
                if not doExaustModel:
                    # HDDM settings - the basic setting
                    # samples=500 # 2000
                    # burn=100
                    # thin=2
                    samples=200 # 2000
                    burn=100
                    thin=2
                    #
                    if not doNogazeOnly:
                        hddm_model_fitted, hddm_model_fitted_nogaze = run_hddm_modeling(hddm_data_tgt_idate, animal_tgt,
                                                                                    samples, burn, thin, doNogazeOnly)
                    elif doNogazeOnly:
                        _, hddm_model_fitted_nogaze = run_hddm_modeling(hddm_data_tgt_idate, animal_tgt,
                                                                        samples, burn, thin, doNogazeOnly)

                elif doExaustModel:
                    # samples=2000 # 2000
                    # burn=1000
                    # thin=4
                    samples=200 # 2000
                    burn=100
                    thin=2
                    #
                    if not doNogazeOnly:
                        hddm_model_fitted, hddm_model_fitted_nogaze = run_hddm_modeling_exaustModel(hddm_data_tgt_idate, animal_tgt, 
                                                                                    samples, burn, thin, doNogazeOnly)
                    elif doNogazeOnly:
                        _, hddm_model_fitted_nogaze = run_hddm_modeling_exaustModel(hddm_data_tgt_idate, animal_tgt,
                                                                                    samples, burn, thin, doNogazeOnly)

            except:
                continue
                
            #
            if not doNogazeOnly:
                hddm_model_fitted_traces = hddm_model_fitted.get_traces()
                hddm_model_fitted_stats  = hddm_model_fitted.gen_stats()
                hddm_model_fitted_dic = hddm_model_fitted.dic_info
            #
            hddm_model_fitted_nogaze_traces = hddm_model_fitted_nogaze.get_traces()
            hddm_model_fitted_nogaze_stats  = hddm_model_fitted_nogaze.gen_stats()
            hddm_model_fitted_nogaze_dic = hddm_model_fitted_nogaze.dic_info

            # save data
            if 0:
                current_dir = data_saved_folder+'data_saved_singlecam_wholebody_neural_and_hddm'+savefile_sufix+'/'+cameraID+'/'+animal_tgt
                add_date_dir = os.path.join(current_dir+'/hddm_model_fitted_allsession_withNeurons/'+session_toana)
                if not os.path.exists(add_date_dir):
                    os.makedirs(add_date_dir)
                #
                if not doNogazeOnly:
                    with open(add_date_dir+'/hddm_model_fitted_traces_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'wb') as f:
                        pickle.dump(hddm_model_fitted_traces, f)

                    with open(add_date_dir+'/hddm_model_fitted_stats_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'wb') as f:
                        pickle.dump(hddm_model_fitted_stats, f)

                    with open(add_date_dir+'/hddm_model_fitted_dic_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'wb') as f:
                        pickle.dump(hddm_model_fitted_dic, f)

                #
                with open(add_date_dir+'/hddm_model_fitted_nogaze_traces_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'wb') as f:
                    pickle.dump(hddm_model_fitted_nogaze_traces, f)

                with open(add_date_dir+'/hddm_model_fitted_nogaze_stats_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'wb') as f:
                    pickle.dump(hddm_model_fitted_nogaze_stats, f)  

                with open(add_date_dir+'/hddm_model_fitted_nogaze_dic_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'wb') as f:
                    pickle.dump(hddm_model_fitted_nogaze_dic, f)
            
          
        #
        # reconstruct V
        #
        df_combined = hddm_data_tgt_idate # your combined trial-level dataframe
        traces = hddm_model_fitted_nogaze_traces  # or use `hddm_model_fitted_traces` if needed
        stats = hddm_model_fitted_nogaze_stats


        # --- STEP 1: Use all posterior samples to compute predicted v for each trial ---
        n_samples = len(traces)
        n_trials = len(df_combined)

        # Allocate space to hold predicted v for each sample x trial
        predicted_v_samples = np.zeros((n_samples, n_trials))

        # Pre-fetch covariate values as numpy arrays
        x1 = df_combined['partner_mean_speed'].values
        x2 = df_combined['partner_speed_std'].values
        x3 = df_combined['self_mean_speed'].values
        x4 = df_combined['self_speed_std'].values
        # x3 = df_combined['failed_pulls_before_reward'].values
        # x3 = df_combined['time_since_last_reward'].values
        # x7 = df_combined['self_gaze_auc'].values

        for i in range(n_samples):
            intercept = traces['v_Intercept'].values[i]
            coef_x1 = traces['v_partner_mean_speed'].values[i]
            coef_x2 = traces['v_partner_speed_std'].values[i]
            coef_x3 = traces['v_self_mean_speed'].values[i]
            coef_x4 = traces['v_self_speed_std'].values[i]
            # coef_x3 = traces['v_failed_pulls_before_reward'].values[i]
            # coef_x3 = traces['v_time_since_last_reward'].values[i]
            # coef_x7 = traces['v_self_gaze_auc'].values[i]

            # Apply regression formula for all trials
            predicted_v_samples[i, :] = (
                # intercept + coef_x1 * x1 + coef_x2 * x2 + coef_x3 * x3
                # intercept + coef_x1 * x1 
                # intercept + coef_x2 * x2 
                # intercept + coef_x1 * x1 + coef_x2 * x2 + coef_x3 * x3
                # intercept + coef_x1 * x1 + coef_x3 * x3
                intercept + coef_x1 * x1 + coef_x2 * x2 + coef_x3 * x3 + coef_x4 * x4
                # intercept + coef_x1 * x1 + coef_x2 * x2 + coef_x3 * x3 + coef_x4 * x4 + coef_x7 * x7
                # intercept + coef_x2 * x2 + coef_x4 * x4
            )

        # --- STEP 2: Compute posterior mean (and optionally CI) of predicted v ---
        df_with_v = df_combined.copy()
        df_with_v['predicted_v'] = predicted_v_samples.mean(axis=0)

        # Optional: also store credible intervals (e.g., 95% CI)
        df_with_v['predicted_v_lower'] = np.percentile(predicted_v_samples, 2.5, axis=0)
        df_with_v['predicted_v_upper'] = np.percentile(predicted_v_samples, 97.5, axis=0) 
        
        
        #
        # put data in the summarizing dataframe
        hddm_RawFullDatas_tgt_idate = hddm_RawFullDatas_tgt_alldates[hddm_RawFullDatas_tgt_alldates['dates']\
                                                                     == session_toana]
        neurons_idate = np.unique(hddm_RawFullDatas_tgt_idate['clusterID'])
        nneurons = np.shape(neurons_idate)[0]
        
        for ineuron in np.arange(0,nneurons,1):
            ineuron_idate = neurons_idate[ineuron]
            hddm_RawFullDatas_tgt_idate_ineuron = hddm_RawFullDatas_tgt_idate[hddm_RawFullDatas_tgt_idate['clusterID']\
                                                                              == ineuron_idate]
            #
            # mean fr
            # hddm_RawFullDatas_tgt_idate_ineuron['mean_fr'] = hddm_RawFullDatas_tgt_idate_ineuron['FR_ievent']\
            #                                                  .apply(np.mean)
            # hddm_RawFullDatas_tgt_idate_ineuron['mean_fr'] = hddm_RawFullDatas_tgt_idate_ineuron['FR_ievent']\
            #                                                     .apply(lambda x: np.mean(x[:int(0.85*fps)]))
            if animal_tgt == 'dodson':
                hddm_RawFullDatas_tgt_idate_ineuron['mean_fr'] = hddm_RawFullDatas_tgt_idate_ineuron['FR_ievent']\
                                                                .apply(lambda x: np.mean(x[:int(1*fps)]))
            else:
                hddm_RawFullDatas_tgt_idate_ineuron['mean_fr'] = hddm_RawFullDatas_tgt_idate_ineuron['FR_ievent']\
                                                                .apply(lambda x: np.mean(x[:int(1*fps)]))
            #
            # fr slope
            # === Fixed slope before 0.85s from pull ===
            pull_margin_frames = int(0.85 * fps)
            def calculate_slope(fr_trace):
                from scipy.stats import linregress
                #
                if isinstance(fr_trace, (list, np.ndarray)) and len(fr_trace) > pull_margin_frames:
                    y = fr_trace[:len(fr_trace) - pull_margin_frames]
                    x = np.arange(len(y))
                    try:
                        slope, *_ = linregress(x, y)
                        return abs(slope)
                    except:
                        return np.nan
                return np.nan
            
            #
            hddm_RawFullDatas_tgt_idate_ineuron['slope_fr'] = hddm_RawFullDatas_tgt_idate_ineuron['FR_ievent']\
                                                              .apply(calculate_slope)

            # merge the ddm fitting result and key bhv metric to it
            columns_to_merge = ['bhv_id', 'rt', 'partner_speed_std', 'predicted_v']
            #
            merged_df = pd.merge(
                                    left=hddm_RawFullDatas_tgt_idate_ineuron,
                                    right=df_with_v[columns_to_merge],
                                    how='left',
                                    on='bhv_id'
                                )
            
            
            #---------------------
            #
            # correlation between neuron measures and bhv measures
            #
            predictor_cols = ['rt', 'partner_speed_std', 'predicted_v']
            dependent_cols = ['mean_fr', 'slope_fr'] 
            
            # Check the correlations between your predictor variables
            # if ineuron == 0:
            #     print(merged_df[['rt', 'partner_speed_std', 'predicted_v']].corr())

            # --- Run the regression for each dependent variable ---
            for dep_var in dependent_cols:

                # --- 1. Initialize the result dictionary for this neuron and DV ---
                result = {
                    'date': merged_df['dates'].iloc[0],
                    'condition': merged_df['condition'].iloc[0],
                    'clusterID': merged_df['clusterID'].iloc[0],
                    'dependent_variable': dep_var,
                    'beta_rt': np.nan, 'pval_rt': np.nan,
                    'beta_partner_speed_std': np.nan, 'pval_partner_speed_std': np.nan,
                    'beta_predicted_v': np.nan, 'pval_predicted_v': np.nan
                }

                # --- 2. Loop through each predictor to run a separate single regression ---
                for predictor in predictor_cols:

                    # Define the simple formula (e.g., 'mean_fr ~ rt')
                    formula = f"{dep_var} ~ {predictor}"

                    # Clean data for this specific predictor and dependent variable
                    clean_df = merged_df.dropna(subset=[predictor, dep_var])

                    # Check for sufficient data (need at least 2 points for a simple regression)
                    if len(clean_df) > 1:
                        model = smf.ols(formula, data=clean_df).fit()

                        # Populate the result dict with the beta and p-value from this model
                        result[f'beta_{predictor}'] = model.params.get(predictor)
                        result[f'pval_{predictor}'] = model.pvalues.get(predictor)

                # --- 3. Append the fully populated result to the master list ---
                summary_list_for_each_neuron.append(result)
     
    #
    summary_list_for_each_neuron_df = pd.DataFrame(summary_list_for_each_neuron)
    
    #------------------
    # do the plotting
    from scipy import stats # Make sure stats is imported from scipy
    #
    # --- 2. Reshape the DataFrame from Wide to Long Format ---
    # This is the most important step for plotting
    long_df_list = []
    predictors = predictor_cols
    
    for pred in predictors:
        temp_df = summary_list_for_each_neuron_df[['dependent_variable', f'beta_{pred}', f'pval_{pred}']].copy()
        temp_df.rename(columns={f'beta_{pred}': 'beta_value', f'pval_{pred}': 'p_value'}, inplace=True)
        temp_df['predictor'] = pred
        long_df_list.append(temp_df)
    plot_df = pd.concat(long_df_list, ignore_index=True)

    # --- 3. Create the Two-Panel Figure ---
    sns.set_style("ticks")
    # Set sharey=False to allow independent y-axes
    fig, axes = plt.subplots(1, 2, figsize=(14, 7), sharey=False)
    fig.suptitle('Regression Coefficients for Neural Activity', fontsize=16)

    # --- Panel 1: Plot and Annotate for mean_fr ---
    axes[0].set_title('Dependent Variable: mean_fr')
    df_mean_fr = plot_df[plot_df['dependent_variable'] == 'mean_fr']
    sns.swarmplot(ax=axes[0], data=df_mean_fr, x='predictor', y='beta_value', color='gray', s=5)
    sns.swarmplot(ax=axes[0], data=df_mean_fr[df_mean_fr['p_value'] < 0.05], x='predictor', y='beta_value', color='red', s=5)

    # --- Panel 2: Plot and Annotate for slope_fr ---
    axes[1].set_title('Dependent Variable: slope_fr')
    # Filter for 'slope_fr'
    df_slope_fr = plot_df[plot_df['dependent_variable'] == 'slope_fr']
    sns.swarmplot(ax=axes[1], data=df_slope_fr, x='predictor', y='beta_value', color='gray', s=5)
    sns.swarmplot(ax=axes[1], data=df_slope_fr[df_slope_fr['p_value'] < 0.05], x='predictor', y='beta_value', color='red', s=5)

    # --- 4. Loop through axes and predictors to add annotations ---
    for i, ax in enumerate(axes):
        # Determine which data to use based on the panel index
        panel_df = df_mean_fr if i == 0 else df_slope_fr

        ymin, ymax = ax.get_ylim()

        for j, pred in enumerate(predictors):
            betas = panel_df[panel_df['predictor'] == pred]['beta_value'].dropna()

            if len(betas) > 1:
                mean_beta = betas.mean()
                t_stat, p_val = stats.ttest_1samp(betas, 0)

                p_text = f"p < 0.001" if p_val < 0.001 else f"p = {p_val:.3f}"
                annotation_text = f"mean β = {mean_beta:.2f}\n({p_text})"

                ax.text(j, ymax * 0.95, annotation_text, ha='center', va='top', fontsize=9, bbox=dict(facecolor='white', alpha=0.5, edgecolor='none'))

    # --- 5. Final Touches ---
    for ax in axes:
        ax.axhline(0, color='black', linestyle='--', linewidth=1)
        ax.set_xlabel('Predictor Variable')
        # Set the y-axis label for both panels
        ax.set_ylabel('Beta Coefficient (β)')
    sns.despine()
    plt.tight_layout(rect=[0, 0, 1, 0.96])
        
    # -------------------
    #
    # second figure, venn diagram    
    from matplotlib_venn import venn3
    # --- 1. Separate the results for each dependent variable ---
    df_mean_fr = summary_list_for_each_neuron_df[summary_list_for_each_neuron_df['dependent_variable'] == 'mean_fr']
    df_slope_fr = summary_list_for_each_neuron_df[summary_list_for_each_neuron_df['dependent_variable'] == 'slope_fr']
    #
    # --- 2. For each dependent variable, get the set of significant neuron IDs for each predictor ---
    # A unique neuron is identified by the combination of 'date' and 'clusterID'.
    # Sets for the 'mean_fr' model
    df_filtered_mean_rt = df_mean_fr[df_mean_fr['pval_rt'] < 0.05]
    mean_fr_rt_neurons = set(zip(df_filtered_mean_rt['date'], df_filtered_mean_rt['clusterID']))
    #
    df_filtered_mean_partner = df_mean_fr[df_mean_fr['pval_partner_speed_std'] < 0.05]
    mean_fr_partner_neurons = set(zip(df_filtered_mean_partner['date'], df_filtered_mean_partner['clusterID']))
    #
    df_filtered_mean_v = df_mean_fr[df_mean_fr['pval_predicted_v'] < 0.05]
    mean_fr_v_neurons = set(zip(df_filtered_mean_v['date'], df_filtered_mean_v['clusterID']))
    # Sets for the 'slope_fr' model
    df_filtered_slope_rt = df_slope_fr[df_slope_fr['pval_rt'] < 0.05]
    slope_fr_rt_neurons = set(zip(df_filtered_slope_rt['date'], df_filtered_slope_rt['clusterID']))
    #
    df_filtered_slope_partner = df_slope_fr[df_slope_fr['pval_partner_speed_std'] < 0.05]
    slope_fr_partner_neurons = set(zip(df_filtered_slope_partner['date'], df_filtered_slope_partner['clusterID']))
    #
    df_filtered_slope_v = df_slope_fr[df_slope_fr['pval_predicted_v'] < 0.05]
    slope_fr_v_neurons = set(zip(df_filtered_slope_v['date'], df_filtered_slope_v['clusterID']))
    #
    # --- 3. Create the new figure (fig2) with two panels for the Venn diagrams ---
    fig2, axes = plt.subplots(1, 2, figsize=(14, 7))
    fig2.suptitle('Overlap of Neurons with Significant Predictors (p < 0.05)', fontsize=16)
    # --- Panel 1: Venn Diagram for mean_fr ---
    ax1 = axes[0]
    venn3(
        subsets=[mean_fr_rt_neurons, mean_fr_partner_neurons, mean_fr_v_neurons],
        set_labels=('rt', 'partner_speed_std', 'predicted_v'),
        ax=ax1
    )
    ax1.set_title('Dependent Variable: mean_fr')
    # --- Panel 2: Venn Diagram for slope_fr ---
    ax2 = axes[1]
    venn3(
        subsets=[slope_fr_rt_neurons, slope_fr_partner_neurons, slope_fr_v_neurons],
        set_labels=('rt', 'partner_speed_std', 'predicted_v'),
        ax=ax2
    )
    ax2.set_title('Dependent Variable: slope_fr')
    
            
    #
    # savefig
    savefig = 0
    if savefig:
        current_dir = data_saved_folder+'fig_for_basic_neural_analysis_allsessions_basicEvents_PCA_Pullfocused_continuousBhv_partnerDistVaris_with_hddm_model'+savefile_sufix+'/'+animal_tgt
        add_fig_dir = os.path.join(current_dir,cameraID+'/fig_for_hddm_model_fitted_allsession_withNeurons')
        #
        
        fig.savefig(add_fig_dir+'/fr_correlation_with_predicted_v_bhv_variables_nogaze_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pdf', 
                        format='pdf', bbox_inches='tight')
        fig2.savefig(add_fig_dir+'/venn_fr_correlation_with_predicted_v_bhv_variables_nogaze_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pdf', 
                        format='pdf', bbox_inches='tight')


### ddm fitting with individual neurons' firing rate 

In [ ]:
# NOTE!!!
# run the hddm fitting using the data set that contained individual neuron
# the data set is calculated in Misha - basic_neural_analysis_allsessions_basicEvents_PCA_makeBhvNeuronVideos_PullStartToPull_section_continuousBhv

# this one is to fit the ddm in each sesssion and each neuron separately

# then combine it with the dataset that look at individual neurons' firing rate

if 1:
    # import warnings
    # warnings.filterwarnings('ignore')

    import seaborn as sns
    from scipy.stats import ttest_1samp

    # In this data set all the rt is calculated using the new definition, but the pull onset is defined slightly different
    #
    # if use the bhv PCA
    dobhvPCA = 1
    #
    doparnterleverspeed = 0 # this one only works when dobhvPCA is 0, it means use partner to lever speed that any speed
    #
    if not dobhvPCA:
        # if use onset of the first increase after min
        doOnsetAfterMin = 1
        if not doOnsetAfterMin:
            doOnsetAfterMin_suffix = ''
        elif doOnsetAfterMin:
            if not doparnterleverspeed:
                doOnsetAfterMin_suffix = 'PullOnsetAfterMin_'
            elif doparnterleverspeed:
                doOnsetAfterMin_suffix = 'PullOnsetAfterMin_PartnerLeverSpeed_'
    #
    elif dobhvPCA:
        # if use onset of the first increase after min
        doOnsetAfterMin = 1
        doBasedonSpeed = 1
        if not doOnsetAfterMin:
            doOnsetAfterMin_suffix = 'bhvPCA_'
        elif doOnsetAfterMin:
            if not doBasedonSpeed:
                doOnsetAfterMin_suffix = 'bhvPCA_PullOnsetAfterMin_'
            elif doBasedonSpeed:
                doOnsetAfterMin_suffix = 'bhvPCA_PullOnsetAfterMinBasedonSpeed_'
                

    # if use a hmm based method to find the trial start
    doHMMmethod = 0
    if doHMMmethod:
        doOnsetAfterMin_suffix = 'HMMmethods_'
        
    exaustModel_sufix = ''
        
    # load the dataset first
    current_dir = data_saved_folder+'data_saved_singlecam_wholebody_neural_and_hddm'+savefile_sufix+'/'+cameraID+'/'+animal_tgt
    add_date_dir = os.path.join(current_dir+'/hddm_model_fitted_combinedsession_withNeurons')
    
    if cond_toplot_type == 'allMC':
        if not (doOnsetAfterMin_suffix == 'bhvPCA_PullOnsetAfterMinBasedonSpeed_') |\
               (doOnsetAfterMin_suffix == 'PullOnsetAfterMin_PartnerLeverSpeed_'):
            with open(add_date_dir+'/hddm_datas_flexibleTW_newRTdefinition_'+doOnsetAfterMin_suffix+'withFRs_all_dates_'+
                      animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
                hddm_data_tgt_alldates = pickle.load(f)
        #
        elif (doOnsetAfterMin_suffix == 'bhvPCA_PullOnsetAfterMinBasedonSpeed_') |\
             (doOnsetAfterMin_suffix == 'PullOnsetAfterMin_PartnerLeverSpeed_'):
            with open(add_date_dir+'/hddm_datas_flexibleTW_newRTdefinition_'+'PullOnsetAfterMin_'+'withFRs_all_dates_'+
                      animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
                hddm_data_tgt_alldates = pickle.load(f)
    #
    else:
        if not (doOnsetAfterMin_suffix == 'bhvPCA_PullOnsetAfterMinBasedonSpeed_') |\
               (doOnsetAfterMin_suffix == 'PullOnsetAfterMin_PartnerLeverSpeed_'):
            with open(add_date_dir+'/hddm_datas_flexibleTW_newRTdefinition_'+doOnsetAfterMin_suffix+'withFRs_all_dates_'+
                      animal1_fixedorders[0]+animal2_fixedorders[0]+'_'+cond_toplot_type+'.pkl', 'rb') as f:
                hddm_data_tgt_alldates = pickle.load(f)
        #
        elif (doOnsetAfterMin_suffix == 'bhvPCA_PullOnsetAfterMinBasedonSpeed_') |\
             (doOnsetAfterMin_suffix == 'PullOnsetAfterMin_PartnerLeverSpeed_'):
            with open(add_date_dir+'/hddm_datas_flexibleTW_newRTdefinition_'+'PullOnsetAfterMin_'+'withFRs_all_dates_'+
                      animal1_fixedorders[0]+animal2_fixedorders[0]+'_'+cond_toplot_type+'.pkl', 'rb') as f:
                hddm_data_tgt_alldates = pickle.load(f)
            

    # make the hddm_data structure consistent with other code
    ind_goodcond = np.isin(hddm_data_tgt_alldates['condition'],conditions_to_ana)
    hddm_data_tgt_alldates = hddm_data_tgt_alldates[ind_goodcond]

    if not dobhvPCA:
        if not doparnterleverspeed:
            rename_dict = {
                'act_animal':'subj_idx',
                'dates': 'date',
                'pull_rt': 'rt',
                'previous_pull_outcome': 'prev_trial_outcome',
                'socialgaze_auc': 'self_gaze_auc',
                'mass_move_speed_mean': 'self_mean_speed',
                'mass_move_speed_std': 'self_speed_std',
                'other_mass_move_speed_mean': 'partner_mean_speed',
                'other_mass_move_speed_std': 'partner_speed_std',
                'num_preceding_failpull': 'failed_pulls_before_reward',
                'time_from_last_reward': 'time_since_last_reward',
                }
        elif doparnterleverspeed:
            rename_dict = {
                'act_animal':'subj_idx',
                'dates': 'date',
                'pull_rt': 'rt',
                'previous_pull_outcome': 'prev_trial_outcome',
                'socialgaze_auc': 'self_gaze_auc',
                'mass_move_speed_mean': 'self_mean_speed',
                'mass_move_speed_std': 'self_speed_std',
                'other_lever_speed_mean': 'partner_mean_speed',
                'other_lever_speed_std': 'partner_speed_std',
                'num_preceding_failpull': 'failed_pulls_before_reward',
                'time_from_last_reward': 'time_since_last_reward',
                }
    elif dobhvPCA:
        rename_dict = {
            'act_animal':'subj_idx',
            'dates': 'date',
            'pull_rt': 'rt',
            'previous_pull_outcome': 'prev_trial_outcome',
            'socialgaze_auc': 'self_gaze_auc',
            # 'self_PC1_mean': 'self_mean_speed',
            # 'self_PC1_std': 'self_speed_std',
            'mass_move_speed_mean': 'self_mean_speed',
            'mass_move_speed_std': 'self_speed_std',
            'other_PC1_mean': 'partner_mean_speed',
            'other_PC1_std': 'partner_speed_std',
            'num_preceding_failpull': 'failed_pulls_before_reward',
            'time_from_last_reward': 'time_since_last_reward',
            }
        
    # Apply the renaming
    hddm_data_tgt_alldates = hddm_data_tgt_alldates.rename(columns=rename_dict)
    hddm_data_tgt_alldates['response'] = 1

    
    #
    # calculated the new partner_mean_speed and partner_speed_std with self_gaze filter
    #
    hddm_data_tgt_alldates['partner_speed_std_nofilter'] = hddm_data_tgt_alldates['partner_speed_std']
    hddm_data_tgt_alldates['partner_mean_speed_nofilter'] = hddm_data_tgt_alldates['partner_mean_speed']
    
    #
    ntrials = np.shape(hddm_data_tgt_alldates)[0]
    for itrial in np.arange(0,ntrials,1):
        gaze_prob_itrial = hddm_data_tgt_alldates['socialgaze_prob'].iloc[itrial]
        #
        if not dobhvPCA:
            if not doparnterleverspeed:
                partner_speed_itrial = hddm_data_tgt_alldates['other_mass_move_speed'].iloc[itrial]
            elif doparnterleverspeed:
                partner_speed_itrial = hddm_data_tgt_alldates['other_lever_speed'].iloc[itrial]
        elif dobhvPCA:
            partner_speed_itrial = hddm_data_tgt_alldates['other_PC1'].iloc[itrial]
        #
        # the shape of 'other_lever_speed' and other variables might be different
        partner_speed_shape = np.shape(partner_speed_itrial)[0]
        gaze_prob_shape = np.shape(gaze_prob_itrial)[0]
        if partner_speed_shape > gaze_prob_shape:
            partner_speed_itrial = partner_speed_itrial[np.arange(0,gaze_prob_shape,1)]
        elif partner_speed_shape < gaze_prob_shape:
            gaze_prob_itrial = gaze_prob_itrial[np.arange(0,partner_speed_shape,1)]

        # smooth the gaze duration a bit more
        gaze_prob_itrial = gaussian_filter1d(gaze_prob_itrial, sigma=8) # the final sigma would be sqrt(4*4+sigma*sigma)

        partner_speed_filtered = partner_speed_itrial[gaze_prob_itrial>0]

        partner_mean_speed_filtered = np.nanmean(partner_speed_filtered)
        partner_std_speed_filtered = np.nanstd(partner_speed_filtered)
        
        hddm_data_tgt_alldates['partner_mean_speed'].iloc[itrial] = partner_mean_speed_filtered
        hddm_data_tgt_alldates['partner_speed_std'].iloc[itrial] = partner_std_speed_filtered
        
    # 
    # remove nan row first for the glm data set
    hddm_data_tgt_alldates = hddm_data_tgt_alldates.dropna()
    
    # 
    # normalize all the predictors
    from scipy.stats import zscore
    #
    # List of predictor columns to normalize (exclude 'rt' and identifiers)
    predictor_columns = [
        'self_gaze_auc',
        'partner_mean_speed',
        'self_mean_speed',
        'partner_speed_std',
        'self_speed_std',
        'failed_pulls_before_reward',
        'time_since_last_reward'
    ]
    # Rename originals and replace with z-scored values
    for col in predictor_columns:
        hddm_data_tgt_alldates[f"{col}_origin"] = hddm_data_tgt_alldates[col]
        hddm_data_tgt_alldates[col] = zscore(hddm_data_tgt_alldates[col])  
        # hddm_data_tgt_alldates[col] = hddm_data_tgt_alldates[col]  
      
    #
    # initialize a dataframe for summarizing all sessions results
    hddm_result_sumary_tgt_alldates_allneurons = []
    
    
    # run the fitting for each session
    for session_toana, hddm_data_tgt_idate in hddm_data_tgt_alldates.groupby('date'):
    
        ind_ = np.isin(dates_list,session_toana)
        task_cond = task_conditions[np.where(ind_)[0][0]]
        
        # Inside the loop:
        # 'session_toana' is the date for the current group
        # 'hddm_data_tgt_idate' is a DataFrame containing only the rows for that date
    
        #
        # # add some semi-artbitrary cretieras
        # hddm_data_tgt_alldates = hddm_data_tgt_idate[hddm_data_tgt_alldates['prev_trial_outcome']==1]
        # hddm_data_tgt_alldates = hddm_data_tgt_idate[hddm_data_tgt_alldates['partner_mean_speed']<=300]
        #
        lower_q = 0.05
        upper_q = 0.95
        rt_lower, rt_upper = hddm_data_tgt_idate['rt'].quantile([lower_q, upper_q])
        hddm_data_tgt_idate = hddm_data_tgt_idate[hddm_data_tgt_idate['rt']>rt_lower]
        hddm_data_tgt_idate = hddm_data_tgt_idate[hddm_data_tgt_idate['rt']<rt_upper]
        #
        tslr_lower, tslr_upper = hddm_data_tgt_idate['time_since_last_reward'].quantile([lower_q, upper_q])
        hddm_data_tgt_idate = hddm_data_tgt_idate[hddm_data_tgt_idate['time_since_last_reward']>tslr_lower]
        hddm_data_tgt_idate = hddm_data_tgt_idate[hddm_data_tgt_idate['time_since_last_reward']<tslr_upper]    
        
        #
        # put data in the summarizing dataframe
        hddm_RawFullDatas_tgt_idate = hddm_RawFullDatas_tgt_alldates[hddm_RawFullDatas_tgt_alldates['dates']\
                                                                     == session_toana]
        neurons_idate = np.unique(hddm_RawFullDatas_tgt_idate['clusterID'])
        nneurons = np.shape(neurons_idate)[0]
        
        for ineuron in np.arange(0,nneurons,1):
            ineuron_idate = neurons_idate[ineuron]
            hddm_RawFullDatas_tgt_idate_ineuron = hddm_RawFullDatas_tgt_idate[hddm_RawFullDatas_tgt_idate['clusterID']\
                                                                              == ineuron_idate]
            #
            # mean fr
            # hddm_RawFullDatas_tgt_idate_ineuron['mean_fr'] = hddm_RawFullDatas_tgt_idate_ineuron['FR_ievent']\
            #                                                  .apply(np.mean)
            # hddm_RawFullDatas_tgt_idate_ineuron['mean_fr'] = hddm_RawFullDatas_tgt_idate_ineuron['FR_ievent']\
            #                                                     .apply(lambda x: np.mean(x[:int(0.85*fps)]))
            if animal_tgt == 'dodson':
                hddm_RawFullDatas_tgt_idate_ineuron['mean_fr'] = hddm_RawFullDatas_tgt_idate_ineuron['FR_ievent']\
                                                                .apply(lambda x: np.mean(x[:int(0.5*fps)]))
            else:
                hddm_RawFullDatas_tgt_idate_ineuron['mean_fr'] = hddm_RawFullDatas_tgt_idate_ineuron['FR_ievent']\
                                                                .apply(lambda x: np.mean(x[:int(1*fps)]))
            
            #
            # fr slope
            # === Fixed slope before 0.85s from pull ===
            pull_margin_frames = int(0.85 * fps)
            def calculate_slope(fr_trace):
                from scipy.stats import linregress
                #
                if isinstance(fr_trace, (list, np.ndarray)) and len(fr_trace) > pull_margin_frames:
                    y = fr_trace[:len(fr_trace) - pull_margin_frames]
                    x = np.arange(len(y))
                    try:
                        slope, *_ = linregress(x, y)
                        return abs(slope)
                    except:
                        return np.nan
                return np.nan
            
            #
            hddm_RawFullDatas_tgt_idate_ineuron['slope_fr'] = hddm_RawFullDatas_tgt_idate_ineuron['FR_ievent']\
                                                              .apply(calculate_slope)

            # merge the ddm fitting result and key bhv metric to it
            columns_to_merge = ['bhv_id', 'rt', 'partner_speed_std', 'response']
            #
            merged_df = pd.merge(
                                    left=hddm_RawFullDatas_tgt_idate_ineuron,
                                    right=hddm_data_tgt_idate[columns_to_merge],
                                    how='left',
                                    on='bhv_id'
                                )
            
            #
            # clear up the merge_df
            #
            # remove nan row first for the glm data set
            merged_df = merged_df.dropna()
            # 
            # normalize all the predictors
            from scipy.stats import zscore
            #
            # List of predictor columns to normalize (exclude 'rt' and identifiers)
            predictor_columns = [
                'mean_fr',
                'slope_fr'
            ]
            # Rename originals and replace with z-scored values
            for col in predictor_columns:
                merged_df[f"{col}_origin"] = merged_df[col]
                merged_df[col] = zscore(merged_df[col])  
                # hddm_data_tgt_alldates[col] = hddm_data_tgt_alldates[col] 
            
            
            # for reviewer1's question
            # - whether ramp slope predicts choice correctness beyond its correlation with RT.
            MIN_TRIALS_PER_GROUP = 5
            
            if (np.sum(merged_df['pull_outcome'] == 1)>=MIN_TRIALS_PER_GROUP) and \
               (np.sum(merged_df['pull_outcome'] == 0)>=MIN_TRIALS_PER_GROUP):
                
                try:
                    from statsmodels.formula.api import logit
                    from statsmodels.formula.api import probit

                    # correctness_model = probit(
                    correctness_model = logit(
                        'pull_outcome ~ slope_fr + rt',
                        data=merged_df
                    ).fit(disp=False)

                    slope_fr_correctness_coef = correctness_model.params['slope_fr']
                    slope_fr_correctness_pval = correctness_model.pvalues['slope_fr']

                except:
                    
                    # from statsmodels.formula.api import logit
                    # from statsmodels.formula.api import probit

                    # # correctness_model = probit(
                    # correctness_model = logit(
                    #     'pull_outcome ~ slope_fr + rt',
                    #     data=merged_df
                    # ).fit(disp=False)
                    
                    
                    slope_fr_correctness_coef = np.nan
                    slope_fr_correctness_pval = np.nan
            
            else:
                
                # from statsmodels.formula.api import logit
                # from statsmodels.formula.api import probit

                # # correctness_model = probit(
                # correctness_model = logit(
                #     'pull_outcome ~ slope_fr + rt',
                #     data=merged_df
                # ).fit(disp=False)
                    
                slope_fr_correctness_coef = np.nan
                slope_fr_correctness_pval = np.nan
                    
            
            # for the same reviewer1's quesiton but use partial regression methods
            from scipy.stats import linregress, mannwhitneyu

            MIN_TRIALS_PER_GROUP = 5   # change if needed
            ALTERNATIVE = 'two-sided'  # 'two-sided' is safest

            # Default outputs (in case of failure / low trial count)
            slope_fr_residRT_mw_medianDiff = np.nan
            slope_fr_residRT_mw_pval = np.nan
            slope_fr_residRT_mw_significant = False
            slope_fr_residRT_mw_n_success = np.nan
            slope_fr_residRT_mw_n_fail = np.nan

            # Clean minimal df
            df_tmp = merged_df.dropna(subset=['slope_fr', 'rt', 'pull_outcome']).copy()
            df_tmp['pull_outcome'] = df_tmp['pull_outcome'].astype(int)

            # Residualize slope_fr against rt
            x = df_tmp['rt'].values
            y = df_tmp['slope_fr'].values

            if (len(df_tmp) >= 2 * MIN_TRIALS_PER_GROUP) and (np.nanstd(x) > 0):
                lr = linregress(x, y)
                y_hat = lr.intercept + lr.slope * x
                df_tmp['slope_fr_resid_rt'] = y - y_hat

                # Put residuals back onto merged_df (aligned by index)
                merged_df.loc[df_tmp.index, 'slope_fr_resid_rt'] = df_tmp['slope_fr_resid_rt']

                # Mann–Whitney on residuals (success vs fail)
                resid_success = df_tmp.loc[df_tmp['pull_outcome'] == 1, 'slope_fr_resid_rt'].values
                resid_fail    = df_tmp.loc[df_tmp['pull_outcome'] == 0, 'slope_fr_resid_rt'].values

                slope_fr_residRT_mw_n_success = len(resid_success)
                slope_fr_residRT_mw_n_fail = len(resid_fail)

                if (slope_fr_residRT_mw_n_success >= MIN_TRIALS_PER_GROUP) and (slope_fr_residRT_mw_n_fail >= MIN_TRIALS_PER_GROUP):
                    mw = mannwhitneyu(resid_success, resid_fail, alternative=ALTERNATIVE)

                    slope_fr_residRT_mw_pval = mw.pvalue
                    slope_fr_residRT_mw_medianDiff = np.nanmedian(resid_success) - np.nanmedian(resid_fail)
                    slope_fr_residRT_mw_significant = bool(slope_fr_residRT_mw_pval < 0.05)
                
                
            # for the same reviewer1's quesiton but use auc on the residual
            from sklearn.metrics import roc_auc_score
            
            MIN_TRIALS_PER_CLASS = MIN_TRIALS_PER_GROUP

            # ---- defaults ----
            auc_M0_rt = np.nan
            auc_M01_evidence = np.nan
            auc_M1_evidence_residRT = np.nan
            auc_M2_slope_residRT = np.nan
            auc_M3_slope_residEvidenceRT = np.nan
            auc_M4_slope_residEvidence = np.nan

            valid_M0_rt = False
            valid_M01_evidence = False
            valid_M1_evidence_residRT = False
            valid_M2_slope_residRT = False
            valid_M3_slope_residEvidenceRT = False
            valid_M4_slope_residEvidence = False

            aucSuite_valid = False
            n_success = np.nan
            n_fail = np.nan

            # ---- prepare data ----
            df_tmp = merged_df.dropna(
                subset=['pull_outcome', 'rt', 'slope_fr', 'partner_speed_std']
            ).copy()

            df_tmp['pull_outcome'] = df_tmp['pull_outcome'].astype(int)
            df_tmp['social_evidence'] = -df_tmp['partner_speed_std'].values

            n_success = int(np.sum(df_tmp['pull_outcome'] == 1))
            n_fail    = int(np.sum(df_tmp['pull_outcome'] == 0))

            def safe_auc(y, score):
                if np.std(score) == 0:
                    return np.nan
                return roc_auc_score(y, score)

            # ---- global validity checks (shared across models) ----
            basic_valid = True
            if (n_success < MIN_TRIALS_PER_CLASS) or (n_fail < MIN_TRIALS_PER_CLASS):
                basic_valid = False
            if np.std(df_tmp['rt'].values) == 0:
                basic_valid = False
            if np.std(df_tmp['social_evidence'].values) == 0:
                basic_valid = False
            if np.std(df_tmp['slope_fr'].values) == 0:
                basic_valid = False

            # ---- run models only if basic checks pass ----
            if basic_valid:

                y = df_tmp['pull_outcome'].values

                # ===== M0: RT only =====
                auc_M0_rt = safe_auc(y, df_tmp['rt'].values)
                valid_M0_rt = np.isfinite(auc_M0_rt)
                
                # ===== M0-1: social evidence only =====
                auc_M01_evidence = safe_auc(y, df_tmp['social_evidence'].values)
                valid_M01_evidence = np.isfinite(auc_M01_evidence)

                # ===== M1: Evidence residualized by RT =====
                lr_e_rt = linregress(df_tmp['rt'].values, df_tmp['social_evidence'].values)
                E_hat_rt = lr_e_rt.intercept + lr_e_rt.slope * df_tmp['rt'].values
                df_tmp['evidence_resid_rt'] = df_tmp['social_evidence'].values - E_hat_rt
                auc_M1_evidence_residRT = safe_auc(y, df_tmp['evidence_resid_rt'].values)
                valid_M1_evidence_residRT = np.isfinite(auc_M1_evidence_residRT)

                # ===== M2: Slope residualized by RT =====
                lr_s_rt = linregress(df_tmp['rt'].values, df_tmp['slope_fr'].values)
                S_hat_rt = lr_s_rt.intercept + lr_s_rt.slope * df_tmp['rt'].values
                df_tmp['slope_resid_rt'] = df_tmp['slope_fr'].values - S_hat_rt
                auc_M2_slope_residRT = safe_auc(y, df_tmp['slope_resid_rt'].values)
                valid_M2_slope_residRT = np.isfinite(auc_M2_slope_residRT)

                # ===== M4: Slope residualized by Evidence =====
                lr_s_e = linregress(df_tmp['social_evidence'].values, df_tmp['slope_fr'].values)
                S_hat_e = lr_s_e.intercept + lr_s_e.slope * df_tmp['social_evidence'].values
                df_tmp['slope_resid_evidence'] = df_tmp['slope_fr'].values - S_hat_e
                auc_M4_slope_residEvidence = safe_auc(y, df_tmp['slope_resid_evidence'].values)
                valid_M4_slope_residEvidence = np.isfinite(auc_M4_slope_residEvidence)

                # ===== M3: Slope residualized by Evidence + RT =====
                X = np.column_stack([
                    np.ones(len(df_tmp)),
                    df_tmp['social_evidence'].values,
                    df_tmp['rt'].values
                ])
                beta, *_ = np.linalg.lstsq(X, df_tmp['slope_fr'].values, rcond=None)
                S_hat_e_rt = X @ beta
                df_tmp['slope_resid_evidence_rt'] = df_tmp['slope_fr'].values - S_hat_e_rt
                auc_M3_slope_residEvidenceRT = safe_auc(y, df_tmp['slope_resid_evidence_rt'].values)
                valid_M3_slope_residEvidenceRT = np.isfinite(auc_M3_slope_residEvidenceRT)

                # ---- master validity: all core models valid ----
                aucSuite_valid = (
                    valid_M0_rt and
                    valid_M1_evidence_residRT and
                    valid_M2_slope_residRT and
                    valid_M3_slope_residEvidenceRT
                )

            # ---- store results ----
            result['aucSuite_valid'] = aucSuite_valid
            result['aucSuite_n_success'] = n_success
            result['aucSuite_n_fail'] = n_fail

            result['auc_M0_rt'] = auc_M0_rt - 0.5
            result['auc_M01_evidence'] = auc_M01_evidence - 0.5
            result['auc_M1_evidence_residRT'] = auc_M1_evidence_residRT - 0.5
            result['auc_M2_slope_residRT'] = auc_M2_slope_residRT - 0.5
            result['auc_M3_slope_residEvidenceRT'] = auc_M3_slope_residEvidenceRT - 0.5
            result['auc_M4_slope_residEvidence'] = auc_M4_slope_residEvidence - 0.5

            result['valid_M0_rt'] = valid_M0_rt
            result['valid_M01_evidence'] = valid_M01_evidence
            result['valid_M1_evidence_residRT'] = valid_M1_evidence_residRT
            result['valid_M2_slope_residRT'] = valid_M2_slope_residRT
            result['valid_M3_slope_residEvidenceRT'] = valid_M3_slope_residEvidenceRT
            result['valid_M4_slope_residEvidence'] = valid_M4_slope_residEvidence
            
            
            # # # 
            # ===== Check: Is slope variance (conditional on evidence) larger on failed trials? =====
            # Idea: within each neuron×session (merged_df), fit slope_fr ~ social_evidence separately
            # for success vs fail trials, then compare residual variance (Var(resid)) between groups.
            #
            # This answers: "conditional on evidence, is slope more variable in failed trials?"
            # It naturally handles unequal trial counts because variance is computed within-group.
            
            # ---- defaults ----
            var_slope_given_evidence_success = np.nan
            var_slope_given_evidence_fail = np.nan
            ratio_var_fail_over_success = np.nan
            log_ratio_var_fail_over_success = np.nan
            var_condEvidence_valid = False
            n_success = np.nan
            n_fail = np.nan

            # ---- prepare data ----
            df_tmp = merged_df.dropna(subset=['pull_outcome', 'slope_fr', 'partner_speed_std']).copy()
            df_tmp['pull_outcome'] = df_tmp['pull_outcome'].astype(int)
            df_tmp['social_evidence'] = -df_tmp['partner_speed_std'].values

            # class counts
            n_success = int(np.sum(df_tmp['pull_outcome'] == 1))
            n_fail    = int(np.sum(df_tmp['pull_outcome'] == 0))

            # enough data?
            if (n_success >= MIN_TRIALS_PER_CLASS) and (n_fail >= MIN_TRIALS_PER_CLASS) and (len(df_tmp) >= 2 * MIN_TRIALS_PER_CLASS):

                # split groups
                df_succ = df_tmp[df_tmp['pull_outcome'] == 1]
                df_fail = df_tmp[df_tmp['pull_outcome'] == 0]

                # require evidence variance within each group, otherwise regression/residuals are degenerate
                ok_succ = (np.std(df_succ['social_evidence'].values) > 0) and (np.std(df_succ['slope_fr'].values) > 0) and (len(df_succ) >= 3)
                ok_fail = (np.std(df_fail['social_evidence'].values) > 0) and (np.std(df_fail['slope_fr'].values) > 0) and (len(df_fail) >= 3)

                if ok_succ and ok_fail:
                    # --- Success: slope_fr ~ evidence ---
                    lr_succ = linregress(df_succ['social_evidence'].values, df_succ['slope_fr'].values)
                    yhat_succ = lr_succ.intercept + lr_succ.slope * df_succ['social_evidence'].values
                    resid_succ = df_succ['slope_fr'].values - yhat_succ

                    # --- Fail: slope_fr ~ evidence ---
                    lr_fail = linregress(df_fail['social_evidence'].values, df_fail['slope_fr'].values)
                    yhat_fail = lr_fail.intercept + lr_fail.slope * df_fail['social_evidence'].values
                    resid_fail = df_fail['slope_fr'].values - yhat_fail

                    # conditional variance = residual variance (use ddof=1 for sample variance)
                    var_slope_given_evidence_success = np.var(resid_succ, ddof=1) if len(resid_succ) > 1 else np.nan
                    var_slope_given_evidence_fail    = np.var(resid_fail, ddof=1) if len(resid_fail) > 1 else np.nan

                    if np.isfinite(var_slope_given_evidence_success) and np.isfinite(var_slope_given_evidence_fail) and (var_slope_given_evidence_success > 0):
                        ratio_var_fail_over_success = var_slope_given_evidence_fail / var_slope_given_evidence_success
                        log_ratio_var_fail_over_success = np.log(ratio_var_fail_over_success)

                        var_condEvidence_valid = True

            # ---- store into result dict (names chosen to avoid overlap) ----
            result['varSlope_condEvidence_valid'] = var_condEvidence_valid
            result['varSlope_condEvidence_n_success'] = n_success
            result['varSlope_condEvidence_n_fail'] = n_fail

            result['varSlope_givenEvidence_success'] = var_slope_given_evidence_success
            result['varSlope_givenEvidence_fail'] = var_slope_given_evidence_fail
            result['varSlope_fail_over_success_ratio'] = ratio_var_fail_over_success
            result['varSlope_fail_over_success_logratio'] = log_ratio_var_fail_over_success
            
            
            
            # fit the ddm model considering the neuron activity
            #
            # try to load the analyzed
            try:

                # dummpy

                current_dir = data_saved_folder+'data_saved_singlecam_wholebody_neural_and_hddm'+savefile_sufix+'/'+cameraID+'/'+animal_tgt
                add_date_dir = os.path.join(current_dir+'/hddm_model_fitted_allsession_withNeurons/'+session_toana+'/'+ineuron_idate)

                #
                # with open(add_date_dir+'/hddm_model_fitted_withNeuronActivity_shortmean_traces_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
                #     hddm_model_fitted_withNeuronActivity_traces = pickle.load(f)
                # with open(add_date_dir+'/hddm_model_fitted_withNeuronActivity_shortmean_stats_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
                #     hddm_model_fitted_withNeuronActivity_stats = pickle.load(f)
                # with open(add_date_dir+'/hddm_model_fitted_withNeuronActivity_shortmean_dic_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
                #     hddm_model_fitted_withNeuronActivity_dic = pickle.load(f)
                    
                with open(add_date_dir+'/hddm_model_fitted_withNeuronActivity_traces_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
                    hddm_model_fitted_withNeuronActivity_traces = pickle.load(f)
                with open(add_date_dir+'/hddm_model_fitted_withNeuronActivity_stats_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
                    hddm_model_fitted_withNeuronActivity_stats = pickle.load(f)
                with open(add_date_dir+'/hddm_model_fitted_withNeuronActivity_dic_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
                    hddm_model_fitted_withNeuronActivity_dic = pickle.load(f)

                print(animal_tgt+' '+cond_toplot_type+' hddm fitting results loaded')

            except:

                print(animal_tgt+' '+cond_toplot_type+' running hddm fitting with neuronal activity')

                try:

                    # run the hddm nodel
                    # HDDM settings - the basic setting
                    # samples=500 # 2000
                    # burn=100
                    # thin=2
                    samples=200 # 2000
                    burn=100
                    thin=2
                    #
                    doNogazeOnly = 1
                    #
                    hddm_model_fitted_withNeuronActivity = run_hddm_modeling_withNeuroActivity(merged_df, 
                                                                            animal_tgt,
                                                                            samples, burn, thin)

                    #
                    hddm_model_fitted_withNeuronActivity_traces = hddm_model_fitted_withNeuronActivity.get_traces()
                    hddm_model_fitted_withNeuronActivity_stats  = hddm_model_fitted_withNeuronActivity.gen_stats()
                    hddm_model_fitted_withNeuronActivity_dic = hddm_model_fitted_withNeuronActivity.dic_info

                except:
                     continue

                # save data
                if 1:
                    current_dir = data_saved_folder+'data_saved_singlecam_wholebody_neural_and_hddm'+savefile_sufix+'/'+cameraID+'/'+animal_tgt
                    add_date_dir = os.path.join(current_dir+'/hddm_model_fitted_allsession_withNeurons/'+session_toana+'/'+ineuron_idate)
                    if not os.path.exists(add_date_dir):
                        os.makedirs(add_date_dir)
                    #
                    with open(add_date_dir+'/hddm_model_fitted_withNeuronActivity_shortmean_traces_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'wb') as f:
                        pickle.dump(hddm_model_fitted_withNeuronActivity_traces, f)
                    with open(add_date_dir+'/hddm_model_fitted_withNeuronActivity_shortmean_stats_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'wb') as f:
                        pickle.dump(hddm_model_fitted_withNeuronActivity_stats, f)  
                    with open(add_date_dir+'/hddm_model_fitted_withNeuronActivity_shortmean_dic_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'wb') as f:
                        pickle.dump(hddm_model_fitted_withNeuronActivity_dic, f)
                    
                    # with open(add_date_dir+'/hddm_model_fitted_withNeuronActivity_traces_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'wb') as f:
                    #     pickle.dump(hddm_model_fitted_withNeuronActivity_traces, f)
                    # with open(add_date_dir+'/hddm_model_fitted_withNeuronActivity_stats_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'wb') as f:
                    #     pickle.dump(hddm_model_fitted_withNeuronActivity_stats, f)  
                    # with open(add_date_dir+'/hddm_model_fitted_withNeuronActivity_dic_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'wb') as f:
                    #     pickle.dump(hddm_model_fitted_withNeuronActivity_dic, f)
                    
                     
            # 
            # neuron ddm without z 
            #
            # try to load the analyzed
            try:

                # dummpy

                current_dir = data_saved_folder+'data_saved_singlecam_wholebody_neural_and_hddm'+savefile_sufix+'/'+cameraID+'/'+animal_tgt
                add_date_dir = os.path.join(current_dir+'/hddm_model_fitted_allsession_withNeurons/'+session_toana+'/'+ineuron_idate)

                #
                with open(add_date_dir+'/hddm_model_fitted_withNeuronActivity_noz_traces_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
                    hddm_model_fitted_withNeuronActivity_noz_traces = pickle.load(f)
                with open(add_date_dir+'/hddm_model_fitted_withNeuronActivity_noz_stats_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
                    hddm_model_fitted_withNeuronActivity_noz_stats = pickle.load(f)
                with open(add_date_dir+'/hddm_model_fitted_withNeuronActivity_noz_dic_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'rb') as f:
                    hddm_model_fitted_withNeuronActivity_noz_dic = pickle.load(f)

                print(animal_tgt+' '+cond_toplot_type+' hddm fitting results loaded')

            except:

                print(animal_tgt+' '+cond_toplot_type+' running hddm fitting with neuronal activity, no consideration of z')

                try:

                    # run the hddm nodel
                    # HDDM settings - the basic setting
                    # samples=500 # 2000
                    # burn=100
                    # thin=2
                    samples=200 # 2000
                    burn=100
                    thin=2
                    #
                    hddm_model_fitted_withNeuronActivity = run_hddm_modeling_withNeuroActivity_noz(merged_df, 
                                                                            animal_tgt,
                                                                            samples, burn, thin)

                    #
                    hddm_model_fitted_withNeuronActivity_noz_traces = hddm_model_fitted_withNeuronActivity.get_traces()
                    hddm_model_fitted_withNeuronActivity_noz_stats  = hddm_model_fitted_withNeuronActivity.gen_stats()
                    hddm_model_fitted_withNeuronActivity_noz_dic = hddm_model_fitted_withNeuronActivity.dic_info

                except:
                     continue

                # save data
                if 1:
                    current_dir = data_saved_folder+'data_saved_singlecam_wholebody_neural_and_hddm'+savefile_sufix+'/'+cameraID+'/'+animal_tgt
                    add_date_dir = os.path.join(current_dir+'/hddm_model_fitted_allsession_withNeurons/'+session_toana+'/'+ineuron_idate)
                    if not os.path.exists(add_date_dir):
                        os.makedirs(add_date_dir)
                    #
                    with open(add_date_dir+'/hddm_model_fitted_withNeuronActivity_noz_traces_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'wb') as f:
                        pickle.dump(hddm_model_fitted_withNeuronActivity_noz_traces, f)
                    with open(add_date_dir+'/hddm_model_fitted_withNeuronActivity_noz_stats_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'wb') as f:
                        pickle.dump(hddm_model_fitted_withNeuronActivity_noz_stats, f)  
                    with open(add_date_dir+'/hddm_model_fitted_withNeuronActivity_noz_dic_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pkl', 'wb') as f:
                        pickle.dump(hddm_model_fitted_withNeuronActivity_noz_dic, f)
    
                    
                        
            
            # put the result together across sessions and neurons
            # --- 1. Initialize the result dictionary for this neuron and DV ---
            result = {
                'date': session_toana,
                'condition': task_cond,
                'clusterID': ineuron_idate,
                'modelDIC': hddm_model_fitted_withNeuronActivity_dic['DIC'],
                'modelPD': hddm_model_fitted_withNeuronActivity_dic['pD'],
                'modelDIC_noz': hddm_model_fitted_withNeuronActivity_noz_dic['DIC'],
            }
            
            # We now iterate over the DataFrame's index (the variable names) and rows.
            for variable_name, row in hddm_model_fitted_withNeuronActivity_stats.iterrows():

                # The significance logic remains the same.
                is_significant = not (row['2.5q'] < 0 and row['97.5q'] > 0)

                # The dictionary is populated in the same way.
                result[f'{variable_name}_mean'] = row['mean']
                result[f'{variable_name}_significant'] = is_significant


            # save the result for answering reviewer1's question
            # - whether ramp slope predicts choice correctness beyond its correlation with RT.
            result['slope_fr_predicts_pull_outcome_coef'] = slope_fr_correctness_coef
            result['slope_fr_predicts_pull_outcome_significant'] = (
                slope_fr_correctness_pval < 0.05
            )
            result['rt_predicts_pull_outcome_coef'] = correctness_model.params['rt']
            
            # for answering the same question, but a different approach - partial regression
            result['slope_fr_residRT_mw_medianDiff'] = slope_fr_residRT_mw_medianDiff
            result['slope_fr_residRT_mw_pval'] = slope_fr_residRT_mw_pval
            result['slope_fr_residRT_mw_significant'] = slope_fr_residRT_mw_significant
            result['slope_fr_residRT_mw_n_success'] = slope_fr_residRT_mw_n_success
            result['slope_fr_residRT_mw_n_fail'] = slope_fr_residRT_mw_n_fail
            
            
                
            # --- 3. Append the fully populated result to the master list ---
            hddm_result_sumary_tgt_alldates_allneurons.append(result)
        
    #
    hddm_result_sumary_tgt_alldates_allneurons_df = pd.DataFrame(hddm_result_sumary_tgt_alldates_allneurons)
    
    # remove some bad fitting neurons
    ind_good = hddm_result_sumary_tgt_alldates_allneurons_df['modelPD'] > 0
    hddm_result_sumary_tgt_alldates_allneurons_df = hddm_result_sumary_tgt_alldates_allneurons_df[ind_good]
    
    # for effect of z, only keep good z
    # ind_goodz = hddm_result_sumary_tgt_alldates_allneurons_df['modelDIC'] < \
    #             hddm_result_sumary_tgt_alldates_allneurons_df['modelDIC_noz']
    # hddm_result_sumary_tgt_alldates_allneurons_df = hddm_result_sumary_tgt_alldates_allneurons_df[ind_goodz]
    
    # for dodson
    ind_baddays = (hddm_result_sumary_tgt_alldates_allneurons_df['date'] == '20250226') | \
                  (hddm_result_sumary_tgt_alldates_allneurons_df['date'] == '20250305')
    
    hddm_result_sumary_tgt_alldates_allneurons_df = hddm_result_sumary_tgt_alldates_allneurons_df[~ind_baddays]
    
    
    # ------------- 
    # do some plotting
    # -------------
    #
    # figure 1, swamp plot (fig)
    #
    # --- 2. Calculate Statistics for Text Box (Using Original Data) ---
    def get_stats_text(data_series):
        """Performs a one-sample t-test and formats the result."""
        mean_val = data_series.mean()
        t_stat, p_val = ttest_1samp(data_series, 0)
        # Format p-value for display
        if p_val < 0.001:
            p_text = "p < 0.001"
        else:
            p_text = f"p = {p_val:.3f}"
        return f"Mean = {mean_val:.2f}\n{p_text}"
    #
    v_stats_text = get_stats_text(hddm_result_sumary_tgt_alldates_allneurons_df['v_slope_fr_mean'])
    z_stats_text = get_stats_text(hddm_result_sumary_tgt_alldates_allneurons_df['z_mean_fr_mean'])
    #
    # --- 3. Remove Outliers for Plotting ---
    def remove_outliers(df, column_name):
        """Removes outliers from a specific column using the IQR method."""
        Q1 = df[column_name].quantile(0.25)
        Q3 = df[column_name].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        #
        return df[(df[column_name] >= lower_bound) & (df[column_name] <= upper_bound)]
    #
    # Create a new DataFrame for plotting with outliers removed
    plot_df = remove_outliers(hddm_result_sumary_tgt_alldates_allneurons_df.copy(), 'v_slope_fr_mean')
    plot_df = remove_outliers(plot_df, 'z_mean_fr_mean')
    #
    print(f"Removed {len(hddm_result_sumary_tgt_alldates_allneurons_df) - len(plot_df)} outliers for plotting.")
    #
    # --- 4. Create the Two-Panel Plot ---
    fig, axes = plt.subplots(1, 2, figsize=(14, 7), sharey=False)
    # Panel 1: v_slope_fr_mean
    ax1 = axes[0]
    sns.swarmplot(y='v_slope_fr_mean', data=plot_df, color='gray', ax=ax1, size=6)
    sns.swarmplot(y='v_slope_fr_mean', data=plot_df[plot_df['v_slope_fr_significant']], color='red', ax=ax1, size=6)
    ax1.axhline(0, color='black', linestyle='--', linewidth=1)
    ax1.set_title('v_slope_fr_mean', fontsize=16)
    ax1.set_xlabel('')
    ax1.set_ylabel('Mean Coefficient Value', fontsize=12)
    ax1.text(0.05, 0.95, v_stats_text, transform=ax1.transAxes, fontsize=12,
             verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    # Panel 2: z_mean_fr_mean
    ax2 = axes[1]
    sns.swarmplot(y='z_mean_fr_mean', data=plot_df, color='gray', ax=ax2, size=6)
    sns.swarmplot(y='z_mean_fr_mean', data=plot_df[plot_df['z_mean_fr_significant']], color='red', ax=ax2, size=6)
    ax2.axhline(0, color='black', linestyle='--', linewidth=1)
    ax2.set_title('z_mean_fr_mean', fontsize=16)
    ax2.set_xlabel('')
    ax2.set_ylabel('')
    ax2.text(0.05, 0.95, z_stats_text, transform=ax2.transAxes, fontsize=12,
             verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        
        
    #
    # figure 2, venn (fig2)
    #    
    ddm_v_slope_fr_neurons = hddm_result_sumary_tgt_alldates_allneurons_df[\
                         hddm_result_sumary_tgt_alldates_allneurons_df\
                         ['v_slope_fr_significant']]
    ddm_v_slope_fr_neuronIDs = set(zip(ddm_v_slope_fr_neurons['date'], ddm_v_slope_fr_neurons['clusterID']))

    #
    from matplotlib_venn import venn2
    #
    fig2, axes = plt.subplots(1, 1, figsize=(7, 7))
    fig2.suptitle('Overlap of Neurons with Significant tuning to predicted_V and significant ddm coefficent', fontsize=16)
    # --- Panel 1: Venn Diagram for mean_fr ---
    ax1 = axes
    venn2(
        subsets=[slope_fr_v_encoding_neuronIDs,ddm_v_slope_fr_neuronIDs],
        set_labels=('FR_slope_tuned_to_predicted_v','significant neuralddm FR_slope coefficent'),
        ax=ax1
    )

    
    # --- NEW FIGURE: Venn Diagram of v_slope and z_mean significance ---
    print("\n--- Generating Venn Diagram for Coefficient Significance ---")

    # Use the original, full DataFrame for accurate counts
    df_full = hddm_result_sumary_tgt_alldates_allneurons_df

    # Get sets of unique neuron identifiers for each significant group
    # Assuming ('date', 'clusterID') uniquely identifies a neuron
    v_slope_sig_neurons = set(
        df_full[df_full['v_slope_fr_significant']].apply(
            lambda row: (row['date'], row['clusterID']), axis=1
        )
    )
    z_mean_sig_neurons = set(
        df_full[df_full['z_mean_fr_significant']].apply(
            lambda row: (row['date'], row['clusterID']), axis=1
        )
    )

    # Create the figure for the Venn diagram
    fig_venn, ax_venn = plt.subplots(figsize=(8, 8))

    # Generate the Venn diagram
    venn2(
        subsets=[v_slope_sig_neurons, z_mean_sig_neurons],
        set_labels=('Significant v_slope_fr', 'Significant z_mean_fr'),
        set_colors=('firebrick', 'darkblue'),
        alpha=0.7,
        ax=ax_venn
    )

    ax_venn.set_title('Overlap of Neurons with Significant DDM Coefficients', fontsize=16)

    
    #
    # savefig
    savefig = 0
    if savefig:
        current_dir = data_saved_folder+'fig_for_basic_neural_analysis_allsessions_basicEvents_PCA_Pullfocused_continuousBhv_partnerDistVaris_with_hddm_model'+savefile_sufix+'/'+animal_tgt
        add_fig_dir = os.path.join(current_dir,cameraID+'/fig_for_hddm_model_fitted_allsession_withNeurons')
        #
        
        fig.savefig(add_fig_dir+'/neuralDDM_v_slopeFR_z_meanFR_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pdf', 
                        format='pdf', bbox_inches='tight')
        fig2.savefig(add_fig_dir+'/venn_fr_correlation_with_predictedv_v_slopeFR_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pdf', 
                        format='pdf', bbox_inches='tight')
        fig_venn.savefig(add_fig_dir+'/venn_neuralDDM_v_slopeFR_z_meanFR_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pdf', 
                        format='pdf', bbox_inches='tight')
        
        

In [ ]:
# plot for answering reviewer 1's comment:
# whether ramp slope predicts choice correctness beyond its correlation with RT.


# --- Helpers ---
def remove_outliers_iqr(df, column_name):
    """Remove outliers using 1.5*IQR rule (for visualization only)."""
    Q1 = df[column_name].quantile(0.15)
    Q3 = df[column_name].quantile(0.85)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return df[(df[column_name] >= lower_bound) & (df[column_name] <= upper_bound)]

def format_p_value(p_val):
    if p_val < 0.001:
        return "p < 0.001"
    return f"p = {p_val:.3f}"

# --- 1) Full dataframe for statistics ---
coef_col = 'slope_fr_predicts_pull_outcome_coef'
sig_col  = 'slope_fr_predicts_pull_outcome_significant'
# coef_col = 'slope_fr_residRT_mw_medianDiff'
# sig_col = 'slope_fr_residRT_mw_significant'

# coef_col = 'auc_M0_rt'
# sig_col = 'valid_M0_rt'
# coef_col = 'auc_M01_evidence'
# sig_col = 'valid_M01_evidence'
# coef_col = 'auc_M1_evidence_residRT'
# sig_col = 'valid_M1_evidence_residRT'
# coef_col = 'auc_M2_slope_residRT'
# sig_col = 'valid_M2_slope_residRT'
# coef_col = 'auc_M3_slope_residEvidenceRT'
# sig_col = 'valid_M3_slope_residEvidenceRT'
# coef_col = 'auc_M4_slope_residEvidence'
# sig_col = 'valid_M4_slope_residEvidence'

# coef_col = 'varSlope_fail_over_success_logratio'
# sig_col = 'valid_M1_evidence_residRT'


df_full = hddm_result_sumary_tgt_alldates_allneurons_df.dropna(subset=[coef_col]).copy()

df_full = df_full.drop_duplicates(subset=[coef_col]).copy()

# --- 2) Outlier removal for plotting ONLY ---
plot_df = remove_outliers_iqr(df_full, coef_col).copy()
print(f"Removed {len(df_full) - len(plot_df)} outliers for plotting (IQR rule).")
 
# plot_df = df_full

# One-sample t-test vs 0 (run on full data; not outlier-removed)
# t_stat, p_val = ttest_1samp(df_full[coef_col], 0)
# mean_val = df_full[coef_col].mean()
# stats_text = f"Mean = {mean_val:.2f}\n{format_p_value(p_val)}"

# One-sample t-test vs 0 (run on full data; with outlier-removed)
t_stat, p_val = ttest_1samp(plot_df[coef_col], 0)  
mean_val = plot_df[coef_col].mean()
stats_text = f"Mean = {mean_val:.2f}\n{format_p_value(p_val)}"

# --- 3) Plot ---
fig, ax = plt.subplots(figsize=(6, 7))

# All neurons (gray)
sns.swarmplot(
    y=coef_col,
    data=plot_df,
    color='gray',
    ax=ax,
    size=6
)

# Significant neurons (red)
sns.swarmplot(
    y=coef_col,
    data=plot_df[plot_df[sig_col]],
    color='red',
    ax=ax,
    size=6
)

ax.axhline(0, color='black', linestyle='--', linewidth=1)
ax.set_title('Ramp slope predicts pull outcome beyond RT', fontsize=14)
ax.set_xlabel('')
# ax.set_ylabel('Coefficient (slope_fr → pull outcome | RT)', fontsize=12)
ax.set_ylabel('AUC (slope_fr → pull outcome | RT)', fontsize=12)

ax.text(
    0.05, 0.95,
    stats_text,
    transform=ax.transAxes,
    fontsize=11,
    verticalalignment='top',
    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5)
)

#
# savefig
savefig = 1
if savefig:
    current_dir = data_saved_folder+'fig_for_basic_neural_analysis_allsessions_basicEvents_PCA_Pullfocused_continuousBhv_partnerDistVaris_with_hddm_model'+savefile_sufix+'/'+animal_tgt
    add_fig_dir = os.path.join(current_dir,cameraID+'/fig_for_hddm_model_fitted_allsession_withNeurons')
    #

    fig.savefig(add_fig_dir+'/neuralslope_outcome_prediction_beyondRT_'+cond_toplot_type+doOnsetAfterMin_suffix+exaustModel_sufix+'.pdf', 
                    format='pdf', bbox_inches='tight')
    

In [ ]:
####
# Reviewer 2: STEP vs RAMP motif fits on FULL TRIAL window (no gaze filter)
# + tau–old_slope sanity checks with:
#   (1) direction handling:
#       - split by step_dir (up/down)
#       - and "signed_old_slope" = old_slope * sign(step_jump)  (or ramp slope sign)
#   (2) strong-trial filtering:
#       - abs(step_jump) percentile gate
#       - optional |deltaBIC| gate
#
# Notes:
# - Uses full trial window: trial start -> (pull - EXCLUDE_TAIL_S)
# - Fits:
#     STEP: 2-level change-point (mu0 then mu1)  [FAST-code definition of tau]
#     RAMP: flat-then-ramp change-point (c then c + b*(t-tau)+)
# - Stores per-trial: tau_step, tau_ramp, step_jump, ramp_slope, BICs, deltaBIC
# - old_slope is ABS(OLS slope) on the same fit window (matches your Fig3 abs usage)
#
# Plots:
#     A) per-neuron median ΔBIC histogram
#     B) per-neuron frac ramp-win histogram
#     C) STEP tau_norm vs old_slope (all step-win trials)
#     D) STEP tau_norm vs signed_old_slope + split by up/down
#     E) STEP tau_norm vs signed_old_slope with "strong step" gate
#     F) RAMP tau_norm vs old_slope (all ramp-win trials)
#     G) RAMP tau_norm vs signed_old_slope + split by up/down
#     H) RAMP tau_norm vs signed_old_slope with "strong ramp" gate
####

if 0:  # old, do not use

    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from scipy.ndimage import gaussian_filter1d
    from scipy.stats import zscore

    # ============================================================
    # USER SETTINGS
    # ============================================================
    fps = 30.0
    dt = 1.0 / fps

    EXCLUDE_TAIL_S = 0.85
    EXCLUDE_TAIL_BINS = int(np.round(EXCLUDE_TAIL_S * fps))  # exclude last 0.85s before pull from fitting window

    # Trial length gate
    MAX_TRIAL_S = 8.0
    MAX_TRIAL_BINS = int(np.floor(MAX_TRIAL_S * fps))

    # Model fitting constraints
    MIN_FIT_BINS = 15
    MIN_VAR = 1e-6

    # STEP min segment length (FAST-code style)
    MIN_SEG_S = 0.20
    MIN_SEG_BINS = max(2, int(np.round(MIN_SEG_S * fps)))  # each side must have >= this many bins

    # RAMP CP constraint (avoid endpoints)
    MIN_CP_GAP = 4

    # Optional extra smoothing on FR within fit window (usually OFF)
    DO_SMOOTH_FR = False
    SMOOTH_FR_SIGMA = 2

    # Plot subsampling (scatter can be huge)
    PLOT_MAX_POINTS = 12000

    # ---- Strong-trial gates (you can tweak)
    STEP_JUMP_ABS_PCTL = 70          # keep top 30% by |step_jump|
    RAMP_SLOPE_ABS_PCTL = 70         # keep top 30% by |ramp_slope|
    USE_DBIC_GATE = False
    DBIC_ABS_MIN = 2.0               # optional: require |deltaBIC| >= this

    # ============================================================
    # Helpers
    # ============================================================
    def _as_1d_float_array(x):
        return np.asarray(x, dtype=float).ravel()

    def _bic_from_sse(sse, n, k):
        # BIC up to constant, smaller is better
        if (not np.isfinite(sse)) or (sse <= 0):
            sse = 1e-12
        return float(n * np.log(sse / n) + k * np.log(n))

    def pearsonr_simple(x, y):
        x = np.asarray(x, float)
        y = np.asarray(y, float)
        m = np.isfinite(x) & np.isfinite(y)
        x = x[m]; y = y[m]
        if len(x) < 3:
            return np.nan
        x0 = x - np.mean(x)
        y0 = y - np.mean(y)
        denom = np.sqrt(np.sum(x0**2) * np.sum(y0**2))
        if denom <= 1e-12:
            return np.nan
        return float(np.sum(x0 * y0) / denom)

    # ============================================================
    # STEP fit (FAST-code style)
    #   Step model: y = mu0 for i<=tau, mu1 for i>tau
    #   tau is the LAST index in segment 0
    # ============================================================
    def fit_step_changepoint_fast(y, min_seg_bins=2):
        """
        Returns: tau_idx (last idx of seg0), mu0, mu1, sse, bic
        k=3 (mu0, mu1, tau)
        """
        y = _as_1d_float_array(y)
        n = len(y)
        if n < (2 * min_seg_bins + 2):
            return None

        csum = np.cumsum(y)
        csum2 = np.cumsum(y * y)

        best_sse = np.inf
        best_tau = None
        best_mu0 = np.nan
        best_mu1 = np.nan

        # tau is last index of segment0
        for tau in range(min_seg_bins - 1, n - min_seg_bins - 1):
            n0 = tau + 1
            n1 = n - n0

            s0 = csum[tau]
            s02 = csum2[tau]
            mu0 = s0 / n0
            sse0 = s02 - (s0 * s0) / n0

            s1 = csum[-1] - s0
            s12 = csum2[-1] - s02
            mu1 = s1 / n1
            sse1 = s12 - (s1 * s1) / n1

            sse = float(sse0 + sse1)
            if sse < best_sse:
                best_sse = sse
                best_tau = int(tau)
                best_mu0 = float(mu0)
                best_mu1 = float(mu1)

        bic = _bic_from_sse(best_sse, n=n, k=3)
        return int(best_tau), float(best_mu0), float(best_mu1), float(best_sse), float(bic)

    # ============================================================
    # RAMP fit (flat-then-ramp changepoint)
    # ============================================================
    def fit_flat_then_ramp_changepoint(y, dt, min_gap=4):
        """
        Flat-then-ramp model:
          y = c                  for i < tau
          y = c + b*(t - t_tau)  for i >= tau
        Params: c, b, tau (k=3)
        Returns: tau_idx, c, b, sse, bic
        """
        y = _as_1d_float_array(y)
        n = len(y)
        if n < (2 * min_gap + 2):
            return None

        t = np.arange(n) * dt

        best_sse = np.inf
        best_tau = None
        best_c = np.nan
        best_b = np.nan

        for tau in range(min_gap, n - min_gap):
            t_tau = t[tau]
            r = np.maximum(0.0, t - t_tau)

            s00 = n
            s01 = np.sum(r)
            s11 = np.sum(r * r)
            y0 = np.sum(y)
            y1 = np.sum(r * y)

            det = s00 * s11 - s01 * s01
            if det <= 1e-12:
                continue

            c = (y0 * s11 - y1 * s01) / det
            b = (y1 * s00 - y0 * s01) / det

            yhat = c + b * r
            resid = y - yhat
            sse = float(np.sum(resid * resid))

            if sse < best_sse:
                best_sse = sse
                best_tau = int(tau)
                best_c = float(c)
                best_b = float(b)

        bic = _bic_from_sse(best_sse, n=n, k=3)
        return int(best_tau), float(best_c), float(best_b), float(best_sse), float(bic)

    # ============================================================
    # Old-slope calculation: ABS(OLS slope) on full fit window
    # ============================================================
    def ols_slope_abs(y, dt):
        y = _as_1d_float_array(y)
        n = len(y)
        if n < 2:
            return np.nan
        t = np.arange(n) * dt
        A = np.column_stack([np.ones(n), t])
        coef, _, _, _ = np.linalg.lstsq(A, y, rcond=None)
        b = float(coef[1])
        return float(abs(b))

    # ============================================================
    # Main
    # ============================================================
    trial_fit_records = []
    neuron_summary_records = []

    for session_toana, hddm_data_tgt_idate in hddm_data_tgt_alldates.groupby('date'):

        # Outlier removal based on rt/time_since_last_reward (kept)
        lower_q = 0.05
        upper_q = 0.95
        if 'rt' in hddm_data_tgt_idate.columns:
            rt_lower, rt_upper = hddm_data_tgt_idate['rt'].quantile([lower_q, upper_q])
            hddm_data_tgt_idate = hddm_data_tgt_idate[(hddm_data_tgt_idate['rt'] > rt_lower) & (hddm_data_tgt_idate['rt'] < rt_upper)]
        if 'time_since_last_reward' in hddm_data_tgt_idate.columns:
            tslr_lower, tslr_upper = hddm_data_tgt_idate['time_since_last_reward'].quantile([lower_q, upper_q])
            hddm_data_tgt_idate = hddm_data_tgt_idate[(hddm_data_tgt_idate['time_since_last_reward'] > tslr_lower) &
                                                      (hddm_data_tgt_idate['time_since_last_reward'] < tslr_upper)]

        hddm_RawFullDatas_tgt_idate = hddm_RawFullDatas_tgt_alldates[hddm_RawFullDatas_tgt_alldates['dates'] == session_toana]
        neurons_idate = np.unique(hddm_RawFullDatas_tgt_idate['clusterID'])

        for ineuron_idate in neurons_idate:

            dfN = hddm_RawFullDatas_tgt_idate[hddm_RawFullDatas_tgt_idate['clusterID'] == ineuron_idate]

            dBICs = []
            taus_step_s = []
            taus_ramp_s = []
            n_used = 0

            for _, row in dfN.iterrows():

                trial_id = int(row['bhv_id']) if 'bhv_id' in row else None

                fr_full = _as_1d_float_array(row['FR_ievent'])
                Lfull = len(fr_full)

                # ---- drop long trials
                if Lfull > MAX_TRIAL_BINS:
                    continue

                end_limit_idx = (Lfull - 1) - EXCLUDE_TAIL_BINS
                if end_limit_idx <= 0:
                    continue

                y = fr_full[:end_limit_idx + 1]

                if DO_SMOOTH_FR and len(y) > 3:
                    y = gaussian_filter1d(y, sigma=SMOOTH_FR_SIGMA)

                if (len(y) < MIN_FIT_BINS) or (not np.all(np.isfinite(y))) or (np.var(y) < MIN_VAR):
                    continue

                n = len(y)
                fit_window_s = n * dt

                # ---- OLD slope (ABS OLS) on same window
                old_slope = ols_slope_abs(y, dt=dt)
                if not np.isfinite(old_slope):
                    continue

                # ---- fits
                step_fit = fit_step_changepoint_fast(y, min_seg_bins=MIN_SEG_BINS)
                ramp_fit = fit_flat_then_ramp_changepoint(y, dt=dt, min_gap=MIN_CP_GAP)
                if (step_fit is None) or (ramp_fit is None):
                    continue

                tau_s_idx, mu0, mu1, sse_s, bic_s = step_fit
                tau_r_idx, c, b, sse_r, bic_r = ramp_fit

                delta_bic = float(bic_s - bic_r)  # >0 => ramp wins

                step_jump = float(mu1 - mu0)
                step_dir = "up" if step_jump > 0 else ("down" if step_jump < 0 else "flat")

                ramp_slope = float(b)  # per second (from CP ramp)
                ramp_dir = "up" if ramp_slope > 0 else ("down" if ramp_slope < 0 else "flat")

                # tau normalized within fit window
                tau_step_norm = float(tau_s_idx / (n - 1)) if (n > 1) else np.nan
                tau_ramp_norm = float(tau_r_idx / (n - 1)) if (n > 1) else np.nan

                # signed old slope (align abs(old_slope) to motif direction)
                signed_old_slope_step = float(old_slope * np.sign(step_jump)) if step_dir != "flat" else np.nan
                signed_old_slope_ramp = float(old_slope * np.sign(ramp_slope)) if ramp_dir != "flat" else np.nan

                trial_fit_records.append({
                    "date": str(session_toana),
                    "clusterID": int(ineuron_idate),
                    "bhv_id": trial_id,

                    "trial_len_bins": int(Lfull),
                    "fit_len_bins": int(n),
                    "fit_window_s": float(fit_window_s),

                    "old_slope": float(old_slope),

                    "bic_step": float(bic_s),
                    "bic_ramp": float(bic_r),
                    "deltaBIC_step_minus_ramp": float(delta_bic),

                    "tau_step_idx": int(tau_s_idx),
                    "tau_ramp_idx": int(tau_r_idx),
                    "tau_step_s": float(tau_s_idx * dt),
                    "tau_ramp_s": float(tau_r_idx * dt),
                    "tau_step_norm": tau_step_norm,
                    "tau_ramp_norm": tau_ramp_norm,

                    "step_jump": step_jump,
                    "step_dir": step_dir,

                    "ramp_slope_per_s": ramp_slope,
                    "ramp_dir": ramp_dir,

                    "signed_old_slope_step": signed_old_slope_step,
                    "signed_old_slope_ramp": signed_old_slope_ramp,
                })

                dBICs.append(delta_bic)
                taus_step_s.append(tau_s_idx * dt)
                taus_ramp_s.append(tau_r_idx * dt)
                n_used += 1

            if n_used > 0:
                dBICs_arr = np.asarray(dBICs, dtype=float)
                neuron_summary_records.append({
                    "date": str(session_toana),
                    "clusterID": int(ineuron_idate),
                    "n_trials_used": int(n_used),
                    "frac_ramp_win": float(np.mean(dBICs_arr > 0)),
                    "median_deltaBIC": float(np.median(dBICs_arr)),
                    "mean_deltaBIC": float(np.mean(dBICs_arr)),
                    "median_tau_step_s": float(np.median(taus_step_s)),
                    "median_tau_ramp_s": float(np.median(taus_ramp_s)),
                })

    trial_fits_df = pd.DataFrame(trial_fit_records)
    neuron_summary_df = pd.DataFrame(neuron_summary_records)

    print("Trial-level fits:", trial_fits_df.shape)
    print("Neuron-level summary:", neuron_summary_df.shape)

    # ============================================================
    # Original plots: per-neuron ΔBIC and frac ramp wins
    # ============================================================
    if len(neuron_summary_df) > 0:
        plt.figure(figsize=(6, 4))
        plt.hist(neuron_summary_df["median_deltaBIC"].values, bins=40)
        plt.axvline(0, linewidth=1)
        plt.xlabel("Per-neuron median ΔBIC (BIC_step - BIC_ramp); >0 => ramp wins")
        plt.ylabel("# neurons")
        plt.title("Step vs ramp (full trial fit window)")
        plt.tight_layout()
        plt.show()

        plt.figure(figsize=(6, 4))
        plt.hist(neuron_summary_df["frac_ramp_win"].values, bins=30, range=(0, 1))
        plt.axvline(0.5, linewidth=1)
        plt.xlabel("Fraction of trials where ramp wins (ΔBIC>0)")
        plt.ylabel("# neurons")
        plt.title("Ramp-win fraction across neurons (full trial window)")
        plt.tight_layout()
        plt.show()

    # ============================================================
    # New plots: tau vs old slope (STEP and RAMP), with gates
    # ============================================================
    if len(trial_fits_df) > 0:

        # winner labels
        m = np.isfinite(trial_fits_df["bic_step"].values) & np.isfinite(trial_fits_df["bic_ramp"].values)
        trial_fits_df = trial_fits_df[m].copy()
        trial_fits_df["win_model"] = np.where(trial_fits_df["bic_ramp"].values < trial_fits_df["bic_step"].values, "ramp", "step")

        def _subsample(df, maxn=PLOT_MAX_POINTS, seed=0):
            if len(df) > maxn:
                return df.sample(maxn, random_state=seed)
            return df

        # ---- STEP: tau_norm vs old_slope (all step-winning)
        step_df = trial_fits_df[trial_fits_df["win_model"] == "step"].copy()
        if len(step_df) > 0:
            dd = _subsample(step_df, seed=0)
            r = pearsonr_simple(dd["tau_step_norm"].values, dd["old_slope"].values)

            plt.figure(figsize=(5.2, 4.2))
            plt.scatter(dd["tau_step_norm"].values, dd["old_slope"].values, s=8, alpha=0.25)
            plt.axhline(0, linewidth=1)
            plt.xlim(-0.02, 1.02)
            plt.xlabel("tau_step (normalized by fit window)")
            plt.ylabel("old slope (ABS OLS; trial start → pull-0.85s)")
            plt.title(f"STEP (win): tau vs old slope (r={r:.3f})")
            plt.tight_layout()
            plt.show()

            # ---- STEP: split by direction + signed slope
            step_up = step_df[step_df["step_dir"] == "up"].copy()
            step_dn = step_df[step_df["step_dir"] == "down"].copy()

            dd_up = _subsample(step_up, seed=1)
            dd_dn = _subsample(step_dn, seed=2)

            r_up = pearsonr_simple(dd_up["tau_step_norm"].values, dd_up["signed_old_slope_step"].values)
            r_dn = pearsonr_simple(dd_dn["tau_step_norm"].values, dd_dn["signed_old_slope_step"].values)

            plt.figure(figsize=(5.2, 4.2))
            if len(dd_up) > 0:
                plt.scatter(dd_up["tau_step_norm"].values, dd_up["signed_old_slope_step"].values, s=8, alpha=0.25, label=f"up (r={r_up:.3f})")
            if len(dd_dn) > 0:
                plt.scatter(dd_dn["tau_step_norm"].values, dd_dn["signed_old_slope_step"].values, s=8, alpha=0.25, label=f"down (r={r_dn:.3f})")
            plt.axhline(0, linewidth=1)
            plt.xlim(-0.02, 1.02)
            plt.xlabel("tau_step (normalized)")
            plt.ylabel("signed old slope (abs(old_slope) × sign(step_jump))")
            plt.title("STEP (win): tau vs signed old slope (split by dir)")
            plt.legend(frameon=False)
            plt.tight_layout()
            plt.show()

            # ---- STEP: strong gate by |step_jump| + optional |deltaBIC|
            abs_jump = step_df["step_jump"].abs().values
            jump_thr = np.nanpercentile(abs_jump, STEP_JUMP_ABS_PCTL) if np.any(np.isfinite(abs_jump)) else np.inf
            strong_step = step_df[step_df["step_jump"].abs() >= jump_thr].copy()

            if USE_DBIC_GATE:
                strong_step = strong_step[strong_step["deltaBIC_step_minus_ramp"].abs() >= DBIC_ABS_MIN].copy()

            dd_strong = _subsample(strong_step, seed=3)
            r_strong = pearsonr_simple(dd_strong["tau_step_norm"].values, dd_strong["signed_old_slope_step"].values)

            plt.figure(figsize=(5.2, 4.2))
            plt.scatter(dd_strong["tau_step_norm"].values, dd_strong["signed_old_slope_step"].values, s=8, alpha=0.25)
            plt.axhline(0, linewidth=1)
            plt.xlim(-0.02, 1.02)
            plt.xlabel("tau_step (normalized)")
            plt.ylabel("signed old slope (aligned to step jump)")
            plt.title(f"STEP strong (|jump|≥p{STEP_JUMP_ABS_PCTL}): r={r_strong:.3f}, n={len(dd_strong)}")
            plt.tight_layout()
            plt.show()

        # ---- RAMP: tau_norm vs old_slope (all ramp-winning)
        ramp_df = trial_fits_df[trial_fits_df["win_model"] == "ramp"].copy()
        if len(ramp_df) > 0:
            dd = _subsample(ramp_df, seed=10)
            r = pearsonr_simple(dd["tau_ramp_norm"].values, dd["old_slope"].values)

            plt.figure(figsize=(5.2, 4.2))
            plt.scatter(dd["tau_ramp_norm"].values, dd["old_slope"].values, s=8, alpha=0.25)
            plt.axhline(0, linewidth=1)
            plt.xlim(-0.02, 1.02)
            plt.xlabel("tau_ramp (normalized by fit window)")
            plt.ylabel("old slope (ABS OLS; trial start → pull-0.85s)")
            plt.title(f"RAMP (win): tau vs old slope (r={r:.3f})")
            plt.tight_layout()
            plt.show()

            # ---- RAMP: split by direction + signed slope
            ramp_up = ramp_df[ramp_df["ramp_dir"] == "up"].copy()
            ramp_dn = ramp_df[ramp_df["ramp_dir"] == "down"].copy()

            dd_up = _subsample(ramp_up, seed=11)
            dd_dn = _subsample(ramp_dn, seed=12)

            r_up = pearsonr_simple(dd_up["tau_ramp_norm"].values, dd_up["signed_old_slope_ramp"].values)
            r_dn = pearsonr_simple(dd_dn["tau_ramp_norm"].values, dd_dn["signed_old_slope_ramp"].values)

            plt.figure(figsize=(5.2, 4.2))
            if len(dd_up) > 0:
                plt.scatter(dd_up["tau_ramp_norm"].values, dd_up["signed_old_slope_ramp"].values, s=8, alpha=0.25, label=f"up (r={r_up:.3f})")
            if len(dd_dn) > 0:
                plt.scatter(dd_dn["tau_ramp_norm"].values, dd_dn["signed_old_slope_ramp"].values, s=8, alpha=0.25, label=f"down (r={r_dn:.3f})")
            plt.axhline(0, linewidth=1)
            plt.xlim(-0.02, 1.02)
            plt.xlabel("tau_ramp (normalized)")
            plt.ylabel("signed old slope (abs(old_slope) × sign(ramp_slope))")
            plt.title("RAMP (win): tau vs signed old slope (split by dir)")
            plt.legend(frameon=False)
            plt.tight_layout()
            plt.show()

            # ---- RAMP: strong gate by |ramp_slope| + optional |deltaBIC|
            abs_b = ramp_df["ramp_slope_per_s"].abs().values
            b_thr = np.nanpercentile(abs_b, RAMP_SLOPE_ABS_PCTL) if np.any(np.isfinite(abs_b)) else np.inf
            strong_ramp = ramp_df[ramp_df["ramp_slope_per_s"].abs() >= b_thr].copy()

            if USE_DBIC_GATE:
                strong_ramp = strong_ramp[strong_ramp["deltaBIC_step_minus_ramp"].abs() >= DBIC_ABS_MIN].copy()

            dd_strong = _subsample(strong_ramp, seed=13)
            r_strong = pearsonr_simple(dd_strong["tau_ramp_norm"].values, dd_strong["signed_old_slope_ramp"].values)

            plt.figure(figsize=(5.2, 4.2))
            plt.scatter(dd_strong["tau_ramp_norm"].values, dd_strong["signed_old_slope_ramp"].values, s=8, alpha=0.25)
            plt.axhline(0, linewidth=1)
            plt.xlim(-0.02, 1.02)
            plt.xlabel("tau_ramp (normalized)")
            plt.ylabel("signed old slope (aligned to ramp slope sign)")
            plt.title(f"RAMP strong (|slope|≥p{RAMP_SLOPE_ABS_PCTL}): r={r_strong:.3f}, n={len(dd_strong)}")
            plt.tight_layout()
            plt.show()

    # Optional: save outputs
    # trial_fits_df.to_csv("step_vs_ramp_fulltrial_trialfits_with_tau_slope_checks.csv", index=False)
    # neuron_summary_df.to_csv("step_vs_ramp_fulltrial_neuron_summary.csv", index=False)


In [ ]:
####
# STEP vs RAMP motif fits (FULL TRIAL window; no gaze filter)
# + Extract behavioral covariates at tau and plot:
#   - distribution of tau-to-pull (STEP tau and RAMP tau)
#   - distributions of socialgaze_prob / other_PC1 / other_mass_move_speed at tau
#   - optional per-neuron mean distributions
#
# Notes:
# - Fit window: trial start -> (pull - EXCLUDE_TAIL_S)
# - STEP: 2-mean change point (FAST style): y=mu0 for i<=tau, y=mu1 for i>tau, tau is LAST idx of seg0
# - RAMP: flat-then-ramp change point
# - old_slope: ABS(OLS slope) on the same fit window (matches your Fig3 usage)
####

if 0:

    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from scipy.ndimage import gaussian_filter1d
    from scipy.stats import zscore

    # ============================================================
    # USER SETTINGS
    # ============================================================
    fps = 30.0
    dt = 1.0 / fps

    EXCLUDE_TAIL_S = 0.85
    EXCLUDE_TAIL_BINS = int(np.round(EXCLUDE_TAIL_S * fps))

    # Trial length gate
    MAX_TRIAL_S = 8.0
    MAX_TRIAL_BINS = int(np.floor(MAX_TRIAL_S * fps))

    # Model fitting constraints
    MIN_FIT_BINS = 15
    MIN_VAR = 1e-6

    # STEP min segment length (FAST-code style)
    MIN_SEG_S = 0.20
    MIN_SEG_BINS = max(2, int(np.round(MIN_SEG_S * fps)))  # each side >= this many bins

    # RAMP CP constraint (avoid endpoints)
    MIN_CP_GAP = 4

    # Optional extra smoothing on FR within fit window (usually OFF)
    DO_SMOOTH_FR = False
    SMOOTH_FR_SIGMA = 2

    # Plot subsampling (scatter can be huge)
    PLOT_MAX_POINTS = 15000

    # What to plot/extract
    PLOT_WIN_MODEL = "step"   # "step", "ramp", or "both"
    USE_POSTSTEP_TAU = False  # if True, for STEP covariates use tau+1 (first bin after step), clipped

    # ============================================================
    # Helpers
    # ============================================================
    def _as_1d_float_array(x):
        return np.asarray(x, dtype=float).ravel()

    def _bic_from_sse(sse, n, k):
        if (not np.isfinite(sse)) or (sse <= 0):
            sse = 1e-12
        return float(n * np.log(sse / n) + k * np.log(n))

    def pearsonr_simple(x, y):
        x = np.asarray(x, float)
        y = np.asarray(y, float)
        m = np.isfinite(x) & np.isfinite(y)
        x = x[m]; y = y[m]
        if len(x) < 3:
            return np.nan
        x0 = x - np.mean(x)
        y0 = y - np.mean(y)
        denom = np.sqrt(np.sum(x0**2) * np.sum(y0**2))
        if denom <= 1e-12:
            return np.nan
        return float(np.sum(x0 * y0) / denom)

    def _subsample_df(df, maxn=PLOT_MAX_POINTS, seed=0):
        if len(df) > maxn:
            return df.sample(maxn, random_state=seed)
        return df

    # ============================================================
    # STEP fit (FAST-code style)
    #   Step model: y = mu0 for i<=tau, mu1 for i>tau
    #   tau is the LAST index in segment 0
    # ============================================================
    def fit_step_changepoint_fast(y, min_seg_bins=2):
        """
        Returns: tau_idx (last idx seg0), mu0, mu1, sse, bic
        k=3 (mu0, mu1, tau)
        """
        y = _as_1d_float_array(y)
        n = len(y)
        if n < (2 * min_seg_bins + 2):
            return None

        csum = np.cumsum(y)
        csum2 = np.cumsum(y * y)

        best_sse = np.inf
        best_tau = None
        best_mu0 = np.nan
        best_mu1 = np.nan

        # tau is last index of segment0
        for tau in range(min_seg_bins - 1, n - min_seg_bins - 1):
            n0 = tau + 1
            n1 = n - n0

            s0 = csum[tau]
            s02 = csum2[tau]
            mu0 = s0 / n0
            sse0 = s02 - (s0 * s0) / n0

            s1 = csum[-1] - s0
            s12 = csum2[-1] - s02
            mu1 = s1 / n1
            sse1 = s12 - (s1 * s1) / n1

            sse = float(sse0 + sse1)
            if sse < best_sse:
                best_sse = sse
                best_tau = int(tau)
                best_mu0 = float(mu0)
                best_mu1 = float(mu1)

        bic = _bic_from_sse(best_sse, n=n, k=3)
        return int(best_tau), float(best_mu0), float(best_mu1), float(best_sse), float(bic)

    # ============================================================
    # RAMP fit (flat-then-ramp changepoint)
    # ============================================================
    def fit_flat_then_ramp_changepoint(y, dt, min_gap=4):
        """
        Flat-then-ramp:
          y = c                  for i < tau
          y = c + b*(t - t_tau)  for i >= tau
        Params: c, b, tau (k=3)
        Returns: tau_idx, c, b, sse, bic
        """
        y = _as_1d_float_array(y)
        n = len(y)
        if n < (2 * min_gap + 2):
            return None

        t = np.arange(n) * dt

        best_sse = np.inf
        best_tau = None
        best_c = np.nan
        best_b = np.nan

        for tau in range(min_gap, n - min_gap):
            t_tau = t[tau]
            r = np.maximum(0.0, t - t_tau)

            s00 = n
            s01 = np.sum(r)
            s11 = np.sum(r * r)
            y0 = np.sum(y)
            y1 = np.sum(r * y)

            det = s00 * s11 - s01 * s01
            if det <= 1e-12:
                continue

            c = (y0 * s11 - y1 * s01) / det
            b = (y1 * s00 - y0 * s01) / det

            yhat = c + b * r
            resid = y - yhat
            sse = float(np.sum(resid * resid))

            if sse < best_sse:
                best_sse = sse
                best_tau = int(tau)
                best_c = float(c)
                best_b = float(b)

        bic = _bic_from_sse(best_sse, n=n, k=3)
        return int(best_tau), float(best_c), float(best_b), float(best_sse), float(bic)

    # ============================================================
    # Old-slope calculation: ABS(OLS slope) on fit window
    # ============================================================
    def ols_slope_abs(y, dt):
        y = _as_1d_float_array(y)
        n = len(y)
        if n < 2:
            return np.nan
        t = np.arange(n) * dt
        A = np.column_stack([np.ones(n), t])
        coef, _, _, _ = np.linalg.lstsq(A, y, rcond=None)
        return float(abs(coef[1]))

    def _get_at(arr, idx):
        if arr is None:
            return np.nan
        if idx < 0 or idx >= len(arr):
            return np.nan
        v = arr[idx]
        return float(v) if np.isfinite(v) else np.nan

    # ============================================================
    # Sanity check: raw df has needed columns
    # ============================================================
    if 'hddm_RawFullDatas_tgt_alldates' not in globals():
        raise ValueError("hddm_RawFullDatas_tgt_alldates not found in globals(). Please load it before running this cell.")
    if 'hddm_data_tgt_alldates' not in globals():
        raise ValueError("hddm_data_tgt_alldates not found in globals(). Please load it before running this cell.")

    needed_cols = ['dates', 'clusterID', 'bhv_id', 'FR_ievent',
                   'socialgaze_prob', 'other_PC1', 'other_mass_move_speed']
    miss = [c for c in needed_cols if c not in hddm_RawFullDatas_tgt_alldates.columns]
    if len(miss) > 0:
        raise ValueError(f"Missing columns in hddm_RawFullDatas_tgt_alldates: {miss}")

    # ============================================================
    # Main
    # ============================================================
    trial_fit_records = []
    neuron_summary_records = []

    for session_toana, hddm_data_tgt_idate in hddm_data_tgt_alldates.groupby('date'):

        # Outlier removal on behavior table (kept from your pipeline)
        lower_q = 0.05
        upper_q = 0.95
        if 'rt' in hddm_data_tgt_idate.columns:
            rt_lower, rt_upper = hddm_data_tgt_idate['rt'].quantile([lower_q, upper_q])
            hddm_data_tgt_idate = hddm_data_tgt_idate[(hddm_data_tgt_idate['rt'] > rt_lower) & (hddm_data_tgt_idate['rt'] < rt_upper)]
        if 'time_since_last_reward' in hddm_data_tgt_idate.columns:
            tslr_lower, tslr_upper = hddm_data_tgt_idate['time_since_last_reward'].quantile([lower_q, upper_q])
            hddm_data_tgt_idate = hddm_data_tgt_idate[(hddm_data_tgt_idate['time_since_last_reward'] > tslr_lower) &
                                                      (hddm_data_tgt_idate['time_since_last_reward'] < tslr_upper)]

        # Restrict raw df to this date
        df_raw_date = hddm_RawFullDatas_tgt_alldates[hddm_RawFullDatas_tgt_alldates['dates'] == session_toana].copy()

        # Optional: restrict to behavior-filtered bhv_ids to stay consistent
        if 'bhv_id' in hddm_data_tgt_idate.columns and 'bhv_id' in df_raw_date.columns:
            good_ids = set(hddm_data_tgt_idate['bhv_id'].astype(int).tolist())
            if len(good_ids) > 0:
                df_raw_date = df_raw_date[df_raw_date['bhv_id'].astype(int).isin(good_ids)].copy()

        neurons_idate = np.unique(df_raw_date['clusterID'])

        for ineuron_idate in neurons_idate:

            dfN = df_raw_date[df_raw_date['clusterID'] == ineuron_idate]

            dBICs = []
            taus_step_s = []
            taus_ramp_s = []
            n_used = 0

            for _, row in dfN.iterrows():

                trial_id = int(row['bhv_id']) if 'bhv_id' in row else None

                fr_full   = _as_1d_float_array(row['FR_ievent'])
                gaze_full = _as_1d_float_array(row['socialgaze_prob'])
                pc1_full  = _as_1d_float_array(row['other_PC1'])
                spd_full  = _as_1d_float_array(row['other_mass_move_speed'])

                # Align lengths across traces
                Lfull = min(len(fr_full), len(gaze_full), len(pc1_full), len(spd_full))
                if Lfull <= 0:
                    continue

                fr_full   = fr_full[:Lfull]
                gaze_full = gaze_full[:Lfull]
                pc1_full  = pc1_full[:Lfull]
                spd_full  = spd_full[:Lfull]

                # Drop long trials
                if Lfull > MAX_TRIAL_BINS:
                    continue

                end_limit_idx = (Lfull - 1) - EXCLUDE_TAIL_BINS
                if end_limit_idx <= 0:
                    continue

                y  = fr_full[:end_limit_idx + 1]
                gz = gaze_full[:end_limit_idx + 1]
                pc = pc1_full[:end_limit_idx + 1]
                sp = spd_full[:end_limit_idx + 1]

                if DO_SMOOTH_FR and len(y) > 3:
                    y = gaussian_filter1d(y, sigma=SMOOTH_FR_SIGMA)

                if (len(y) < MIN_FIT_BINS) or (not np.all(np.isfinite(y))) or (np.var(y) < MIN_VAR):
                    continue

                n = len(y)
                fit_window_s = n * dt

                # Old slope (ABS OLS)
                old_slope = ols_slope_abs(y, dt=dt)
                if not np.isfinite(old_slope):
                    continue

                # Fits
                step_fit = fit_step_changepoint_fast(y, min_seg_bins=MIN_SEG_BINS)
                ramp_fit = fit_flat_then_ramp_changepoint(y, dt=dt, min_gap=MIN_CP_GAP)
                if (step_fit is None) or (ramp_fit is None):
                    continue

                tau_s_idx, mu0, mu1, sse_s, bic_s = step_fit
                tau_r_idx, c, b, sse_r, bic_r = ramp_fit

                delta_bic = float(bic_s - bic_r)  # >0 => ramp wins

                step_jump = float(mu1 - mu0)
                step_dir = "up" if step_jump > 0 else ("down" if step_jump < 0 else "flat")

                ramp_slope = float(b)
                ramp_dir = "up" if ramp_slope > 0 else ("down" if ramp_slope < 0 else "flat")

                tau_step_norm = float(tau_s_idx / (n - 1)) if (n > 1) else np.nan
                tau_ramp_norm = float(tau_r_idx / (n - 1)) if (n > 1) else np.nan

                signed_old_slope_step = float(old_slope * np.sign(step_jump)) if step_dir != "flat" else np.nan
                signed_old_slope_ramp = float(old_slope * np.sign(ramp_slope)) if ramp_dir != "flat" else np.nan

                # Distance to pull for each tau
                dist_to_pull_step_s = float((n - 1 - tau_s_idx) * dt)
                dist_to_pull_ramp_s = float((n - 1 - tau_r_idx) * dt)

                # Covariates at tau
                # For STEP, tau is last baseline bin; optionally use tau+1 for post-step state.
                tau_s_use = min(tau_s_idx + 1, n - 1) if USE_POSTSTEP_TAU else tau_s_idx

                gaze_at_tau_step = _get_at(gz, tau_s_use)
                pc1_at_tau_step  = _get_at(pc, tau_s_use)
                spd_at_tau_step  = _get_at(sp, tau_s_use)

                gaze_at_tau_ramp = _get_at(gz, tau_r_idx)
                pc1_at_tau_ramp  = _get_at(pc, tau_r_idx)
                spd_at_tau_ramp  = _get_at(sp, tau_r_idx)

                trial_fit_records.append({
                    "date": str(session_toana),
                    "clusterID": int(ineuron_idate),
                    "bhv_id": trial_id,

                    "trial_len_bins": int(Lfull),
                    "fit_len_bins": int(n),
                    "fit_window_s": float(fit_window_s),

                    "old_slope": float(old_slope),

                    "bic_step": float(bic_s),
                    "bic_ramp": float(bic_r),
                    "deltaBIC_step_minus_ramp": float(delta_bic),

                    "tau_step_idx": int(tau_s_idx),
                    "tau_ramp_idx": int(tau_r_idx),
                    "tau_step_s": float(tau_s_idx * dt),
                    "tau_ramp_s": float(tau_r_idx * dt),
                    "tau_step_norm": tau_step_norm,
                    "tau_ramp_norm": tau_ramp_norm,

                    "dist_to_pull_step_s": float(dist_to_pull_step_s),
                    "dist_to_pull_ramp_s": float(dist_to_pull_ramp_s),

                    "step_jump": step_jump,
                    "step_dir": step_dir,

                    "ramp_slope_per_s": ramp_slope,
                    "ramp_dir": ramp_dir,

                    "signed_old_slope_step": signed_old_slope_step,
                    "signed_old_slope_ramp": signed_old_slope_ramp,

                    # covariates at tau
                    "gaze_at_tau_step": gaze_at_tau_step,
                    "other_pc1_at_tau_step": pc1_at_tau_step,
                    "other_speed_at_tau_step": spd_at_tau_step,

                    "gaze_at_tau_ramp": gaze_at_tau_ramp,
                    "other_pc1_at_tau_ramp": pc1_at_tau_ramp,
                    "other_speed_at_tau_ramp": spd_at_tau_ramp,
                })

                dBICs.append(delta_bic)
                taus_step_s.append(tau_s_idx * dt)
                taus_ramp_s.append(tau_r_idx * dt)
                n_used += 1

            if n_used > 0:
                dBICs_arr = np.asarray(dBICs, dtype=float)
                neuron_summary_records.append({
                    "date": str(session_toana),
                    "clusterID": int(ineuron_idate),
                    "n_trials_used": int(n_used),
                    "frac_ramp_win": float(np.mean(dBICs_arr > 0)),
                    "median_deltaBIC": float(np.median(dBICs_arr)),
                    "mean_deltaBIC": float(np.mean(dBICs_arr)),
                    "median_tau_step_s": float(np.median(taus_step_s)),
                    "median_tau_ramp_s": float(np.median(taus_ramp_s)),
                })

    trial_fits_df = pd.DataFrame(trial_fit_records)
    neuron_summary_df = pd.DataFrame(neuron_summary_records)

    print("Trial-level fits:", trial_fits_df.shape)
    print("Neuron-level summary:", neuron_summary_df.shape)

    # ============================================================
    # Original plots: per-neuron ΔBIC and frac ramp wins
    # ============================================================
    if len(neuron_summary_df) > 0:
        plt.figure(figsize=(6, 4))
        plt.hist(neuron_summary_df["median_deltaBIC"].values, bins=40)
        plt.axvline(0, linewidth=1)
        plt.xlabel("Per-neuron median ΔBIC (BIC_step - BIC_ramp); >0 => ramp wins")
        plt.ylabel("# neurons")
        plt.title("Step vs ramp (full trial fit window)")
        plt.tight_layout()
        plt.show()

        plt.figure(figsize=(6, 4))
        plt.hist(neuron_summary_df["frac_ramp_win"].values, bins=30, range=(0, 1))
        plt.axvline(0.5, linewidth=1)
        plt.xlabel("Fraction of trials where ramp wins (ΔBIC>0)")
        plt.ylabel("# neurons")
        plt.title("Ramp-win fraction across neurons (full trial window)")
        plt.tight_layout()
        plt.show()

    # ============================================================
    # Winner labels
    # ============================================================
    if len(trial_fits_df) == 0:
        raise ValueError("No valid trial fits found. Check EXCLUDE_TAIL_S, MIN_FIT_BINS, data columns, etc.")

    m = np.isfinite(trial_fits_df["bic_step"].values) & np.isfinite(trial_fits_df["bic_ramp"].values)
    trial_fits_df = trial_fits_df[m].copy()
    trial_fits_df["win_model"] = np.where(trial_fits_df["bic_ramp"].values < trial_fits_df["bic_step"].values, "ramp", "step")

    # ============================================================
    # NEW: tau-to-pull + covariates@tau plots
    # ============================================================
    def plot_tau_and_covariates(df_plot, which="STEP"):
        if len(df_plot) == 0:
            print(f"[plot_tau_and_covariates] Empty df for {which}")
            return

        dfp = _subsample_df(df_plot, maxn=PLOT_MAX_POINTS, seed=0)

        if which.upper() == "STEP":
            dist_col = "dist_to_pull_step_s"
            covs = [
                ("gaze_at_tau_step", "socialgaze_prob at STEP tau"),
                ("other_pc1_at_tau_step", "other_PC1 at STEP tau"),
                ("other_speed_at_tau_step", "other_mass_move_speed at STEP tau"),
            ]
            ttl_prefix = "STEP"
        else:
            dist_col = "dist_to_pull_ramp_s"
            covs = [
                ("gaze_at_tau_ramp", "socialgaze_prob at RAMP tau"),
                ("other_pc1_at_tau_ramp", "other_PC1 at RAMP tau"),
                ("other_speed_at_tau_ramp", "other_mass_move_speed at RAMP tau"),
            ]
            ttl_prefix = "RAMP"

        # dist-to-pull
        x = dfp[dist_col].values
        x = x[np.isfinite(x)]
        plt.figure(figsize=(6, 4))
        plt.hist(x, bins=50)
        plt.xlabel(f"{dist_col} (seconds)")
        plt.ylabel("# trials")
        plt.title(f"{ttl_prefix}: distance-to-pull at tau (pooled)")
        plt.tight_layout()
        plt.show()

        # covariates at tau
        for col, title in covs:
            v = dfp[col].values
            v = v[np.isfinite(v)]
            plt.figure(figsize=(6, 4))
            plt.hist(v, bins=50)
            plt.xlabel(col)
            plt.ylabel("# trials")
            plt.title(f"{ttl_prefix}: {title} (pooled)")
            plt.tight_layout()
            plt.show()

        # per-neuron means (across trials)
        per_neuron = []
        for (date, clusterID), g in df_plot.groupby(["date", "clusterID"], sort=False):
            per_neuron.append({
                "date": str(date),
                "clusterID": int(clusterID),
                "n_trials": int(len(g)),
                "mean_dist_to_pull": float(np.nanmean(g[dist_col].values)),
                "mean_gaze": float(np.nanmean(g[covs[0][0]].values)),
                "mean_pc1":  float(np.nanmean(g[covs[1][0]].values)),
                "mean_spd":  float(np.nanmean(g[covs[2][0]].values)),
            })
        per_neuron_df = pd.DataFrame(per_neuron)

        if len(per_neuron_df) > 0:
            for col, title in [
                ("mean_dist_to_pull", f"{ttl_prefix}: per-neuron mean dist-to-pull at tau"),
                ("mean_gaze", f"{ttl_prefix}: per-neuron mean gaze@tau"),
                ("mean_pc1", f"{ttl_prefix}: per-neuron mean other_PC1@tau"),
                ("mean_spd", f"{ttl_prefix}: per-neuron mean other_speed@tau"),
            ]:
                vv = per_neuron_df[col].values
                vv = vv[np.isfinite(vv)]
                plt.figure(figsize=(6, 4))
                plt.hist(vv, bins=40)
                plt.xlabel(col)
                plt.ylabel("# neurons")
                plt.title(title)
                plt.tight_layout()
                plt.show()

        return per_neuron_df

    if PLOT_WIN_MODEL == "step":
        df_step_win = trial_fits_df[trial_fits_df["win_model"] == "step"].copy()
        _ = plot_tau_and_covariates(df_step_win, which="STEP")

    elif PLOT_WIN_MODEL == "ramp":
        df_ramp_win = trial_fits_df[trial_fits_df["win_model"] == "ramp"].copy()
        _ = plot_tau_and_covariates(df_ramp_win, which="RAMP")

    elif PLOT_WIN_MODEL == "both":
        df_step_win = trial_fits_df[trial_fits_df["win_model"] == "step"].copy()
        df_ramp_win = trial_fits_df[trial_fits_df["win_model"] == "ramp"].copy()
        _ = plot_tau_and_covariates(df_step_win, which="STEP")
        _ = plot_tau_and_covariates(df_ramp_win, which="RAMP")
    else:
        raise ValueError("PLOT_WIN_MODEL must be one of: 'step', 'ramp', 'both'")

    # ============================================================
    # Outputs in memory:
    #   trial_fits_df, neuron_summary_df
    # ============================================================
    # Optional save:
    # trial_fits_df.to_csv("step_ramp_trialfits_with_tau_covariates.csv", index=False)
    # neuron_summary_df.to_csv("step_ramp_neuron_summary.csv", index=False)


In [ ]:
####
# Per-neuron tau consistency analysis (copy/paste cell)
#
# Input:
#   - trial_fits_df from your STEP/RAMP fitting code (must contain at least):
#       ['date','clusterID','fit_len_bins','tau_step_idx']  (for STEP)
#     or
#       ['date','clusterID','fit_len_bins','tau_ramp_idx']  (for RAMP)
#
# What it computes (per neuron within session):
#   - median dist_to_pull (seconds)
#   - MAD / IQR of dist_to_pull (seconds)
#   - split-half reliability of median dist_to_pull (repeat splits)
#   - bootstrap CI width for median dist_to_pull
#
# Outputs:
#   - neuron_tau_consistency_df
#   - Summary plots
####

if 0:
    
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt

    # =========================
    # USER SETTINGS
    # =========================
    fps = 30.0
    dt = 1.0 / fps

    # Choose which tau to analyze
    TAU_MODE = "step"   # "step" or "ramp"

    # If you only want e.g. step-winning trials, set this True and ensure win_model exists
    USE_WINNER_FILTER = False
    WINNER_MODEL = "step"  # "step" or "ramp"

    # Split-half
    N_SPLIT_REPEATS = 300
    MIN_TRIALS_FOR_SPLIT = 8   # require at least this many trials for split-half metrics

    # Bootstrap CI for median
    N_BOOT = 800
    CI_PCT = (2.5, 97.5)

    RNG_SEED = 0

    # =========================
    # Helpers
    # =========================
    def _mad(x):
        x = np.asarray(x, float)
        x = x[np.isfinite(x)]
        if len(x) == 0:
            return np.nan
        med = np.nanmedian(x)
        return float(np.nanmedian(np.abs(x - med)))

    def _iqr(x):
        x = np.asarray(x, float)
        x = x[np.isfinite(x)]
        if len(x) == 0:
            return np.nan
        q25, q75 = np.nanpercentile(x, [25, 75])
        return float(q75 - q25)

    def split_half_reliability_median(x, n_repeats=200, seed=0):
        """
        x: 1D array of per-trial values for one neuron (e.g., dist_to_pull_s)
        Returns:
          r_split: correlation across repeats of (median_half1, median_half2)
          median_abs_diff: median(|med1 - med2|)
          med1_mean, med2_mean: mean medians across repeats (sanity)
        """
        rng = np.random.default_rng(seed)
        x = np.asarray(x, float)
        x = x[np.isfinite(x)]
        n = len(x)
        if n < 4:
            return np.nan, np.nan, np.nan, np.nan

        med1 = np.empty(n_repeats, float)
        med2 = np.empty(n_repeats, float)

        idx_all = np.arange(n)
        for k in range(n_repeats):
            perm = rng.permutation(idx_all)
            h = n // 2
            i1 = perm[:h]
            i2 = perm[h:]
            if len(i1) == 0 or len(i2) == 0:
                med1[k] = np.nan
                med2[k] = np.nan
                continue
            med1[k] = np.nanmedian(x[i1])
            med2[k] = np.nanmedian(x[i2])

        m = np.isfinite(med1) & np.isfinite(med2)
        if np.sum(m) < 5:
            return np.nan, np.nan, np.nan, np.nan

        a = med1[m]; b = med2[m]
        # correlation (Pearson) across repeats
        a0 = a - np.mean(a)
        b0 = b - np.mean(b)
        denom = np.sqrt(np.sum(a0*a0) * np.sum(b0*b0))
        r = np.nan if denom <= 1e-12 else float(np.sum(a0*b0) / denom)

        med_abs_diff = float(np.nanmedian(np.abs(a - b)))
        return r, med_abs_diff, float(np.nanmean(a)), float(np.nanmean(b))

    def bootstrap_median_ci(x, n_boot=500, ci_pct=(2.5,97.5), seed=0):
        rng = np.random.default_rng(seed)
        x = np.asarray(x, float)
        x = x[np.isfinite(x)]
        n = len(x)
        if n == 0:
            return np.nan, np.nan, np.nan, np.nan
        boots = np.empty(n_boot, float)
        for k in range(n_boot):
            samp = rng.choice(x, size=n, replace=True)
            boots[k] = np.nanmedian(samp)
        lo, hi = np.nanpercentile(boots, ci_pct)
        return float(lo), float(hi), float(hi - lo), float(np.nanstd(boots))

    # =========================
    # Validate input and pick tau column
    # =========================
    if "trial_fits_df" not in globals():
        raise ValueError("trial_fits_df not found. Run your STEP/RAMP fitting cell first.")

    df = trial_fits_df.copy()

    required_common = ["date", "clusterID", "fit_len_bins"]
    for c in required_common:
        if c not in df.columns:
            raise ValueError(f"trial_fits_df missing required column: {c}")

    if TAU_MODE == "step":
        tau_col = "tau_step_idx"
    elif TAU_MODE == "ramp":
        tau_col = "tau_ramp_idx"
    else:
        raise ValueError('TAU_MODE must be "step" or "ramp"')

    if tau_col not in df.columns:
        raise ValueError(f"trial_fits_df missing tau column: {tau_col}")

    # Optional: winner filter if present
    if USE_WINNER_FILTER:
        if "win_model" not in df.columns:
            raise ValueError("USE_WINNER_FILTER=True but win_model column not found in trial_fits_df.")
        df = df[df["win_model"] == WINNER_MODEL].copy()

    # Basic finite checks
    m = np.isfinite(df[tau_col].values) & np.isfinite(df["fit_len_bins"].values)
    df = df[m].copy()

    # Compute dist_to_pull_s per trial
    # dist_to_pull_s = (n-1 - tau_idx) * dt
    df["dist_to_pull_s"] = (df["fit_len_bins"].astype(float) - 1.0 - df[tau_col].astype(float)) * dt
    df["tau_norm"] = df[tau_col].astype(float) / (df["fit_len_bins"].astype(float) - 1.0)

    # =========================
    # Per-neuron consistency metrics
    # =========================
    rng = np.random.default_rng(RNG_SEED)
    rows = []

    for (date, cid), g in df.groupby(["date", "clusterID"], sort=False):
        x = g["dist_to_pull_s"].values
        x = x[np.isfinite(x)]
        n = len(x)
        if n == 0:
            continue

        med = float(np.nanmedian(x))
        mad = _mad(x)
        iqr = _iqr(x)

        # split-half + bootstrap only if enough trials
        if n >= MIN_TRIALS_FOR_SPLIT:
            r_split, med_abs_diff, med1_mean, med2_mean = split_half_reliability_median(
                x, n_repeats=N_SPLIT_REPEATS, seed=int(rng.integers(0, 2**31 - 1))
            )
        else:
            r_split, med_abs_diff, med1_mean, med2_mean = np.nan, np.nan, np.nan, np.nan

        lo, hi, ciw, boot_sd = bootstrap_median_ci(
            x, n_boot=N_BOOT, ci_pct=CI_PCT, seed=int(rng.integers(0, 2**31 - 1))
        )

        rows.append({
            "date": str(date),
            "clusterID": int(cid),
            "n_trials": int(n),

            "median_dist_to_pull_s": med,
            "mad_dist_to_pull_s": mad,
            "iqr_dist_to_pull_s": iqr,

            "split_half_r_median": r_split,
            "split_half_median_absdiff_s": med_abs_diff,

            "boot_median_ci_lo_s": lo,
            "boot_median_ci_hi_s": hi,
            "boot_median_ci_width_s": ciw,
            "boot_median_sd_s": boot_sd,
        })

    neuron_tau_consistency_df = pd.DataFrame(rows)
    print("neuron_tau_consistency_df:", neuron_tau_consistency_df.shape)
    display(neuron_tau_consistency_df.head())

    # =========================
    # Summary plots
    # =========================
    if len(neuron_tau_consistency_df) > 0:

        # 1) distribution of median dist-to-pull
        plt.figure(figsize=(6,4))
        plt.hist(neuron_tau_consistency_df["median_dist_to_pull_s"].values, bins=40)
        plt.axvline(0, linewidth=1)
        plt.xlabel("Per-neuron median (tau → pull) distance (s)")
        plt.ylabel("# neurons")
        plt.title(f"{TAU_MODE.upper()} tau: per-neuron median dist-to-pull")
        plt.tight_layout()
        plt.show()

        # 2) distribution of MAD (consistency)
        plt.figure(figsize=(6,4))
        plt.hist(neuron_tau_consistency_df["mad_dist_to_pull_s"].values[np.isfinite(neuron_tau_consistency_df["mad_dist_to_pull_s"].values)], bins=40)
        plt.axvline(0, linewidth=1)
        plt.xlabel("Per-neuron MAD of dist-to-pull (s)")
        plt.ylabel("# neurons")
        plt.title(f"{TAU_MODE.upper()} tau: per-neuron consistency (MAD)")
        plt.tight_layout()
        plt.show()

        # 3) split-half reliability distribution
        plt.figure(figsize=(6,4))
        rr = neuron_tau_consistency_df["split_half_r_median"].values
        rr = rr[np.isfinite(rr)]
        if len(rr) > 0:
            plt.hist(rr, bins=35, range=(-1,1))
            plt.axvline(0, linewidth=1)
            plt.xlabel("Split-half reliability (corr of half medians across repeats)")
            plt.ylabel("# neurons")
            plt.title(f"{TAU_MODE.upper()} tau: split-half reliability (median dist-to-pull)")
            plt.tight_layout()
            plt.show()

        # 4) bootstrap CI width distribution
        plt.figure(figsize=(6,4))
        ww = neuron_tau_consistency_df["boot_median_ci_width_s"].values
        ww = ww[np.isfinite(ww)]
        if len(ww) > 0:
            plt.hist(ww, bins=40)
            plt.axvline(0, linewidth=1)
            plt.xlabel("Bootstrap CI width of median dist-to-pull (s)")
            plt.ylabel("# neurons")
            plt.title(f"{TAU_MODE.upper()} tau: uncertainty of per-neuron median")
            plt.tight_layout()
            plt.show()

        # 5) sanity: MAD vs n_trials
        plt.figure(figsize=(6,4))
        plt.scatter(neuron_tau_consistency_df["n_trials"].values,
                    neuron_tau_consistency_df["mad_dist_to_pull_s"].values,
                    s=12, alpha=0.35)
        plt.xlabel("n_trials per neuron")
        plt.ylabel("MAD dist-to-pull (s)")
        plt.title(f"{TAU_MODE.upper()} tau: consistency vs trial count")
        plt.tight_layout()
        plt.show()

    # =========================
    # Notes:
    # - neuron_tau_consistency_df is now in memory.
    # - If you want “distance-to-pull” for step-win only, set:
    #     USE_WINNER_FILTER=True; WINNER_MODEL="step"; TAU_MODE="step"
    # =========================


In [ ]:
####
# analysis for reviewer 2: step vs ramp on neural firing DURING SOCIAL GAZE
# - Single-trial, variable length
# - Uses LAST social-gaze bout that ends BEFORE (pull - 0.85 s)
# - Fits Step (change-point) vs Flat-then-Ramp (change-point + slope) within that bout window
# - fps = 30 Hz
# - Saves per neuron per trial: tau_step, tau_ramp, BICs, deltaBIC, bout start/end/duration
# NOTE: This is intentionally WITHOUT the matched no-gaze control (per your request)
####

if 0: # old, do not use

    import os, pickle
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from scipy.ndimage import gaussian_filter1d
    from scipy.stats import zscore

    # ============================================================
    # USER SETTINGS
    # ============================================================
    fps = 30.0
    dt = 1.0 / fps

    EXCLUDE_TAIL_S = 0.85
    EXCLUDE_TAIL_BINS = int(np.round(EXCLUDE_TAIL_S * fps))  # last 0.85s before pull is excluded from eligibility

    # social gaze bout definition
    GAZE_THRESH = 0.00
    SMOOTH_GAZE = False
    SMOOTH_GAZE_SIGMA = 8  # matches your earlier usage (frames)

    # bout duration threshold
    MIN_BOUT_DUR_S = 0.3
    MIN_BOUT_BINS = int(np.ceil(MIN_BOUT_DUR_S * fps))

    # model fitting thresholds inside selected bout window
    MIN_FIT_BINS = 15          # within bout, need enough points
    MIN_CP_GAP = 4             # change-point cannot be too close to ends (within the bout segment)
    MIN_VAR = 1e-6             # skip almost-flat traces

    # optional: mild smoothing on FR within bout (usually keep off initially)
    DO_SMOOTH_FR = False
    SMOOTH_FR_SIGMA = 2

    # ============================================================
    # Helpers: step vs ramp fits
    # ============================================================
    def _as_1d_float_array(x):
        return np.asarray(x, dtype=float).ravel()

    def _bic_from_sse(sse, n, k):
        if (not np.isfinite(sse)) or (sse <= 0):
            sse = 1e-12
        return n * np.log(sse / n) + k * np.log(n)

    def fit_step_changepoint(y, min_gap=4):
        """
        Step model: y = c1 for i < tau; y = c2 for i >= tau
        Params: c1, c2, tau (k=3)
        Returns: tau_idx, c1, c2, sse, bic
        """
        y = _as_1d_float_array(y)
        n = len(y)
        if n < (2 * min_gap + 2):
            return None

        csum = np.cumsum(y)
        csum2 = np.cumsum(y * y)

        best_sse = np.inf
        best_tau = None
        best_c1 = np.nan
        best_c2 = np.nan

        for tau in range(min_gap, n - min_gap):
            n1 = tau
            n2 = n - tau

            s1 = csum[tau - 1]
            s12 = csum2[tau - 1]
            m1 = s1 / n1
            sse1 = s12 - (s1 * s1) / n1

            s2 = csum[-1] - s1
            s22 = csum2[-1] - s12
            m2 = s2 / n2
            sse2 = s22 - (s2 * s2) / n2

            sse = sse1 + sse2
            if sse < best_sse:
                best_sse = sse
                best_tau = tau
                best_c1 = m1
                best_c2 = m2

        bic = _bic_from_sse(best_sse, n=n, k=3)
        return best_tau, float(best_c1), float(best_c2), float(best_sse), float(bic)

    def fit_flat_then_ramp_changepoint(y, dt, min_gap=4):
        """
        Flat-then-ramp model:
          y = c                  for i < tau
          y = c + b*(t - t_tau)  for i >= tau
        Params: c, b, tau (k=3)
        Returns: tau_idx, c, b, sse, bic
        """
        y = _as_1d_float_array(y)
        n = len(y)
        if n < (2 * min_gap + 2):
            return None

        t = np.arange(n) * dt

        best_sse = np.inf
        best_tau = None
        best_c = np.nan
        best_b = np.nan

        for tau in range(min_gap, n - min_gap):
            t_tau = t[tau]
            r = np.maximum(0.0, t - t_tau)

            s00 = n
            s01 = np.sum(r)
            s11 = np.sum(r * r)
            y0 = np.sum(y)
            y1 = np.sum(r * y)

            det = s00 * s11 - s01 * s01
            if det <= 1e-12:
                continue

            c = (y0 * s11 - y1 * s01) / det
            b = (y1 * s00 - y0 * s01) / det

            yhat = c + b * r
            resid = y - yhat
            sse = float(np.sum(resid * resid))

            if sse < best_sse:
                best_sse = sse
                best_tau = tau
                best_c = c
                best_b = b

        bic = _bic_from_sse(best_sse, n=n, k=3)
        return best_tau, float(best_c), float(best_b), float(best_sse), float(bic)

    # ============================================================
    # Helpers: bout finding
    # ============================================================
    def find_last_bout_ending_before(gaze_trace, end_limit_idx, thresh=0.05, smooth=True, sigma=8):
        """
        Find the last contiguous segment where gaze > thresh,
        with segment_end <= end_limit_idx.
        Returns (start_idx, end_idx) or None.
        """
        g = _as_1d_float_array(gaze_trace)

        if smooth and len(g) > 3:
            g = gaussian_filter1d(g, sigma=sigma)

        on = g > thresh
        if not np.any(on):
            return None

        # find all segments of True
        # segment boundaries where on changes
        idx = np.where(on)[0]
        # build segments by scanning idx
        segments = []
        s = idx[0]
        prev = idx[0]
        for k in idx[1:]:
            if k == prev + 1:
                prev = k
            else:
                segments.append((s, prev))
                s = k
                prev = k
        segments.append((s, prev))

        # keep only those ending before end_limit_idx
        eligible = [(a, b) for (a, b) in segments if b <= end_limit_idx]
        if len(eligible) == 0:
            return None

        return eligible[-1]  # last in time

    # ============================================================
    # Your original flags / loading logic (kept as-is)
    # ============================================================

    dobhvPCA = 1
    doparnterleverspeed = 0

    if not dobhvPCA:
        doOnsetAfterMin = 1
        if not doOnsetAfterMin:
            doOnsetAfterMin_suffix = ''
        else:
            if not doparnterleverspeed:
                doOnsetAfterMin_suffix = 'PullOnsetAfterMin_'
            else:
                doOnsetAfterMin_suffix = 'PullOnsetAfterMin_PartnerLeverSpeed_'
    else:
        doOnsetAfterMin = 1
        doBasedonSpeed = 1
        if not doOnsetAfterMin:
            doOnsetAfterMin_suffix = 'bhvPCA_'
        else:
            if not doBasedonSpeed:
                doOnsetAfterMin_suffix = 'bhvPCA_PullOnsetAfterMin_'
            else:
                doOnsetAfterMin_suffix = 'bhvPCA_PullOnsetAfterMinBasedonSpeed_'

    doHMMmethod = 0
    if doHMMmethod:
        doOnsetAfterMin_suffix = 'HMMmethods_'

    exaustModel_sufix = ''

    # load the dataset first
    current_dir = data_saved_folder + 'data_saved_singlecam_wholebody_neural_and_hddm' + savefile_sufix + '/' + cameraID + '/' + animal_tgt
    add_date_dir = os.path.join(current_dir + '/hddm_model_fitted_combinedsession_withNeurons')

    if cond_toplot_type == 'allMC':
        if not (doOnsetAfterMin_suffix == 'bhvPCA_PullOnsetAfterMinBasedonSpeed_') | \
               (doOnsetAfterMin_suffix == 'PullOnsetAfterMin_PartnerLeverSpeed_'):
            with open(add_date_dir + '/hddm_datas_flexibleTW_newRTdefinition_' + doOnsetAfterMin_suffix + 'withFRs_all_dates_' +
                      animal1_fixedorders[0] + animal2_fixedorders[0] + '.pkl', 'rb') as f:
                hddm_data_tgt_alldates = pickle.load(f)
        else:
            with open(add_date_dir + '/hddm_datas_flexibleTW_newRTdefinition_' + 'PullOnsetAfterMin_' + 'withFRs_all_dates_' +
                      animal1_fixedorders[0] + animal2_fixedorders[0] + '.pkl', 'rb') as f:
                hddm_data_tgt_alldates = pickle.load(f)
    else:
        if not (doOnsetAfterMin_suffix == 'bhvPCA_PullOnsetAfterMinBasedonSpeed_') | \
               (doOnsetAfterMin_suffix == 'PullOnsetAfterMin_PartnerLeverSpeed_'):
            with open(add_date_dir + '/hddm_datas_flexibleTW_newRTdefinition_' + doOnsetAfterMin_suffix + 'withFRs_all_dates_' +
                      animal1_fixedorders[0] + animal2_fixedorders[0] + '_' + cond_toplot_type + '.pkl', 'rb') as f:
                hddm_data_tgt_alldates = pickle.load(f)
        else:
            with open(add_date_dir + '/hddm_datas_flexibleTW_newRTdefinition_' + 'PullOnsetAfterMin_' + 'withFRs_all_dates_' +
                      animal1_fixedorders[0] + animal2_fixedorders[0] + '_' + cond_toplot_type + '.pkl', 'rb') as f:
                hddm_data_tgt_alldates = pickle.load(f)

    # make the hddm_data structure consistent with other code
    ind_goodcond = np.isin(hddm_data_tgt_alldates['condition'], conditions_to_ana)
    hddm_data_tgt_alldates = hddm_data_tgt_alldates[ind_goodcond]

    if not dobhvPCA:
        if not doparnterleverspeed:
            rename_dict = {
                'act_animal': 'subj_idx',
                'dates': 'date',
                'pull_rt': 'rt',
                'previous_pull_outcome': 'prev_trial_outcome',
                'socialgaze_auc': 'self_gaze_auc',
                'mass_move_speed_mean': 'self_mean_speed',
                'mass_move_speed_std': 'self_speed_std',
                'other_mass_move_speed_mean': 'partner_mean_speed',
                'other_mass_move_speed_std': 'partner_speed_std',
                'num_preceding_failpull': 'failed_pulls_before_reward',
                'time_from_last_reward': 'time_since_last_reward',
            }
        else:
            rename_dict = {
                'act_animal': 'subj_idx',
                'dates': 'date',
                'pull_rt': 'rt',
                'previous_pull_outcome': 'prev_trial_outcome',
                'socialgaze_auc': 'self_gaze_auc',
                'mass_move_speed_mean': 'self_mean_speed',
                'mass_move_speed_std': 'self_speed_std',
                'other_lever_speed_mean': 'partner_mean_speed',
                'other_lever_speed_std': 'partner_speed_std',
                'num_preceding_failpull': 'failed_pulls_before_reward',
                'time_from_last_reward': 'time_since_last_reward',
            }
    else:
        rename_dict = {
            'act_animal': 'subj_idx',
            'dates': 'date',
            'pull_rt': 'rt',
            'previous_pull_outcome': 'prev_trial_outcome',
            'socialgaze_auc': 'self_gaze_auc',
            'mass_move_speed_mean': 'self_mean_speed',
            'mass_move_speed_std': 'self_speed_std',
            'other_PC1_mean': 'partner_mean_speed',
            'other_PC1_std': 'partner_speed_std',
            'num_preceding_failpull': 'failed_pulls_before_reward',
            'time_from_last_reward': 'time_since_last_reward',
        }

    hddm_data_tgt_alldates = hddm_data_tgt_alldates.rename(columns=rename_dict)
    hddm_data_tgt_alldates['response'] = 1

    # remove nan row first for the glm data set
    hddm_data_tgt_alldates = hddm_data_tgt_alldates.dropna()

    # normalize predictors (kept; not required for step/ramp but kept for compatibility)
    predictor_columns = [
        'self_gaze_auc',
        'partner_mean_speed',
        'self_mean_speed',
        'partner_speed_std',
        'self_speed_std',
        'failed_pulls_before_reward',
        'time_since_last_reward'
    ]
    for col in predictor_columns:
        if col in hddm_data_tgt_alldates.columns:
            hddm_data_tgt_alldates[f"{col}_origin"] = hddm_data_tgt_alldates[col]
            hddm_data_tgt_alldates[col] = zscore(hddm_data_tgt_alldates[col])

    # ============================================================
    # NEW: results containers
    # ============================================================
    trial_fit_records = []
    neuron_summary_records = []

    # ============================================================
    # Run for each session
    # ============================================================
    for session_toana, hddm_data_tgt_idate in hddm_data_tgt_alldates.groupby('date'):

        # Outlier removal based on rt/time_since_last_reward (same as before)
        lower_q = 0.05
        upper_q = 0.95
        if 'rt' in hddm_data_tgt_idate.columns:
            rt_lower, rt_upper = hddm_data_tgt_idate['rt'].quantile([lower_q, upper_q])
            hddm_data_tgt_idate = hddm_data_tgt_idate[(hddm_data_tgt_idate['rt'] > rt_lower) & (hddm_data_tgt_idate['rt'] < rt_upper)]
        if 'time_since_last_reward' in hddm_data_tgt_idate.columns:
            tslr_lower, tslr_upper = hddm_data_tgt_idate['time_since_last_reward'].quantile([lower_q, upper_q])
            hddm_data_tgt_idate = hddm_data_tgt_idate[(hddm_data_tgt_idate['time_since_last_reward'] > tslr_lower) &
                                                      (hddm_data_tgt_idate['time_since_last_reward'] < tslr_upper)]

        # raw neural rows for this date
        hddm_RawFullDatas_tgt_idate = hddm_RawFullDatas_tgt_alldates[hddm_RawFullDatas_tgt_alldates['dates'] == session_toana]
        neurons_idate = np.unique(hddm_RawFullDatas_tgt_idate['clusterID'])

        for ineuron_idate in neurons_idate:

            dfN = hddm_RawFullDatas_tgt_idate[hddm_RawFullDatas_tgt_idate['clusterID'] == ineuron_idate]

            dBICs = []
            taus_step_s = []
            taus_ramp_s = []
            n_used = 0

            for _, row in dfN.iterrows():

                # trial identifiers
                trial_id = int(row['bhv_id']) if 'bhv_id' in row else None

                # traces
                fr_full = _as_1d_float_array(row['FR_ievent'])
                gaze_full = _as_1d_float_array(row['socialgaze_prob']) if 'socialgaze_prob' in row else None
                if gaze_full is None:
                    continue

                # ensure equal length (trim to min)
                L = min(len(fr_full), len(gaze_full))
                if L <= (EXCLUDE_TAIL_BINS + MIN_FIT_BINS):
                    continue
                fr_full = fr_full[:L]
                gaze_full = gaze_full[:L]

                # define eligibility: bout must END before end_limit_idx
                end_limit_idx = (L - 1) - EXCLUDE_TAIL_BINS
                if end_limit_idx <= 0:
                    continue

                # find last bout ending before the limit
                bout = find_last_bout_ending_before(
                    gaze_full, end_limit_idx=end_limit_idx,
                    thresh=GAZE_THRESH, smooth=SMOOTH_GAZE, sigma=SMOOTH_GAZE_SIGMA
                )
                if bout is None:
                    continue

                b0, b1 = bout
                bout_len = b1 - b0 + 1
                if bout_len < MIN_BOUT_BINS:
                    continue

                # extract FR within bout
                y = fr_full[b0:b1 + 1]
                if DO_SMOOTH_FR and len(y) > 3:
                    y = gaussian_filter1d(y, sigma=SMOOTH_FR_SIGMA)

                # QC
                if (len(y) < MIN_FIT_BINS) or (not np.all(np.isfinite(y))) or (np.var(y) < MIN_VAR):
                    continue

                # fit models
                step_fit = fit_step_changepoint(y, min_gap=MIN_CP_GAP)
                ramp_fit = fit_flat_then_ramp_changepoint(y, dt=dt, min_gap=MIN_CP_GAP)
                if (step_fit is None) or (ramp_fit is None):
                    continue

                tau_s_idx, c1, c2, sse_s, bic_s = step_fit
                tau_r_idx, c, b, sse_r, bic_r = ramp_fit

                delta_bic = float(bic_s - bic_r)  # >0 => ramp better

                # store trial-level
                trial_fit_records.append({
                    "date": str(session_toana),
                    "clusterID": int(ineuron_idate),
                    "bhv_id": trial_id,
                    "trial_len_bins": int(L),
                    "exclude_tail_s": float(EXCLUDE_TAIL_S),
                    "end_limit_idx": int(end_limit_idx),

                    "bout_start_idx": int(b0),
                    "bout_end_idx": int(b1),
                    "bout_dur_s": float(bout_len * dt),

                    # change points in ABSOLUTE trial coordinates (seconds from trial start)
                    "tau_step_abs_s": float((b0 + tau_s_idx) * dt),
                    "tau_ramp_abs_s": float((b0 + tau_r_idx) * dt),

                    # change points relative to bout start (seconds)
                    "tau_step_inbout_s": float(tau_s_idx * dt),
                    "tau_ramp_inbout_s": float(tau_r_idx * dt),

                    "bic_step": float(bic_s),
                    "bic_ramp": float(bic_r),
                    "deltaBIC_step_minus_ramp": float(delta_bic),

                    "step_jump": float(c2 - c1),
                    "ramp_slope_per_s": float(b),
                    "y_var": float(np.var(y)),
                })

                dBICs.append(delta_bic)
                taus_step_s.append((b0 + tau_s_idx) * dt)
                taus_ramp_s.append((b0 + tau_r_idx) * dt)
                n_used += 1

            # store neuron-level summary
            if n_used > 0:
                dBICs_arr = np.asarray(dBICs, dtype=float)
                neuron_summary_records.append({
                    "date": str(session_toana),
                    "clusterID": int(ineuron_idate),
                    "n_trials_used": int(n_used),
                    "frac_ramp_win": float(np.mean(dBICs_arr > 0)),
                    "median_deltaBIC": float(np.median(dBICs_arr)),
                    "mean_deltaBIC": float(np.mean(dBICs_arr)),
                    "median_tau_step_abs_s": float(np.median(taus_step_s)),
                    "median_tau_ramp_abs_s": float(np.median(taus_ramp_s)),
                })

    # ============================================================
    # Output DataFrames + quick summary plots
    # ============================================================
    trial_fits_df = pd.DataFrame(trial_fit_records)
    neuron_summary_df = pd.DataFrame(neuron_summary_records)

    print("Trial-level fits:", trial_fits_df.shape)
    print("Neuron-level summary:", neuron_summary_df.shape)

    if len(neuron_summary_df) > 0:
        plt.figure(figsize=(6, 4))
        plt.hist(neuron_summary_df["median_deltaBIC"].values, bins=40)
        plt.axvline(0, linewidth=1)
        plt.xlabel("Per-neuron median ΔBIC (BIC_step - BIC_ramp); >0 => ramp wins")
        plt.ylabel("# neurons")
        plt.title("Step vs ramp during last eligible social-gaze bout")
        plt.tight_layout()
        plt.show()

        plt.figure(figsize=(6, 4))
        plt.hist(neuron_summary_df["frac_ramp_win"].values, bins=30, range=(0, 1))
        plt.axvline(0.5, linewidth=1)
        plt.xlabel("Fraction of trials where ramp wins (ΔBIC>0)")
        plt.ylabel("# neurons")
        plt.title("Ramp-win fraction across neurons (gaze bouts)")
        plt.tight_layout()
        plt.show()

    # Optional: Save
    # trial_fits_df.to_csv("step_vs_ramp_gaze_lastbout_trial_fits.csv", index=False)
    # neuron_summary_df.to_csv("step_vs_ramp_gaze_lastbout_neuron_summary.csv", index=False)


In [ ]:
####
# Gaze-bout continuity vs reset test (Reviewer 1 concept; also helps R2)
# Question: across multiple social-gaze bouts within a trial, does neural activity "carry over" or "reset"?
#
# Strategy (per neuron, across trials):
# - Find social-gaze bouts from socialgaze_prob (thresholded; optional smoothing)
# - Consider only bouts that END before (pull - EXCLUDE_TAIL_S)  [default 0.85s]
# - For trials with >=2 eligible bouts:
#     x = mean FR near END of the 2nd-last eligible bout
#     y = mean FR near START of the last eligible bout
#   If accumulation carries over, x and y should covary across trials.
#
# Bidirectional note:
# - Ramping/stepping can be up or down, so we report both signed correlation r and |r|.
# - We do NOT assume positive effects.
####

if 0: # old version, do not use

    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from scipy.ndimage import gaussian_filter1d

    # ----------------------------
    # User settings
    # ----------------------------
    fps = 30.0
    dt = 1.0 / fps

    EXCLUDE_TAIL_S = 0.3 # 0.85
    EXCLUDE_TAIL_BINS = int(np.round(EXCLUDE_TAIL_S * fps))

    # gaze bout detection
    GAZE_THRESH = 0.05
    SMOOTH_GAZE = True
    SMOOTH_GAZE_SIGMA = 8  # frames, consistent with your earlier usage

    # bout eligibility + snippet windows
    MIN_BOUT_DUR_S = 0.30
    MIN_BOUT_BINS = int(np.ceil(MIN_BOUT_DUR_S * fps))

    SNIP_WIN_S = 0.20                  # use short windows to reduce noise
    SNIP_WIN_BINS = max(1, int(np.round(SNIP_WIN_S * fps)))  # ~6 frames at 30Hz

    # optional FR smoothing (keep off initially)
    SMOOTH_FR = False
    SMOOTH_FR_SIGMA = 2

    # per-neuron stats: require enough trials with >=2 eligible bouts
    MIN_PAIRS_PER_NEURON = 6

    # ----------------------------
    # Helpers
    # ----------------------------
    def _as_1d_float_array(x):
        return np.asarray(x, dtype=float).ravel()

    def _clip(a, lo, hi):
        return max(lo, min(hi, a))

    def find_bouts(gaze_trace, thresh=0.05, smooth=True, sigma=8):
        """
        Return list of (start_idx, end_idx) inclusive for contiguous gaze>thresh segments.
        """
        g = _as_1d_float_array(gaze_trace)
        if smooth and len(g) > 3:
            g = gaussian_filter1d(g, sigma=sigma)
        on = g > thresh
        if not np.any(on):
            return []

        idx = np.where(on)[0]
        bouts = []
        s = idx[0]
        prev = idx[0]
        for k in idx[1:]:
            if k == prev + 1:
                prev = k
            else:
                bouts.append((s, prev))
                s = k
                prev = k
        bouts.append((s, prev))
        return bouts

    def mean_in_window(y, start, end):
        """
        Mean y[start:end] inclusive after bounds clipping.
        """
        y = _as_1d_float_array(y)
        n = len(y)
        start = _clip(int(start), 0, n - 1)
        end = _clip(int(end), 0, n - 1)
        if end < start:
            return np.nan
        seg = y[start:end + 1]
        if seg.size == 0:
            return np.nan
        return float(np.nanmean(seg))

    def corr_pearson(x, y):
        """
        Pearson correlation (manual) returning np.nan if insufficient.
        """
        x = np.asarray(x, dtype=float)
        y = np.asarray(y, dtype=float)
        m = np.isfinite(x) & np.isfinite(y)
        x = x[m]
        y = y[m]
        if len(x) < 3:
            return np.nan
        x0 = x - np.mean(x)
        y0 = y - np.mean(y)
        denom = np.sqrt(np.sum(x0**2) * np.sum(y0**2))
        if denom <= 1e-12:
            return np.nan
        return float(np.sum(x0 * y0) / denom)

    # ----------------------------
    # Main: build bout-pair table across all (trial, neuron)
    # ----------------------------
    pair_records = []

    # Expecting this table exists in your workspace
    # hddm_RawFullDatas_tgt_alldates with columns:
    #   dates, clusterID, bhv_id, FR_ievent, socialgaze_prob
    df_all = hddm_RawFullDatas_tgt_alldates.copy()

    # small sanity filter
    needed_cols = ['dates', 'clusterID', 'bhv_id', 'FR_ievent', 'socialgaze_prob']
    missing = [c for c in needed_cols if c not in df_all.columns]
    if len(missing) > 0:
        raise ValueError(f"Missing required columns in hddm_RawFullDatas_tgt_alldates: {missing}")

    for (date, clusterID), gN in df_all.groupby(['dates', 'clusterID'], sort=False):

        for _, row in gN.iterrows():

            trial_id = int(row['bhv_id'])
            fr = _as_1d_float_array(row['FR_ievent'])
            gaze = _as_1d_float_array(row['socialgaze_prob'])

            # match lengths
            L = min(len(fr), len(gaze))
            if L <= (EXCLUDE_TAIL_BINS + 5):
                continue
            fr = fr[:L]
            gaze = gaze[:L]

            if SMOOTH_FR and len(fr) > 3:
                fr = gaussian_filter1d(fr, sigma=SMOOTH_FR_SIGMA)

            end_limit_idx = (L - 1) - EXCLUDE_TAIL_BINS
            if end_limit_idx <= 1:
                continue

            bouts = find_bouts(gaze, thresh=GAZE_THRESH, smooth=SMOOTH_GAZE, sigma=SMOOTH_GAZE_SIGMA)

            # keep eligible bouts that end before cutoff and meet duration
            eligible = []
            for b0, b1 in bouts:
                if b1 <= end_limit_idx and (b1 - b0 + 1) >= MIN_BOUT_BINS:
                    eligible.append((b0, b1))

            if len(eligible) < 2:
                continue

            # last two eligible bouts
            (b0_prev, b1_prev) = eligible[-2]
            (b0_last, b1_last) = eligible[-1]

            # Define snippets:
            # x = mean FR near end of previous bout
            x_start = b1_prev - (SNIP_WIN_BINS - 1)
            x_end = b1_prev
            x = mean_in_window(fr, x_start, x_end)

            # y = mean FR near start of last bout
            y_start = b0_last
            y_end = b0_last + (SNIP_WIN_BINS - 1)
            y = mean_in_window(fr, y_start, y_end)

            # Optional "gap" (for debugging / later): middle of the gap between bouts
            gap_start = b1_prev + 1
            gap_end = b0_last - 1
            if gap_end >= gap_start:
                # take a short window in the middle of the gap
                gap_mid = (gap_start + gap_end) // 2
                gap0 = gap_mid - (SNIP_WIN_BINS // 2)
                gap1 = gap0 + (SNIP_WIN_BINS - 1)
                gap = mean_in_window(fr, gap0, gap1)
            else:
                gap = np.nan

            pair_records.append({
                "date": str(date),
                "clusterID": int(clusterID),
                "bhv_id": trial_id,
                "trial_len_s": float(L * dt),
                "end_limit_s": float(end_limit_idx * dt),

                "prev_bout_start_s": float(b0_prev * dt),
                "prev_bout_end_s": float(b1_prev * dt),
                "last_bout_start_s": float(b0_last * dt),
                "last_bout_end_s": float(b1_last * dt),

                "prev_bout_dur_s": float((b1_prev - b0_prev + 1) * dt),
                "last_bout_dur_s": float((b1_last - b0_last + 1) * dt),
                "gap_dur_s": float(max(0, (b0_last - b1_prev - 1) * dt)),

                "FR_prev_end": float(x),
                "FR_last_start": float(y),
                "FR_gap_mid": float(gap),
            })

    pair_df = pd.DataFrame(pair_records)
    print("Bout-pair table:", pair_df.shape)
    print(pair_df.head())

    # ----------------------------
    # Per-neuron summary: signed and absolute correlations
    # ----------------------------
    neuron_rows = []
    for (date, clusterID), gg in pair_df.groupby(['date', 'clusterID'], sort=False):
        x = gg["FR_prev_end"].values
        y = gg["FR_last_start"].values
        r = corr_pearson(x, y)
        n = int(np.sum(np.isfinite(x) & np.isfinite(y)))

        if n >= MIN_PAIRS_PER_NEURON:
            neuron_rows.append({
                "date": str(date),
                "clusterID": int(clusterID),
                "n_pairs": n,
                "r_signed": float(r),
                "r_abs": float(np.abs(r)) if np.isfinite(r) else np.nan,
            })

    neuron_df = pd.DataFrame(neuron_rows)
    print("Per-neuron correlation table:", neuron_df.shape)
    print(neuron_df.head())

    # ----------------------------
    # Quick plots
    # ----------------------------
    # 1) Pooled scatter (all neurons/trials) — just a sanity check
    if len(pair_df) > 0:
        m = np.isfinite(pair_df["FR_prev_end"].values) & np.isfinite(pair_df["FR_last_start"].values)
        xx = pair_df.loc[m, "FR_prev_end"].values
        yy = pair_df.loc[m, "FR_last_start"].values
        r_pool = corr_pearson(xx, yy)

        plt.figure(figsize=(5, 5))
        plt.scatter(xx, yy, s=8, alpha=0.3)
        plt.xlabel(f"FR near end of 2nd-last gaze bout (mean {SNIP_WIN_S:.2f}s)")
        plt.ylabel(f"FR near start of last gaze bout (mean {SNIP_WIN_S:.2f}s)")
        plt.title(f"Pooled (all neurons/trials) r = {r_pool:.3f}")
        plt.axhline(0, linewidth=1)
        plt.axvline(0, linewidth=1)
        plt.tight_layout()
        plt.show()

    # 2) Distribution of signed r across neurons (bidirectional)
    if len(neuron_df) > 0:
        plt.figure(figsize=(6, 4))
        plt.hist(neuron_df["r_signed"].values, bins=40, range=(-1, 1))
        plt.axvline(0, linewidth=1)
        plt.xlabel("Per-neuron r (signed): FR_prev_end vs FR_last_start")
        plt.ylabel("# neurons")
        plt.title(f"Carry-over test (signed), n_neurons={len(neuron_df)}")
        plt.tight_layout()
        plt.show()

        # 3) Distribution of |r| across neurons (strength, direction-agnostic)
        plt.figure(figsize=(6, 4))
        plt.hist(neuron_df["r_abs"].values, bins=30, range=(0, 1))
        plt.axvline(0.2, linewidth=1)
        plt.xlabel("Per-neuron |r|")
        plt.ylabel("# neurons")
        plt.title(f"Carry-over strength (|r|), n_neurons={len(neuron_df)}")
        plt.tight_layout()
        plt.show()

    # Optional: Save tables
    # pair_df.to_csv("gaze_bout_carryover_pairs.csv", index=False)
    # neuron_df.to_csv("gaze_bout_carryover_per_neuron.csv", index=False)


In [ ]:
####
# Reviewer 2: PEAK (piecewise linear up+down, optimized tau) vs RAMP (OLS line)
# FAST option (Option 2): grid-search tau in O(n) using prefix sums (no nonlinear optimizer)
#
# Fit window per trial:
#   trial start -> (pull - EXCLUDE_TAIL_S)   where pull is end of FR_ievent vector
#
# Outputs:
#   trial_fit_df  : per (date, clusterID, bhv_id) with BICs, tau_peak, ramp_slope, old_slope_abs, etc.
#   neuron_sum_df : per neuron: frac_peak_win, median_dBIC (BIC_peak - BIC_ramp), etc.
#
# Plots:
#   (1) Trial-level winner counts
#   (2) Per-neuron median ΔBIC histogram  (ΔBIC = BIC_peak - BIC_ramp; <0 => peak wins)
#   (3) Per-neuron frac peak-win histogram
#   (4) tau_peak_norm vs old_slope_abs scatter + correlation (all trials)
#   (5) tau_peak_norm vs old_slope_abs scatter + correlation (peak-winning only)
####

if 0:

    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from scipy.ndimage import gaussian_filter1d

    # ----------------------------
    # USER SETTINGS
    # ----------------------------
    fps = 30.0
    dt = 1.0 / fps

    EXCLUDE_TAIL_S = 0.85
    EXCLUDE_TAIL_BINS = int(np.round(EXCLUDE_TAIL_S * fps))

    # Requested earlier
    MAX_TRIAL_S = 8.0
    MAX_TRIAL_BINS = int(np.floor(MAX_TRIAL_S * fps))

    # Optional: extra smoothing (your FR is already smoothed/z-scored; usually keep OFF)
    SMOOTH_FR = False
    SMOOTH_FR_SIGMA = 2

    # Optional: downsample for speed (2 means 30Hz -> 15Hz). Tau_norm still meaningful.
    DOWNSAMPLE = 1  # set to 2 to speed up a lot

    # Peak-model tau search constraints
    MIN_SEG_S = 0.20
    MIN_SEG_BINS = max(2, int(np.round(MIN_SEG_S * fps / DOWNSAMPLE)))  # each side at least this many points

    MIN_POINTS = 12  # after downsample, minimum total points in fit window

    # Scatter subsampling
    PLOT_MAX_POINTS = 12000

    # ----------------------------
    # Helpers
    # ----------------------------
    def _as_1d_float_array(x):
        return np.asarray(x, dtype=float).ravel()

    def bic_from_sse(sse, n, k):
        # BIC up to constant; smaller is better
        if (not np.isfinite(sse)) or (sse <= 0) or (n <= (k + 1)):
            return np.nan
        return float(n * np.log(sse / n) + k * np.log(n))

    def pearsonr_simple(x, y):
        x = np.asarray(x, float)
        y = np.asarray(y, float)
        m = np.isfinite(x) & np.isfinite(y)
        x = x[m]; y = y[m]
        if len(x) < 3:
            return np.nan
        x0 = x - np.mean(x)
        y0 = y - np.mean(y)
        denom = np.sqrt(np.sum(x0**2) * np.sum(y0**2))
        if denom <= 1e-12:
            return np.nan
        return float(np.sum(x0 * y0) / denom)

    def _subsample(df, maxn=PLOT_MAX_POINTS, seed=0):
        if len(df) > maxn:
            return df.sample(maxn, random_state=seed)
        return df

    # ---- OLS ramp (y = a + b t)
    def fit_ramp_ols(y, dt):
        y = _as_1d_float_array(y)
        n = len(y)
        if n < 2:
            return np.nan, np.nan, np.nan
        t = np.arange(n, dtype=float) * dt
        A = np.column_stack([np.ones(n), t])
        coef, _, _, _ = np.linalg.lstsq(A, y, rcond=None)
        a, b = float(coef[0]), float(coef[1])
        yhat = a + b * t
        sse = float(np.nansum((y - yhat) ** 2))
        return sse, a, b

    # ---- Prefix sums for fast segment linear regression SSE with intercept
    def _prefix_sums(t, y):
        # returns prefix sums arrays with leading 0
        # so sum over [lo, hi) = pre[hi] - pre[lo]
        pre_t  = np.concatenate([[0.0], np.cumsum(t)])
        pre_t2 = np.concatenate([[0.0], np.cumsum(t*t)])
        pre_y  = np.concatenate([[0.0], np.cumsum(y)])
        pre_y2 = np.concatenate([[0.0], np.cumsum(y*y)])
        pre_ty = np.concatenate([[0.0], np.cumsum(t*y)])
        return pre_t, pre_t2, pre_y, pre_y2, pre_ty

    def _seg_sums(pre_t, pre_t2, pre_y, pre_y2, pre_ty, lo, hi):
        # hi is exclusive
        n = hi - lo
        st  = pre_t[hi]  - pre_t[lo]
        st2 = pre_t2[hi] - pre_t2[lo]
        sy  = pre_y[hi]  - pre_y[lo]
        sy2 = pre_y2[hi] - pre_y2[lo]
        sty = pre_ty[hi] - pre_ty[lo]
        return n, st, st2, sy, sy2, sty

    def _ols_sse_from_sums(n, st, st2, sy, sy2, sty):
        """
        Fit y = a + b t on this segment using sums.
        SSE = y'y - beta^T X'y
        where X'y = [sum_y, sum_ty]
              X'X = [[n, sum_t],
                     [sum_t, sum_t2]]
        """
        if n < 2:
            return np.nan, np.nan, np.nan

        det = n * st2 - st * st
        if det <= 1e-12:
            # t has no variance -> only intercept possible
            a = sy / n
            b = 0.0
            sse = sy2 - (sy * sy) / n
            return float(sse), float(a), float(b)

        a = (sy * st2 - sty * st) / det
        b = (sty * n  - sy  * st) / det

        # SSE = y'y - [a b]·[sum_y sum_ty]
        sse = sy2 - (a * sy + b * sty)
        return float(sse), float(a), float(b)

    # ---- Peak model: two independent lines split at tau (no continuity constraint)
    # y = a1 + b1 t   for i <= tau
    # y = a2 + b2 t   for i >  tau
    # tau optimized by grid search; O(n) using prefix sums
    def fit_peak_piecewise_linear(y, dt, min_seg_bins=3):
        y = _as_1d_float_array(y)
        n = len(y)
        if n < (2 * min_seg_bins + 2):
            return None

        t = np.arange(n, dtype=float) * dt

        pre_t, pre_t2, pre_y, pre_y2, pre_ty = _prefix_sums(t, y)

        best_sse = np.inf
        best_tau = None
        best_params = None  # (a1,b1,a2,b2)

        # tau is last index of left segment (<= tau), so left = [0, tau+1), right = [tau+1, n)
        for tau in range(min_seg_bins - 1, n - min_seg_bins - 1):
            lo1, hi1 = 0, tau + 1
            lo2, hi2 = tau + 1, n

            n1, st1, st21, sy1, sy21, sty1 = _seg_sums(pre_t, pre_t2, pre_y, pre_y2, pre_ty, lo1, hi1)
            n2, st2, st22, sy2, sy22, sty2 = _seg_sums(pre_t, pre_t2, pre_y, pre_y2, pre_ty, lo2, hi2)

            sse1, a1, b1 = _ols_sse_from_sums(n1, st1, st21, sy1, sy21, sty1)
            sse2, a2, b2 = _ols_sse_from_sums(n2, st2, st22, sy2, sy22, sty2)

            if not np.isfinite(sse1) or not np.isfinite(sse2):
                continue

            sse = sse1 + sse2
            if sse < best_sse:
                best_sse = float(sse)
                best_tau = int(tau)
                best_params = (float(a1), float(b1), float(a2), float(b2))

        if best_tau is None:
            return None

        return best_tau, best_params[0], best_params[1], best_params[2], best_params[3], best_sse

    # ----------------------------
    # Main loop
    # ----------------------------
    df_all = hddm_RawFullDatas_tgt_alldates  # assume it exists in your notebook already

    needed_cols = ['dates', 'clusterID', 'bhv_id', 'FR_ievent']
    missing = [c for c in needed_cols if c not in df_all.columns]
    if missing:
        raise ValueError(f"Missing required columns in hddm_RawFullDatas_tgt_alldates: {missing}")

    trial_records = []

    # itertuples is much faster than iterrows
    for row in df_all.itertuples(index=False):
        date = getattr(row, 'dates')
        clusterID = getattr(row, 'clusterID')
        bhv_id = getattr(row, 'bhv_id')
        fr_full = _as_1d_float_array(getattr(row, 'FR_ievent'))

        Lfull = len(fr_full)

        # long trial gate
        if Lfull > MAX_TRIAL_BINS:
            continue

        end_idx = (Lfull - 1) - EXCLUDE_TAIL_BINS
        if end_idx < 0:
            continue

        y = fr_full[:end_idx + 1]

        # downsample for speed
        if DOWNSAMPLE > 1:
            y = y[::DOWNSAMPLE]

        if SMOOTH_FR and len(y) > 3:
            y = gaussian_filter1d(y, sigma=SMOOTH_FR_SIGMA)

        # QC
        if len(y) < MIN_POINTS:
            continue
        if (not np.all(np.isfinite(y))) or (np.var(y) < 1e-6):
            continue

        n = len(y)
        dt_eff = dt * DOWNSAMPLE
        fit_window_s = n * dt_eff

        # ---- Ramp OLS
        sse_r, a_r, b_r = fit_ramp_ols(y, dt_eff)
        bic_r = bic_from_sse(sse_r, n=n, k=2)

        # Old slope metric for your sanity-check plot:
        # (this matches your “abs(slope)” convention)
        old_slope_abs = abs(b_r) if np.isfinite(b_r) else np.nan

        # ---- Peak piecewise linear
        peak_fit = fit_peak_piecewise_linear(y, dt_eff, min_seg_bins=MIN_SEG_BINS)
        if peak_fit is None:
            continue
        tau_p, a1, b1, a2, b2, sse_p = peak_fit
        # count tau as a parameter as well (k=5 total) for BIC comparison
        bic_p = bic_from_sse(sse_p, n=n, k=5)

        if (not np.isfinite(bic_r)) or (not np.isfinite(bic_p)):
            continue

        dBIC_peak_minus_ramp = float(bic_p - bic_r)   # <0 => peak wins
        win_model = "peak" if bic_p < bic_r else "ramp"

        tau_peak_norm = float(tau_p / (n - 1)) if n > 1 else np.nan
        # peak time in seconds within fit window:
        tau_peak_s = float(tau_p * dt_eff)

        trial_records.append({
            "date": str(date),
            "clusterID": int(clusterID),
            "bhv_id": int(bhv_id),

            "trial_len_bins": int(Lfull),
            "fit_len_bins": int(n),
            "fit_window_s": float(fit_window_s),
            "dt_eff": float(dt_eff),

            "bic_ramp": float(bic_r),
            "bic_peak": float(bic_p),
            "dBIC_peak_minus_ramp": float(dBIC_peak_minus_ramp),
            "win_model": win_model,

            "ramp_a": float(a_r) if np.isfinite(a_r) else np.nan,
            "ramp_slope_per_s": float(b_r) if np.isfinite(b_r) else np.nan,
            "old_slope_abs": float(old_slope_abs),

            "tau_peak_idx": int(tau_p),
            "tau_peak_s": float(tau_peak_s),
            "tau_peak_norm": float(tau_peak_norm),

            # piecewise params (useful for QC)
            "b1_rise_slope": float(b1),
            "b2_fall_slope": float(b2),
        })

    trial_fit_df = pd.DataFrame(trial_records)
    print("trial_fit_df:", trial_fit_df.shape)
    if len(trial_fit_df) == 0:
        raise RuntimeError("No trials passed QC/gates. Consider loosening MAX_TRIAL_S, MIN_SEG_S, or MIN_POINTS.")

    # ----------------------------
    # Per-neuron summary
    # ----------------------------
    neuron_rows = []
    for (date, clusterID), gg in trial_fit_df.groupby(["date", "clusterID"], sort=False):
        d = gg["dBIC_peak_minus_ramp"].values
        neuron_rows.append({
            "date": str(date),
            "clusterID": int(clusterID),
            "n_trials_used": int(len(gg)),
            "frac_peak_win": float(np.mean(d < 0)),
            "median_dBIC_peak_minus_ramp": float(np.nanmedian(d)),
            "mean_dBIC_peak_minus_ramp": float(np.nanmean(d)),
        })
    neuron_sum_df = pd.DataFrame(neuron_rows)
    print("neuron_sum_df:", neuron_sum_df.shape)

    # ----------------------------
    # Plots
    # ----------------------------

    # (1) winner count (trials)
    win_counts = trial_fit_df["win_model"].value_counts().reindex(["peak", "ramp"]).fillna(0)
    plt.figure(figsize=(4.8, 4))
    plt.bar(win_counts.index, win_counts.values)
    plt.ylabel("# trials")
    plt.title("Single-trial best motif (BIC): PEAK vs RAMP")
    plt.tight_layout()
    plt.show()

    # (2) per-neuron median ΔBIC histogram
    plt.figure(figsize=(6.2, 4))
    plt.hist(neuron_sum_df["median_dBIC_peak_minus_ramp"].values, bins=40)
    plt.axvline(0, linewidth=1)
    plt.xlabel("Per-neuron median ΔBIC (BIC_peak - BIC_ramp); <0 => PEAK wins")
    plt.ylabel("# neurons")
    plt.title("PEAK vs RAMP (full trial fit window)")
    plt.tight_layout()
    plt.show()

    # (3) per-neuron frac peak-win histogram
    plt.figure(figsize=(6.2, 4))
    plt.hist(neuron_sum_df["frac_peak_win"].values, bins=30, range=(0, 1))
    plt.axvline(0.5, linewidth=1)
    plt.xlabel("Fraction of trials where PEAK wins (ΔBIC<0)")
    plt.ylabel("# neurons")
    plt.title("PEAK-win fraction across neurons")
    plt.tight_layout()
    plt.show()

    # (4) tau_peak_norm vs old_slope_abs (all trials)
    dd_all = _subsample(trial_fit_df, seed=0)
    r_all = pearsonr_simple(dd_all["tau_peak_norm"].values, dd_all["old_slope_abs"].values)

    plt.figure(figsize=(5.4, 4.3))
    plt.scatter(dd_all["tau_peak_norm"].values, dd_all["old_slope_abs"].values, s=8, alpha=0.25)
    plt.axhline(0, linewidth=1)
    plt.xlim(-0.02, 1.02)
    plt.xlabel("tau_peak (normalized within fit window)")
    plt.ylabel("old slope abs (|OLS slope| on same window)")
    plt.title(f"All trials: tau_peak vs |old slope| (r={r_all:.3f}, n={len(dd_all)})")
    plt.tight_layout()
    plt.show()

    # (5) tau_peak_norm vs old_slope_abs (peak-winning trials)
    peak_df = trial_fit_df[trial_fit_df["win_model"] == "peak"].copy()
    if len(peak_df) > 0:
        dd_peak = _subsample(peak_df, seed=1)
        r_peak = pearsonr_simple(dd_peak["tau_peak_norm"].values, dd_peak["old_slope_abs"].values)

        plt.figure(figsize=(5.4, 4.3))
        plt.scatter(dd_peak["tau_peak_norm"].values, dd_peak["old_slope_abs"].values, s=8, alpha=0.25)
        plt.axhline(0, linewidth=1)
        plt.xlim(-0.02, 1.02)
        plt.xlabel("tau_peak (normalized within fit window)")
        plt.ylabel("old slope abs (|OLS slope| on same window)")
        plt.title(f"PEAK-win trials: tau_peak vs |old slope| (r={r_peak:.3f}, n={len(dd_peak)})")
        plt.tight_layout()
        plt.show()

    # Optional saves
    # trial_fit_df.to_csv("peak_vs_ramp_fulltrial_trialfits.csv", index=False)
    # neuron_sum_df.to_csv("peak_vs_ramp_fulltrial_neuron_summary.csv", index=False)


In [ ]:
####
# Peak-at-tau behavioral state analysis (FULL TRIAL window, no gaze filtering)
#
# Goal:
# 1) For each neuron x trial:
#    - define tau as argmax (if overall slope >=0) OR argmin (if slope <0)
#    - extract distance to pull, and covariates at tau:
#        socialgaze_prob, other_pc1, other_mass_move_speed
# 2) Summaries:
#    A) pooled (all neuron/trials): hist + binned means vs dist_to_pull
#    B) per-neuron means: plot across neurons
# 3) Shuffle controls (CHOOSE ONE):
#    - SHUFFLE_METHOD="trial_id_full":
#        real "trial-id shuffle": pick a donor trial (within same session+neuron),
#        and take donor's (tau, dist_to_pull, cov@tau). This breaks alignment to
#        each trial's own pull timing.
#    - SHUFFLE_METHOD="tau_perm_within_trial":
#        permute tau indices across trials within neuron/session, but read covariates
#        from the target trial at the permuted tau (dist_to_pull changes accordingly).
#    - SHUFFLE_METHOD="tau_random_within_trial":
#        choose a random tau within each trial and read covariates there.
# 4) Method 1: split-half reliability per neuron (mean@tau stability across trials)
# 5) NEW PANELS:
#    - Real vs shuffle for distance_to_pull (histograms)
#    - Scatter: distance_to_pull consistency vs (real-shuffle) effect size
####

if 1:

    import os, pickle
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from scipy.ndimage import gaussian_filter1d
    from scipy.stats import zscore

    # ============================================================
    # USER SETTINGS
    # ============================================================
    fps = 30.0
    dt = 1.0 / fps

    EXCLUDE_TAIL_S = 0.05  # trial start -> pull-0.85s
    EXCLUDE_TAIL_BINS = int(np.round(EXCLUDE_TAIL_S * fps))

    MAX_TRIAL_S = 8.0
    MAX_TRIAL_BINS = int(np.floor(MAX_TRIAL_S * fps))

    MIN_FIT_BINS = 15
    MIN_VAR = 1e-6

    DO_SMOOTH_FR = False
    SMOOTH_FR_SIGMA = 2

    USE_PEAK_PROM_FILTER = True
    PEAK_PROM_ABS_MIN = 0.25

    # ----------------------------
    # SHUFFLE CONTROLS
    # ----------------------------
    DO_SHUFFLE = True
    # Choose ONE:
    #   "trial_id_full" | "tau_perm_within_trial" | "tau_random_within_trial"
    SHUFFLE_METHOD = "tau_random_within_trial"
    N_SHUFFLE_REPEATS = 1
    RNG_SEED = 0

    # Random tau bounds (used only for tau_random_within_trial)
    RANDOM_TAU_MIN_FRAC = 0.00   # set 0.05 to avoid first 5%
    RANDOM_TAU_MAX_FRAC = 1.00   # set 0.95 to avoid last 5%

    # ----------------------------
    # Split-half reliability
    # ----------------------------
    DO_SPLIT_HALF = True
    N_SPLIT_REPEATS = 200
    MIN_TRIALS_FOR_SPLIT = 8

    # ----------------------------
    # plotting
    # ----------------------------
    PLOT_MAX_POINTS = 15000
    N_BINS_DIST = 18
    SHOW_BINNED_PLOTS = True

    # ============================================================
    # Helpers
    # ============================================================
    def _as_1d_float_array(x):
        return np.asarray(x, dtype=float).ravel()

    def pearsonr_simple(x, y):
        x = np.asarray(x, float)
        y = np.asarray(y, float)
        m = np.isfinite(x) & np.isfinite(y)
        x = x[m]; y = y[m]
        if len(x) < 3:
            return np.nan
        x0 = x - np.mean(x)
        y0 = y - np.mean(y)
        denom = np.sqrt(np.sum(x0**2) * np.sum(y0**2))
        if denom <= 1e-12:
            return np.nan
        return float(np.sum(x0 * y0) / denom)

    def _subsample(df, maxn=PLOT_MAX_POINTS, seed=0):
        if len(df) > maxn:
            return df.sample(maxn, random_state=seed)
        return df

    def ols_slope(y, dt):
        y = _as_1d_float_array(y)
        n = len(y)
        if n < 2:
            return np.nan
        t = np.arange(n) * dt
        A = np.column_stack([np.ones(n), t])
        coef, _, _, _ = np.linalg.lstsq(A, y, rcond=None)
        return float(coef[1])

    def peak_prominence_1d(y, peak_idx):
        if peak_idx is None or (not np.isfinite(peak_idx)):
            return np.nan
        peak_idx = int(peak_idx)
        if peak_idx < 0 or peak_idx >= len(y):
            return np.nan
        return float(np.abs(y[peak_idx] - np.nanmedian(y)))

    def binned_mean(x, y, n_bins=12, x_min=None, x_max=None):
        x = np.asarray(x, float)
        y = np.asarray(y, float)
        m = np.isfinite(x) & np.isfinite(y)
        x = x[m]; y = y[m]
        if len(x) == 0:
            return None
        if x_min is None: x_min = np.nanmin(x)
        if x_max is None: x_max = np.nanmax(x)
        edges = np.linspace(x_min, x_max, n_bins + 1)
        centers = 0.5 * (edges[:-1] + edges[1:])
        means = np.full(n_bins, np.nan)
        counts = np.zeros(n_bins, int)
        for i in range(n_bins):
            mm = (x >= edges[i]) & (x < edges[i+1]) if i < n_bins-1 else (x >= edges[i]) & (x <= edges[i+1])
            if np.any(mm):
                means[i] = np.nanmean(y[mm])
                counts[i] = int(np.sum(mm))
        return centers, means, counts

    def split_half_mean_stability(x, n_repeats=200, rng=None, min_trials=8, eps=1e-9):
        x = np.asarray(x, float)
        x = x[np.isfinite(x)]
        n = len(x)
        if n < min_trials:
            return np.nan, np.nan
        sd = float(np.nanstd(x))
        half = n // 2
        if half < 2:
            return np.nan, np.nan
        absdiffs = []
        for _ in range(n_repeats):
            perm = rng.permutation(n)
            a = x[perm[:half]]
            b = x[perm[half:2*half]]
            absdiffs.append(np.abs(np.mean(a) - np.mean(b)))
        mean_absdiff = float(np.mean(absdiffs))
        mean_rel = float(1.0 - mean_absdiff / (sd + eps)) if np.isfinite(sd) else np.nan
        return mean_absdiff, mean_rel

    def mad(x):
        x = np.asarray(x, float)
        x = x[np.isfinite(x)]
        if len(x) == 0:
            return np.nan
        med = np.median(x)
        return float(np.median(np.abs(x - med)))

    def draw_random_tau_idx(n, rng, min_frac=0.0, max_frac=1.0):
        if n <= 1:
            return 0
        lo = int(np.ceil(min_frac * (n - 1)))
        hi = int(np.floor(max_frac * (n - 1)))
        lo = max(0, min(lo, n - 1))
        hi = max(0, min(hi, n - 1))
        if hi < lo:
            lo, hi = 0, n - 1
        return int(rng.integers(lo, hi + 1))  # inclusive hi

    # ============================================================
    # (A) Load hddm_data_tgt_alldates as in your pipeline (optional)
    # ============================================================
    if 'hddm_data_tgt_alldates' not in globals():

        dobhvPCA = 1
        doparnterleverspeed = 0

        if not dobhvPCA:
            doOnsetAfterMin = 1
            if not doOnsetAfterMin:
                doOnsetAfterMin_suffix = ''
            else:
                if not doparnterleverspeed:
                    doOnsetAfterMin_suffix = 'PullOnsetAfterMin_'
                else:
                    doOnsetAfterMin_suffix = 'PullOnsetAfterMin_PartnerLeverSpeed_'
        else:
            doOnsetAfterMin = 1
            doBasedonSpeed = 1
            if not doOnsetAfterMin:
                doOnsetAfterMin_suffix = 'bhvPCA_'
            else:
                if not doBasedonSpeed:
                    doOnsetAfterMin_suffix = 'bhvPCA_PullOnsetAfterMin_'
                else:
                    doOnsetAfterMin_suffix = 'bhvPCA_PullOnsetAfterMinBasedonSpeed_'

        doHMMmethod = 0
        if doHMMmethod:
            doOnsetAfterMin_suffix = 'HMMmethods_'

        current_dir = data_saved_folder + 'data_saved_singlecam_wholebody_neural_and_hddm' + savefile_sufix + '/' + cameraID + '/' + animal_tgt
        add_date_dir = os.path.join(current_dir + '/hddm_model_fitted_combinedsession_withNeurons')

        if cond_toplot_type == 'allMC':
            if not (doOnsetAfterMin_suffix == 'bhvPCA_PullOnsetAfterMinBasedonSpeed_') | \
                   (doOnsetAfterMin_suffix == 'PullOnsetAfterMin_PartnerLeverSpeed_'):
                with open(add_date_dir + '/hddm_datas_flexibleTW_newRTdefinition_' + doOnsetAfterMin_suffix + 'withFRs_all_dates_' +
                          animal1_fixedorders[0] + animal2_fixedorders[0] + '.pkl', 'rb') as f:
                    hddm_data_tgt_alldates = pickle.load(f)
            else:
                with open(add_date_dir + '/hddm_datas_flexibleTW_newRTdefinition_' + 'PullOnsetAfterMin_' + 'withFRs_all_dates_' +
                          animal1_fixedorders[0] + animal2_fixedorders[0] + '.pkl', 'rb') as f:
                    hddm_data_tgt_alldates = pickle.load(f)
        else:
            if not (doOnsetAfterMin_suffix == 'bhvPCA_PullOnsetAfterMinBasedonSpeed_') | \
                   (doOnsetAfterMin_suffix == 'PullOnsetAfterMin_PartnerLeverSpeed_'):
                with open(add_date_dir + '/hddm_datas_flexibleTW_newRTdefinition_' + doOnsetAfterMin_suffix + 'withFRs_all_dates_' +
                          animal1_fixedorders[0] + animal2_fixedorders[0] + '_' + cond_toplot_type + '.pkl', 'rb') as f:
                    hddm_data_tgt_alldates = pickle.load(f)
            else:
                with open(add_date_dir + '/hddm_datas_flexibleTW_newRTdefinition_' + 'PullOnsetAfterMin_' + 'withFRs_all_dates_' +
                          animal1_fixedorders[0] + animal2_fixedorders[0] + '_' + cond_toplot_type + '.pkl', 'rb') as f:
                    hddm_data_tgt_alldates = pickle.load(f)

        ind_goodcond = np.isin(hddm_data_tgt_alldates['condition'], conditions_to_ana)
        hddm_data_tgt_alldates = hddm_data_tgt_alldates[ind_goodcond]

        if not dobhvPCA:
            if not doparnterleverspeed:
                rename_dict = {
                    'act_animal': 'subj_idx',
                    'dates': 'date',
                    'pull_rt': 'rt',
                    'previous_pull_outcome': 'prev_trial_outcome',
                    'socialgaze_auc': 'self_gaze_auc',
                    'mass_move_speed_mean': 'self_mean_speed',
                    'mass_move_speed_std': 'self_speed_std',
                    'other_mass_move_speed_mean': 'partner_mean_speed',
                    'other_mass_move_speed_std': 'partner_speed_std',
                    'num_preceding_failpull': 'failed_pulls_before_reward',
                    'time_from_last_reward': 'time_since_last_reward',
                }
            else:
                rename_dict = {
                    'act_animal': 'subj_idx',
                    'dates': 'date',
                    'pull_rt': 'rt',
                    'previous_pull_outcome': 'prev_trial_outcome',
                    'socialgaze_auc': 'self_gaze_auc',
                    'mass_move_speed_mean': 'self_mean_speed',
                    'mass_move_speed_std': 'self_speed_std',
                    'other_lever_speed_mean': 'partner_mean_speed',
                    'other_lever_speed_std': 'partner_speed_std',
                    'num_preceding_failpull': 'failed_pulls_before_reward',
                    'time_from_last_reward': 'time_since_last_reward',
                }
        else:
            rename_dict = {
                'act_animal': 'subj_idx',
                'dates': 'date',
                'pull_rt': 'rt',
                'previous_pull_outcome': 'prev_trial_outcome',
                'socialgaze_auc': 'self_gaze_auc',
                'mass_move_speed_mean': 'self_mean_speed',
                'mass_move_speed_std': 'self_speed_std',
                'other_PC1_mean': 'partner_mean_speed',
                'other_PC1_std': 'partner_speed_std',
                'num_preceding_failpull': 'failed_pulls_before_reward',
                'time_from_last_reward': 'time_since_last_reward',
            }

        hddm_data_tgt_alldates = hddm_data_tgt_alldates.rename(columns=rename_dict)
        hddm_data_tgt_alldates['response'] = 1
        hddm_data_tgt_alldates = hddm_data_tgt_alldates.dropna()

        predictor_columns = [
            'self_gaze_auc',
            'partner_mean_speed',
            'self_mean_speed',
            'partner_speed_std',
            'self_speed_std',
            'failed_pulls_before_reward',
            'time_since_last_reward'
        ]
        for col in predictor_columns:
            if col in hddm_data_tgt_alldates.columns:
                hddm_data_tgt_alldates[f"{col}_origin"] = hddm_data_tgt_alldates[col]
                hddm_data_tgt_alldates[col] = zscore(hddm_data_tgt_alldates[col])

    # ============================================================
    # (B) Validate required columns in raw traces dataframe
    # ============================================================
    if 'hddm_RawFullDatas_tgt_alldates' not in globals():
        raise ValueError("hddm_RawFullDatas_tgt_alldates not found in globals(). Please load it before running this cell.")

    df_raw = hddm_RawFullDatas_tgt_alldates.copy()

    needed = ['dates', 'clusterID', 'bhv_id', 'FR_ievent',
              'socialgaze_prob', 'other_PC1', 'other_mass_move_speed']
    missing = [c for c in needed if c not in df_raw.columns]
    if missing:
        raise ValueError(f"Missing required columns in hddm_RawFullDatas_tgt_alldates: {missing}")

    rng = np.random.default_rng(RNG_SEED)

    # ============================================================
    # (C) Main loop: build per-trial peak-at-tau table + shuffle
    # ============================================================
    trial_rows = []
    shuffle_rows = []

    for session_toana, hddm_data_tgt_idate in hddm_data_tgt_alldates.groupby('date'):

        # behavior-table outlier removal (kept for consistency)
        lower_q = 0.05
        upper_q = 0.95
        if 'rt' in hddm_data_tgt_idate.columns:
            rt_lower, rt_upper = hddm_data_tgt_idate['rt'].quantile([lower_q, upper_q])
            hddm_data_tgt_idate = hddm_data_tgt_idate[(hddm_data_tgt_idate['rt'] > rt_lower) & (hddm_data_tgt_idate['rt'] < rt_upper)]
        if 'time_since_last_reward' in hddm_data_tgt_idate.columns:
            tslr_lower, tslr_upper = hddm_data_tgt_idate['time_since_last_reward'].quantile([lower_q, upper_q])
            hddm_data_tgt_idate = hddm_data_tgt_idate[(hddm_data_tgt_idate['time_since_last_reward'] > tslr_lower) &
                                                      (hddm_data_tgt_idate['time_since_last_reward'] < tslr_upper)]

        dfS = df_raw[df_raw['dates'] == session_toana].copy()

        if 'bhv_id' in hddm_data_tgt_idate.columns:
            good_ids = set(hddm_data_tgt_idate['bhv_id'].astype(int).tolist())
            if len(good_ids) > 0:
                dfS = dfS[dfS['bhv_id'].astype(int).isin(good_ids)].copy()

        for clusterID, gN in dfS.groupby('clusterID', sort=False):

            trial_list = []

            for _, row in gN.iterrows():

                bhv_id = int(row['bhv_id'])

                fr_full   = _as_1d_float_array(row['FR_ievent'])
                gaze_full = _as_1d_float_array(row['socialgaze_prob'])
                pc1_full  = _as_1d_float_array(row['other_PC1'])
                spd_full  = _as_1d_float_array(row['other_mass_move_speed'])

                L = min(len(fr_full), len(gaze_full), len(pc1_full), len(spd_full))
                if L <= (EXCLUDE_TAIL_BINS + MIN_FIT_BINS):
                    continue
                if L > MAX_TRIAL_BINS:
                    continue

                end_idx = (L - 1) - EXCLUDE_TAIL_BINS
                if end_idx < MIN_FIT_BINS:
                    continue

                y  = fr_full[:end_idx + 1]
                gz = gaze_full[:end_idx + 1]
                pc = pc1_full[:end_idx + 1]
                sp = spd_full[:end_idx + 1]

                if DO_SMOOTH_FR and len(y) > 3:
                    y = gaussian_filter1d(y, sigma=SMOOTH_FR_SIGMA)

                if (not np.all(np.isfinite(y))) or (np.var(y) < MIN_VAR):
                    continue

                n = len(y)
                fit_window_s = n * dt

                slope_signed = ols_slope(y, dt=dt)

                if np.isfinite(slope_signed) and slope_signed < 0:
                    tau_idx = int(np.nanargmin(y))
                    peak_dir = "down"
                else:
                    tau_idx = int(np.nanargmax(y))
                    peak_dir = "up"

                prom = peak_prominence_1d(y, tau_idx)
                if USE_PEAK_PROM_FILTER and (not np.isfinite(prom) or prom < PEAK_PROM_ABS_MIN):
                    continue

                fr_at_tau    = float(y[tau_idx])
                gaze_at_tau  = float(gz[tau_idx]) if np.isfinite(gz[tau_idx]) else np.nan
                pc1_at_tau   = float(pc[tau_idx]) if np.isfinite(pc[tau_idx]) else np.nan
                spd_at_tau   = float(sp[tau_idx]) if np.isfinite(sp[tau_idx]) else np.nan

                dist_to_pull_s = float((n - 1 - tau_idx) * dt)
                tau_norm       = float(tau_idx / (n - 1)) if n > 1 else np.nan
                old_slope_abs   = float(np.abs(slope_signed))

                trial_rows.append({
                    "date": str(session_toana),
                    "clusterID": int(clusterID),
                    "bhv_id": int(bhv_id),

                    "trial_len_bins": int(L),
                    "fit_len_bins": int(n),
                    "fit_window_s": float(fit_window_s),

                    "tau_idx": int(tau_idx),
                    "tau_s": float(tau_idx * dt),
                    "tau_norm": float(tau_norm),
                    "dist_to_pull_s": float(dist_to_pull_s),

                    "peak_dir": peak_dir,
                    "slope_signed": float(slope_signed),
                    "old_slope_abs": float(old_slope_abs),
                    "peak_prom_abs": float(prom),
                    "fr_at_tau": float(fr_at_tau),

                    "gaze_at_tau": gaze_at_tau,
                    "other_pc1_at_tau": pc1_at_tau,
                    "other_speed_at_tau": spd_at_tau,
                })

                if DO_SHUFFLE:
                    trial_list.append({
                        "bhv_id": int(bhv_id),
                        "n": int(n),
                        "tau_idx": int(tau_idx),
                        "dist_to_pull_s": float(dist_to_pull_s),

                        "gz": gz,
                        "pc": pc,
                        "sp": sp,

                        "gaze_at_tau": float(gaze_at_tau),
                        "pc1_at_tau": float(pc1_at_tau),
                        "spd_at_tau": float(spd_at_tau),
                    })

            # ---- Shuffle (within this session+neuron)
            if DO_SHUFFLE and len(trial_list) >= 2:
                T = len(trial_list)
                for rep in range(N_SHUFFLE_REPEATS):
                    perm = rng.permutation(T)

                    if SHUFFLE_METHOD == "trial_id_full":
                        # donor supplies tau + cov@tau + dist_to_pull
                        for i in range(T):
                            tgt = trial_list[i]
                            donor = trial_list[perm[i]]
                            shuffle_rows.append({
                                "date": str(session_toana),
                                "clusterID": int(clusterID),
                                "bhv_id": int(tgt["bhv_id"]),

                                "tau_idx_shuf": int(donor["tau_idx"]),
                                "tau_norm_shuf": float(donor["tau_idx"] / (donor["n"] - 1)) if donor["n"] > 1 else np.nan,
                                "dist_to_pull_s_shuf": float(donor["dist_to_pull_s"]),

                                "gaze_at_tau_shuf": float(donor["gaze_at_tau"]),
                                "other_pc1_at_tau_shuf": float(donor["pc1_at_tau"]),
                                "other_speed_at_tau_shuf": float(donor["spd_at_tau"]),
                            })

                    elif SHUFFLE_METHOD == "tau_perm_within_trial":
                        # permute tau across trials, but sample cov from target trial at permuted tau
                        for i in range(T):
                            tgt = trial_list[i]
                            donor = trial_list[perm[i]]

                            tau_i = int(donor["tau_idx"])
                            n_t = int(tgt["n"])
                            tau_use = min(tau_i, n_t - 1)

                            shuffle_rows.append({
                                "date": str(session_toana),
                                "clusterID": int(clusterID),
                                "bhv_id": int(tgt["bhv_id"]),

                                "tau_idx_shuf": int(tau_i),
                                "tau_norm_shuf": float(tau_i / (n_t - 1)) if n_t > 1 else np.nan,
                                "dist_to_pull_s_shuf": float((n_t - 1 - tau_i) * dt),

                                "gaze_at_tau_shuf": float(tgt["gz"][tau_use]) if np.isfinite(tgt["gz"][tau_use]) else np.nan,
                                "other_pc1_at_tau_shuf": float(tgt["pc"][tau_use]) if np.isfinite(tgt["pc"][tau_use]) else np.nan,
                                "other_speed_at_tau_shuf": float(tgt["sp"][tau_use]) if np.isfinite(tgt["sp"][tau_use]) else np.nan,
                            })

                    elif SHUFFLE_METHOD == "tau_random_within_trial":
                        # random tau within each trial
                        for i in range(T):
                            tgt = trial_list[i]
                            n_t = int(tgt["n"])
                            tau_i = draw_random_tau_idx(
                                n_t, rng,
                                min_frac=RANDOM_TAU_MIN_FRAC,
                                max_frac=RANDOM_TAU_MAX_FRAC
                            )
                            tau_use = min(tau_i, n_t - 1)

                            shuffle_rows.append({
                                "date": str(session_toana),
                                "clusterID": int(clusterID),
                                "bhv_id": int(tgt["bhv_id"]),

                                "tau_idx_shuf": int(tau_i),
                                "tau_norm_shuf": float(tau_i / (n_t - 1)) if n_t > 1 else np.nan,
                                "dist_to_pull_s_shuf": float((n_t - 1 - tau_i) * dt),

                                "gaze_at_tau_shuf": float(tgt["gz"][tau_use]) if np.isfinite(tgt["gz"][tau_use]) else np.nan,
                                "other_pc1_at_tau_shuf": float(tgt["pc"][tau_use]) if np.isfinite(tgt["pc"][tau_use]) else np.nan,
                                "other_speed_at_tau_shuf": float(tgt["sp"][tau_use]) if np.isfinite(tgt["sp"][tau_use]) else np.nan,
                            })

                    else:
                        raise ValueError(f"Unknown SHUFFLE_METHOD: {SHUFFLE_METHOD}")

    trial_peak_df = pd.DataFrame(trial_rows)
    shuffle_df = pd.DataFrame(shuffle_rows) if (DO_SHUFFLE and len(shuffle_rows) > 0) else None

    print("trial_peak_df:", trial_peak_df.shape)
    if DO_SHUFFLE:
        print("shuffle_df:", None if shuffle_df is None else shuffle_df.shape)

    # ============================================================
    # (D) Neuron-level summaries + split-half reliability
    # ============================================================
    neuron_rows = []
    for (date, clusterID), g in trial_peak_df.groupby(["date", "clusterID"], sort=False):

        out = {
            "date": str(date),
            "clusterID": int(clusterID),
            "n_trials": int(len(g)),

            "mean_gaze_at_tau": float(np.nanmean(g["gaze_at_tau"].values)),
            "median_gaze_at_tau": float(np.nanmedian(g["gaze_at_tau"].values)),

            "mean_other_pc1_at_tau": float(np.nanmean(g["other_pc1_at_tau"].values)),
            "median_other_pc1_at_tau": float(np.nanmedian(g["other_pc1_at_tau"].values)),

            "mean_other_speed_at_tau": float(np.nanmean(g["other_speed_at_tau"].values)),
            "median_other_speed_at_tau": float(np.nanmedian(g["other_speed_at_tau"].values)),

            "frac_peak_up": float(np.mean(g["peak_dir"].values == "up")),
            "mean_dist_to_pull_s": float(np.nanmean(g["dist_to_pull_s"].values)),
            "mad_dist_to_pull_s": mad(g["dist_to_pull_s"].values),
        }

        # split-half reliability on mean@tau
        if DO_SPLIT_HALF and int(len(g)) >= MIN_TRIALS_FOR_SPLIT:
            for short, col in [
                ("gaze", "gaze_at_tau"),
                ("pc1", "other_pc1_at_tau"),
                ("speed", "other_speed_at_tau"),
            ]:
                mean_absdiff, mean_rel = split_half_mean_stability(
                    g[col].values,
                    n_repeats=N_SPLIT_REPEATS,
                    rng=rng,
                    min_trials=MIN_TRIALS_FOR_SPLIT
                )
                out[f"split_half_mean_absdiff_{short}"] = mean_absdiff
                out[f"split_half_mean_rel_{short}"] = mean_rel
        else:
            for short in ["gaze", "pc1", "speed"]:
                out[f"split_half_mean_absdiff_{short}"] = np.nan
                out[f"split_half_mean_rel_{short}"] = np.nan

        neuron_rows.append(out)

    neuron_peak_df = pd.DataFrame(neuron_rows)
    print("neuron_peak_df:", neuron_peak_df.shape)

    # Shuffle neuron-level means (null)
    neuron_null_df = None
    if DO_SHUFFLE and shuffle_df is not None and len(shuffle_df) > 0:
        null_rows = []
        for (date, clusterID), g in shuffle_df.groupby(["date", "clusterID"], sort=False):
            null_rows.append({
                "date": str(date),
                "clusterID": int(clusterID),
                "n_trials_shuf": int(len(g)),

                "mean_gaze_at_tau_shuf": float(np.nanmean(g["gaze_at_tau_shuf"].values)),
                "mean_other_pc1_at_tau_shuf": float(np.nanmean(g["other_pc1_at_tau_shuf"].values)),
                "mean_other_speed_at_tau_shuf": float(np.nanmean(g["other_speed_at_tau_shuf"].values)),

                "mean_dist_to_pull_s_shuf": float(np.nanmean(g["dist_to_pull_s_shuf"].values)),
                "mad_dist_to_pull_s_shuf": mad(g["dist_to_pull_s_shuf"].values),
            })
        neuron_null_df = pd.DataFrame(null_rows)

    
    # ===============================
    # FIGURE SAVING SETTINGS
    # ===============================
    savefig = 1   # <-- set to 1 to save figures

    if savefig:
        current_dir = (
            data_saved_folder
            + 'fig_for_basic_neural_analysis_allsessions_basicEvents_PCA_Pullfocused_continuousBhv_partnerDistVaris_with_hddm_model'
            + savefile_sufix
            + '/'
            + animal_tgt
        )
        add_fig_dir = os.path.join(
            current_dir,
            cameraID + '/fig_for_hddm_model_fitted_allsession_withNeurons'
        )
        os.makedirs(add_fig_dir, exist_ok=True)
        
        
    # ============================================================
    # (E) Plots
    # ============================================================
    # 1) pooled distributions
    if len(trial_peak_df) > 0:
        for col, title in [
            ("gaze_at_tau", "socialgaze_prob at tau"),
            ("other_pc1_at_tau", "other_pc1 at tau"),
            ("other_speed_at_tau", "other_mass_move_speed at tau"),
            ("dist_to_pull_s", "distance to pull (s) at tau"),
        ]:
            plt.figure(figsize=(6, 4))
            plt.hist(trial_peak_df[col].values[np.isfinite(trial_peak_df[col].values)], bins=50)
            plt.xlabel(col)
            plt.ylabel("# trials")
            plt.title(f"Pooled (all neurons/trials): {title}")
            plt.tight_layout()
            plt.show()

    # 2) pooled binned means vs dist_to_pull_s
    if SHOW_BINNED_PLOTS and len(trial_peak_df) > 0:
        x = trial_peak_df["dist_to_pull_s"].values
        for ycol, ttl in [
            ("gaze_at_tau", "Mean socialgaze_prob at tau vs distance-to-pull"),
            ("other_pc1_at_tau", "Mean other_pc1 at tau vs distance-to-pull"),
            ("other_speed_at_tau", "Mean other_speed at tau vs distance-to-pull"),
        ]:
            y = trial_peak_df[ycol].values
            out = binned_mean(x, y, n_bins=N_BINS_DIST, x_min=0.0, x_max=np.nanmax(x))
            if out is None:
                continue
            centers, means, counts = out
            plt.figure(figsize=(6, 4))
            plt.plot(centers, means, marker='o')
            plt.xlabel("dist_to_pull_s (seconds)")
            plt.ylabel(f"mean {ycol}")
            plt.title(ttl)
            plt.tight_layout()
            plt.show()

    # 3) neuron-level distributions (means)
    if len(neuron_peak_df) > 0:
        for col, title in [
            ("mean_gaze_at_tau", "Per-neuron mean gaze_at_tau"),
            ("mean_other_pc1_at_tau", "Per-neuron mean other_pc1_at_tau"),
            ("mean_other_speed_at_tau", "Per-neuron mean other_speed_at_tau"),
            ("frac_peak_up", "Per-neuron fraction peak_dir == up"),
            ("mean_dist_to_pull_s", "Per-neuron mean dist_to_pull_s (real)"),
            ("mad_dist_to_pull_s", "Per-neuron MAD dist_to_pull_s (consistency)"),
        ]:
            plt.figure(figsize=(6, 4))
            v = neuron_peak_df[col].values
            v = v[np.isfinite(v)]
            if len(v) == 0:
                continue
            plt.hist(v, bins=40)
            plt.xlabel(col)
            plt.ylabel("# neurons")
            plt.title(title)
            plt.tight_layout()
            plt.show()

    # 4) merged real vs shuffle + deltas + NEW distance panels
    mrg = None
    if neuron_null_df is not None and len(neuron_null_df) > 0 and len(neuron_peak_df) > 0:
        mrg = neuron_peak_df.merge(neuron_null_df, on=["date", "clusterID"], how="inner")
        if len(mrg) > 0:

            mrg["delta_mean_gaze_at_tau"]  = mrg["mean_gaze_at_tau"]        - mrg["mean_gaze_at_tau_shuf"]
            mrg["delta_mean_other_pc1"]    = mrg["mean_other_pc1_at_tau"]   - mrg["mean_other_pc1_at_tau_shuf"]
            mrg["delta_mean_other_speed"]  = mrg["mean_other_speed_at_tau"] - mrg["mean_other_speed_at_tau_shuf"]

            # ---- NEW: distance_to_pull real vs shuffle (hist)
            fig = plt.figure(figsize=(6,4))
            a = mrg["mean_dist_to_pull_s"].values
            b = mrg["mean_dist_to_pull_s_shuf"].values
            a = a[np.isfinite(a)]; b = b[np.isfinite(b)]
            plt.hist(a, bins=40, alpha=0.5, label="real")
            plt.hist(b, bins=40, alpha=0.5, label=f"shuffle ({SHUFFLE_METHOD})")
            plt.xlabel("per-neuron mean dist_to_pull_s")
            plt.ylabel("# neurons")
            plt.title("Distance-to-pull: real vs shuffle")
            plt.legend(frameon=False)
            plt.tight_layout()
            #
            if savefig:
                fig.savefig(
                    os.path.join(
                        add_fig_dir,
                        f'dist_to_pull_mean_real_vs_shuffle_{cond_toplot_type}{doOnsetAfterMin_suffix}.pdf'
                    ),
                    format='pdf',
                    bbox_inches='tight'
                )
            #
            plt.show()

            # ---- NEW: distance consistency (MAD) real vs shuffle (hist)
            fig = plt.figure(figsize=(6,4))
            a = mrg["mad_dist_to_pull_s"].values
            b = mrg["mad_dist_to_pull_s_shuf"].values
            a = a[np.isfinite(a)]; b = b[np.isfinite(b)]
            plt.hist(a, bins=40, alpha=0.5, label="real")
            plt.hist(b, bins=40, alpha=0.5, label=f"shuffle ({SHUFFLE_METHOD})")
            plt.xlabel("per-neuron MAD dist_to_pull_s")
            plt.ylabel("# neurons")
            plt.title("Distance-to-pull consistency: real vs shuffle")
            plt.legend(frameon=False)
            plt.tight_layout()
            #
            if savefig:
                fig.savefig(
                    os.path.join(
                        add_fig_dir,
                        f'dist_to_pull_MAD_real_vs_shuffle_{cond_toplot_type}{doOnsetAfterMin_suffix}.pdf'
                    ),
                    format='pdf',
                    bbox_inches='tight'
                )
            #
            plt.show()
            
            

            # ---- helper: two-panel overlay + delta
            def _two_panel_real_vs_shuf_and_delta(real_col, shuf_col, delta_col, xlabel, title, bins=35):
                fig, ax = plt.subplots(1, 2, figsize=(11, 4))

                ax[0].hist(mrg[real_col].values, bins=bins, alpha=0.5, label="real")
                ax[0].hist(mrg[shuf_col].values, bins=bins, alpha=0.5, label=f"shuffle ({SHUFFLE_METHOD})")
                ax[0].set_xlabel(xlabel)
                ax[0].set_ylabel("# neurons")
                ax[0].set_title(title)
                ax[0].legend(frameon=False)

                dd = mrg[delta_col].values
                dd = dd[np.isfinite(dd)]
                ax[1].hist(dd, bins=bins)
                ax[1].axvline(0, linewidth=1)
                ax[1].set_xlabel(f"Δ = real − shuffle ({xlabel})")
                ax[1].set_ylabel("# neurons")
                if len(dd) > 0:
                    ax[1].set_title(f"Δ (mean={np.nanmean(dd):.3g}, median={np.nanmedian(dd):.3g})")
                else:
                    ax[1].set_title("Δ distribution")

                plt.tight_layout()
                plt.show()

            _two_panel_real_vs_shuf_and_delta(
                real_col="mean_other_pc1_at_tau",
                shuf_col="mean_other_pc1_at_tau_shuf",
                delta_col="delta_mean_other_pc1",
                xlabel="per-neuron mean other_pc1@tau",
                title="Neuron mean other_pc1@tau: real vs shuffle"
            )

            _two_panel_real_vs_shuf_and_delta(
                real_col="mean_gaze_at_tau",
                shuf_col="mean_gaze_at_tau_shuf",
                delta_col="delta_mean_gaze_at_tau",
                xlabel="per-neuron mean gaze@tau",
                title="Neuron mean gaze@tau: real vs shuffle"
            )

            _two_panel_real_vs_shuf_and_delta(
                real_col="mean_other_speed_at_tau",
                shuf_col="mean_other_speed_at_tau_shuf",
                delta_col="delta_mean_other_speed",
                xlabel="per-neuron mean other_speed@tau",
                title="Neuron mean other_speed@tau: real vs shuffle"
            )

            # ---- NEW: scatter showing distance consistency vs real-shuffle difference
            # use x = MAD(dist_to_pull) (smaller = more consistent), y = delta_mean_other_pc1
            x = mrg["mad_dist_to_pull_s"].values
            y = mrg["delta_mean_other_pc1"].values
            mm = np.isfinite(x) & np.isfinite(y)
            if np.sum(mm) >= 10:
                r = pearsonr_simple(x[mm], y[mm])
                plt.figure(figsize=(6, 4.5))
                plt.scatter(x[mm], y[mm], s=10, alpha=0.35)
                plt.axhline(0, linewidth=1)
                plt.xlabel("Distance-to-pull consistency (MAD dist_to_pull_s; lower=more consistent)")
                plt.ylabel("Δ mean other_pc1@tau (real − shuffle)")
                plt.title(f"PC1 effect vs distance-to-pull consistency (r={r:.3f}, n={int(np.sum(mm))})")
                plt.tight_layout()
                plt.show()

    # 5) split-half reliability plots (across neurons)
    if DO_SPLIT_HALF and len(neuron_peak_df) > 0:
        for short, col in [
            ("gaze", "split_half_mean_rel_gaze"),
            ("pc1", "split_half_mean_rel_pc1"),
            ("speed", "split_half_mean_rel_speed"),
        ]:
            vals = neuron_peak_df[col].values
            vals = vals[np.isfinite(vals)]
            if len(vals) == 0:
                continue
            plt.figure(figsize=(6, 4))
            plt.hist(vals, bins=40)
            plt.axvline(0, linewidth=1)
            plt.xlabel(col)
            plt.ylabel("# neurons")
            plt.title(f"Split-half reliability (mean@tau) across neurons: {short}")
            plt.tight_layout()
            plt.show()

        # Optional: Δ vs reliability
        if mrg is not None and len(mrg) > 0:
            for short, dcol, rcol in [
                ("pc1", "delta_mean_other_pc1", "split_half_mean_rel_pc1"),
                ("gaze", "delta_mean_gaze_at_tau", "split_half_mean_rel_gaze"),
                ("speed", "delta_mean_other_speed", "split_half_mean_rel_speed"),
            ]:
                x = mrg[dcol].values
                y = mrg[rcol].values
                mm = np.isfinite(x) & np.isfinite(y)
                if np.sum(mm) < 10:
                    continue
                r = pearsonr_simple(x[mm], y[mm])
                plt.figure(figsize=(6, 4.5))
                plt.scatter(x[mm], y[mm], s=10, alpha=0.35)
                plt.axvline(0, linewidth=1)
                plt.axhline(0, linewidth=1)
                plt.xlabel(f"{dcol}  (real - shuffle)")
                plt.ylabel(f"{rcol}  (split-half reliability)")
                plt.title(f"{short}: Δ vs reliability (r={r:.3f}, n={int(np.sum(mm))})")
                plt.tight_layout()
                plt.show()

    # 6) Optional: tau vs |old slope| sanity check (pooled)
    if len(trial_peak_df) > 0:
        dd = _subsample(trial_peak_df, seed=123)
        r = pearsonr_simple(dd["tau_norm"].values, dd["old_slope_abs"].values)
        plt.figure(figsize=(6, 4.5))
        plt.scatter(dd["tau_norm"].values, dd["old_slope_abs"].values, s=8, alpha=0.20)
        plt.axhline(0, linewidth=1)
        plt.xlim(-0.02, 1.02)
        plt.xlabel("tau_peak (normalized within fit window)")
        plt.ylabel("old slope abs (OLS slope on same window)")
        plt.title(f"All trials: tau_peak vs |old slope| (r={r:.3f}, n={len(dd)})")
        plt.tight_layout()
        plt.show()

    # Outputs in memory:
    #   trial_peak_df, shuffle_df, neuron_peak_df, neuron_null_df, mrg (if merged)


In [ ]:
hddm_RawFullDatas_tgt_alldates.keys()

In [ ]:
neuron_summary_df

In [ ]:
np.shape(trial_fit_records)

In [ ]:
hddm_model_fitted_withNeuronActivity_dic

In [ ]:
animal_tgt == 'dodson'

In [ ]:
np.nanmean(hddm_result_sumary_tgt_alldates_allneurons_df['t_mean'])

In [ ]:
hddm_result_sumary_tgt_alldates_allneurons_df

In [ ]:
# hddm_model_fitted_stats

In [ ]:
# hddm_data_tgt_idate['partner_mean_speed']

In [ ]:
if 0:
    # List of z-scored predictor columns
    predictor_columns = [
        'self_gaze_auc',
        'partner_mean_speed',
        'self_mean_speed',
        'partner_speed_std',
        'self_speed_std',
        'failed_pulls_before_reward',
        'time_since_last_reward'
    ]

    # Compute correlation matrix
    correlation_matrix = hddm_data_tgt_alldates[predictor_columns].corr()

    # Round for readability
    correlation_matrix_rounded = correlation_matrix.round(2)

    # Display the table
    # print("Correlation matrix among predictors:")
    print(correlation_matrix_rounded)

    plt.figure(figsize=(10, 8))
    sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap="coolwarm", square=True)
    plt.title("Correlation Matrix of Predictors")
    plt.tight_layout()
    plt.show()

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant
import pandas as pd

# List of predictors (already z-scored)
predictor_columns = [
    'self_gaze_auc',
    'partner_mean_speed',
    'self_mean_speed',
    'partner_speed_std',
    'self_speed_std',
    'failed_pulls_before_reward',
    'time_since_last_reward'
]

# Extract predictor data
X = hddm_data_tgt_alldates[predictor_columns].copy()

# Add a constant term for intercept (required by VIF calc)
X_const = add_constant(X)

# Compute VIF for each feature
vif_data = pd.DataFrame()
vif_data['Variable'] = X_const.columns
vif_data['VIF'] = [variance_inflation_factor(X_const.values, i) for i in range(X_const.shape[1])]

# Drop the constant from the results
vif_data = vif_data[vif_data['Variable'] != 'const']

# Display VIFs
print(vif_data)


In [ ]:
X

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from statsmodels.stats.multitest import multipletests

# List of variables
predictor_columns = [
    'rt',
    'self_gaze_auc',
    'partner_mean_speed',
    'self_mean_speed',
    'partner_speed_std',
    'self_speed_std',
    'predicted_v',
    'fr_slope',
    'fr_mean',
]

# Drop NaNs
df_corr = df_with_v[predictor_columns].dropna()

n = len(predictor_columns)
corr_matrix = np.zeros((n, n))
pval_matrix = np.ones((n, n))

# Compute correlations and p-values (upper triangle only)
triu_indices = np.triu_indices(n, k=1)
pvals = []

for i, j in zip(*triu_indices):
    r, p = pearsonr(df_corr[predictor_columns[i]], df_corr[predictor_columns[j]])
    corr_matrix[i, j] = corr_matrix[j, i] = r
    pval_matrix[i, j] = pval_matrix[j, i] = p
    pvals.append(p)

# Apply FDR correction
pvals_corrected = multipletests(pvals, method='fdr_bh')[1]

# Create corrected p-value matrix
pval_corrected_matrix = np.ones((n, n))
for idx, (i, j) in enumerate(zip(*triu_indices)):
    pval_corrected_matrix[i, j] = pval_corrected_matrix[j, i] = pvals_corrected[idx]

# Annotate with asterisks if corrected p < 0.01
annot = np.empty_like(corr_matrix, dtype=object)
for i in range(n):
    for j in range(n):
        val = f"{corr_matrix[i, j]:.2f}"
        if i != j and pval_corrected_matrix[i, j] < 0.01:
            val += "*"
        annot[i, j] = val

# Plot
plt.figure(figsize=(10, 8))
sns.heatmap(
    corr_matrix, annot=annot, fmt='', cmap='coolwarm', square=True,
    xticklabels=predictor_columns, yticklabels=predictor_columns,
    cbar_kws={"label": "Pearson r"}
)
plt.title("Correlation Matrix of Predictors, predicted drift rate v and FR properties \
          \n(* FDR-corrected p < 0.01)")
plt.tight_layout()
plt.show()


In [ ]:
pval_corrected_matrix

In [ ]:
# ind_ = df_with_v['condition']=='MC'
# ind_ = df_with_v['date']==np.unique(df_with_v['date'])[10]
# ind_ = (df_with_v['condition']=='MC_withDodson')  & (df_with_v['rt'] > 5.5)
# ind_ = df_with_v['prev_trial_outcome'] == 1
# ind_ = df_with_v['rt']>5
ind_ = df_with_v['rt']>=0
print(np.sum(ind_))
# df_with_v[df_with_v['prev_trial_outcome'] == 1]['rt'] = df_with_v[df_with_v['prev_trial_outcome'] == 1]['rt']-1

# plt.plot(df_with_v[ind_]['rt'],df_with_v[ind_]['self_gaze_auc']/df_with_v[ind_]['rt'],'.')
plt.plot(df_with_v[ind_]['partner_mean_speed'],df_with_v[ind_]['self_gaze_auc'],'.')

# plt.plot(df_with_v[ind_]['rt'],df_with_v[ind_]['partner_mean_speed'],'.')

from scipy.stats import pearsonr
from scipy.stats import linregress

x = df_with_v[ind_]['partner_mean_speed']
y = df_with_v[ind_]['self_gaze_auc']#/df_with_v[ind_]['rt']

slope, intercept, _, _, _ = linregress(x, y)
r_value, p_value = pearsonr(x, y)

print(slope)
print(r_value)
print(p_value)

In [ ]:
plt.hist(df_with_v['predicted_v'])

In [ ]:
np.sum(df_with_v['prev_trial_outcome']==0)

In [ ]:
# lower_q = 0.025
# upper_q = 0.975
# tslr_lower, tslr_upper = df_with_v['time_since_last_reward'].quantile([lower_q, upper_q])
# df_with_v = df_with_v[df_with_v['time_since_last_reward']>tslr_lower]
# df_with_v = df_with_v[df_with_v['time_since_last_reward']<tslr_upper]
# df_with_v = df_with_v[df_with_v['prev_trial_outcome']==1]

plt.figure(figsize=(6, 4))
plt.plot(df_with_v['rt'], df_with_v['time_since_last_reward'], '.')
plt.xlabel('Reaction Time')
plt.ylabel('time_since_last_reward')

from scipy.stats import pearsonr
from scipy.stats import linregress

x = df_with_v['rt']
y = df_with_v['time_since_last_reward']

slope, intercept, _, _, _ = linregress(x, y)
r_value, p_value = pearsonr(x, y)

print(slope)
print(r_value)
print(p_value)



In [ ]:
# lower_q = 0.025
# upper_q = 0.975
# tslr_lower, tslr_upper = df_with_v['time_since_last_reward'].quantile([lower_q, upper_q])
# df_with_v = df_with_v[df_with_v['time_since_last_reward']>tslr_lower]
# df_with_v = df_with_v[df_with_v['time_since_last_reward']<tslr_upper]
# df_with_v = df_with_v[df_with_v['prev_trial_outcome']==1]

plt.figure(figsize=(6, 4))
plt.plot(df_with_v['rt'], df_with_v['predicted_v'], '.')
plt.xlabel('Reaction Time')
plt.ylabel('predicted_v')

from scipy.stats import pearsonr
from scipy.stats import linregress

x = df_with_v['rt']
y = df_with_v['predicted_v']

slope, intercept, _, _, _ = linregress(x, y)
r_value, p_value = pearsonr(x, y)

print(slope)
print(r_value)
print(p_value)



In [ ]:
# lower_q = 0.025
# upper_q = 0.975
# tslr_lower, tslr_upper = df_with_v['time_since_last_reward'].quantile([lower_q, upper_q])
# df_with_v = df_with_v[df_with_v['time_since_last_reward']>tslr_lower]
# df_with_v = df_with_v[df_with_v['time_since_last_reward']<tslr_upper]
# df_with_v = df_with_v[df_with_v['prev_trial_outcome']==1]

plt.figure(figsize=(6, 4))
plt.plot(df_with_v['rt'], df_with_v['partner_mean_speed'], '.')
plt.xlabel('rt')
plt.ylabel('partner_mean_speed')

from scipy.stats import pearsonr
from scipy.stats import linregress

x = df_with_v['rt']
y = df_with_v['partner_mean_speed']

slope, intercept, _, _, _ = linregress(x, y)
r_value, p_value = pearsonr(x, y)

print(slope)
print(r_value)
print(p_value)



In [ ]:
# lower_q = 0.025
# upper_q = 0.975
# tslr_lower, tslr_upper = df_with_v['time_since_last_reward'].quantile([lower_q, upper_q])
# df_with_v = df_with_v[df_with_v['time_since_last_reward']>tslr_lower]
# df_with_v = df_with_v[df_with_v['time_since_last_reward']<tslr_upper]
# df_with_v = df_with_v[df_with_v['prev_trial_outcome']==1]

plt.figure(figsize=(6, 4))
plt.plot(df_with_v['rt'], df_with_v['self_gaze_auc'], '.')
#plt.plot(df_with_v['rt'], df_with_v['self_gaze_auc_origin']/df_with_v['rt'], '.')
plt.xlabel('rt')
plt.ylabel('self_gaze_auc')

from scipy.stats import pearsonr
from scipy.stats import linregress

x = df_with_v['rt']
y = df_with_v['self_gaze_auc']
# y = df_with_v['self_gaze_auc_origin']/df_with_v['rt']

slope, intercept, _, _, _ = linregress(x, y)
r_value, p_value = pearsonr(x, y)

print(slope)
print(r_value)
print(p_value)



In [ ]:
# lower_q = 0.025
# upper_q = 0.975
# tslr_lower, tslr_upper = df_with_v['time_since_last_reward'].quantile([lower_q, upper_q])
# df_with_v = df_with_v[df_with_v['time_since_last_reward']>tslr_lower]
# df_with_v = df_with_v[df_with_v['time_since_last_reward']<tslr_upper]
# df_with_v = df_with_v[df_with_v['prev_trial_outcome']==1]

plt.figure(figsize=(6, 4))
plt.plot(df_with_v['predicted_v'], df_with_v['partner_mean_speed'], '.')
plt.xlabel('predicted_v')
plt.ylabel('partner_mean_speed')

from scipy.stats import pearsonr
from scipy.stats import linregress

x = df_with_v['predicted_v']
y = df_with_v['partner_mean_speed']

slope, intercept, _, _, _ = linregress(x, y)
r_value, p_value = pearsonr(x, y)

print(slope)
print(r_value)
print(p_value)



In [ ]:
# lower_q = 0.025
# upper_q = 0.975
# tslr_lower, tslr_upper = df_with_v['time_since_last_reward'].quantile([lower_q, upper_q])
# df_with_v = df_with_v[df_with_v['time_since_last_reward']>tslr_lower]
# df_with_v = df_with_v[df_with_v['time_since_last_reward']<tslr_upper]
# df_with_v = df_with_v[df_with_v['prev_trial_outcome']==1]

plt.figure(figsize=(6, 4))
plt.plot(df_with_v['predicted_v'], df_with_v['self_speed_std'], '.')
plt.xlabel('predicted_v')
plt.ylabel('self_speed_std')

from scipy.stats import pearsonr
from scipy.stats import linregress

x = df_with_v['predicted_v']
y = df_with_v['self_speed_std']

slope, intercept, _, _, _ = linregress(x, y)
r_value, p_value = pearsonr(x, y)

print(slope)
print(r_value)
print(p_value)



In [ ]:
# lower_q = 0.025
# upper_q = 0.975
# tslr_lower, tslr_upper = df_with_v['time_since_last_reward'].quantile([lower_q, upper_q])
# df_with_v = df_with_v[df_with_v['time_since_last_reward']>tslr_lower]
# df_with_v = df_with_v[df_with_v['time_since_last_reward']<tslr_upper]
# df_with_v = df_with_v[df_with_v['prev_trial_outcome']==1]

plt.figure(figsize=(6, 4))
plt.plot(df_with_v['time_since_last_reward'], df_with_v['partner_mean_speed'], '.')
plt.xlabel('time_since_last_reward')
plt.ylabel('partner_mean_speed')

from scipy.stats import pearsonr
from scipy.stats import linregress

x = df_with_v['time_since_last_reward']
y = df_with_v['partner_mean_speed']

slope, intercept, _, _, _ = linregress(x, y)
r_value, p_value = pearsonr(x, y)

print(slope)
print(r_value)
print(p_value)



In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

X = df_with_v[['partner_mean_speed', 'time_since_last_reward']]
vif_data = pd.DataFrame()
vif_data['feature'] = X.columns
vif_data['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print(vif_data)

In [ ]:
hddm_model_fitted_nogaze_stats

In [ ]:
hddm_model_fitted_nogaze_dic

In [ ]:
hddm_model_fitted_stats

In [ ]:
hddm_model_fitted_dic

In [ ]:
np.sum(df_with_v['prev_trial_outcome']==1)

In [ ]:
plt.plot(traces['v_Intercept'])

In [ ]:
lower_q = 0.025
upper_q = 0.975
rt_lower, rt_upper = hddm_data_tgt_alldates['rt'].quantile([lower_q, upper_q])
hddm_data_tgt_alldates = hddm_data_tgt_alldates[hddm_data_tgt_alldates['rt']>rt_lower]
hddm_data_tgt_alldates = hddm_data_tgt_alldates[hddm_data_tgt_alldates['rt']<rt_upper]
#
tslr_lower, tslr_upper = hddm_data_tgt_alldates['time_since_last_reward'].quantile([lower_q, upper_q])
hddm_data_tgt_alldates = hddm_data_tgt_alldates[hddm_data_tgt_alldates['time_since_last_reward']>tslr_lower]
hddm_data_tgt_alldates = hddm_data_tgt_alldates[hddm_data_tgt_alldates['time_since_last_reward']<tslr_upper]
#
plt.figure(figsize=(6, 4))
plt.plot(hddm_data_tgt_alldates['rt'], hddm_data_tgt_alldates['time_since_last_reward'], '.')


In [ ]:
# use a prediction model to predict the pull based on the hddm model fitting

# --- Extract 'a' (new step) ---
# my_model = hddm_model_fitted_nogaze
# df_combined = hddm_data['hddm_data_'+animal_id_toana]
df_combined = hddm_data_tgt_alldates

# --- STEP 1: Get the mean posterior estimates for all 'v' coefficients ---
# traces = my_model.get_traces()
traces = hddm_model_fitted_nogaze_traces

# Get the mean value for the intercept
v_intercept_mean = traces['v_Intercept'].mean()

# Get the mean value for the coefficient of each predictor
# v_gaze_coef_mean = traces['v_self_gaze_auc'].mean()
v_speed_coef_mean = traces['v_partner_mean_speed'].mean()
v_failed_pulls_before_reward_mean = traces['v_failed_pulls_before_reward'].mean()
v_time_since_last_reward_mean = traces['v_time_since_last_reward'].mean()

# --- STEP 2: Apply the regression equation to your dataframe ---
# This calculates the predicted 'v' for each trial based on its unique covariate values.
df_lifelines = df_combined.copy() # Work with a copy

df_lifelines['predicted_v']  = (v_intercept_mean +
                              # (v_gaze_coef_mean * df_with_v['self_gaze_auc']) +
                              (v_speed_coef_mean * df_with_v['partner_mean_speed'])+
                              (v_failed_pulls_before_reward_mean * df_with_v['failed_pulls_before_reward'])+
                              (v_time_since_last_reward_mean * df_with_v['time_since_last_reward']))



a_intercept_mean = traces['a_Intercept'].mean()
a_failed_pulls_coef = traces['a_failed_pulls_before_reward'].mean()
# ... other a coefficients ...
df_lifelines['predicted_a'] = (a_intercept_mean + (a_failed_pulls_coef * df_lifelines['failed_pulls_before_reward']))

# --- Extract 'z' (new step) ---
# Get the mean z values for each condition
z_after_failure = traces['z_trans(0)'].mean()
z_after_success = traces['z_trans(1)'].mean()
# Create the predicted_z column based on the previous trial's outcome
df_lifelines['predicted_z'] = np.where(df_lifelines['prev_trial_outcome'] == 0, z_after_failure, z_after_success)


## --- Part 2: Fit the Cox Proportional Hazards Model ---
print("\n--- Step 2: Fitting the Cox Proportional Hazards Model ---")

df_for_fitting = df_lifelines[['rt', 'response', 'predicted_v', 'predicted_a', 'predicted_z']]
# df_for_fitting = df_lifelines[['rt', 'response', 'predicted_v']]

# cph = CoxPHFitter()
cph = CoxPHFitter(penalizer=0.01)

cph.fit(df_for_fitting, duration_col='rt', event_col='response')
print("Model Fit Summary:")
cph.print_summary()
print("-" * 40)


# --- Part 3: Prepare Continuous Timeline for Prediction ---
print("\n--- Step 3: Preparing Continuous Timeline ---")
# Define the total length and resolution of your prediction timeline
total_experiment_seconds = df_for_fitting['rt'].sum()
prediction_timestep = 0.1 # Make a prediction every 100ms (10 Hz)
# Create the continuous timeline dataframe
timeline_df = pd.DataFrame({'time': np.arange(0, total_experiment_seconds, prediction_timestep)})
# Get the absolute time when each pull (event) occurred
event_times = df_for_fitting['rt'].cumsum().rename('time')

# Create a dataframe of the covariate values at the time of each event
covariates_at_events = df_for_fitting[['predicted_v', 'predicted_a', 'predicted_z']].set_index(event_times)
# covariates_at_events = df_for_fitting[['predicted_v']].set_index(event_times)

# Map the covariate values onto the continuous timeline
timeline_with_covariates = pd.merge_asof(timeline_df, covariates_at_events, on='time')
timeline_with_covariates = timeline_with_covariates.ffill().dropna()
print("Timeline prepared for prediction.")
print("-" * 40)


# --- Part 4: Run Stochastic Simulation to Predict Pulls ---
# This part replaces the simple thresholding method.
print("\n--- Step 4: Running Stochastic Simulation ---")
# 4a: Predict the partial hazard (the step function based on covariates)
partial_hazard = cph.predict_partial_hazard(timeline_with_covariates)
timeline_with_covariates['partial_hazard'] = partial_hazard.values

# 4b: Get the baseline hazard from the fitted model
baseline_hazard_df = cph.baseline_hazard_

# 4c: Map the baseline hazard onto our continuous timeline
# MODIFICATION 3: Add .bfill() to fix the NaN issue at the beginning of the timeline.
timeline_with_full_hazard = pd.merge(timeline_with_covariates, baseline_hazard_df,
                                     left_on='time', right_index=True, how='left').ffill().bfill()

# 4d: Calculate the full, time-varying hazard
# Full Hazard h(t) = Baseline Hazard h₀(t) * Partial Hazard (from covariates)
timeline_with_full_hazard['full_hazard'] = (timeline_with_full_hazard['baseline hazard'] *
                                             timeline_with_full_hazard['partial_hazard'])

# 4e: Run the simulation loop
binary_prediction_list = []
pulls_predicted_count = 0
for index, row in timeline_with_full_hazard.iterrows():
    # Calculate the probability of a pull in this specific time step
    prob_of_pull = row['full_hazard'] * prediction_timestep
    # Ensure probability is not > 1
    prob_of_pull = min(prob_of_pull, 1.0)
    # Simulate a "coin flip" weighted by this probability
    if np.random.rand() < prob_of_pull:
        binary_prediction_list.append(1)
        pulls_predicted_count += 1
    else:
        binary_prediction_list.append(0)

# Add the final prediction to our dataframe
timeline_with_full_hazard['stochastic_prediction'] = binary_prediction_list
print(f"Simulation complete. Predicted {pulls_predicted_count} pulls.")
print("Sample of timeline with final predictions:")
print(timeline_with_full_hazard[['time', 'full_hazard', 'stochastic_prediction']].head(20))
print("-" * 40)


# --- Part 5: Visualize the Results of the Simulation ---
print("\n--- Step 5: Visualizing the Prediction ---")
plt.style.use('seaborn-v0_8-whitegrid')
fig, ax = plt.subplots(figsize=(16, 7))

# Plot the full predicted hazard over time
ax.plot(timeline_with_full_hazard['time'], timeline_with_full_hazard['full_hazard'],
        label='Full Predicted Hazard h(t)', color='mediumseagreen', alpha=0.9)

# Mark the actual pull times from the data
ax.vlines(event_times, ymin=0, ymax=ax.get_ylim()[1], color='black', linestyle='-',
          alpha=0.7, label='Actual Pulls', linewidth=1.5)

# Mark the predicted pull times from the stochastic simulation
predicted_pull_df = timeline_with_full_hazard[timeline_with_full_hazard['stochastic_prediction'] == 1]
ax.scatter(predicted_pull_df['time'], [ax.get_ylim()[1] * 0.95] * len(predicted_pull_df),
           color='red', marker='v', s=80, label='Predicted Pulls (from Simulation)', zorder=5)

ax.set_xlabel('Time (seconds)', fontsize=12)
ax.set_ylabel('Full Predicted Hazard Rate', fontsize=12)
ax.set_title('Stochastic Simulation of Pull Events Based on Predicted Hazard', fontsize=16)
ax.legend()
ax.set_xlim(0, 500) # Zoom in on the first 150 seconds for clarity
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()


In [ ]:
# --- Prerequisite: You must have a fitted Cox model 'cph' ---
# and your dataframe 'df_for_fitting' from the previous steps.

print("\n--- Evaluating Model with a Deterministic Method ---")

# --- Step 1: Predict the Median Survival Time for each Interval ---
# This gives us the model's single best guess for the duration of each IPI.
predicted_durations = cph.predict_median(df_for_fitting)
df_for_fitting['predicted_rt'] = predicted_durations.values

# --- Step 2: Calculate the Predicted Pull Times ---
# We can see when the model predicted pulls would happen by taking the
# cumulative sum of the predicted durations.
actual_pull_times = df_for_fitting['rt'].cumsum()
predicted_pull_times = df_for_fitting['predicted_rt'].cumsum()

print("Comparison of first 10 actual vs. predicted pull times:")
comparison_df = pd.DataFrame({
    'Actual Pull Time': actual_pull_times,
    'Predicted Pull Time': predicted_pull_times
})
print(comparison_df.head(10))

# --- Step 3: Visualize the Comparison ---
plt.style.use('seaborn-v0_8-whitegrid')
fig, ax = plt.subplots(figsize=(12, 8))

# Plot actual vs. predicted times in a scatter plot
ax.scatter(actual_pull_times, predicted_pull_times, alpha=0.7, edgecolors='k')

# Add a y=x line. Perfect predictions would fall on this line.
perfect_line_max = max(actual_pull_times.max(), predicted_pull_times.max())
ax.plot([0, perfect_line_max], [0, perfect_line_max], 'r--', label='Perfect Prediction (y=x)')

ax.set_xlabel('Actual Pull Time (seconds)', fontsize=12)
ax.set_ylabel('Predicted Pull Time (seconds)', fontsize=12)
ax.set_title('Model Predictive Accuracy: Actual vs. Predicted Pull Times', fontsize=16)
ax.legend()
ax.axis('equal') # Ensure the plot is square for easy comparison
plt.tight_layout()
plt.show()